In [ ]:
from pathlib import Path

# Localiza a raiz do projeto (um nível acima de notebooks/)
PROJECT_ROOT = Path.cwd().parent

# Caminho do arquivo de agosto
arquivo_agosto = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "senatran"
    / "2026-08"
    / "I_Frota_por_UF_Municipio_Marca_e_Modelo_Ano_Agosto_2026.TXT"
)

print(f"Raiz do projeto: {PROJECT_ROOT}")
print(f"Arquivo encontrado: {arquivo_agosto.exists()}")
print(f"Tamanho: {arquivo_agosto.stat().st_size / (1024**3):.2f} GiB")

In [ ]:
import pandas as pd

df_amostra = pd.read_csv(
    arquivo_agosto,
    sep=";",
    encoding="utf-8",
    nrows=10
)

display(df_amostra)

In [ ]:
df_amostra.dtypes

In [ ]:
total_registros = 0
quantidades_nulas = 0
quantidades_fracionarias = 0
quantidades_nao_positivas = 0

for bloco in pd.read_csv(
    arquivo_agosto,
    sep=";",
    encoding="utf-8",
    chunksize=500_000
):
    total_registros += len(bloco)

    quantidade = bloco["Qtd. Veículos"]

    quantidades_nulas += quantidade.isna().sum()

    quantidades_fracionarias += (
        quantidade.notna() & (quantidade % 1 != 0)
    ).sum()

    quantidades_nao_positivas += (
        quantidade.notna() & (quantidade <= 0)
    ).sum()

print(f"Total de registros: {total_registros:,}")
print(f"Quantidades nulas: {quantidades_nulas:,}")
print(f"Quantidades fracionárias: {quantidades_fracionarias:,}")
print(f"Quantidades iguais ou inferiores a zero: {quantidades_nao_positivas:,}")

In [ ]:
arquivo_setembro = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "senatran"
    / "2026-09"
    / "I_Frota_por_UF_Municipio_Marca_e_Modelo_Ano_Setembro_2026.TXT"
)

print(f"Arquivo encontrado: {arquivo_setembro.exists()}")

In [ ]:
total_registros = 0
quantidades_nulas = 0
quantidades_fracionarias = 0
quantidades_nao_positivas = 0

for bloco in pd.read_csv(
    arquivo_setembro,
    sep=";",
    encoding="utf-8",
    chunksize=500_000
):
    total_registros += len(bloco)

    quantidade = bloco["Qtd. Veículos"]

    quantidades_nulas += quantidade.isna().sum()

    quantidades_fracionarias += (
        quantidade.notna() & (quantidade % 1 != 0)
    ).sum()

    quantidades_nao_positivas += (
        quantidade.notna() & (quantidade <= 0)
    ).sum()

print(f"Total de registros: {total_registros:,}")
print(f"Quantidades nulas: {quantidades_nulas:,}")
print(f"Quantidades fracionárias: {quantidades_fracionarias:,}")
print(f"Quantidades iguais ou inferiores a zero: {quantidades_nao_positivas:,}")

In [ ]:
colunas_chave = [
    "UF",
    "Município",
    "Marca Modelo",
    "Ano Fabricação Veículo CRV"
]

total_duplicados = 0

for bloco in pd.read_csv(
    arquivo_agosto,
    sep=";",
    encoding="utf-8",
    usecols=colunas_chave,
    chunksize=500_000
):
    total_duplicados += bloco.duplicated(
        subset=colunas_chave
    ).sum()

print(f"Duplicidades encontradas: {total_duplicados:,}")

In [ ]:
total_registros = 0
quantidades_nulas = 0
quantidades_fracionarias = 0
quantidades_nao_positivas = 0

for bloco in pd.read_csv(
    arquivo_agosto,
    sep=";",
    encoding="utf-8",
    chunksize=500_000
):
    total_registros += len(bloco)

    quantidade = bloco["Qtd. Veículos"]

    quantidades_nulas += quantidade.isna().sum()

    quantidades_fracionarias += (
        quantidade.notna() & (quantidade % 1 != 0)
    ).sum()

    quantidades_nao_positivas += (
        quantidade.notna() & (quantidade <= 0)
    ).sum()

print(f"Total de registros: {total_registros:,}")
print(f"Quantidades nulas: {quantidades_nulas:,}")
print(f"Quantidades fracionárias: {quantidades_fracionarias:,}")
print(f"Quantidades iguais ou inferiores a zero: {quantidades_nao_positivas:,}")

In [ ]:
total_registros = 0
duplicados_consecutivos = 0
quebras_ordenacao = 0

ultimo_registro = None

for bloco in pd.read_csv(
    arquivo_setembro,
    sep=";",
    encoding="utf-8",
    usecols=colunas_chave,
    chunksize=500_000
):
    registros = list(
        bloco.itertuples(index=False, name=None)
    )

    if ultimo_registro is not None and registros:
        if registros[0] == ultimo_registro:
            duplicados_consecutivos += 1

        if registros[0] < ultimo_registro:
            quebras_ordenacao += 1

    for anterior, atual in zip(registros, registros[1:]):
        if atual == anterior:
            duplicados_consecutivos += 1

        if atual < anterior:
            quebras_ordenacao += 1

    if registros:
        ultimo_registro = registros[-1]

    total_registros += len(bloco)

print(f"Total de registros: {total_registros:,}")
print(f"Duplicidades consecutivas: {duplicados_consecutivos:,}")
print(f"Quebras de ordenação: {quebras_ordenacao:,}")

In [ ]:
nulos_por_coluna = pd.Series(0, index=colunas_chave + ["Qtd. Veículos"])

for bloco in pd.read_csv(
    arquivo_setembro,
    sep=";",
    encoding="utf-8",
    chunksize=500_000
):
    nulos_por_coluna += bloco.isna().sum()

print("Valores ausentes por coluna:")
display(nulos_por_coluna)

In [ ]:
anos_minimo = None
anos_maximo = None
anos_zerados = 0
anos_futuros = 0
total_registros = 0

for bloco in pd.read_csv(
    arquivo_setembro,
    sep=";",
    encoding="utf-8",
    usecols=["Ano Fabricação Veículo CRV"],
    chunksize=500_000
):
    anos = pd.to_numeric(
        bloco["Ano Fabricação Veículo CRV"],
        errors="coerce"
    )

    minimo_bloco = anos.min()
    maximo_bloco = anos.max()

    anos_minimo = (
        minimo_bloco if anos_minimo is None
        else min(anos_minimo, minimo_bloco)
    )

    anos_maximo = (
        maximo_bloco if anos_maximo is None
        else max(anos_maximo, maximo_bloco)
    )

    anos_zerados += (anos == 0).sum()
    anos_futuros += (anos > 2026).sum()
    total_registros += len(bloco)

print(f"Total de registros: {total_registros:,}")
print(f"Menor ano: {anos_minimo}")
print(f"Maior ano: {anos_maximo}")
print(f"Anos iguais a zero: {anos_zerados:,}")
print(f"Anos posteriores a 2026: {anos_futuros:,}")

In [ ]:
distribuicao_anos_agosto = (
    pd.read_csv(
        arquivo_agosto,
        sep=";",
        encoding="utf-8",
        usecols=["Ano Fabricação Veículo CRV", "Qtd. Veículos"],
        chunksize=500_000
    )
)

contagem_anos_agosto = {}

for bloco in distribuicao_anos_agosto:
    bloco["Ano Fabricação Veículo CRV"] = pd.to_numeric(
        bloco["Ano Fabricação Veículo CRV"],
        errors="coerce"
    )

    bloco["Qtd. Veículos"] = pd.to_numeric(
        bloco["Qtd. Veículos"],
        errors="coerce"
    )

    agrupado = (
        bloco.groupby("Ano Fabricação Veículo CRV")["Qtd. Veículos"]
        .sum()
    )

    for ano, quantidade in agrupado.items():
        contagem_anos_agosto[ano] = (
            contagem_anos_agosto.get(ano, 0) + quantidade
        )

distribuicao_agosto = (
    pd.Series(contagem_anos_agosto, name="Quantidade de veículos")
    .sort_index()
)

display(distribuicao_agosto.to_frame())

In [ ]:
distribuicao_anos_setembro = (
    pd.read_csv(
        arquivo_setembro,
        sep=";",
        encoding="utf-8",
        usecols=["Ano Fabricação Veículo CRV", "Qtd. Veículos"],
        chunksize=500_000
    )
)

contagem_anos_setembro = {}

for bloco in distribuicao_anos_setembro:
    bloco["Ano Fabricação Veículo CRV"] = pd.to_numeric(
        bloco["Ano Fabricação Veículo CRV"],
        errors="coerce"
    )

    bloco["Qtd. Veículos"] = pd.to_numeric(
        bloco["Qtd. Veículos"],
        errors="coerce"
    )

    agrupado = (
        bloco.groupby("Ano Fabricação Veículo CRV")["Qtd. Veículos"]
        .sum()
    )

    for ano, quantidade in agrupado.items():
        contagem_anos_setembro[ano] = (
            contagem_anos_setembro.get(ano, 0) + quantidade
        )

distribuicao_setembro = (
    pd.Series(contagem_anos_setembro, name="Quantidade de veículos")
    .sort_index()
)

display(distribuicao_setembro.to_frame())

In [ ]:
comparacao_anos = pd.DataFrame({
    "Agosto": distribuicao_agosto,
    "Setembro": distribuicao_setembro
}).fillna(0)

comparacao_anos["Diferenca"] = (
    comparacao_anos["Setembro"] - comparacao_anos["Agosto"]
)

comparacao_anos["Variacao percentual"] = (
    comparacao_anos["Diferenca"]
    .div(comparacao_anos["Agosto"].replace(0, pd.NA))
    .mul(100)
)

display(
    comparacao_anos.sort_values(
        "Diferenca",
        ascending=False
    ).head(15).round(2)
)

In [ ]:
# Resumo geral das diferenças entre os levantamentos

print("Anos com aumento:", (comparacao_anos["Diferenca"] > 0).sum())
print("Anos sem alteração:", (comparacao_anos["Diferenca"] == 0).sum())
print("Anos com redução:", (comparacao_anos["Diferenca"] < 0).sum())

print("\nMaiores reduções:")
display(
    comparacao_anos
    .sort_values("Diferenca")
    .head(15)
    .round(2)
)

In [ ]:
print("Anos com aumento:", (comparacao_anos["Diferenca"] > 0).sum())
print("Anos sem alteração:", (comparacao_anos["Diferenca"] == 0).sum())
print("Anos com redução:", (comparacao_anos["Diferenca"] < 0).sum())

In [ ]:
marcas_modelos = set()

for bloco in pd.read_csv(
    arquivo_setembro,
    sep=";",
    encoding="utf-8",
    usecols=["Marca Modelo"],
    chunksize=500_000
):
    marcas_modelos.update(
        bloco["Marca Modelo"].dropna().unique()
    )

print(f"Total de combinações distintas: {len(marcas_modelos):,}")

In [ ]:
from collections import defaultdict

quantidade_por_modelo = defaultdict(float)

for bloco in pd.read_csv(
    arquivo_setembro,
    sep=";",
    encoding="utf-8",
    usecols=["Marca Modelo", "Qtd. Veículos"],
    chunksize=500_000
):
    bloco["Qtd. Veículos"] = pd.to_numeric(
        bloco["Qtd. Veículos"],
        errors="coerce"
    )

    agrupado = (
        bloco.groupby("Marca Modelo")["Qtd. Veículos"]
        .sum()
    )

    for modelo, quantidade in agrupado.items():
        quantidade_por_modelo[modelo] += quantidade

ranking_modelos = (
    pd.Series(quantidade_por_modelo, name="Quantidade de veículos")
    .sort_values(ascending=False)
)

display(ranking_modelos.head(30).to_frame())

In [ ]:
quantidade_por_marca = defaultdict(float)

for bloco in pd.read_csv(
    arquivo_setembro,
    sep=";",
    encoding="utf-8",
    usecols=["Marca Modelo", "Qtd. Veículos"],
    chunksize=500_000
):
    bloco["Qtd. Veículos"] = pd.to_numeric(
        bloco["Qtd. Veículos"],
        errors="coerce"
    )

    bloco["Marca"] = (
        bloco["Marca Modelo"]
        .str.split("/", n=1)
        .str[0]
    )

    agrupado = (
        bloco.groupby("Marca")["Qtd. Veículos"]
        .sum()
    )

    for marca, quantidade in agrupado.items():
        quantidade_por_marca[marca] += quantidade

ranking_marcas = (
    pd.Series(quantidade_por_marca, name="Quantidade de veículos")
    .sort_values(ascending=False)
)

print(f"Total de marcas distintas: {len(ranking_marcas):,}")

display(ranking_marcas.head(30).to_frame())

In [ ]:
import pandas as pd

df_marcas_modelos = pd.DataFrame(
    quantidade_por_modelo.items(),
    columns=["Marca Modelo", "Quantidade de veículos"]
)

df_marcas_modelos["Marca"] = (
    df_marcas_modelos["Marca Modelo"]
    .str.split("/", n=1)
    .str[0]
)

marcas_investigar = [
    "I", "R", "IMP", "SR", "REB", "JTA",
    "GM", "CHEV", "CHEVROLET"
]

for marca in marcas_investigar:
    print(f"\n--- {marca} ---")

    display(
        df_marcas_modelos[
            df_marcas_modelos["Marca"] == marca
        ]
        .sort_values(
            "Quantidade de veículos",
            ascending=False
        )
        .head(10)
        [["Marca Modelo", "Quantidade de veículos"]]
    )

In [ ]:
resultados_investigacao = []

for marca in marcas_investigar:
    resultado = (
        df_marcas_modelos[
            df_marcas_modelos["Marca"] == marca
        ]
        .sort_values(
            "Quantidade de veículos",
            ascending=False
        )
        .head(10)
        .copy()
    )

    resultados_investigacao.append(resultado)

tabela_investigacao = pd.concat(
    resultados_investigacao,
    ignore_index=True
)

display(
    tabela_investigacao[
        ["Marca", "Marca Modelo", "Quantidade de veículos"]
    ]
)

In [ ]:
frequencia_prefixos = (
    df_marcas_modelos
    .groupby("Marca")["Quantidade de veículos"]
    .agg(
        modelos_distintos="count",
        quantidade_veiculos="sum"
    )
    .sort_values(
        "quantidade_veiculos",
        ascending=False
    )
)

print(f"Total de prefixos distintos: {len(frequencia_prefixos):,}")

display(frequencia_prefixos.head(50))

In [ ]:
prefixos_criticos = ["I", "R", "IMP", "SR", "REB", "JTA"]

amostra_prefixos = (
    df_marcas_modelos[
        df_marcas_modelos["Marca"].isin(prefixos_criticos)
    ]
    .sort_values(
        ["Marca", "Quantidade de veículos"],
        ascending=[True, False]
    )
    .groupby("Marca")
    .head(8)
    .reset_index(drop=True)
)

display(amostra_prefixos)

In [ ]:
# Seleciona registros com os prefixos I/ e IMP/
df_importados = df_marcas_modelos[
    df_marcas_modelos["Marca Modelo"].str.startswith(("I/", "IMP/"))
].copy()

# Remove apenas o primeiro prefixo, preservando o campo original
df_importados["Descricao sem prefixo"] = (
    df_importados["Marca Modelo"].str.split("/", n=1).str[1]
)

# Extrai o primeiro identificador após a barra
df_importados["Identificador candidato"] = (
    df_importados["Descricao sem prefixo"].str.split().str[0]
)

# Resume por identificador e prefixo original
resumo_importados = (
    df_importados
    .groupby(["Marca", "Identificador candidato"])
    .agg(
        modelos_distintos=("Marca Modelo", "count"),
        quantidade_veiculos=("Quantidade de veículos", "sum")
    )
    .sort_values(
        ["Marca", "quantidade_veiculos"],
        ascending=[True, False]
    )
)

display(resumo_importados.groupby(level=0, group_keys=False).head(20))

In [ ]:
resumo_importados_formatado = (
    resumo_importados
    .reset_index()
    .rename(columns={"Marca": "Prefixo original"})
)

display(
    resumo_importados_formatado[
        resumo_importados_formatado["Identificador candidato"].isin(
            ["TOYOTA", "FIAT", "VW", "FORD", "CHEVROLET",
             "CHEV", "GM", "M.BENZ", "MBENZ", "BYD", "GWM"]
        )
    ]
    .sort_values(
        ["Prefixo original", "quantidade_veiculos"],
        ascending=[True, False]
    )
)

In [ ]:
# Separa o campo original na primeira barra
partes = df_marcas_modelos["Marca Modelo"].str.split(
    "/", n=1, expand=True
)

prefixo = partes[0].str.strip()
descricao = partes[1].fillna("").str.strip()

# Mantém o prefixo como fabricante candidata
fabricante_candidata = prefixo.copy()

# Para I/ e IMP/, utiliza o primeiro identificador após a barra
mascara_importados = prefixo.isin(["I", "IMP"])

fabricante_candidata.loc[mascara_importados] = (
    descricao.loc[mascara_importados].str.split().str[0]
)

# Cria nova coluna sem alterar os dados originais
df_marcas_modelos["Fabricante candidata"] = fabricante_candidata

# Ranking exploratório
ranking_fabricantes_candidatas = (
    df_marcas_modelos
    .groupby("Fabricante candidata")
    .agg(
        modelos_distintos=("Marca Modelo", "count"),
        quantidade_veiculos=("Quantidade de veículos", "sum")
    )
    .sort_values("quantidade_veiculos", ascending=False)
)

display(ranking_fabricantes_candidatas.head(40))

In [ ]:
from pathlib import Path

pasta_senatran = PROJECT_ROOT / "data" / "raw" / "senatran"

for arquivo in sorted(pasta_senatran.rglob("*")):
    if arquivo.is_file():
        print(arquivo.relative_to(PROJECT_ROOT))

In [ ]:
from pathlib import Path

pasta_senatran = PROJECT_ROOT / "data" / "raw" / "senatran"

for arquivo in sorted(pasta_senatran.rglob("*.xlsx")):
    print(
        f"Arquivo: {arquivo.name}\n"
        f"Tamanho: {arquivo.stat().st_size / (1024**2):.2f} MB\n"
        f"Caminho: {arquivo.relative_to(PROJECT_ROOT)}\n"
    )

In [ ]:
arquivo_setembro_tipo = (
    PROJECT_ROOT
    / "data" / "raw" / "senatran" / "2026-09"
    / "G_Frota_por_UF_Municipio_TIPO_ESPECIE_EIXOS_Setembro_2026.xlsx"
)

print("Existe:", arquivo_setembro_tipo.exists())

if arquivo_setembro_tipo.exists():
    print(f"Tamanho: {arquivo_setembro_tipo.stat().st_size / (1024**2):.2f} MB")

In [ ]:
arquivo_agosto_tipo = (
    PROJECT_ROOT
    / "data" / "raw" / "senatran" / "2026-08"
    / "G_Frota_por_UF_Municipio_TIPO_ESPECIE_EIXOS_Agosto_2026.xlsx"
)

arquivo_setembro_tipo = (
    PROJECT_ROOT
    / "data" / "raw" / "senatran" / "2026-09"
    / "G_Frota_por_UF_Municipio_TIPO_ESPECIE_EIXOS_Setembro_2026.xlsx"
)

print("Agosto:", arquivo_agosto_tipo.exists())
print("Setembro:", arquivo_setembro_tipo.exists())

In [ ]:
import pandas as pd

arquivos_tipo_especie = {
    "Agosto": arquivo_agosto_tipo,
    "Setembro": arquivo_setembro_tipo
}

for mes, arquivo in arquivos_tipo_especie.items():
    print(f"\n{'=' * 60}")
    print(f"{mes} de 2026")
    print(f"{'=' * 60}")

    excel = pd.ExcelFile(arquivo)

    print("Abas disponíveis:", excel.sheet_names)

    for aba in excel.sheet_names:
        df_amostra = pd.read_excel(
            arquivo,
            sheet_name=aba,
            nrows=5
        )

        print(f"\nAba: {aba}")
        print("Colunas:", df_amostra.columns.tolist())
        print("\nAmostra:")
        display(df_amostra)

In [ ]:
# Carrega somente a aba Layout G
df_tipo_agosto = pd.read_excel(
    arquivo_agosto_tipo,
    sheet_name="Layout G"
)

df_tipo_setembro = pd.read_excel(
    arquivo_setembro_tipo,
    sheet_name="Layout G"
)

for mes, df in [
    ("Agosto", df_tipo_agosto),
    ("Setembro", df_tipo_setembro)
]:
    print(f"\n{'=' * 55}")
    print(f"{mes} de 2026")
    print(f"{'=' * 55}")

    print(f"Total de registros: {len(df):,}")
    print(f"Total de veículos: {df['Qtd. Veículos'].sum():,.0f}")

    print("\nTipos de veículos:")
    display(
        df.groupby("Tipo Veículo")["Qtd. Veículos"]
        .sum()
        .sort_values(ascending=False)
        .to_frame()
    )

    print("\nEspécies de veículos:")
    display(
        df.groupby("Espécie Veículo")["Qtd. Veículos"]
        .sum()
        .sort_values(ascending=False)
        .to_frame()
    )

    print("\nValores de eixos:")
    display(
        df.groupby("Eixos")["Qtd. Veículos"]
        .sum()
        .sort_values(ascending=False)
        .to_frame()
    )

In [ ]:
tipos_candidatos = [
    "AUTOMOVEL",
    "CAMINHONETE",
    "CAMIONETA",
    "UTILITARIO"
]

for mes, df in [
    ("Agosto", df_tipo_agosto),
    ("Setembro", df_tipo_setembro)
]:
    print(f"\n{'=' * 60}")
    print(f"{mes} de 2026 — Tipos candidatos")
    print(f"{'=' * 60}")

    analise = (
        df[df["Tipo Veículo"].isin(tipos_candidatos)]
        .groupby(
            ["Tipo Veículo", "Espécie Veículo"],
            dropna=False
        )["Qtd. Veículos"]
        .sum()
        .reset_index()
        .sort_values(
            ["Tipo Veículo", "Qtd. Veículos"],
            ascending=[True, False]
        )
    )

    display(analise)

In [ ]:
import numpy as np

In [ ]:
# Totais por tipo de veículo em cada mês
total_agosto = (
    df_tipo_agosto
    .groupby("Tipo Veículo")["Qtd. Veículos"]
    .sum()
    .rename("Agosto")
)

total_setembro = (
    df_tipo_setembro
    .groupby("Tipo Veículo")["Qtd. Veículos"]
    .sum()
    .rename("Setembro")
)

comparacao_tipos = pd.concat(
    [total_agosto, total_setembro],
    axis=1
).fillna(0)

comparacao_tipos["Diferença"] = (
    comparacao_tipos["Setembro"]
    - comparacao_tipos["Agosto"]
)

comparacao_tipos["Variação percentual (%)"] = (
    comparacao_tipos["Diferença"]
    / comparacao_tipos["Agosto"].replace(0, np.nan)
    * 100
)

display(
    comparacao_tipos
    .loc[tipos_candidatos]
    .sort_values("Setembro", ascending=False)
    .round(2)
)

In [ ]:
colunas_chave_tipo = [
    "UF",
    "Município",
    "Tipo Veículo",
    "Espécie Veículo",
    "Eixos"
]

for mes, df in [
    ("Agosto", df_tipo_agosto),
    ("Setembro", df_tipo_setembro)
]:
    print(f"\n{mes} de 2026")
    print("-" * 45)

    print("Registros:", len(df))
    print("Valores nulos por coluna:")
    display(df.isna().sum().to_frame("Nulos"))

    duplicados = df.duplicated(
        subset=colunas_chave_tipo,
        keep=False
    )

    print("Registros envolvidos em duplicidades:", duplicados.sum())

    if duplicados.any():
        display(
            df.loc[duplicados]
            .sort_values(colunas_chave_tipo)
            .head(20)
        )

In [ ]:
# Localiza a linha com valores nulos em agosto
linhas_nulas_agosto = df_tipo_agosto[
    df_tipo_agosto.isna().any(axis=1)
]

display(linhas_nulas_agosto)

print(
    "Quantidade de veículos nessa linha:",
    linhas_nulas_agosto["Qtd. Veículos"].sum()
)

print(
    "Índice da linha:",
    linhas_nulas_agosto.index.tolist()
)

In [ ]:
total_geral_agosto = linhas_nulas_agosto["Qtd. Veículos"].sum()

total_classificado_agosto = (
    df_tipo_agosto.loc[
        ~df_tipo_agosto.index.isin(linhas_nulas_agosto.index),
        "Qtd. Veículos"
    ].sum()
)

total_setembro = df_tipo_setembro["Qtd. Veículos"].sum()

print(f"Total geral informado em agosto: {total_geral_agosto:,.0f}")
print(f"Soma das linhas classificadas em agosto: {total_classificado_agosto:,.0f}")
print(f"Diferença agosto: {total_geral_agosto - total_classificado_agosto:,.0f}")
print(f"Total de setembro: {total_setembro:,.0f}")

In [ ]:
total_setembro = df_tipo_setembro["Qtd. Veículos"].sum()

print(f"Total geral de setembro: {total_setembro:,.0f}")
print(f"Total de registros dimensionais: {len(df_tipo_setembro):,}")
print(
    "Valores nulos:",
    df_tipo_setembro.isna().sum().sum()
)

In [ ]:
# Remove somente a linha de total geral de agosto na cópia de trabalho
df_tipo_agosto_dim = df_tipo_agosto.dropna(
    subset=colunas_chave_tipo
).copy()

# Preserva apenas registros dimensionais de setembro
df_tipo_setembro_dim = df_tipo_setembro.dropna(
    subset=colunas_chave_tipo
).copy()

# Conjuntos de chaves
chaves_agosto = set(
    map(
        tuple,
        df_tipo_agosto_dim[colunas_chave_tipo].itertuples(
            index=False,
            name=None
        )
    )
)

chaves_setembro = set(
    map(
        tuple,
        df_tipo_setembro_dim[colunas_chave_tipo].itertuples(
            index=False,
            name=None
        )
    )
)

# Comparação
print(f"Combinações em agosto: {len(chaves_agosto):,}")
print(f"Combinações em setembro: {len(chaves_setembro):,}")
print(f"Presentes nos dois meses: {len(chaves_agosto & chaves_setembro):,}")
print(f"Somente em agosto: {len(chaves_agosto - chaves_setembro):,}")
print(f"Somente em setembro: {len(chaves_setembro - chaves_agosto):,}")

In [ ]:
# Combinações exclusivas de cada mês
exclusivas_agosto = pd.DataFrame(
    list(chaves_agosto - chaves_setembro),
    columns=colunas_chave_tipo
).merge(
    df_tipo_agosto_dim,
    on=colunas_chave_tipo,
    how="left"
)

exclusivas_setembro = pd.DataFrame(
    list(chaves_setembro - chaves_agosto),
    columns=colunas_chave_tipo
).merge(
    df_tipo_setembro_dim,
    on=colunas_chave_tipo,
    how="left"
)

print("COMBINAÇÕES EXCLUSIVAS DE AGOSTO")
print("Quantidade de veículos:", 
      f"{exclusivas_agosto['Qtd. Veículos'].sum():,.0f}")

display(
    exclusivas_agosto
    .sort_values("Qtd. Veículos", ascending=False)
    .head(20)
)

print("\nCOMBINAÇÕES EXCLUSIVAS DE SETEMBRO")
print("Quantidade de veículos:",
      f"{exclusivas_setembro['Qtd. Veículos'].sum():,.0f}")

display(
    exclusivas_setembro
    .sort_values("Qtd. Veículos", ascending=False)
    .head(20)
)

In [ ]:
for mes, df in [
    ("Agosto", exclusivas_agosto),
    ("Setembro", exclusivas_setembro)
]:
    df_escopo = df[
        df["Tipo Veículo"].isin(tipos_candidatos)
    ]

    print(f"\n{'=' * 55}")
    print(f"Exclusividades do escopo — {mes}")
    print(f"{'=' * 55}")

    print("Combinações exclusivas:", len(df_escopo))
    print(
        "Quantidade de veículos:",
        f"{df_escopo['Qtd. Veículos'].sum():,.0f}"
    )

    display(
        df_escopo.groupby("Tipo Veículo")
        .agg(
            combinacoes=("Qtd. Veículos", "size"),
            veiculos=("Qtd. Veículos", "sum")
        )
        .sort_values("veiculos", ascending=False)
    )

In [ ]:
# Recorte do escopo em cópias de trabalho
df_escopo_agosto = df_tipo_agosto_dim[
    df_tipo_agosto_dim["Tipo Veículo"].isin(tipos_candidatos)
].copy()

df_escopo_setembro = df_tipo_setembro_dim[
    df_tipo_setembro_dim["Tipo Veículo"].isin(tipos_candidatos)
].copy()

# Comparação por UF
uf_agosto = (
    df_escopo_agosto.groupby("UF")["Qtd. Veículos"]
    .sum()
    .rename("Agosto")
)

uf_setembro = (
    df_escopo_setembro.groupby("UF")["Qtd. Veículos"]
    .sum()
    .rename("Setembro")
)

comparacao_uf = pd.concat(
    [uf_agosto, uf_setembro],
    axis=1
).fillna(0)

comparacao_uf["Diferença"] = (
    comparacao_uf["Setembro"] - comparacao_uf["Agosto"]
)

comparacao_uf["Variação percentual (%)"] = (
    comparacao_uf["Diferença"]
    / comparacao_uf["Agosto"].replace(0, np.nan)
    * 100
)

display(
    comparacao_uf
    .sort_values("Setembro", ascending=False)
    .round(2)
)

In [ ]:
for mes, df in [
    ("Agosto", df_escopo_agosto),
    ("Setembro", df_escopo_setembro)
]:
    print(f"\n{'=' * 55}")
    print(f"Qualidade geográfica — {mes}")
    print(f"{'=' * 55}")

    mascara = (
        df["UF"].eq("Sem Informação")
        | df["Município"].eq("Sem Informação")
    )

    registros_sem_localizacao = df.loc[mascara]

    print(
        "Combinações com UF ou município sem informação:",
        len(registros_sem_localizacao)
    )

    print(
        "Quantidade de veículos:",
        f"{registros_sem_localizacao['Qtd. Veículos'].sum():,.0f}"
    )

    display(
        registros_sem_localizacao
        .groupby(["UF", "Município"])["Qtd. Veículos"]
        .sum()
        .sort_values(ascending=False)
        .to_frame()
    )

In [ ]:
for mes, df in [
    ("Agosto", df_escopo_agosto),
    ("Setembro", df_escopo_setembro)
]:
    df_validos = df[
        (df["UF"] != "Sem Informação")
        & (df["Município"] != "Sem Informação")
    ]

    municipios = (
        df_validos.groupby(["UF", "Município"])["Qtd. Veículos"]
        .sum()
        .sort_values(ascending=False)
    )

    print(f"\n{'=' * 55}")
    print(f"Distribuição municipal — {mes}")
    print(f"{'=' * 55}")

    print("Combinações UF/Município:", len(municipios))
    print("Municípios distintos:", 
          df_validos["Município"].nunique())

    print("\n20 municípios com maior frota:")
    display(municipios.head(20).to_frame())

In [ ]:
# Municípios associados a mais de uma UF
municipios_por_uf = (
    df_escopo_setembro[
        (df_escopo_setembro["UF"] != "Sem Informação")
        & (df_escopo_setembro["Município"] != "Sem Informação")
    ]
    .groupby("Município")["UF"]
    .nunique()
    .sort_values(ascending=False)
)

municipios_repetidos = municipios_por_uf[
    municipios_por_uf > 1
]

print(
    "Nomes de municípios associados a mais de uma UF:",
    len(municipios_repetidos)
)

print("\nExemplos:")
display(municipios_repetidos.head(30).to_frame("Quantidade de UFs"))

In [ ]:
# Comparação das combinações UF/Município entre os arquivos I e G

colunas_geo = ["UF", "Município"]

geo_tipo = set(
    df_escopo_setembro[colunas_geo]
    .drop_duplicates()
    .itertuples(index=False, name=None)
)

geo_modelo = set(
    df_marcas_modelos[colunas_geo]
    .drop_duplicates()
    .itertuples(index=False, name=None)
) if all(col in df_marcas_modelos.columns for col in colunas_geo) else None

print("Combinações geográficas no arquivo G:", len(geo_tipo))
print("Arquivo de marca/modelo possui colunas geográficas:", geo_modelo is not None)

In [ ]:
# Combinações geográficas completas do arquivo G de setembro
geo_setembro_completo = set(
    df_tipo_setembro_dim[["UF", "Município"]]
    .drop_duplicates()
    .itertuples(index=False, name=None)
)

# Combinações geográficas do escopo selecionado
geo_setembro_escopo = set(
    df_escopo_setembro[["UF", "Município"]]
    .drop_duplicates()
    .itertuples(index=False, name=None)
)

print("Combinações no arquivo G completo:", len(geo_setembro_completo))
print("Combinações no escopo:", len(geo_setembro_escopo))

print("\nCombinações presentes no arquivo completo, mas ausentes no escopo:")

display(
    pd.DataFrame(
        sorted(geo_setembro_completo - geo_setembro_escopo),
        columns=["UF", "Município"]
    )
)

In [ ]:
print("DataFrames disponíveis:")

for nome in [
    "df_agosto",
    "df_setembro",
    "df_marcas_modelos",
    "df_escopo_agosto",
    "df_escopo_setembro"
]:
    print(f"{nome}: {nome in globals()}")

In [ ]:
from pathlib import Path

arquivo_i_agosto = (
    PROJECT_ROOT
    / "data/processed/senatran/2026-08/"
    / "I_Frota_por_UF_Municipio_Marca_e_Modelo_Ano_Agosto_2026.TXT"
)

arquivo_i_setembro = (
    PROJECT_ROOT
    / "data/processed/senatran/2026-09/"
    / "I_Frota_por_UF_Municipio_Marca_e_Modelo_Ano_Setembro_2026.TXT"
)

print("Arquivo I — Agosto:", arquivo_i_agosto.exists())
print("Arquivo I — Setembro:", arquivo_i_setembro.exists())

print("\nTamanhos:")
print(f"Agosto: {arquivo_i_agosto.stat().st_size / (1024**3):.2f} GB")
print(f"Setembro: {arquivo_i_setembro.stat().st_size / (1024**3):.2f} GB")

In [ ]:
def extrair_combinacoes_geograficas(caminho, chunksize=100_000):
    combinacoes = set()

    for chunk in pd.read_csv(
        caminho,
        sep=";",
        encoding="utf-8",
        usecols=["UF", "Município"],
        dtype={"UF": "string", "Município": "string"},
        chunksize=chunksize
    ):
        combinacoes.update(
            chunk[["UF", "Município"]]
            .drop_duplicates()
            .itertuples(index=False, name=None)
        )

    return combinacoes


geo_i_agosto = extrair_combinacoes_geograficas(arquivo_i_agosto)
geo_i_setembro = extrair_combinacoes_geograficas(arquivo_i_setembro)

print("Arquivo I — Agosto:", len(geo_i_agosto))
print("Arquivo I — Setembro:", len(geo_i_setembro))

In [ ]:
# Comparação das combinações geográficas — setembro

print("Arquivo I:", len(geo_i_setembro))
print("Arquivo G:", len(geo_setembro_completo))

somente_i = geo_i_setembro - geo_setembro_completo
somente_g = geo_setembro_completo - geo_i_setembro

print("\nCombinações presentes somente no arquivo I:", len(somente_i))
print("Combinações presentes somente no arquivo G:", len(somente_g))

print("\nPresentes somente no arquivo I:")
display(pd.DataFrame(sorted(somente_i), columns=["UF", "Município"]).head(30))

print("\nPresentes somente no arquivo G:")
display(pd.DataFrame(sorted(somente_g), columns=["UF", "Município"]).head(30))

In [ ]:
# Combinações geográficas do arquivo G — agosto
geo_g_agosto = set(
    df_tipo_agosto_dim[["UF", "Município"]]
    .drop_duplicates()
    .itertuples(index=False, name=None)
)

somente_i_agosto = geo_i_agosto - geo_g_agosto
somente_g_agosto = geo_g_agosto - geo_i_agosto

print("Arquivo I — Agosto:", len(geo_i_agosto))
print("Arquivo G — Agosto:", len(geo_g_agosto))

print("\nSomente no arquivo I:", len(somente_i_agosto))
print("Somente no arquivo G:", len(somente_g_agosto))

In [ ]:
# Base geográfica preliminar — setembro de 2026
df_geografia_base = pd.DataFrame(
    sorted(geo_i_setembro),
    columns=["UF", "Município"]
)

# Classificação preliminar dos registros geográficos
df_geografia_base["Status geográfico"] = np.where(
    df_geografia_base["UF"].isin(
        ["Sem Informação", "Não Identificado", "Não se Aplica"]
    ),
    "Registro especial",
    "Identificação informada"
)

print("Total de combinações:", len(df_geografia_base))

print("\nDistribuição por status:")
display(
    df_geografia_base["Status geográfico"]
    .value_counts()
    .to_frame("Quantidade")
)

print("\nRegistros especiais:")
display(
    df_geografia_base[
        df_geografia_base["Status geográfico"] == "Registro especial"
    ]
)

In [ ]:
# Separação dos municípios identificados
df_geografia_municipios = (
    df_geografia_base[
        df_geografia_base["Status geográfico"]
        == "Identificação informada"
    ]
    .copy()
    .reset_index(drop=True)
)

print("Municípios identificados:")
print("Combinações UF/Município:", len(df_geografia_municipios))
print("Nomes distintos:", df_geografia_municipios["Município"].nunique())
print("UFs distintas:", df_geografia_municipios["UF"].nunique())

print("\nDistribuição de municípios por UF:")
display(
    df_geografia_municipios
    .groupby("UF")["Município"]
    .nunique()
    .sort_values(ascending=False)
    .to_frame("Municípios distintos")
)

In [ ]:
# Verificar registros do Distrito Federal e de Pernambuco
print("Distrito Federal:")
display(
    df_geografia_municipios[
        df_geografia_municipios["UF"] == "DISTRITO FEDERAL"
    ]
)

print("\nRegistros de Pernambuco relacionados a Fernando de Noronha:")
display(
    df_geografia_municipios[
        (df_geografia_municipios["UF"] == "PERNAMBUCO")
        & (
            df_geografia_municipios["Município"]
            .str.contains("NORONHA", case=False, na=False)
        )
    ]
)

print("\nVerificar Boa Esperança do Norte:")
display(
    df_geografia_municipios[
        (df_geografia_municipios["UF"] == "MATO GROSSO")
        & (
            df_geografia_municipios["Município"]
            .str.contains("BOA ESPERANCA DO NORTE", case=False, na=False)
        )
    ]
)

In [ ]:
# Correspondências especiais verificadas no IBGE
mapeamento_especial_ibge = {
    ("DISTRITO FEDERAL", "BRASILIA"): "5300108",
    ("PERNAMBUCO", "FERNANDO DE NORONHA"): "2605459",
    ("MATO GROSSO", "BOA ESPERANCA DO NORTE"): "5101837"
}

df_mapeamento_especial = pd.DataFrame(
    [
        {
            "UF": uf,
            "Município": municipio,
            "Código IBGE": codigo
        }
        for (uf, municipio), codigo
        in mapeamento_especial_ibge.items()
    ]
)

display(df_mapeamento_especial)

In [ ]:
import requests

url_ibge = (
    "https://servicodados.ibge.gov.br/api/v1/localidades/municipios"
)

resposta_ibge = requests.get(url_ibge, timeout=60)
resposta_ibge.raise_for_status()

dados_ibge = resposta_ibge.json()

print("Registros retornados:", len(dados_ibge))
print("\nExemplo:")
print(dados_ibge[0])

In [ ]:
import unicodedata

def normalizar_texto(valor):
    texto = unicodedata.normalize("NFKD", str(valor))
    texto = "".join(
        caractere for caractere in texto
        if not unicodedata.combining(caractere)
    )
    return " ".join(texto.upper().split())


# Transformar o retorno da API em DataFrame
df_ibge = pd.DataFrame([
    {
        "Código IBGE": item["id"],
        "Município IBGE": item["nome"],
        "UF IBGE": item["microrregiao"]["mesorregiao"]["UF"]["nome"],
        "Sigla UF": item["microrregiao"]["mesorregiao"]["UF"]["sigla"],
        "Região": item["microrregiao"]["mesorregiao"]["UF"]["regiao"]["nome"]
    }
    for item in dados_ibge
])

# Criar chaves normalizadas para comparação
df_ibge["UF normalizada"] = df_ibge["UF IBGE"].apply(normalizar_texto)
df_ibge["Município normalizado"] = df_ibge["Município IBGE"].apply(normalizar_texto)

df_geografia_municipios["UF normalizada"] = (
    df_geografia_municipios["UF"].apply(normalizar_texto)
)

df_geografia_municipios["Município normalizado"] = (
    df_geografia_municipios["Município"].apply(normalizar_texto)
)

print("Registros na referência IBGE:", len(df_ibge))
print("Códigos IBGE distintos:", df_ibge["Código IBGE"].nunique())
print("UFs na referência IBGE:", df_ibge["Sigla UF"].nunique())

print("\nExemplo da referência:")
display(df_ibge.head())

In [ ]:
# Identificar registros sem a hierarquia esperada
registros_irregulares = []

for item in dados_ibge:
    microrregiao = item.get("microrregiao")

    if microrregiao is None:
        registros_irregulares.append(item)
        continue

    mesorregiao = microrregiao.get("mesorregiao")

    if mesorregiao is None or mesorregiao.get("UF") is None:
        registros_irregulares.append(item)

print("Registros com estrutura diferente:", len(registros_irregulares))

for item in registros_irregulares:
    print(item)


In [ ]:
def extrair_uf_ibge(item):
    # Estrutura tradicional
    microrregiao = item.get("microrregiao")

    if microrregiao:
        mesorregiao = microrregiao.get("mesorregiao")

        if mesorregiao and mesorregiao.get("UF"):
            return mesorregiao["UF"]

    # Estrutura alternativa para municípios sem microrregião
    regiao_imediata = item.get("regiao-imediata")

    if regiao_imediata:
        regiao_intermediaria = regiao_imediata.get(
            "regiao-intermediaria"
        )

        if regiao_intermediaria:
            return regiao_intermediaria.get("UF")

    return None


registros_ibge = []

for item in dados_ibge:
    uf = extrair_uf_ibge(item)

    registros_ibge.append({
        "Código IBGE": item["id"],
        "Município IBGE": item["nome"],
        "UF IBGE": uf["nome"] if uf else None,
        "Sigla UF": uf["sigla"] if uf else None,
        "Região": uf["regiao"]["nome"] if uf else None
    })

df_ibge = pd.DataFrame(registros_ibge)

print("Registros:", len(df_ibge))
print("Códigos distintos:", df_ibge["Código IBGE"].nunique())
print("Registros sem UF:", df_ibge["UF IBGE"].isna().sum())

print("\nBoa Esperança do Norte:")
display(
    df_ibge[
        df_ibge["Código IBGE"] == 5101837
    ]
)

In [ ]:
import unicodedata

def normalizar_texto(valor):
    texto = unicodedata.normalize("NFKD", str(valor))
    texto = "".join(
        c for c in texto
        if not unicodedata.combining(c)
    )
    return " ".join(texto.upper().split())


# Preparar chaves de comparação no IBGE
df_ibge["UF normalizada"] = df_ibge["UF IBGE"].apply(normalizar_texto)
df_ibge["Município normalizado"] = (
    df_ibge["Município IBGE"].apply(normalizar_texto)
)

# Preparar chaves de comparação no SENATRAN
df_geografia_municipios["UF normalizada"] = (
    df_geografia_municipios["UF"].apply(normalizar_texto)
)

df_geografia_municipios["Município normalizado"] = (
    df_geografia_municipios["Município"].apply(normalizar_texto)
)

# Cruzamento das duas referências
df_validacao_geo = df_geografia_municipios.merge(
    df_ibge[
        [
            "Código IBGE",
            "Município IBGE",
            "UF IBGE",
            "UF normalizada",
            "Município normalizado"
        ]
    ],
    on=["UF normalizada", "Município normalizado"],
    how="outer",
    indicator=True,
    validate="one_to_one"
)

print("Resultado do cruzamento:")
print(df_validacao_geo["_merge"].value_counts())

print("\nTotal de registros:", len(df_validacao_geo))

print("\nRegistros sem correspondência no IBGE:")
display(
    df_validacao_geo[
        df_validacao_geo["_merge"] == "left_only"
    ][["UF", "Município"]]
)

print("\nRegistros do IBGE ausentes no SENATRAN:")
display(
    df_validacao_geo[
        df_validacao_geo["_merge"] == "right_only"
    ][["UF IBGE", "Município IBGE", "Código IBGE"]]
)

In [ ]:
from difflib import SequenceMatcher

# Separar registros sem correspondência
senatran_pendentes = df_validacao_geo[
    df_validacao_geo["_merge"] == "left_only"
].copy()

ibge_pendentes = df_validacao_geo[
    df_validacao_geo["_merge"] == "right_only"
].copy()

candidatos = []

for _, sen in senatran_pendentes.iterrows():

    ibge_mesma_uf = ibge_pendentes[
        ibge_pendentes["UF normalizada"]
        == sen["UF normalizada"]
    ]

    for _, ibge in ibge_mesma_uf.iterrows():

        similaridade = SequenceMatcher(
            None,
            sen["Município normalizado"],
            ibge["Município normalizado"]
        ).ratio()

        candidatos.append({
            "UF": sen["UF"],
            "Município SENATRAN": sen["Município"],
            "Município IBGE": ibge["Município IBGE"],
            "Código IBGE": ibge["Código IBGE"],
            "Similaridade": round(similaridade, 3)
        })

df_candidatos_geo = pd.DataFrame(candidatos)

# Três candidatos mais próximos para cada município SENATRAN
df_candidatos_geo = (
    df_candidatos_geo
    .sort_values(
        ["Município SENATRAN", "Similaridade"],
        ascending=[True, False]
    )
    .groupby(["UF", "Município SENATRAN"])
    .head(3)
    .reset_index(drop=True)
)

display(df_candidatos_geo)

In [ ]:
resumo_candidatos = (
    df_candidatos_geo
    .groupby(["UF", "Município SENATRAN"])
    .agg(
        **{
            "Melhor similaridade": ("Similaridade", "max"),
            "Segundo melhor": (
                "Similaridade",
                lambda x: x.nlargest(2).iloc[-1] if len(x) >= 2 else None
            ),
            "Candidato principal": ("Município IBGE", "first"),
            "Código candidato": ("Código IBGE", "first")
        }
    )
    .reset_index()
)

resumo_candidatos["Diferença entre candidatos"] = (
    resumo_candidatos["Melhor similaridade"]
    - resumo_candidatos["Segundo melhor"]
)

display(
    resumo_candidatos.sort_values(
        ["Melhor similaridade", "Diferença entre candidatos"],
        ascending=False
    )
)

In [ ]:
resumo_candidatos["Faixa de similaridade"] = pd.cut(
    resumo_candidatos["Melhor similaridade"],
    bins=[-1, 0.75, 0.90, 1.00],
    labels=[
        "Baixa — investigar",
        "Intermediária — revisar",
        "Alta — candidato forte"
    ]
)

print("Quantidade de municípios por faixa:")
display(
    resumo_candidatos["Faixa de similaridade"]
    .value_counts()
    .to_frame("Quantidade")
)

print("\nMunicípios que exigem maior atenção:")
display(
    resumo_candidatos[
        resumo_candidatos["Faixa de similaridade"]
        == "Baixa — investigar"
    ].sort_values("UF")
)

In [ ]:
registros_mapeamento = [
    {
        "UF": "GOIAS",
        "Município SENATRAN": "BOM JESUS",
        "Município IBGE candidato": "Bom Jesus de Goiás",
        "Código IBGE candidato": "5203500",
        "Tipo de divergência": "Nome abreviado ou variante",
        "Justificativa": "Histórico do IBGE registra a denominação Bom Jesus de Goiás; verificar referência usada pelo SENATRAN.",
        "Status": "Pendente de validação"
    },
    {
        "UF": "PARAIBA",
        "Município SENATRAN": "SANTAREM",
        "Município IBGE candidato": None,
        "Código IBGE candidato": None,
        "Tipo de divergência": "Sem candidato confiável",
        "Justificativa": "Não foi identificada correspondência geográfica confiável.",
        "Status": "Pendente de validação"
    },
    {
        "UF": "PARAIBA",
        "Município SENATRAN": "SAO DOMINGOS DE POMBAL",
        "Município IBGE candidato": "São Domingos",
        "Código IBGE candidato": "2513968",
        "Tipo de divergência": "Possível alteração de denominação",
        "Justificativa": "Investigar histórico de denominação e correspondência territorial.",
        "Status": "Pendente de validação"
    },
    {
        "UF": "RIO GRANDE DO NORTE",
        "Município SENATRAN": "ARES",
        "Município IBGE candidato": "Arês",
        "Código IBGE candidato": "2401206",
        "Tipo de divergência": "Variação de grafia",
        "Justificativa": "Candidato correspondente na listagem oficial do IBGE.",
        "Status": "Pendente de validação"
    },
    {
        "UF": "RIO GRANDE DO NORTE",
        "Município SENATRAN": "BOA SAUDE",
        "Município IBGE candidato": "Januário Cicco",
        "Código IBGE candidato": "2405306",
        "Tipo de divergência": "Possível alteração de denominação",
        "Justificativa": "O histórico territorial do IBGE registra Januário Cicco como ex-Boa Saúde; confirmar a referência temporal.",
        "Status": "Pendente de validação"
    },
    {
        "UF": "RORAIMA",
        "Município SENATRAN": "SAO LUIZ",
        "Município IBGE candidato": "São Luiz do Anauá",
        "Código IBGE candidato": "1400605",
        "Tipo de divergência": "Nome abreviado ou variante",
        "Justificativa": "Candidato oficial do IBGE para Roraima; confirmar a equivalência.",
        "Status": "Pendente de validação"
    },
    {
        "UF": "TOCANTINS",
        "Município SENATRAN": "FORTALEZA DO TABOCAO",
        "Município IBGE candidato": "Tabocão",
        "Código IBGE candidato": "1708254",
        "Tipo de divergência": "Possível alteração de denominação",
        "Justificativa": "Documentação do IBGE de 2022 ainda utiliza Fortaleza do Tabocão; investigar a divergência.",
        "Status": "Pendente de validação"
    },
    {
        "UF": "TOCANTINS",
        "Município SENATRAN": "SAO VALERIO DA NATIVIDADE",
        "Município IBGE candidato": "São Valério",
        "Código IBGE candidato": "1720499",
        "Tipo de divergência": "Possível alteração de denominação",
        "Justificativa": "Candidato textual; falta comprovação da equivalência histórica.",
        "Status": "Pendente de validação"
    }
]

df_mapeamento_manual_geo = pd.DataFrame(registros_mapeamento)

display(df_mapeamento_manual_geo)

In [ ]:
# Criar cópias para preservar os DataFrames originais
df_mapeamento_validacao = df_mapeamento_manual_geo.copy()
df_ibge_validacao = df_ibge.copy()

# Padronizar os códigos IBGE
df_mapeamento_validacao["Código IBGE candidato"] = pd.to_numeric(
    df_mapeamento_validacao["Código IBGE candidato"],
    errors="coerce"
).astype("Int64")

df_ibge_validacao["Código IBGE"] = pd.to_numeric(
    df_ibge_validacao["Código IBGE"],
    errors="coerce"
).astype("Int64")

# Cruzar os candidatos com a base oficial
df_validacao_mapeamento = df_mapeamento_validacao.merge(
    df_ibge_validacao[
        ["Código IBGE", "Município IBGE", "Sigla UF", "UF IBGE"]
    ],
    left_on="Código IBGE candidato",
    right_on="Código IBGE",
    how="left",
    indicator=True
)

# Relacionar as UFs
mapa_ufs = {
    "GOIAS": "GO",
    "PARAIBA": "PB",
    "RIO GRANDE DO NORTE": "RN",
    "RORAIMA": "RR",
    "TOCANTINS": "TO"
}

df_validacao_mapeamento["UF esperada"] = (
    df_validacao_mapeamento["UF"].map(mapa_ufs)
)

df_validacao_mapeamento["UF confere"] = (
    df_validacao_mapeamento["UF esperada"]
    == df_validacao_mapeamento["Sigla UF"]
)

display(
    df_validacao_mapeamento[
        [
            "UF",
            "Município SENATRAN",
            "Município IBGE candidato",
            "Código IBGE candidato",
            "Município IBGE",
            "Sigla UF",
            "UF confere",
            "_merge"
        ]
    ]
)

In [ ]:
df_validacao_mapeamento["Validação código e UF"] = np.where(
    df_validacao_mapeamento["_merge"].eq("both")
    & df_validacao_mapeamento["UF confere"],
    "Código e UF conferem",
    "Sem correspondência validada"
)

df_validacao_mapeamento["Validação da equivalência"] = (
    "Pendente de comprovação histórica"
)

df_validacao_mapeamento.loc[
    df_validacao_mapeamento["_merge"].eq("left_only"),
    "Validação da equivalência"
] = "Sem candidato — investigar"

display(
    df_validacao_mapeamento[
        [
            "UF",
            "Município SENATRAN",
            "Município IBGE candidato",
            "Código IBGE candidato",
            "Validação código e UF",
            "Validação da equivalência"
        ]
    ]
)

In [ ]:
# Recomeçar a partir da tabela original, sem aproveitar alterações parciais
df_mapeamento_geo_final = df_mapeamento_manual_geo.copy()

# Converter a coluna de códigos para inteiro anulável
df_mapeamento_geo_final["Código IBGE candidato"] = (
    pd.to_numeric(
        df_mapeamento_geo_final["Código IBGE candidato"],
        errors="coerce"
    ).astype("Int64")
)

# Aplicar as equivalências documentadas
for indice, linha in df_mapeamento_geo_final.iterrows():

    equivalencia = equivalencias_confirmadas[
        linha["Município SENATRAN"]
    ]

    df_mapeamento_geo_final.loc[indice, "Município IBGE candidato"] = (
        equivalencia["nome"]
    )

    df_mapeamento_geo_final.loc[indice, "Código IBGE candidato"] = (
        equivalencia["codigo"]
    )

    df_mapeamento_geo_final.loc[indice, "Tipo de divergência"] = (
        equivalencia["tipo"]
    )

    df_mapeamento_geo_final.loc[indice, "Justificativa"] = (
        equivalencia["justificativa"]
    )

    df_mapeamento_geo_final.loc[indice, "Fonte de validação"] = (
        equivalencia["fonte"]
    )

    df_mapeamento_geo_final.loc[indice, "Status"] = (
        "Equivalência documentada"
    )

display(df_mapeamento_geo_final)

In [ ]:
# Cópias para validação, preservando as tabelas originais
df_mapa_validacao = df_mapeamento_geo_final.copy()
df_ibge_validacao = df_ibge.copy()

# Padronizar os tipos dos códigos
df_mapa_validacao["Código IBGE candidato"] = (
    pd.to_numeric(
        df_mapa_validacao["Código IBGE candidato"],
        errors="coerce"
    ).astype("Int64")
)

df_ibge_validacao["Código IBGE"] = (
    pd.to_numeric(
        df_ibge_validacao["Código IBGE"],
        errors="coerce"
    ).astype("Int64")
)

# Relacionar cada UF do SENATRAN à sigla oficial
mapa_ufs = {
    "GOIAS": "GO",
    "PARAIBA": "PB",
    "RIO GRANDE DO NORTE": "RN",
    "RORAIMA": "RR",
    "TOCANTINS": "TO"
}

df_mapa_validacao["UF esperada"] = (
    df_mapa_validacao["UF"].map(mapa_ufs)
)

# Conferir os códigos na base oficial
df_validacao_final = df_mapa_validacao.merge(
    df_ibge_validacao[
        ["Código IBGE", "Município IBGE", "Sigla UF"]
    ],
    left_on="Código IBGE candidato",
    right_on="Código IBGE",
    how="left",
    indicator=True,
    validate="many_to_one"
)

# Validar existência do código e correspondência da UF
df_validacao_final["Código encontrado"] = (
    df_validacao_final["_merge"].eq("both")
)

df_validacao_final["UF confere"] = (
    df_validacao_final["UF esperada"]
    == df_validacao_final["Sigla UF"]
)

df_validacao_final["Validação técnica"] = np.where(
    df_validacao_final["Código encontrado"]
    & df_validacao_final["UF confere"],
    "Aprovado",
    "Revisar"
)

print("Resultado da validação técnica:")
print(df_validacao_final["Validação técnica"].value_counts())

display(
    df_validacao_final[
        [
            "UF",
            "Município SENATRAN",
            "Município IBGE candidato",
            "Código IBGE candidato",
            "Município IBGE",
            "Sigla UF",
            "Validação técnica"
        ]
    ]
)

In [ ]:
# Criar uma cópia para preservar o resultado anterior
df_mapeamento_geo_final = df_mapeamento_geo_final.copy()

# Padronizar o nome de São Luiz conforme o cadastro oficial do IBGE
df_mapeamento_geo_final.loc[
    df_mapeamento_geo_final["Município SENATRAN"].eq("SAO LUIZ"),
    "Município IBGE candidato"
] = "São Luiz do Anauá"

# Incorporar os resultados da validação técnica
df_mapeamento_geo_final = df_mapeamento_geo_final.merge(
    df_validacao_final[
        [
            "Município SENATRAN",
            "Código encontrado",
            "UF confere",
            "Validação técnica"
        ]
    ],
    on="Município SENATRAN",
    how="left",
    validate="one_to_one"
)

# Classificação: validação técnica não substitui evidência histórica
df_mapeamento_geo_final["Status"] = np.where(
    df_mapeamento_geo_final["Código encontrado"]
    & df_mapeamento_geo_final["UF confere"],
    "Código e UF validados — equivalência documentada",
    "Pendente de revisão"
)

display(df_mapeamento_geo_final)

In [ ]:
# Recuperar os 42 registros sem correspondência exata no IBGE
df_pendentes_geo = df_validacao_geo.loc[
    df_validacao_geo["_merge"].eq("left_only"),
    ["UF", "Município"]
].copy()

# Identificar os oito municípios já documentados
municipios_mapeados = set(
    zip(
        df_mapeamento_geo_final["UF"],
        df_mapeamento_geo_final["Município SENATRAN"]
    )
)

# Separar os municípios ainda não contemplados
df_municipios_restantes = df_pendentes_geo.loc[
    ~df_pendentes_geo.apply(
        lambda linha: (linha["UF"], linha["Município"])
        in municipios_mapeados,
        axis=1
    )
].copy()

print("Divergências originais:", len(df_pendentes_geo))
print("Municípios já mapeados:", len(municipios_mapeados))
print("Municípios restantes:", len(df_municipios_restantes))

display(
    df_municipios_restantes.sort_values(
        ["UF", "Município"]
    ).reset_index(drop=True)
)

In [ ]:
# Candidatos identificados na consulta ao IBGE
candidatos_34 = {
    ("BAHIA", "LAGEDO DO TABOCAL"): ("Lajedo do Tabocal", 2919058),
    ("BAHIA", "SANTA TERESINHA"): ("Santa Terezinha", 2928505),
    ("MARANHAO", "PINDARE MIRIM"): ("Pindaré-Mirim", 2108504),
    ("MATO GROSSO", "SANTO ANTONIO DO LEVERGER"): ("Santo Antônio de Leverger", 5107800),
    ("MINAS GERAIS", "AMPARO DA SERRA"): ("Amparo do Serra", 3102506),
    ("MINAS GERAIS", "BARAO D0 MONTE ALTO"): ("Barão do Monte Alto", 3105509),
    ("MINAS GERAIS", "PINGO D'AGUA"): ("Pingo-d'Água", 3150539),
    ("MINAS GERAIS", "QUELUZITA"): (None, None),
    ("PARA", "ELDORADO DOS CARAJAS"): ("Eldorado do Carajás", 1502954),
    ("PARA", "SANTA ISABEL DO PARA"): ("Santa Izabel do Pará", 1506500),
    ("PARANA", "BELA VISTA DO CAROBA"): ("Bela Vista da Caroba", 4102752),
    ("PARANA", "MUNHOZ DE MELLO"): ("Munhoz de Melo", 4116307),
    ("PARANA", "PINHAL DO SAO BENTO"): ("Pinhal de São Bento", 4119251),
    ("PARANA", "SANTA CRUZ DO MONTE CASTELO"): ("Santa Cruz de Monte Castelo", 4123303),
    ("PERNAMBUCO", "BELEM DE SAO FRANCISCO"): ("Belém do São Francisco", 2601607),
    ("PERNAMBUCO", "IGUARACI"): ("Iguaracy", 2606903),
    ("PERNAMBUCO", "LAGOA DO ITAENGA"): ("Lagoa de Itaenga", 2608503),
    ("RIO DE JANEIRO", "PARATI"): ("Paraty", 3303807),
    ("RIO DE JANEIRO", "TRAJANO DE MORAIS"): ("Trajano de Moraes", 3305901),
    ("RIO GRANDE DO NORTE", "CERRO-CORA"): ("Cerro Corá", 2402709),
    ("RIO GRANDE DO NORTE", "LAGOA DANTA"): ("Lagoa d'Anta", 2406205),
    ("RIO GRANDE DO SUL", "ENTRE IJUIS"): ("Entre-Ijuís", 4306932),
    ("RIO GRANDE DO SUL", "SANTANA DO LIVRAMENTO"): ("Sant'Ana do Livramento", 4317103),
    ("SANTA CATARINA", "BALNEARIO DE PICARRAS"): ("Balneário Piçarras", 4212809),
    ("SANTA CATARINA", "GRAO PARA"): ("Grão-Pará", 4206108),
    ("SANTA CATARINA", "LAGEADO GRANDE"): ("Lajeado Grande", 4209458),
    ("SANTA CATARINA", "PRESIDENTE CASTELO BRANCO"): ("Presidente Castello Branco", 4213906),
    ("SANTA CATARINA", "SAO LOURENCO D'OESTE"): ("São Lourenço do Oeste", 4216909),
    ("SANTA CATARINA", "SAO MIGUEL D'OESTE"): ("São Miguel do Oeste", 4217204),
    ("SAO PAULO", "BIRITIBA-MIRIM"): ("Biritiba Mirim", 3506607),
    ("SAO PAULO", "MOGI-GUACU"): ("Mogi Guaçu", 3530706),
    ("SAO PAULO", "MOGI-MIRIM"): ("Mogi Mirim", 3530805),
    ("SERGIPE", "AMPARO DE SAO FRANCISCO"): ("Amparo do São Francisco", 2800100),
    ("TOCANTINS", "COUTO DE MAGALHAES"): ("Couto Magalhães", 1706001)
}

# Construir tabela de candidatos sem alterar os dados originais
registros_candidatos = []

for _, linha in df_municipios_restantes.iterrows():
    chave = (linha["UF"], linha["Município"])
    nome, codigo = candidatos_34[chave]

    registros_candidatos.append({
        "UF": linha["UF"],
        "Município SENATRAN": linha["Município"],
        "Município IBGE candidato": nome,
        "Código IBGE candidato": codigo,
        "Status": (
            "Sem candidato confiável — investigar"
            if codigo is None
            else "Candidato — validar código e UF"
        )
    })

df_candidatos_34 = pd.DataFrame(registros_candidatos)

df_candidatos_34["Código IBGE candidato"] = (
    pd.to_numeric(
        df_candidatos_34["Código IBGE candidato"],
        errors="coerce"
    ).astype("Int64")
)

print("Total de candidatos:", len(df_candidatos_34))
print("\nDistribuição dos status:")
print(df_candidatos_34["Status"].value_counts())

display(df_candidatos_34)

In [ ]:
# Criar cópias para validação
df_candidatos_validacao = df_candidatos_34.copy()
df_ibge_validacao_34 = df_ibge.copy()

# Padronizar códigos
df_candidatos_validacao["Código IBGE candidato"] = (
    pd.to_numeric(
        df_candidatos_validacao["Código IBGE candidato"],
        errors="coerce"
    ).astype("Int64")
)

df_ibge_validacao_34["Código IBGE"] = (
    pd.to_numeric(
        df_ibge_validacao_34["Código IBGE"],
        errors="coerce"
    ).astype("Int64")
)

# Mapa de UFs
mapa_ufs_completo = {
    "BAHIA": "BA",
    "MARANHAO": "MA",
    "MATO GROSSO": "MT",
    "MINAS GERAIS": "MG",
    "PARA": "PA",
    "PARANA": "PR",
    "PERNAMBUCO": "PE",
    "RIO DE JANEIRO": "RJ",
    "RIO GRANDE DO NORTE": "RN",
    "RIO GRANDE DO SUL": "RS",
    "SANTA CATARINA": "SC",
    "SAO PAULO": "SP",
    "SERGIPE": "SE",
    "TOCANTINS": "TO"
}

df_candidatos_validacao["UF esperada"] = (
    df_candidatos_validacao["UF"].map(mapa_ufs_completo)
)

# Cruzar com o cadastro oficial do IBGE
df_validacao_candidatos_34 = df_candidatos_validacao.merge(
    df_ibge_validacao_34[
        ["Código IBGE", "Município IBGE", "Sigla UF"]
    ],
    left_on="Código IBGE candidato",
    right_on="Código IBGE",
    how="left",
    indicator=True,
    validate="many_to_one"
)

# Verificar existência do código e UF
df_validacao_candidatos_34["Código encontrado"] = (
    df_validacao_candidatos_34["_merge"].eq("both")
)

df_validacao_candidatos_34["UF confere"] = (
    df_validacao_candidatos_34["UF esperada"]
    == df_validacao_candidatos_34["Sigla UF"]
)

df_validacao_candidatos_34["Validação técnica"] = np.select(
    [
        df_validacao_candidatos_34["Código IBGE candidato"].isna(),
        ~df_validacao_candidatos_34["Código encontrado"],
        ~df_validacao_candidatos_34["UF confere"]
    ],
    [
        "Sem candidato",
        "Código não encontrado",
        "UF divergente"
    ],
    default="Código e UF conferem"
)

# Resumo
print("Resultado da validação técnica:")
print(df_validacao_candidatos_34["Validação técnica"].value_counts())

print(
    "\nCódigos candidatos duplicados:",
    df_validacao_candidatos_34[
        "Código IBGE candidato"
    ].dropna().duplicated().sum()
)

display(
    df_validacao_candidatos_34[
        [
            "UF",
            "Município SENATRAN",
            "Município IBGE candidato",
            "Código IBGE candidato",
            "Município IBGE",
            "Sigla UF",
            "Validação técnica"
        ]
    ]
)

In [ ]:
# Consultar municípios de Minas Gerais na base oficial já carregada
df_ibge_mg = df_ibge[
    df_ibge["Sigla UF"].eq("MG")
].copy()

# Buscar nomes que contenham "QUELUZ"
df_queluz_mg = df_ibge_mg[
    df_ibge_mg["Município IBGE"]
    .str.contains("queluz", case=False, na=False)
].copy()

print("Municípios encontrados em Minas Gerais:")
display(
    df_queluz_mg[
        ["Código IBGE", "Município IBGE", "UF IBGE", "Sigla UF"]
    ].sort_values("Município IBGE")
)

In [ ]:
df_candidatos_34.loc[
    df_candidatos_34["Município SENATRAN"].eq("QUELUZITA"),
    "Município IBGE candidato"
] = None

df_candidatos_34.loc[
    df_candidatos_34["Município SENATRAN"].eq("QUELUZITA"),
    "Código IBGE candidato"
] = pd.NA

df_candidatos_34.loc[
    df_candidatos_34["Município SENATRAN"].eq("QUELUZITA"),
    "Status"
] = "Pendente de investigação — sem equivalência comprovada"

display(
    df_candidatos_34[
        df_candidatos_34["Município SENATRAN"].eq("QUELUZITA")
    ]
)

In [ ]:
# Preparar os oito mapeamentos documentados
df_mapa_8 = df_mapeamento_geo_final.copy()

# Preparar os 34 candidatos
df_mapa_34 = df_candidatos_34.copy()

# Padronizar a estrutura das duas tabelas
colunas_mapa = [
    "UF",
    "Município SENATRAN",
    "Município IBGE candidato",
    "Código IBGE candidato",
    "Status"
]

df_mapa_8 = df_mapa_8[colunas_mapa].copy()
df_mapa_34 = df_mapa_34[colunas_mapa].copy()

# Concatenar os registros
df_equivalencias_geo = pd.concat(
    [df_mapa_8, df_mapa_34],
    ignore_index=True
)

# Garantir o tipo inteiro anulável para os códigos
df_equivalencias_geo["Código IBGE candidato"] = (
    pd.to_numeric(
        df_equivalencias_geo["Código IBGE candidato"],
        errors="coerce"
    ).astype("Int64")
)

# Ordenar para facilitar a auditoria
df_equivalencias_geo = (
    df_equivalencias_geo
    .sort_values(["UF", "Município SENATRAN"])
    .reset_index(drop=True)
)

# Resumo da consolidação
print("Total de equivalências:", len(df_equivalencias_geo))
print(
    "Municípios com código:",
    df_equivalencias_geo["Código IBGE candidato"].notna().sum()
)
print(
    "Municípios sem código:",
    df_equivalencias_geo["Código IBGE candidato"].isna().sum()
)
print(
    "Municípios SENATRAN duplicados:",
    df_equivalencias_geo["Município SENATRAN"].duplicated().sum()
)

display(df_equivalencias_geo)

In [ ]:
# Atualizar o status dos 33 candidatos com código e UF validados
df_mapa_34.loc[
    df_mapa_34["Código IBGE candidato"].notna()
    & df_mapa_34["Status"].eq("Candidato — validar código e UF"),
    "Status"
] = "Código e UF conferem — equivalência nominal pendente de documentação"

# Reunir novamente os dois conjuntos
df_equivalencias_geo = pd.concat(
    [df_mapa_8, df_mapa_34],
    ignore_index=True
)

df_equivalencias_geo = (
    df_equivalencias_geo
    .sort_values(["UF", "Município SENATRAN"])
    .reset_index(drop=True)
)

# Verificar duplicidade pela chave composta
duplicados = df_equivalencias_geo.duplicated(
    subset=["UF", "Município SENATRAN"]
).sum()

print("Total de equivalências:", len(df_equivalencias_geo))
print("Com código IBGE:", df_equivalencias_geo["Código IBGE candidato"].notna().sum())
print("Sem código IBGE:", df_equivalencias_geo["Código IBGE candidato"].isna().sum())
print("Duplicidades na chave UF + Município:", duplicados)

print("\nDistribuição dos status:")
display(df_equivalencias_geo["Status"].value_counts().to_frame("Quantidade"))

display(df_equivalencias_geo)

In [ ]:
from difflib import SequenceMatcher

# Analisar somente os 33 casos ainda pendentes de documentação
df_auditoria_nominal = df_equivalencias_geo[
    df_equivalencias_geo["Status"].eq(
        "Código e UF conferem — equivalência nominal pendente de documentação"
    )
].copy()

def calcular_similaridade(linha):
    nome_senatran = normalizar_texto(linha["Município SENATRAN"])
    nome_ibge = normalizar_texto(linha["Município IBGE candidato"])

    return round(
        SequenceMatcher(None, nome_senatran, nome_ibge).ratio(),
        3
    )

df_auditoria_nominal["Similaridade nominal"] = (
    df_auditoria_nominal.apply(calcular_similaridade, axis=1)
)

df_auditoria_nominal["Triagem"] = np.where(
    df_auditoria_nominal["Similaridade nominal"] >= 0.85,
    "Provável variação gráfica — revisar",
    "Investigar possível divergência histórica"
)

df_auditoria_nominal = df_auditoria_nominal.sort_values(
    ["Triagem", "Similaridade nominal"],
    ascending=[True, True]
).reset_index(drop=True)

display(
    df_auditoria_nominal[
        [
            "UF",
            "Município SENATRAN",
            "Município IBGE candidato",
            "Código IBGE candidato",
            "Similaridade nominal",
            "Triagem"
        ]
    ]
)

In [ ]:
def classificar_divergencia(linha):
    senatran = normalizar_texto(linha["Município SENATRAN"])
    ibge = normalizar_texto(linha["Município IBGE candidato"])

    if senatran == ibge:
        return "Acentuação ou formatação"

    if "0" in senatran:
        return "Possível erro de digitação"

    if linha["Similaridade nominal"] < 0.85:
        return "Investigar denominação histórica"

    if (
        senatran.replace("-", " ").replace("'", " ")
        == ibge.replace("-", " ").replace("'", " ")
    ):
        return "Pontuação ou separação de palavras"

    return "Diferença de grafia ou denominação — revisar"


df_auditoria_nominal["Natureza preliminar"] = (
    df_auditoria_nominal.apply(classificar_divergencia, axis=1)
)

display(
    df_auditoria_nominal[
        [
            "UF",
            "Município SENATRAN",
            "Município IBGE candidato",
            "Similaridade nominal",
            "Natureza preliminar"
        ]
    ].sort_values(["Natureza preliminar", "UF"])
)

print("\nDistribuição preliminar:")
display(
    df_auditoria_nominal["Natureza preliminar"]
    .value_counts()
    .to_frame("Quantidade")
)

In [ ]:
# Criar cópia para auditoria documental
df_auditoria_documental = df_auditoria_nominal.copy()

# Campos para registrar a investigação
df_auditoria_documental["Fonte de validação"] = pd.NA
df_auditoria_documental["Evidência encontrada"] = pd.NA
df_auditoria_documental["Decisão final"] = "Pendente"
df_auditoria_documental["Justificativa da decisão"] = pd.NA

print("Registros preparados para auditoria:", len(df_auditoria_documental))
print("\nDistribuição inicial das decisões:")
display(
    df_auditoria_documental["Decisão final"]
    .value_counts()
    .to_frame("Quantidade")
)

display(
    df_auditoria_documental[
        [
            "UF",
            "Município SENATRAN",
            "Município IBGE candidato",
            "Código IBGE candidato",
            "Natureza preliminar",
            "Fonte de validação",
            "Decisão final"
        ]
    ]
)

In [ ]:
# Registrar a evidência oficial para Parati / Paraty
filtro_parati = (
    df_auditoria_documental["Município SENATRAN"].eq("PARATI")
    & df_auditoria_documental["UF"].eq("RIO DE JANEIRO")
)

df_auditoria_documental.loc[
    filtro_parati, "Fonte de validação"
] = (
    "IBGE — Alterações Toponímicas Municipais; "
    "IBGE — atualização de nomes de municípios (07/02/2019)"
)

df_auditoria_documental.loc[
    filtro_parati, "Evidência encontrada"
] = (
    "O IBGE documenta a alteração ortográfica de Parati para Paraty. "
    "A mudança foi oficializada pela Lei Municipal nº 1.553, "
    "de 22/03/2007, e incorporada à base do IBGE em 2008."
)

df_auditoria_documental.loc[
    filtro_parati, "Decisão final"
] = "Equivalência confirmada — alteração ortográfica"

df_auditoria_documental.loc[
    filtro_parati, "Justificativa da decisão"
] = (
    "Parati e Paraty são denominações do mesmo município. "
    "Código IBGE 3303807, conforme documentação oficial."
)

display(
    df_auditoria_documental.loc[
        filtro_parati,
        [
            "UF",
            "Município SENATRAN",
            "Município IBGE candidato",
            "Código IBGE candidato",
            "Fonte de validação",
            "Decisão final",
            "Justificativa da decisão"
        ]
    ]
)

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

arquivo_agosto = (
    PROJECT_ROOT
    / "data/raw/senatran/2026-08/G_Frota_por_UF_Municipio_TIPO_ESPECIE_EIXOS_Agosto_2026.xlsx"
)

arquivo_setembro = (
    PROJECT_ROOT
    / "data/raw/senatran/2026-09/G_Frota_por_UF_Municipio_TIPO_ESPECIE_EIXOS_Setembro_2026.xlsx"
)

print("Arquivo de agosto:", arquivo_agosto.exists())
print("Arquivo de setembro:", arquivo_setembro.exists())

In [ ]:
# Carregar os dados SENATRAN de agosto e setembro de 2026

df_tipo_agosto = pd.read_excel(
    arquivo_agosto,
    sheet_name="Layout G"
)

df_tipo_setembro = pd.read_excel(
    arquivo_setembro,
    sheet_name="Layout G"
)

print("Agosto:", df_tipo_agosto.shape)
print("Setembro:", df_tipo_setembro.shape)

display(df_tipo_setembro.head())

In [ ]:
# Criar cópias de trabalho, preservando os dados originais

colunas_identificacao = [
    "UF",
    "Tipo Veículo",
    "Espécie Veículo",
    "Eixos"
]

df_tipo_agosto_dim = (
    df_tipo_agosto
    .dropna(subset=colunas_identificacao)
    .copy()
)

df_tipo_setembro_dim = (
    df_tipo_setembro
    .dropna(subset=colunas_identificacao)
    .copy()
)

for mes, df in {
    "Agosto": df_tipo_agosto_dim,
    "Setembro": df_tipo_setembro_dim
}.items():
    print(f"\n{mes} de 2026")
    print(f"Registros: {len(df):,}")
    print(f"Quantidade de veículos: {df['Qtd. Veículos'].sum():,}")

In [ ]:
# Definir as categorias incluídas na primeira versão

tipos_candidatos = [
    "AUTOMOVEL",
    "CAMINHONETE",
    "CAMIONETA",
    "UTILITARIO"
]

df_frota_agosto = df_tipo_agosto_dim[
    df_tipo_agosto_dim["Tipo Veículo"].isin(tipos_candidatos)
].copy()

df_frota_setembro = df_tipo_setembro_dim[
    df_tipo_setembro_dim["Tipo Veículo"].isin(tipos_candidatos)
].copy()

for mes, df in {
    "Agosto": df_frota_agosto,
    "Setembro": df_frota_setembro
}.items():
    print(f"\n{mes} de 2026")
    print(f"Registros: {len(df):,}")
    print(f"Quantidade de veículos: {df['Qtd. Veículos'].sum():,}")
    print("\nCategorias:")
    print(df.groupby("Tipo Veículo")["Qtd. Veículos"].sum())

In [ ]:
# Verificar as unidades federativas presentes nas duas bases

ufs_agosto = set(df_frota_agosto["UF"].unique())
ufs_setembro = set(df_frota_setembro["UF"].unique())

print("Quantidade de UFs em agosto:", len(ufs_agosto))
print("Quantidade de UFs em setembro:", len(ufs_setembro))

print("\nUFs presentes apenas em agosto:")
print(sorted(ufs_agosto - ufs_setembro))

print("\nUFs presentes apenas em setembro:")
print(sorted(ufs_setembro - ufs_agosto))

print("\nValores geográficos presentes:")
print(sorted(ufs_agosto | ufs_setembro))

In [ ]:
# Quantificar registros sem identificação geográfica

for mes, df in {
    "Agosto": df_frota_agosto,
    "Setembro": df_frota_setembro
}.items():

    df_sem_uf = df[df["UF"] == "Sem Informação"]

    quantidade = df_sem_uf["Qtd. Veículos"].sum()
    total = df["Qtd. Veículos"].sum()

    percentual = quantidade / total * 100

    print(f"\n{mes} de 2026")
    print(f"Registros: {len(df_sem_uf):,}")
    print(f"Veículos: {quantidade:,}")
    print(f"Participação na frota: {percentual:.3f}%")

In [ ]:
# Criar dimensão geográfica por Unidade Federativa

dados_ufs = [
    ("AC", "Acre", "Norte"),
    ("AL", "Alagoas", "Nordeste"),
    ("AP", "Amapá", "Norte"),
    ("AM", "Amazonas", "Norte"),
    ("BA", "Bahia", "Nordeste"),
    ("CE", "Ceará", "Nordeste"),
    ("DF", "Distrito Federal", "Centro-Oeste"),
    ("ES", "Espírito Santo", "Sudeste"),
    ("GO", "Goiás", "Centro-Oeste"),
    ("MA", "Maranhão", "Nordeste"),
    ("MT", "Mato Grosso", "Centro-Oeste"),
    ("MS", "Mato Grosso do Sul", "Centro-Oeste"),
    ("MG", "Minas Gerais", "Sudeste"),
    ("PA", "Pará", "Norte"),
    ("PB", "Paraíba", "Nordeste"),
    ("PR", "Paraná", "Sul"),
    ("PE", "Pernambuco", "Nordeste"),
    ("PI", "Piauí", "Nordeste"),
    ("RJ", "Rio de Janeiro", "Sudeste"),
    ("RN", "Rio Grande do Norte", "Nordeste"),
    ("RS", "Rio Grande do Sul", "Sul"),
    ("RO", "Rondônia", "Norte"),
    ("RR", "Roraima", "Norte"),
    ("SC", "Santa Catarina", "Sul"),
    ("SP", "São Paulo", "Sudeste"),
    ("SE", "Sergipe", "Nordeste"),
    ("TO", "Tocantins", "Norte"),
]

df_dim_uf = pd.DataFrame(
    dados_ufs,
    columns=["Sigla UF", "Nome UF", "Região"]
)

# Adicionar membro especial para registros sem identificação
df_dim_uf.loc[len(df_dim_uf)] = [
    "SI",
    "Sem Informação",
    "Não identificada"
]

df_dim_uf.insert(
    0,
    "ID UF",
    range(1, len(df_dim_uf) + 1)
)

display(df_dim_uf)
print(f"\nTotal de registros na dimensão: {len(df_dim_uf)}")

In [ ]:
# Validar correspondência entre as bases de frota e a dimensão UF

mapa_uf = dict(
    zip(df_dim_uf["Nome UF"].str.upper(), df_dim_uf["Sigla UF"])
)

mapa_uf["SEM INFORMAÇÃO"] = "SI"

for mes, df in {
    "Agosto": df_frota_agosto,
    "Setembro": df_frota_setembro
}.items():

    ufs_base = set(df["UF"].unique())
    ufs_dimensao = set(mapa_uf.keys())

    sem_correspondencia = ufs_base - ufs_dimensao

    print(f"\n{mes} de 2026")
    print("Categorias geográficas na base:", len(ufs_base))
    print("Categorias sem correspondência:", len(sem_correspondencia))
    print("Valores sem correspondência:", sorted(sem_correspondencia))

In [ ]:
import unicodedata

def normalizar_uf(valor):
    texto = unicodedata.normalize("NFKD", str(valor))
    texto = "".join(
        caractere
        for caractere in texto
        if not unicodedata.combining(caractere)
    )
    return " ".join(texto.upper().split())


# Criar mapa usando nomes normalizados
mapa_uf = {
    normalizar_uf(nome): sigla
    for nome, sigla in zip(
        df_dim_uf["Nome UF"],
        df_dim_uf["Sigla UF"]
    )
}

for mes, df in {
    "Agosto": df_frota_agosto,
    "Setembro": df_frota_setembro
}.items():

    ufs_base = {
        normalizar_uf(valor)
        for valor in df["UF"].unique()
    }

    sem_correspondencia = ufs_base - set(mapa_uf.keys())

    print(f"\n{mes} de 2026")
    print("Categorias geográficas na base:", len(ufs_base))
    print("Categorias sem correspondência:", len(sem_correspondencia))
    print("Valores sem correspondência:", sorted(sem_correspondencia))

In [ ]:
# Criar mapa de nomes normalizados para ID UF

mapa_id_uf = {
    normalizar_uf(nome): id_uf
    for nome, id_uf in zip(
        df_dim_uf["Nome UF"],
        df_dim_uf["ID UF"]
    )
}

# Criar cópias com a chave geográfica
df_frota_agosto_modelo = df_frota_agosto.copy()
df_frota_setembro_modelo = df_frota_setembro.copy()

# Associar ID UF
for df in [df_frota_agosto_modelo, df_frota_setembro_modelo]:
    df["ID UF"] = df["UF"].apply(
        lambda valor: mapa_id_uf[normalizar_uf(valor)]
    )

# Conferir o resultado
for mes, df in {
    "Agosto": df_frota_agosto_modelo,
    "Setembro": df_frota_setembro_modelo
}.items():

    print(f"\n{mes} de 2026")
    print("Registros:", f"{len(df):,}")
    print("Registros sem ID UF:", df["ID UF"].isna().sum())
    print("Quantidade de veículos:", f"{df['Qtd. Veículos'].sum():,}")

display(df_frota_setembro_modelo.head())

In [ ]:
# Validar duplicidades na granularidade original da fonte

colunas_chave_fato = [
    "ID UF",
    "Município",
    "Tipo Veículo",
    "Espécie Veículo",
    "Eixos"
]

for mes, df in {
    "Agosto": df_frota_agosto_modelo,
    "Setembro": df_frota_setembro_modelo
}.items():

    duplicados = df.duplicated(
        subset=colunas_chave_fato,
        keep=False
    )

    print(f"\n{mes} de 2026")
    print("Total de registros:", f"{len(df):,}")
    print("Registros envolvidos em duplicidades:", f"{duplicados.sum():,}")
    print("Combinações duplicadas:", f"{df.loc[duplicados, colunas_chave_fato].drop_duplicates().shape[0]:,}")

In [ ]:
# Comparar as combinações da granularidade entre os dois meses

chaves_agosto = set(
    map(
        tuple,
        df_frota_agosto_modelo[colunas_chave_fato].itertuples(
            index=False,
            name=None
        )
    )
)

chaves_setembro = set(
    map(
        tuple,
        df_frota_setembro_modelo[colunas_chave_fato].itertuples(
            index=False,
            name=None
        )
    )
)

comuns = chaves_agosto & chaves_setembro
somente_agosto = chaves_agosto - chaves_setembro
somente_setembro = chaves_setembro - chaves_agosto

print("Combinações em agosto:", f"{len(chaves_agosto):,}")
print("Combinações em setembro:", f"{len(chaves_setembro):,}")
print("Combinações presentes nos dois meses:", f"{len(comuns):,}")
print("Exclusivas de agosto:", f"{len(somente_agosto):,}")
print("Exclusivas de setembro:", f"{len(somente_setembro):,}")

In [ ]:
# Construir base histórica de frota

df_frota_agosto_final = df_frota_agosto_modelo.copy()
df_frota_setembro_final = df_frota_setembro_modelo.copy()

df_frota_agosto_final["Data Referência"] = pd.Timestamp("2026-08-01")
df_frota_setembro_final["Data Referência"] = pd.Timestamp("2026-09-01")

df_frota_historica = pd.concat(
    [
        df_frota_agosto_final,
        df_frota_setembro_final
    ],
    ignore_index=True
)

# Conferir a base consolidada
print("Total de registros:", f"{len(df_frota_historica):,}")
print(
    "Quantidade total de veículos nas observações:",
    f"{df_frota_historica['Qtd. Veículos'].sum():,}"
)

print("\nRegistros por mês:")
display(
    df_frota_historica.groupby("Data Referência").agg(
        Registros=("Qtd. Veículos", "size"),
        Veículos=("Qtd. Veículos", "sum")
    )
)

In [ ]:
# Validação da base histórica consolidada

print("Total de registros:", len(df_frota_historica))
print("Valores nulos por coluna:")
display(df_frota_historica.isna().sum())

print("\nDatas de referência:")
print(sorted(df_frota_historica["Data Referência"].unique()))

print("\nQuantidade de IDs UF distintos:")
print(df_frota_historica["ID UF"].nunique())

print("\nQuantidade de veículos por mês:")
display(
    df_frota_historica.groupby("Data Referência")["Qtd. Veículos"].sum()
)

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Raiz do projeto:", PROJECT_ROOT)
print("Pasta src existe:", (PROJECT_ROOT / "src").exists())

In [ ]:
arquivo_database = PROJECT_ROOT / "src" / "database.py"

print(arquivo_database.read_text(encoding="utf-8"))

In [ ]:
from sqlalchemy import text
from src.database import criar_engine

engine = criar_engine()

with engine.connect() as conexao:
    resultado = conexao.execute(
        text("SELECT version(), current_database()")
    ).fetchone()

print("Banco conectado:", resultado[1])
print("Versão PostgreSQL:", resultado[0])

In [ ]:
from sqlalchemy import text

consulta = text("""
    SELECT
        table_schema,
        table_name
    FROM information_schema.tables
    WHERE table_schema IN ('staging', 'analytics')
    ORDER BY table_schema, table_name;
""")

with engine.connect() as conexao:
    tabelas = conexao.execute(consulta).fetchall()

for schema, tabela in tabelas:
    print(f"{schema}.{tabela}")

if not tabelas:
    print("Nenhuma tabela encontrada nos schemas staging e analytics.")

In [ ]:
consulta = text("""
    SELECT schema_name
    FROM information_schema.schemata
    WHERE schema_name IN ('staging', 'analytics')
    ORDER BY schema_name;
""")

with engine.connect() as conexao:
    schemas = conexao.execute(consulta).fetchall()

print("Schemas encontrados:")

for (schema,) in schemas:
    print(schema)

In [ ]:
consulta = text("""
    CREATE TABLE analytics.dim_uf (
        id_uf INTEGER PRIMARY KEY,
        sigla_uf VARCHAR(2) NOT NULL UNIQUE,
        nome_uf VARCHAR(50) NOT NULL,
        regiao VARCHAR(30) NOT NULL
    );
""")

with engine.begin() as conexao:
    conexao.execute(consulta)

print("Tabela analytics.dim_uf criada.")

In [ ]:
from sqlalchemy import text

registros_uf = df_dim_uf.rename(columns={
    "ID UF": "id_uf",
    "Sigla UF": "sigla_uf",
    "Nome UF": "nome_uf",
    "Região": "regiao"
})

registros_uf.to_sql(
    name="dim_uf",
    con=engine,
    schema="analytics",
    if_exists="append",
    index=False
)

print("Registros preparados:", len(registros_uf))

In [ ]:
consulta = text("""
    SELECT
        COUNT(*) AS total_registros,
        COUNT(DISTINCT sigla_uf) AS ufs_distintas
    FROM analytics.dim_uf;
""")

with engine.connect() as conexao:
    resultado = conexao.execute(consulta).fetchone()

print("Total de registros:", resultado[0])
print("UFs distintas:", resultado[1])

In [ ]:
consulta = text("""
    SELECT id_uf, sigla_uf, nome_uf, regiao
    FROM analytics.dim_uf
    ORDER BY id_uf;
""")

with engine.connect() as conexao:
    resultado = conexao.execute(consulta).fetchall()

for registro in resultado:
    print(registro)

In [ ]:
consulta = text("""
    CREATE TABLE analytics.fato_frota (
        data_referencia DATE NOT NULL,
        id_uf INTEGER NOT NULL,
        municipio VARCHAR(100) NOT NULL,
        tipo_veiculo VARCHAR(50) NOT NULL,
        especie_veiculo VARCHAR(50) NOT NULL,
        eixos VARCHAR(20) NOT NULL,
        qtd_veiculos BIGINT NOT NULL,

        CONSTRAINT fk_fato_frota_uf
            FOREIGN KEY (id_uf)
            REFERENCES analytics.dim_uf (id_uf),

        CONSTRAINT ck_fato_frota_qtd
            CHECK (qtd_veiculos > 0),

        CONSTRAINT pk_fato_frota
            PRIMARY KEY (
                data_referencia,
                id_uf,
                municipio,
                tipo_veiculo,
                especie_veiculo,
                eixos
            )
    );
""")

with engine.begin() as conexao:
    conexao.execute(consulta)

print("Tabela analytics.fato_frota criada.")

In [ ]:
df_frota_carga = df_frota_historica.rename(columns={
    "Data Referência": "data_referencia",
    "ID UF": "id_uf",
    "Município": "municipio",
    "Tipo Veículo": "tipo_veiculo",
    "Espécie Veículo": "especie_veiculo",
    "Eixos": "eixos",
    "Qtd. Veículos": "qtd_veiculos"
}).copy()

df_frota_carga["data_referencia"] = pd.to_datetime(
    df_frota_carga["data_referencia"]
).dt.date

df_frota_carga["id_uf"] = df_frota_carga["id_uf"].astype("int64")
df_frota_carga["qtd_veiculos"] = df_frota_carga["qtd_veiculos"].astype("int64")

display(df_frota_carga.head())
print("Total de registros:", len(df_frota_carga))
print("Total de veículos:", df_frota_carga["qtd_veiculos"].sum())

In [ ]:
print("Colunas:")
print(df_frota_carga.columns.tolist())

print("\nTipos de dados:")
display(df_frota_carga.dtypes.to_frame("tipo"))

print("\nValores distintos de eixos:")
print(sorted(df_frota_carga["eixos"].astype(str).unique()))

print("\nValores nulos:")
display(df_frota_carga.isna().sum().to_frame("nulos"))

In [ ]:
print("Menor valor:", df_frota_carga["eixos"].min())
print("Maior valor:", df_frota_carga["eixos"].max())

valores_fracionarios = (
    df_frota_carga["eixos"] % 1 != 0
).sum()

print("Valores fracionários:", valores_fracionarios)

print("\nFrequência dos valores negativos:")
display(
    df_frota_carga.loc[
        df_frota_carga["eixos"] < 0, "eixos"
    ].value_counts().sort_index()
)

In [ ]:
df_frota_carga = df_frota_carga.drop(columns=["UF"]).copy()

df_frota_carga["eixos"] = (
    df_frota_carga["eixos"].astype("int64").astype(str)
)

colunas_fato = [
    "data_referencia",
    "id_uf",
    "municipio",
    "tipo_veiculo",
    "especie_veiculo",
    "eixos",
    "qtd_veiculos"
]

df_frota_carga = df_frota_carga[colunas_fato]

print("Colunas:", df_frota_carga.columns.tolist())
print("Total de registros:", len(df_frota_carga))
print("Total de veículos:", df_frota_carga["qtd_veiculos"].sum())

display(df_frota_carga.head())

In [ ]:
df_frota_carga.to_sql(
    name="fato_frota",
    con=engine,
    schema="analytics",
    if_exists="append",
    index=False,
    chunksize=5000
)

print("Carga concluída.")
print("Registros enviados:", len(df_frota_carga))

In [ ]:
consulta = text("""
    SELECT
        data_referencia,
        COUNT(*) AS total_registros,
        SUM(qtd_veiculos) AS total_veiculos,
        COUNT(DISTINCT id_uf) AS ufs_distintas
    FROM analytics.fato_frota
    GROUP BY data_referencia
    ORDER BY data_referencia;
""")

with engine.connect() as conexao:
    resultado = conexao.execute(consulta).fetchall()

for linha in resultado:
    print(
        f"Referência: {linha[0]} | "
        f"Registros: {linha[1]:,} | "
        f"Veículos: {linha[2]:,} | "
        f"UFs: {linha[3]}"
    )

In [ ]:
consulta = text("""
    SELECT
        COUNT(*) AS total_registros,
        COUNT(d.id_uf) AS registros_com_uf,
        COUNT(*) - COUNT(d.id_uf) AS registros_sem_uf
    FROM analytics.fato_frota f
    LEFT JOIN analytics.dim_uf d
        ON f.id_uf = d.id_uf;
""")

with engine.connect() as conexao:
    resultado = conexao.execute(consulta).fetchone()

print("Total de registros:", resultado[0])
print("Registros com UF correspondente:", resultado[1])
print("Registros sem UF correspondente:", resultado[2])

In [ ]:
consulta = text("""
    SELECT
        COUNT(*) AS total_registros,
        COUNT(DISTINCT (
            data_referencia,
            id_uf,
            municipio,
            tipo_veiculo,
            especie_veiculo,
            eixos
        )) AS combinacoes_distintas
    FROM analytics.fato_frota;
""")

with engine.connect() as conexao:
    resultado = conexao.execute(consulta).fetchone()

print("Total de registros:", resultado[0])
print("Combinações distintas:", resultado[1])
print("Duplicidades:", resultado[0] - resultado[1])

In [ ]:
consulta = text("""
    WITH frota_por_uf AS (
        SELECT
            f.data_referencia,
            d.sigla_uf,
            d.nome_uf,
            d.regiao,
            SUM(f.qtd_veiculos) AS total_veiculos
        FROM analytics.fato_frota f
        INNER JOIN analytics.dim_uf d
            ON f.id_uf = d.id_uf
        GROUP BY
            f.data_referencia,
            d.sigla_uf,
            d.nome_uf,
            d.regiao
    )

    SELECT
        data_referencia,
        sigla_uf,
        nome_uf,
        regiao,
        total_veiculos,
        ROUND(
            100.0 * total_veiculos
            / SUM(total_veiculos) OVER (
                PARTITION BY data_referencia
            ),
            2
        ) AS participacao_percentual
    FROM frota_por_uf
    ORDER BY
        data_referencia,
        total_veiculos DESC;
""")

with engine.connect() as conexao:
    df_frota_por_uf = pd.read_sql(consulta, conexao)

display(df_frota_por_uf)

In [ ]:
consulta = text("""
    SELECT
        f.data_referencia,
        d.regiao,
        SUM(f.qtd_veiculos) AS total_veiculos,
        ROUND(
            100.0 * SUM(f.qtd_veiculos)
            / SUM(SUM(f.qtd_veiculos)) OVER (
                PARTITION BY f.data_referencia
            ),
            2
        ) AS participacao_percentual
    FROM analytics.fato_frota f
    INNER JOIN analytics.dim_uf d
        ON f.id_uf = d.id_uf
    WHERE d.regiao <> 'Não identificada'
    GROUP BY
        f.data_referencia,
        d.regiao
    ORDER BY
        f.data_referencia,
        total_veiculos DESC;
""")

with engine.connect() as conexao:
    df_frota_por_regiao = pd.read_sql(consulta, conexao)

display(df_frota_por_regiao)

In [ ]:
consulta = text("""
    WITH frota_mensal AS (
        SELECT
            f.data_referencia,
            d.sigla_uf,
            d.nome_uf,
            d.regiao,
            SUM(f.qtd_veiculos) AS total_veiculos
        FROM analytics.fato_frota f
        INNER JOIN analytics.dim_uf d
            ON f.id_uf = d.id_uf
        GROUP BY
            f.data_referencia,
            d.sigla_uf,
            d.nome_uf,
            d.regiao
    ),

    comparacao AS (
        SELECT
            sigla_uf,
            nome_uf,
            regiao,
            MAX(total_veiculos) FILTER (
                WHERE data_referencia = DATE '2026-08-01'
            ) AS frota_agosto,
            MAX(total_veiculos) FILTER (
                WHERE data_referencia = DATE '2026-09-01'
            ) AS frota_setembro
        FROM frota_mensal
        GROUP BY
            sigla_uf,
            nome_uf,
            regiao
    )

    SELECT
        sigla_uf,
        nome_uf,
        regiao,
        frota_agosto,
        frota_setembro,
        frota_setembro - frota_agosto AS variacao_absoluta,
        ROUND(
            100.0 * (frota_setembro - frota_agosto)
            / NULLIF(frota_agosto, 0),
            2
        ) AS variacao_percentual
    FROM comparacao
    WHERE frota_agosto IS NOT NULL
      AND frota_setembro IS NOT NULL
    ORDER BY variacao_absoluta DESC;
""")

with engine.connect() as conexao:
    df_variacao_uf = pd.read_sql(consulta, conexao)

display(df_variacao_uf)

In [ ]:
consulta = text("""
    WITH frota_categoria AS (
        SELECT
            data_referencia,
            tipo_veiculo,
            especie_veiculo,
            SUM(qtd_veiculos) AS total_veiculos
        FROM analytics.fato_frota
        GROUP BY
            data_referencia,
            tipo_veiculo,
            especie_veiculo
    )

    SELECT
        data_referencia,
        tipo_veiculo,
        especie_veiculo,
        total_veiculos,
        ROUND(
            100.0 * total_veiculos
            / SUM(total_veiculos) OVER (
                PARTITION BY data_referencia
            ),
            2
        ) AS participacao_percentual
    FROM frota_categoria
    ORDER BY
        data_referencia,
        total_veiculos DESC;
""")

with engine.connect() as conexao:
    df_frota_categoria = pd.read_sql(consulta, conexao)

display(df_frota_categoria)

In [ ]:
consulta = text("""
    WITH frota_tipo AS (
        SELECT
            data_referencia,
            tipo_veiculo,
            SUM(qtd_veiculos) AS total_veiculos
        FROM analytics.fato_frota
        GROUP BY
            data_referencia,
            tipo_veiculo
    ),

    comparacao AS (
        SELECT
            tipo_veiculo,
            MAX(total_veiculos) FILTER (
                WHERE data_referencia = DATE '2026-08-01'
            ) AS frota_agosto,
            MAX(total_veiculos) FILTER (
                WHERE data_referencia = DATE '2026-09-01'
            ) AS frota_setembro
        FROM frota_tipo
        GROUP BY tipo_veiculo
    )

    SELECT
        tipo_veiculo,
        frota_agosto,
        frota_setembro,
        frota_setembro - frota_agosto AS variacao_absoluta,
        ROUND(
            100.0 * (frota_setembro - frota_agosto)
            / NULLIF(frota_agosto, 0),
            2
        ) AS variacao_percentual
    FROM comparacao
    ORDER BY frota_setembro DESC;
""")

with engine.connect() as conexao:
    df_variacao_tipo = pd.read_sql(consulta, conexao)

display(df_variacao_tipo)

In [ ]:
consulta = text("""
    SELECT
        data_referencia,
        tipo_veiculo,
        SUM(qtd_veiculos) AS total_veiculos,
        ROUND(
            100.0 * SUM(qtd_veiculos)
            / SUM(SUM(qtd_veiculos)) OVER (
                PARTITION BY data_referencia
            ),
            2
        ) AS participacao_percentual
    FROM analytics.fato_frota
    GROUP BY
        data_referencia,
        tipo_veiculo
    ORDER BY
        data_referencia,
        total_veiculos DESC;
""")

with engine.connect() as conexao:
    df_participacao_tipo = pd.read_sql(consulta, conexao)

display(df_participacao_tipo)

In [ ]:
from pathlib import Path
import pandas as pd

pasta_senatran = PROJECT_ROOT / "data" / "raw" / "senatran"

arquivos = sorted(
    pasta_senatran.rglob("G_Frota_por_UF_Municipio_TIPO_ESPECIE_EIXOS_*.xlsx")
)

df_arquivos = pd.DataFrame([
    {
        "mes_referencia": arquivo.parent.name,
        "nome_arquivo": arquivo.name,
        "tamanho_mb": round(arquivo.stat().st_size / (1024 ** 2), 2),
        "caminho": str(arquivo.relative_to(PROJECT_ROOT))
    }
    for arquivo in arquivos
])

display(df_arquivos)

print(f"Total de arquivos encontrados: {len(df_arquivos)}")

In [ ]:
from pathlib import Path
import pandas as pd

periodo = pd.period_range(
    start="2025-01",
    end="2026-09",
    freq="M"
)

meses_disponiveis = {"2026-08", "2026-09"}

meses_faltantes = [
    str(mes)
    for mes in periodo
    if str(mes) not in meses_disponiveis
]

for mes in meses_faltantes:
    (pasta_senatran / mes).mkdir(
        parents=True,
        exist_ok=True
    )

df_meses_faltantes = pd.DataFrame({
    "mes_referencia": meses_faltantes,
    "status": "Aguardando download"
})

display(df_meses_faltantes)

print(f"Meses a obter: {len(meses_faltantes)}")
print(f"Meses já disponíveis: {len(meses_disponiveis)}")

In [ ]:
from html.parser import HTMLParser
from urllib.request import Request, urlopen
from urllib.parse import urljoin
import re
import pandas as pd

class ExtratorLinks(HTMLParser):
    def __init__(self):
        super().__init__()
        self.links = []
        self.href_atual = None

    def handle_starttag(self, tag, attrs):
        if tag == "a":
            atributos = dict(attrs)
            self.href_atual = atributos.get("href")

    def handle_data(self, data):
        if self.href_atual:
            self.links.append((data.strip(), self.href_atual))

    def handle_endtag(self, tag):
        if tag == "a":
            self.href_atual = None


paginas = {
    "2025": (
        "https://www.gov.br/transportes/pt-br/assuntos/"
        "transito/conteudo-Senatran/frota-de-veiculos-2025"
    ),
    "2026": (
        "https://www.gov.br/transportes/pt-br/assuntos/"
        "transito/conteudo-Senatran/frota-de-veiculos-2026"
    )
}

links_encontrados = []

for ano, url_pagina in paginas.items():
    requisicao = Request(
        url_pagina,
        headers={"User-Agent": "Mozilla/5.0"}
    )

    with urlopen(requisicao, timeout=60) as resposta:
        html = resposta.read().decode("utf-8", errors="replace")

    extrator = ExtratorLinks()
    extrator.feed(html)

    for texto, href in extrator.links:
        if "g_frota_por_uf_municipio_tipo_especie_eixos" in href.lower():
            url_arquivo = urljoin(url_pagina, href)

            correspondencia = re.search(
                r"(janeiro|fevereiro|marco|março|abril|maio|junho|"
                r"julho|agosto|setembro|outubro|novembro|dezembro)"
                r"_(2025|2026)\.xlsx",
                href,
                flags=re.IGNORECASE
            )

            if correspondencia:
                mes = correspondencia.group(1).lower()
                ano_arquivo = correspondencia.group(2)

                meses = {
                    "janeiro": "01",
                    "fevereiro": "02",
                    "marco": "03",
                    "março": "03",
                    "abril": "04",
                    "maio": "05",
                    "junho": "06",
                    "julho": "07",
                    "agosto": "08",
                    "setembro": "09",
                    "outubro": "10",
                    "novembro": "11",
                    "dezembro": "12"
                }

                referencia = f"{ano_arquivo}-{meses[mes]}"

                links_encontrados.append({
                    "mes_referencia": referencia,
                    "url_download": url_arquivo
                })

df_links_senatran = (
    pd.DataFrame(links_encontrados)
    .drop_duplicates(subset="mes_referencia")
    .sort_values("mes_referencia")
    .reset_index(drop=True)
)

display(df_links_senatran)

print(f"Links encontrados: {len(df_links_senatran)}")

In [ ]:
for ano, url_pagina in paginas.items():
    requisicao = Request(
        url_pagina,
        headers={"User-Agent": "Mozilla/5.0"}
    )

    with urlopen(requisicao, timeout=60) as resposta:
        html = resposta.read().decode("utf-8", errors="replace")

    extrator = ExtratorLinks()
    extrator.feed(html)

    print(f"\n{'=' * 70}")
    print(f"ARQUIVOS ENCONTRADOS NA PÁGINA DE {ano}")
    print("=" * 70)

    for texto, href in extrator.links:
        if "frota" in href.lower() or "frota" in texto.lower():
            print(f"Texto: {texto}")
            print(f"Link: {urljoin(url_pagina, href)}")
            print("-" * 70)

In [ ]:
import re
from urllib.parse import urljoin
from urllib.request import Request, urlopen
import pandas as pd

meses = {
    "janeiro": "01",
    "fevereiro": "02",
    "marco": "03",
    "março": "03",
    "maro": "03",
    "abril": "04",
    "maio": "05",
    "junho": "06",
    "julho": "07",
    "agosto": "08",
    "setembro": "09",
    "outubro": "10",
    "novembro": "11",
    "dezembro": "12"
}

links_corrigidos = []

for ano, url_pagina in paginas.items():
    requisicao = Request(
        url_pagina,
        headers={"User-Agent": "Mozilla/5.0"}
    )

    with urlopen(requisicao, timeout=60) as resposta:
        html = resposta.read().decode("utf-8", errors="replace")

    extrator = ExtratorLinks()
    extrator.feed(html)

    for texto, href in extrator.links:
        nome = href.lower()

        if "g_frota_por_uf_municipio_tipo_especie_eixos" not in nome:
            continue

        correspondencia = re.search(
            r"(janeiro|fevereiro|marco|março|maro|abril|maio|"
            r"junho|julho|agosto|setembro|outubro|novembro|dezembro)"
            r"[_-](2025|2026)(?:-\d+)?\.xlsx",
            nome,
            flags=re.IGNORECASE
        )

        if correspondencia:
            mes = correspondencia.group(1)
            ano_arquivo = correspondencia.group(2)

            referencia = (
                f"{ano_arquivo}-{meses[mes]}"
            )

            links_corrigidos.append({
                "mes_referencia": referencia,
                "url_download": urljoin(url_pagina, href)
            })

df_links_senatran = (
    pd.DataFrame(links_corrigidos)
    .drop_duplicates(subset="mes_referencia")
    .sort_values("mes_referencia")
    .reset_index(drop=True)
)

display(df_links_senatran)

print(f"Total de competências encontradas: {len(df_links_senatran)}")

In [ ]:
from pathlib import Path
import pandas as pd

diretorio_senatran = PROJECT_ROOT / "data" / "raw" / "senatran"

arquivos_existentes = []

for arquivo in diretorio_senatran.glob("*/*.xlsx"):
    arquivos_existentes.append({
        "arquivo": arquivo.name,
        "mes_referencia": arquivo.parent.name,
        "tamanho_mb": round(arquivo.stat().st_size / (1024 ** 2), 2)
    })

df_arquivos_existentes = (
    pd.DataFrame(arquivos_existentes)
    .sort_values("mes_referencia")
    .reset_index(drop=True)
)

display(df_arquivos_existentes)

print(f"Total de arquivos XLSX encontrados: {len(df_arquivos_existentes)}")

In [ ]:
from pathlib import Path
from io import BytesIO
from urllib.request import Request, urlopen
from zipfile import ZipFile
from openpyxl import load_workbook
import pandas as pd

# Meses esperados no histórico
meses_pt = {
    "01": "Janeiro",
    "02": "Fevereiro",
    "03": "Marco",
    "04": "Abril",
    "05": "Maio",
    "06": "Junho",
    "07": "Julho",
    "08": "Agosto",
    "09": "Setembro",
    "10": "Outubro",
    "11": "Novembro",
    "12": "Dezembro"
}

competencias_esperadas = pd.period_range(
    "2025-01",
    "2026-09",
    freq="M"
).astype(str).tolist()

competencias_encontradas = (
    df_links_senatran["mes_referencia"].tolist()
)

if competencias_encontradas != competencias_esperadas:
    raise ValueError(
        "A lista de links não corresponde às 21 competências esperadas."
    )

# Identifica os meses que ainda não possuem arquivo
competencias_existentes = {
    arquivo.parent.name
    for arquivo in diretorio_senatran.glob("*/*.xlsx")
}

df_pendentes = df_links_senatran[
    ~df_links_senatran["mes_referencia"].isin(competencias_existentes)
].copy()

print(f"Arquivos a baixar: {len(df_pendentes)}")

resultados_download = []

for _, registro in df_pendentes.iterrows():

    referencia = registro["mes_referencia"]
    url = registro["url_download"]

    ano, mes = referencia.split("-")
    nome_mes = meses_pt[mes]

    diretorio_destino = diretorio_senatran / referencia
    diretorio_destino.mkdir(parents=True, exist_ok=True)

    nome_arquivo = (
        f"G_Frota_por_UF_Municipio_TIPO_ESPECIE_EIXOS_"
        f"{nome_mes}_{ano}.xlsx"
    )

    caminho_destino = diretorio_destino / nome_arquivo

    try:
        requisicao = Request(
            url,
            headers={"User-Agent": "Mozilla/5.0"}
        )

        with urlopen(requisicao, timeout=120) as resposta:
            conteudo = resposta.read()
            status_http = resposta.status

        # Verifica se o arquivo é um ZIP válido
        with ZipFile(BytesIO(conteudo)) as arquivo_zip:
            arquivo_corrompido = arquivo_zip.testzip()

        if arquivo_corrompido is not None:
            raise ValueError(
                f"Arquivo ZIP corrompido: {arquivo_corrompido}"
            )

        # Verifica se é uma planilha XLSX legível
        planilha = load_workbook(
            BytesIO(conteudo),
            read_only=True,
            data_only=True
        )

        abas = planilha.sheetnames
        planilha.close()

        if "Layout G" not in abas:
            raise ValueError(
                f"Aba 'Layout G' não encontrada. Abas: {abas}"
            )

        # Salva somente depois das validações
        caminho_destino.write_bytes(conteudo)

        resultados_download.append({
            "mes_referencia": referencia,
            "status": "OK",
            "status_http": status_http,
            "tamanho_mb": round(len(conteudo) / (1024 ** 2), 2)
        })

        print(f"{referencia}: download e validação concluídos.")

    except Exception as erro:
        resultados_download.append({
            "mes_referencia": referencia,
            "status": "ERRO",
            "status_http": None,
            "tamanho_mb": None
        })

        print(f"{referencia}: ERRO — {erro}")

df_resultados_download = pd.DataFrame(resultados_download)

display(df_resultados_download)

print(
    "\nDownloads concluídos:",
    (df_resultados_download["status"] == "OK").sum()
)

print(
    "Downloads com erro:",
    (df_resultados_download["status"] == "ERRO").sum()
)

In [ ]:
from pathlib import Path
from zipfile import ZipFile, BadZipFile
import pandas as pd

meses_esperados = pd.period_range(
    start="2025-01",
    end="2026-09",
    freq="M"
).astype(str)

resultados = []

for mes in meses_esperados:
    pasta_mes = diretorio_senatran / mes
    arquivos = list(pasta_mes.glob("*.xlsx"))

    if len(arquivos) == 0:
        resultados.append({
            "mes_referencia": mes,
            "quantidade_arquivos": 0,
            "arquivo": None,
            "tamanho_mb": None,
            "integridade": "AUSENTE"
        })
        continue

    for arquivo in arquivos:
        try:
            with ZipFile(arquivo, "r") as zip_file:
                integridade = (
                    "OK" if zip_file.testzip() is None
                    else "ARQUIVO CORROMPIDO"
                )
        except BadZipFile:
            integridade = "ARQUIVO CORROMPIDO"

        resultados.append({
            "mes_referencia": mes,
            "quantidade_arquivos": len(arquivos),
            "arquivo": arquivo.name,
            "tamanho_mb": round(
                arquivo.stat().st_size / (1024 ** 2), 2
            ),
            "integridade": integridade
        })

df_validacao_historico = pd.DataFrame(resultados)

display(df_validacao_historico)

print("Competências esperadas:", len(meses_esperados))
print(
    "Competências com exatamente um arquivo:",
    (
        df_validacao_historico
        .groupby("mes_referencia")["quantidade_arquivos"]
        .first()
        .eq(1)
        .sum()
    )
)
print(
    "Arquivos íntegros:",
    (df_validacao_historico["integridade"] == "OK").sum()
)

In [ ]:
from pathlib import Path
from openpyxl import load_workbook
import pandas as pd

arquivos_senatran = sorted(
    diretorio_senatran.glob("*/*.xlsx")
)

inspecao_arquivos = []

for arquivo in arquivos_senatran:
    workbook = load_workbook(
        arquivo,
        read_only=True,
        data_only=True
    )

    planilha = workbook["Layout G"]

    inspecao_arquivos.append({
        "mes_referencia": arquivo.parent.name,
        "arquivo": arquivo.name,
        "planilha": planilha.title,
        "quantidade_linhas": planilha.max_row,
        "quantidade_colunas": planilha.max_column,
        "colunas": [
            celula.value
            for celula in next(planilha.iter_rows(
                min_row=1,
                max_row=1
            ))
        ]
    })

    workbook.close()

df_inspecao_historico = pd.DataFrame(inspecao_arquivos)

display(df_inspecao_historico)

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

diretorio_senatran = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "senatran"
)

print("Raiz do projeto:", PROJECT_ROOT)
print("Diretório SENATRAN:", diretorio_senatran)
print("Diretório existe:", diretorio_senatran.exists())

In [ ]:
from openpyxl import load_workbook
import pandas as pd

arquivos_senatran = sorted(
    diretorio_senatran.glob("*/*.xlsx")
)

inspecao_arquivos = []

for arquivo in arquivos_senatran:
    workbook = load_workbook(
        arquivo,
        read_only=True,
        data_only=True
    )

    planilha = workbook["Layout G"]

    inspecao_arquivos.append({
        "mes_referencia": arquivo.parent.name,
        "arquivo": arquivo.name,
        "planilha": planilha.title,
        "quantidade_linhas": planilha.max_row,
        "quantidade_colunas": planilha.max_column,
        "colunas": [
            celula.value
            for celula in next(planilha.iter_rows(
                min_row=1,
                max_row=1
            ))
        ]
    })

    workbook.close()

df_inspecao_historico = pd.DataFrame(inspecao_arquivos)

display(df_inspecao_historico)

In [ ]:
from openpyxl import load_workbook

cabecalhos_historicos = []

for arquivo in arquivos_senatran:
    workbook = load_workbook(
        arquivo,
        read_only=True,
        data_only=True
    )

    planilha = workbook["Layout G"]

    cabecalho = [
        celula.value
        for celula in next(
            planilha.iter_rows(
                min_row=1,
                max_row=1
            )
        )
    ]

    cabecalhos_historicos.append({
        "mes_referencia": arquivo.parent.name,
        "cabecalho": cabecalho
    })

    workbook.close()

cabecalho_referencia = cabecalhos_historicos[0]["cabecalho"]

for item in cabecalhos_historicos:
    print(
        item["mes_referencia"],
        "IGUAL" if item["cabecalho"] == cabecalho_referencia else "DIFERENTE",
        item["cabecalho"]
    )

In [ ]:
import pandas as pd

arquivo_teste = (
    diretorio_senatran
    / "2025-01"
    / next((diretorio_senatran / "2025-01").glob("*.xlsx")).name
)

df_teste = pd.read_excel(
    arquivo_teste,
    sheet_name="Layout G"
)

print("Dimensões:", df_teste.shape)

print("\nTipos de dados:")
print(df_teste.dtypes)

print("\nPrimeiras 5 linhas:")
display(df_teste.head())

print("\nÚltimas 5 linhas:")
display(df_teste.tail())

print("\nValores nulos por coluna:")
display(df_teste.isna().sum())

In [ ]:
colunas_chave = [
    "UF",
    "Município",
    "Tipo Veículo",
    "Espécie Veículo",
    "Eixos"
]

print("Quantidade total de registros:", len(df_teste))

print("\nQuantidade de UFs:")
print(df_teste["UF"].nunique())

print("\nValores distintos de Eixos:")
print(sorted(df_teste["Eixos"].unique()))

print("\nEstatísticas de quantidade de veículos:")
display(df_teste["Qtd. Veículos"].describe())

print("\nQuantidades inválidas:")
print(
    "Zero:",
    (df_teste["Qtd. Veículos"] == 0).sum()
)
print(
    "Negativas:",
    (df_teste["Qtd. Veículos"] < 0).sum()
)

print("\nRegistros duplicados pela chave candidata:")
print(df_teste.duplicated(subset=colunas_chave).sum())

print("\nCategorias de Tipo Veículo:")
display(
    df_teste["Tipo Veículo"]
    .value_counts()
    .to_frame("quantidade_registros")
)

print("\nCategorias de Espécie Veículo:")
display(
    df_teste["Espécie Veículo"]
    .value_counts()
    .to_frame("quantidade_registros")
)

In [ ]:
print("Valores distintos de UF:")
display(
    df_teste["UF"]
    .value_counts(dropna=False)
    .rename_axis("UF")
    .reset_index(name="quantidade_registros")
    .sort_values("UF")
    .reset_index(drop=True)
)

In [ ]:
resumo_uf_jan2025 = (
    df_teste
    .groupby("UF", as_index=False)
    .agg(
        quantidade_registros=("Qtd. Veículos", "size"),
        total_veiculos=("Qtd. Veículos", "sum")
    )
    .sort_values("total_veiculos", ascending=False)
    .reset_index(drop=True)
)

display(resumo_uf_jan2025)

print(
    "Total geral de veículos:",
    f"{resumo_uf_jan2025['total_veiculos'].sum():,}"
)

print(
    "\nTotal de veículos nas classificações especiais:",
    f"{resumo_uf_jan2025.loc[
        resumo_uf_jan2025['UF'].isin([
            'Sem Informação',
            'Não Identificado',
            'Não se Aplica'
        ]),
        'total_veiculos'
    ].sum():,}"
)

In [ ]:
df_uf_especial = df_teste[
    df_teste["UF"].isin([
        "Sem Informação",
        "Não Identificado",
        "Não se Aplica"
    ])
].copy()

resumo_uf_especial = (
    df_uf_especial
    .groupby(
        ["UF", "Tipo Veículo", "Espécie Veículo"],
        as_index=False
    )
    .agg(
        quantidade_registros=("Qtd. Veículos", "size"),
        total_veiculos=("Qtd. Veículos", "sum")
    )
    .sort_values(
        ["UF", "total_veiculos"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

display(resumo_uf_especial)

print(
    "Total de registros especiais:",
    len(df_uf_especial)
)

print(
    "Total de veículos especiais:",
    f"{df_uf_especial['Qtd. Veículos'].sum():,}"
)

In [ ]:
tipos_escopo = [
    "AUTOMOVEL",
    "CAMINHONETE",
    "CAMIONETA",
    "UTILITARIO"
]

df_escopo_jan2025 = df_teste[
    df_teste["Tipo Veículo"].isin(tipos_escopo)
].copy()

resumo_escopo_uf = (
    df_escopo_jan2025
    .assign(
        classificacao_uf=lambda df: df["UF"].where(
            ~df["UF"].isin([
                "Sem Informação",
                "Não Identificado",
                "Não se Aplica"
            ]),
            "Classificação especial"
        )
    )
    .groupby("classificacao_uf", as_index=False)
    .agg(
        quantidade_registros=("Qtd. Veículos", "size"),
        total_veiculos=("Qtd. Veículos", "sum")
    )
)

resumo_escopo_uf["participacao_pct"] = (
    resumo_escopo_uf["total_veiculos"]
    / resumo_escopo_uf["total_veiculos"].sum()
    * 100
)

display(
    resumo_escopo_uf.sort_values(
        "total_veiculos",
        ascending=False
    )
)

print(
    "Total de veículos no escopo:",
    f"{df_escopo_jan2025['Qtd. Veículos'].sum():,}"
)

In [ ]:
df_escopo_especial = df_escopo_jan2025[
    df_escopo_jan2025["UF"].isin([
        "Sem Informação",
        "Não Identificado",
        "Não se Aplica"
    ])
].copy()

resumo_escopo_especial = (
    df_escopo_especial
    .groupby(
        ["UF", "Tipo Veículo", "Espécie Veículo"],
        as_index=False
    )
    .agg(
        quantidade_registros=("Qtd. Veículos", "size"),
        total_veiculos=("Qtd. Veículos", "sum")
    )
    .sort_values(
        "total_veiculos",
        ascending=False
    )
    .reset_index(drop=True)
)

display(resumo_escopo_especial)

print("Registros:", len(df_escopo_especial))
print(
    "Total de veículos:",
    f"{df_escopo_especial['Qtd. Veículos'].sum():,}"
)

In [ ]:
from collections import Counter

resumo_classificacoes_uf = []

for arquivo in arquivos_senatran:
    mes = arquivo.parent.name

    df_mes = pd.read_excel(
        arquivo,
        sheet_name="Layout G",
        usecols=["UF", "Tipo Veículo", "Qtd. Veículos"]
    )

    df_mes_escopo = df_mes[
        df_mes["Tipo Veículo"].isin(tipos_escopo)
    ]

    resumo = (
        df_mes_escopo
        .groupby("UF", as_index=False)
        .agg(
            quantidade_registros=("Qtd. Veículos", "size"),
            total_veiculos=("Qtd. Veículos", "sum")
        )
    )

    especiais = resumo[
        ~resumo["UF"].isin([
            "ACRE", "ALAGOAS", "AMAPA", "AMAZONAS",
            "BAHIA", "CEARA", "DISTRITO FEDERAL",
            "ESPIRITO SANTO", "GOIAS", "MARANHAO",
            "MATO GROSSO", "MATO GROSSO DO SUL",
            "MINAS GERAIS", "PARA", "PARAIBA", "PARANA",
            "PERNAMBUCO", "PIAUI", "RIO DE JANEIRO",
            "RIO GRANDE DO NORTE", "RIO GRANDE DO SUL",
            "RONDONIA", "RORAIMA", "SANTA CATARINA",
            "SAO PAULO", "SERGIPE", "TOCANTINS"
        ]) 
    ].copy()

    for _, linha in especiais.iterrows():
        resumo_classificacoes_uf.append({
            "mes": mes,
            "UF": linha["UF"],
            "quantidade_registros": linha["quantidade_registros"],
            "total_veiculos": linha["total_veiculos"]
        })

df_classificacoes_uf = pd.DataFrame(resumo_classificacoes_uf)

display(
    df_classificacoes_uf.sort_values(["mes", "UF"])
)

print("Classificações especiais encontradas:")
print(sorted(df_classificacoes_uf["UF"].unique()))

In [ ]:
from openpyxl import load_workbook
from collections import defaultdict

resumo_classificacoes_uf = []

for i, arquivo in enumerate(arquivos_senatran, start=1):
    mes = arquivo.parent.name

    wb = load_workbook(
        arquivo,
        read_only=True,
        data_only=True
    )

    ws = wb["Layout G"]

    acumulado = defaultdict(lambda: [0, 0])

    for uf, tipo, qtd in ws.iter_rows(
        min_row=2,
        min_col=1,
        max_col=6,
        values_only=True
    ):
        if tipo not in tipos_escopo:
            continue

        if uf not in {
            "ACRE", "ALAGOAS", "AMAPA", "AMAZONAS",
            "BAHIA", "CEARA", "DISTRITO FEDERAL",
            "ESPIRITO SANTO", "GOIAS", "MARANHAO",
            "MATO GROSSO", "MATO GROSSO DO SUL",
            "MINAS GERAIS", "PARA", "PARAIBA", "PARANA",
            "PERNAMBUCO", "PIAUI", "RIO DE JANEIRO",
            "RIO GRANDE DO NORTE", "RIO GRANDE DO SUL",
            "RONDONIA", "RORAIMA", "SANTA CATARINA",
            "SAO PAULO", "SERGIPE", "TOCANTINS"
        }:
            acumulado[uf][0] += 1
            acumulado[uf][1] += qtd or 0

    for uf, valores in acumulado.items():
        resumo_classificacoes_uf.append({
            "mes": mes,
            "UF": uf,
            "quantidade_registros": valores[0],
            "total_veiculos": valores[1]
        })

    wb.close()

    print(f"{i}/{len(arquivos_senatran)} — {mes} concluído")

df_classificacoes_uf = pd.DataFrame(resumo_classificacoes_uf)

display(
    df_classificacoes_uf.sort_values(["mes", "UF"])
)

print(
    "\nClassificações especiais encontradas:",
    sorted(df_classificacoes_uf["UF"].unique())
)

In [ ]:
df_participacao_si = df_classificacoes_uf[
    ["mes", "total_veiculos"]
].copy()

df_participacao_si = df_participacao_si.rename(
    columns={"total_veiculos": "veiculos_sem_informacao"}
)

df_participacao_si["mes"] = pd.PeriodIndex(
    df_participacao_si["mes"],
    freq="M"
)

df_participacao_si = df_participacao_si.sort_values("mes")

display(df_participacao_si)

In [ ]:
from openpyxl import load_workbook

totais_mensais = []

for i, arquivo in enumerate(arquivos_senatran, start=1):
    mes = arquivo.parent.name

    wb = load_workbook(
        arquivo,
        read_only=True,
        data_only=True
    )
    ws = wb["Layout G"]

    total_escopo = 0

    for uf, municipio, tipo, especie, eixos, qtd in ws.iter_rows(
        min_row=2,
        min_col=1,
        max_col=6,
        values_only=True
    ):
        if tipo in tipos_escopo:
            total_escopo += qtd or 0

    wb.close()

    totais_mensais.append({
        "mes": mes,
        "total_frota_escopo": total_escopo
    })

    print(f"{i}/{len(arquivos_senatran)} — {mes} concluído")

df_totais_mensais = pd.DataFrame(totais_mensais)

df_participacao_si["mes"] = (
    df_participacao_si["mes"].astype(str)
)

df_participacao_si = df_participacao_si.merge(
    df_totais_mensais,
    on="mes",
    how="left",
    validate="one_to_one"
)

df_participacao_si["participacao_si_pct"] = (
    df_participacao_si["veiculos_sem_informacao"]
    / df_participacao_si["total_frota_escopo"]
    * 100
)

display(
    df_participacao_si[
        [
            "mes",
            "total_frota_escopo",
            "veiculos_sem_informacao",
            "participacao_si_pct"
        ]
    ].round(3)
)

In [ ]:
print("Competências:", len(df_totais_mensais))
print("Meses distintos:", df_totais_mensais["mes"].nunique())
print("Valores nulos:")
display(df_totais_mensais.isna().sum())

print("\nTotais mensais:")
display(
    df_totais_mensais.sort_values("mes").reset_index(drop=True)
)

In [ ]:
from openpyxl import load_workbook

classificacoes_por_mes = []

for i, arquivo in enumerate(arquivos_senatran, start=1):
    mes = arquivo.parent.name

    wb = load_workbook(
        arquivo,
        read_only=True,
        data_only=True
    )
    ws = wb["Layout G"]

    ufs = set()

    for linha in ws.iter_rows(
        min_row=2,
        min_col=1,
        max_col=3,
        values_only=True
    ):
        uf, municipio, tipo = linha

        if tipo in tipos_escopo:
            ufs.add(uf)

    wb.close()

    classificacoes_por_mes.append({
        "mes": mes,
        "quantidade_classificacoes": len(ufs),
        "classificacoes": sorted(ufs)
    })

    print(f"{i}/{len(arquivos_senatran)} — {mes} concluído")

df_ufs_por_mes = pd.DataFrame(classificacoes_por_mes)

display(
    df_ufs_por_mes[
        ["mes", "quantidade_classificacoes"]
    ]
)

print("\nClassificações encontradas em toda a série:")
print(
    sorted(
        set().union(*df_ufs_por_mes["classificacoes"])
    )
)

In [ ]:
referencia_ufs = set(df_ufs_por_mes.iloc[0]["classificacoes"])

df_ufs_por_mes["faltantes"] = df_ufs_por_mes["classificacoes"].apply(
    lambda ufs: sorted(referencia_ufs - set(ufs))
)

df_ufs_por_mes["adicionais"] = df_ufs_por_mes["classificacoes"].apply(
    lambda ufs: sorted(set(ufs) - referencia_ufs)
)

display(
    df_ufs_por_mes[
        ["mes", "quantidade_classificacoes", "faltantes", "adicionais"]
    ]
)

print(
    "\nTodas as competências possuem a mesma cobertura?",
    df_ufs_por_mes["faltantes"].apply(len).eq(0).all()
    and df_ufs_por_mes["adicionais"].apply(len).eq(0).all()
)

In [ ]:
from openpyxl import load_workbook

validacao_chaves = []

for i, arquivo in enumerate(arquivos_senatran, start=1):
    mes = arquivo.parent.name

    wb = load_workbook(
        arquivo,
        read_only=True,
        data_only=True
    )
    ws = wb["Layout G"]

    chaves = set()
    duplicidades = 0
    registros_escopo = 0

    for uf, municipio, tipo, especie, eixos, qtd in ws.iter_rows(
        min_row=2,
        min_col=1,
        max_col=6,
        values_only=True
    ):
        if tipo not in tipos_escopo:
            continue

        registros_escopo += 1

        chave = (uf, municipio, tipo, especie, str(eixos))

        if chave in chaves:
            duplicidades += 1
        else:
            chaves.add(chave)

    wb.close()

    validacao_chaves.append({
        "mes": mes,
        "registros_escopo": registros_escopo,
        "chaves_distintas": len(chaves),
        "duplicidades": duplicidades
    })

    print(f"{i}/{len(arquivos_senatran)} — {mes} concluído")

df_validacao_chaves = pd.DataFrame(validacao_chaves)

display(df_validacao_chaves)

print(
    "\nTotal de duplicidades:",
    df_validacao_chaves["duplicidades"].sum()
)

In [ ]:
from openpyxl import load_workbook

validacao_quantidades = []

for i, arquivo in enumerate(arquivos_senatran, start=1):
    mes = arquivo.parent.name

    wb = load_workbook(
        arquivo,
        read_only=True,
        data_only=True
    )
    ws = wb["Layout G"]

    registros = 0
    nulos = 0
    zeros = 0
    negativos = 0
    nao_inteiros = 0
    total_veiculos = 0

    for uf, municipio, tipo, especie, eixos, qtd in ws.iter_rows(
        min_row=2,
        min_col=1,
        max_col=6,
        values_only=True
    ):
        if tipo not in tipos_escopo:
            continue

        registros += 1

        if qtd is None:
            nulos += 1
            continue

        total_veiculos += qtd

        if qtd == 0:
            zeros += 1
        elif qtd < 0:
            negativos += 1

        if not isinstance(qtd, (int, float)) or qtd != int(qtd):
            nao_inteiros += 1

    wb.close()

    validacao_quantidades.append({
        "mes": mes,
        "registros": registros,
        "total_veiculos": total_veiculos,
        "nulos": nulos,
        "zeros": zeros,
        "negativos": negativos,
        "nao_inteiros": nao_inteiros
    })

    print(f"{i}/{len(arquivos_senatran)} — {mes} concluído")

df_validacao_quantidades = pd.DataFrame(validacao_quantidades)

display(df_validacao_quantidades)

print("\nOcorrências totais:")
display(
    df_validacao_quantidades[
        ["nulos", "zeros", "negativos", "nao_inteiros"]
    ].sum()
)

In [ ]:
from openpyxl import load_workbook

colunas_chave = [
    "UF",
    "Município",
    "Tipo Veículo",
    "Espécie Veículo",
    "Eixos"
]

validacao_nulos_chave = []

for i, arquivo in enumerate(arquivos_senatran, start=1):
    mes = arquivo.parent.name

    wb = load_workbook(
        arquivo,
        read_only=True,
        data_only=True
    )
    ws = wb["Layout G"]

    nulos = {coluna: 0 for coluna in colunas_chave}
    registros = 0

    for linha in ws.iter_rows(
        min_row=2,
        min_col=1,
        max_col=6,
        values_only=True
    ):
        uf, municipio, tipo, especie, eixos, qtd = linha

        if tipo not in tipos_escopo:
            continue

        registros += 1

        valores = [uf, municipio, tipo, especie, eixos]

        for coluna, valor in zip(colunas_chave, valores):
            if valor is None:
                nulos[coluna] += 1

    wb.close()

    validacao_nulos_chave.append({
        "mes": mes,
        "registros": registros,
        **nulos
    })

    print(f"{i}/{len(arquivos_senatran)} — {mes} concluído")

df_validacao_nulos_chave = pd.DataFrame(
    validacao_nulos_chave
)

display(df_validacao_nulos_chave)

print("\nTotal de valores ausentes por coluna:")
display(
    df_validacao_nulos_chave[colunas_chave].sum()
)

In [ ]:
from openpyxl import load_workbook

combinacoes_tipo_especie = set()

for i, arquivo in enumerate(arquivos_senatran, start=1):
    mes = arquivo.parent.name

    wb = load_workbook(
        arquivo,
        read_only=True,
        data_only=True
    )
    ws = wb["Layout G"]

    for linha in ws.iter_rows(
        min_row=2,
        min_col=1,
        max_col=6,
        values_only=True
    ):
        uf, municipio, tipo, especie, eixos, qtd = linha

        if tipo in tipos_escopo:
            combinacoes_tipo_especie.add((tipo, especie))

    wb.close()

    print(f"{i}/{len(arquivos_senatran)} — {mes} concluído")

df_combinacoes_tipo_especie = pd.DataFrame(
    sorted(combinacoes_tipo_especie),
    columns=["Tipo Veículo", "Espécie Veículo"]
)

display(df_combinacoes_tipo_especie)

print("\nQuantidade de combinações:", len(df_combinacoes_tipo_especie))

In [ ]:
from openpyxl import load_workbook

combinacoes_por_mes = []

for i, arquivo in enumerate(arquivos_senatran, start=1):
    mes = arquivo.parent.name
    combinacoes = set()

    wb = load_workbook(
        arquivo,
        read_only=True,
        data_only=True
    )
    ws = wb["Layout G"]

    for uf, municipio, tipo, especie, eixos, qtd in ws.iter_rows(
        min_row=2,
        min_col=1,
        max_col=6,
        values_only=True
    ):
        if tipo in tipos_escopo:
            combinacoes.add((tipo, especie))

    wb.close()

    combinacoes_por_mes.append({
        "mes": mes,
        "combinacoes": combinacoes
    })

    print(f"{i}/{len(arquivos_senatran)} — {mes} concluído")

referencia_combinacoes = combinacoes_por_mes[0]["combinacoes"]

resultado_combinacoes = []

for item in combinacoes_por_mes:
    resultado_combinacoes.append({
        "mes": item["mes"],
        "quantidade_combinacoes": len(item["combinacoes"]),
        "faltantes": sorted(
            referencia_combinacoes - item["combinacoes"]
        ),
        "adicionais": sorted(
            item["combinacoes"] - referencia_combinacoes
        )
    })

df_validacao_combinacoes = pd.DataFrame(resultado_combinacoes)

display(df_validacao_combinacoes)

In [ ]:
from openpyxl import load_workbook

valores_eixos_por_mes = []

for i, arquivo in enumerate(arquivos_senatran, start=1):
    mes = arquivo.parent.name
    valores_eixos = set()

    wb = load_workbook(
        arquivo,
        read_only=True,
        data_only=True
    )
    ws = wb["Layout G"]

    for uf, municipio, tipo, especie, eixos, qtd in ws.iter_rows(
        min_row=2,
        min_col=1,
        max_col=6,
        values_only=True
    ):
        if tipo in tipos_escopo:
            valores_eixos.add(eixos)

    wb.close()

    valores_eixos_por_mes.append({
        "mes": mes,
        "valores_eixos": valores_eixos
    })

    print(f"{i}/{len(arquivos_senatran)} — {mes} concluído")

referencia_eixos = valores_eixos_por_mes[0]["valores_eixos"]

resultado_eixos = []

for item in valores_eixos_por_mes:
    resultado_eixos.append({
        "mes": item["mes"],
        "quantidade_valores": len(item["valores_eixos"]),
        "valores": sorted(item["valores_eixos"], key=lambda x: str(x)),
        "faltantes": sorted(
            referencia_eixos - item["valores_eixos"],
            key=lambda x: str(x)
        ),
        "adicionais": sorted(
            item["valores_eixos"] - referencia_eixos,
            key=lambda x: str(x)
        )
    })

df_validacao_eixos = pd.DataFrame(resultado_eixos)

display(df_validacao_eixos)

In [ ]:
from openpyxl import load_workbook

arquivo = next(
    arquivo for arquivo in arquivos_senatran
    if arquivo.parent.name == "2026-09"
)

distribuicao_eixos = {}

wb = load_workbook(
    arquivo,
    read_only=True,
    data_only=True
)
ws = wb["Layout G"]

for uf, municipio, tipo, especie, eixos, qtd in ws.iter_rows(
    min_row=2,
    min_col=1,
    max_col=6,
    values_only=True
):
    if tipo in tipos_escopo:
        if eixos not in distribuicao_eixos:
            distribuicao_eixos[eixos] = {
                "registros": 0,
                "veiculos": 0
            }

        distribuicao_eixos[eixos]["registros"] += 1
        distribuicao_eixos[eixos]["veiculos"] += qtd

wb.close()

df_distribuicao_eixos = (
    pd.DataFrame.from_dict(distribuicao_eixos, orient="index")
    .rename_axis("eixos")
    .reset_index()
    .sort_values("eixos", key=lambda coluna: coluna.astype(str))
    .reset_index(drop=True)
)

display(df_distribuicao_eixos)

In [ ]:
display(
    df_distribuicao_eixos
    .sort_values("veiculos", ascending=False)
    .reset_index(drop=True)
)

In [ ]:
from openpyxl import load_workbook

distribuicao_tipo_eixos = {}

wb = load_workbook(
    arquivo,
    read_only=True,
    data_only=True
)
ws = wb["Layout G"]

for uf, municipio, tipo, especie, eixos, qtd in ws.iter_rows(
    min_row=2,
    min_col=1,
    max_col=6,
    values_only=True
):
    if tipo in tipos_escopo:
        chave = (tipo, eixos)

        if chave not in distribuicao_tipo_eixos:
            distribuicao_tipo_eixos[chave] = {
                "registros": 0,
                "veiculos": 0
            }

        distribuicao_tipo_eixos[chave]["registros"] += 1
        distribuicao_tipo_eixos[chave]["veiculos"] += qtd

wb.close()

df_tipo_eixos = (
    pd.DataFrame([
        {
            "tipo_veiculo": tipo,
            "eixos": eixos,
            **valores
        }
        for (tipo, eixos), valores in distribuicao_tipo_eixos.items()
    ])
    .sort_values(
        ["tipo_veiculo", "veiculos"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

display(df_tipo_eixos)

In [ ]:
df_top_eixos_tipo = (
    df_tipo_eixos
    .sort_values(
        ["tipo_veiculo", "veiculos"],
        ascending=[True, False]
    )
    .groupby("tipo_veiculo", group_keys=False)
    .head(5)
    .reset_index(drop=True)
)

display(df_top_eixos_tipo)

In [ ]:
from openpyxl import load_workbook

eixos_prioritarios = {0, 1, 2, 20}
distribuicao_eixos_mensal = []

for i, arquivo_mes in enumerate(arquivos_senatran, start=1):
    mes = arquivo_mes.parent.name
    totais = {eixo: 0 for eixo in eixos_prioritarios}

    wb = load_workbook(
        arquivo_mes,
        read_only=True,
        data_only=True
    )
    ws = wb["Layout G"]

    for uf, municipio, tipo, especie, eixos, qtd in ws.iter_rows(
        min_row=2,
        min_col=1,
        max_col=6,
        values_only=True
    ):
        if tipo in tipos_escopo and eixos in eixos_prioritarios:
            totais[eixos] += qtd

    wb.close()

    distribuicao_eixos_mensal.append({
        "mes": mes,
        **{f"eixos_{eixo}": totais[eixo] for eixo in sorted(eixos_prioritarios)}
    })

    print(f"{i}/{len(arquivos_senatran)} — {mes} concluído")

df_eixos_mensal = pd.DataFrame(distribuicao_eixos_mensal)

display(df_eixos_mensal)

In [ ]:
from openpyxl import load_workbook

meses_investigacao = ["2026-01", "2026-02"]

resultado_eixos_20 = []

for mes in meses_investigacao:
    arquivo_mes = next(
        arquivo for arquivo in arquivos_senatran
        if arquivo.parent.name == mes
    )

    wb = load_workbook(
        arquivo_mes,
        read_only=True,
        data_only=True
    )
    ws = wb["Layout G"]

    totais_tipo = {}

    for uf, municipio, tipo, especie, eixos, qtd in ws.iter_rows(
        min_row=2,
        min_col=1,
        max_col=6,
        values_only=True
    ):
        if tipo in tipos_escopo and eixos == 20:
            totais_tipo[tipo] = (
                totais_tipo.get(tipo, 0) + qtd
            )

    wb.close()

    for tipo, total in totais_tipo.items():
        resultado_eixos_20.append({
            "mes": mes,
            "tipo_veiculo": tipo,
            "veiculos": total
        })

df_eixos_20_por_tipo = pd.DataFrame(resultado_eixos_20)

display(
    df_eixos_20_por_tipo
    .pivot(
        index="tipo_veiculo",
        columns="mes",
        values="veiculos"
    )
    .fillna(0)
    .astype(int)
)

In [ ]:
from openpyxl import load_workbook

resultado_especies_eixos_20 = []

for mes in ["2026-01", "2026-02"]:
    arquivo_mes = next(
        arquivo for arquivo in arquivos_senatran
        if arquivo.parent.name == mes
    )

    wb = load_workbook(
        arquivo_mes,
        read_only=True,
        data_only=True
    )
    ws = wb["Layout G"]

    totais_especie = {}

    for uf, municipio, tipo, especie, eixos, qtd in ws.iter_rows(
        min_row=2,
        min_col=1,
        max_col=6,
        values_only=True
    ):
        if tipo == "AUTOMOVEL" and eixos == 20:
            totais_especie[especie] = (
                totais_especie.get(especie, 0) + qtd
            )

    wb.close()

    for especie, total in totais_especie.items():
        resultado_especies_eixos_20.append({
            "mes": mes,
            "especie_veiculo": especie,
            "veiculos": total
        })

df_especies_eixos_20 = pd.DataFrame(
    resultado_especies_eixos_20
)

display(
    df_especies_eixos_20
    .pivot(
        index="especie_veiculo",
        columns="mes",
        values="veiculos"
    )
    .fillna(0)
    .astype(int)
)

In [ ]:
from openpyxl import load_workbook

resultado_municipios_eixos_20 = []

for mes in ["2026-01", "2026-02"]:
    arquivo_mes = next(
        arquivo for arquivo in arquivos_senatran
        if arquivo.parent.name == mes
    )

    wb = load_workbook(
        arquivo_mes,
        read_only=True,
        data_only=True
    )
    ws = wb["Layout G"]

    for uf, municipio, tipo, especie, eixos, qtd in ws.iter_rows(
        min_row=2,
        min_col=1,
        max_col=6,
        values_only=True
    ):
        if (
            tipo == "AUTOMOVEL"
            and especie == "PASSAGEIRO"
            and eixos == 20
        ):
            resultado_municipios_eixos_20.append({
                "mes": mes,
                "uf": uf,
                "municipio": municipio,
                "veiculos": qtd
            })

    wb.close()

df_municipios_eixos_20 = pd.DataFrame(
    resultado_municipios_eixos_20
)

display(
    df_municipios_eixos_20
    .sort_values(
        ["mes", "veiculos"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

In [ ]:
from openpyxl import load_workbook

colunas_textuais = {
    "UF": 1,
    "Município": 2,
    "Tipo Veículo": 3,
    "Espécie Veículo": 4
}

problemas_textuais = []

for i, arquivo_mes in enumerate(arquivos_senatran, start=1):
    mes = arquivo_mes.parent.name

    wb = load_workbook(
        arquivo_mes,
        read_only=True,
        data_only=True
    )
    ws = wb["Layout G"]

    contagens = {
        coluna: {
            "nulos": 0,
            "vazios": 0,
            "espacos_externos": 0,
            "variacao_maiusculas": 0
        }
        for coluna in colunas_textuais
    }

    for linha in ws.iter_rows(
        min_row=2,
        min_col=1,
        max_col=6,
        values_only=True
    ):
        if linha[2] in tipos_escopo:
            for coluna, indice in colunas_textuais.items():
                valor = linha[indice - 1]

                if valor is None:
                    contagens[coluna]["nulos"] += 1
                    continue

                if not isinstance(valor, str):
                    continue

                if valor.strip() == "":
                    contagens[coluna]["vazios"] += 1

                if valor != valor.strip():
                    contagens[coluna]["espacos_externos"] += 1

                if valor != valor.upper():
                    contagens[coluna]["variacao_maiusculas"] += 1

    wb.close()

    for coluna, valores in contagens.items():
        problemas_textuais.append({
            "mes": mes,
            "coluna": coluna,
            **valores
        })

    print(f"{i}/{len(arquivos_senatran)} — {mes} concluído")

df_validacao_textual = pd.DataFrame(problemas_textuais)

display(df_validacao_textual)

In [ ]:
from openpyxl import load_workbook
from collections import Counter, defaultdict

valores_fora_caixa_alta = defaultdict(Counter)

for i, arquivo_mes in enumerate(arquivos_senatran, start=1):
    mes = arquivo_mes.parent.name

    wb = load_workbook(
        arquivo_mes,
        read_only=True,
        data_only=True
    )
    ws = wb["Layout G"]

    for linha in ws.iter_rows(
        min_row=2,
        min_col=1,
        max_col=6,
        values_only=True
    ):
        if linha[2] in tipos_escopo:
            for coluna, indice in colunas_textuais.items():
                valor = linha[indice - 1]

                if isinstance(valor, str) and valor != valor.upper():
                    valores_fora_caixa_alta[(mes, coluna)][valor] += 1

    wb.close()

resultado_textos = []

for (mes, coluna), valores in valores_fora_caixa_alta.items():
    for valor, quantidade in valores.items():
        resultado_textos.append({
            "mes": mes,
            "coluna": coluna,
            "valor": valor,
            "ocorrencias": quantidade
        })

df_textos_fora_caixa_alta = pd.DataFrame(resultado_textos)

display(df_textos_fora_caixa_alta)

In [ ]:
resumo_validacao_senatran = pd.DataFrame([
    {
        "validacao": "Arquivos mensais",
        "resultado": "21 arquivos íntegros",
        "status": "Aprovado"
    },
    {
        "validacao": "Estrutura e cabeçalhos",
        "resultado": "6 colunas padronizadas",
        "status": "Aprovado"
    },
    {
        "validacao": "Cobertura temporal",
        "resultado": "Jan/2025 a Set/2026",
        "status": "Aprovado"
    },
    {
        "validacao": "Cobertura geográfica",
        "resultado": "27 UFs + Sem Informação",
        "status": "Aprovado"
    },
    {
        "validacao": "Chave candidata",
        "resultado": "Nenhuma duplicidade nos 21 meses",
        "status": "Aprovado"
    },
    {
        "validacao": "Valores da quantidade",
        "resultado": "Sem nulos, zeros, negativos ou frações",
        "status": "Aprovado"
    },
    {
        "validacao": "Nulos na chave",
        "resultado": "Nenhum nulo",
        "status": "Aprovado"
    },
    {
        "validacao": "Combinações tipo-espécie",
        "resultado": "30 combinações estáveis",
        "status": "Aprovado"
    },
    {
        "validacao": "Valores de Eixos",
        "resultado": "84 valores estáveis; sem interpretação completa",
        "status": "Documentar"
    },
    {
        "validacao": "Padronização textual",
        "resultado": "Sem inconsistências; Sem Informação preservado",
        "status": "Aprovado"
    },
    {
        "validacao": "Anomalia Eixos = 20",
        "resultado": "Aumento em AUTOMOVEL/PASSAGEIRO em Fev/2026",
        "status": "Documentar"
    }
])

display(resumo_validacao_senatran)

In [ ]:
from sqlalchemy import text

engine = criar_engine()

with engine.connect() as conexao:
    consulta = text("""
        SELECT
            column_name,
            data_type,
            is_nullable
        FROM information_schema.columns
        WHERE table_schema = 'analytics'
          AND table_name = 'fato_frota'
        ORDER BY ordinal_position;
    """)

    df_estrutura_fato = pd.read_sql(
        consulta,
        conexao
    )

display(df_estrutura_fato)

In [ ]:
from sqlalchemy import text
from src.database import criar_engine

engine = criar_engine()

with engine.connect() as conexao:
    consulta = text("""
        SELECT
            column_name,
            data_type,
            is_nullable
        FROM information_schema.columns
        WHERE table_schema = 'analytics'
          AND table_name = 'fato_frota'
        ORDER BY ordinal_position;
    """)

    df_estrutura_fato = pd.read_sql(
        consulta,
        conexao
    )

display(df_estrutura_fato)

In [ ]:
import sys
from pathlib import Path
from sqlalchemy import text

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.database import criar_engine

engine = criar_engine()

with engine.connect() as conexao:
    consulta = text("""
        SELECT
            column_name,
            data_type,
            is_nullable
        FROM information_schema.columns
        WHERE table_schema = 'analytics'
          AND table_name = 'fato_frota'
        ORDER BY ordinal_position;
    """)

    df_estrutura_fato = pd.read_sql(consulta, conexao)

display(df_estrutura_fato)

In [ ]:
consulta = text("""
    SELECT
        data_referencia,
        COUNT(*) AS quantidade_registros,
        SUM(qtd_veiculos) AS total_veiculos,
        COUNT(DISTINCT id_uf) AS quantidade_ufs
    FROM analytics.fato_frota
    GROUP BY data_referencia
    ORDER BY data_referencia;
""")

with engine.connect() as conexao:
    df_datas_carregadas = pd.read_sql(
        consulta,
        conexao
    )

display(df_datas_carregadas)

In [ ]:
consulta = text("""
    SELECT
        con.conname AS nome_restricao,
        con.contype AS tipo,
        pg_get_constraintdef(con.oid) AS definicao
    FROM pg_constraint con
    JOIN pg_class tab
        ON tab.oid = con.conrelid
    JOIN pg_namespace ns
        ON ns.oid = tab.relnamespace
    WHERE ns.nspname = 'analytics'
      AND tab.relname = 'fato_frota'
    ORDER BY con.contype, con.conname;
""")

with engine.connect() as conexao:
    df_restricoes_fato = pd.read_sql(
        consulta,
        conexao
    )

display(df_restricoes_fato)

In [ ]:
consulta = text("""
    SELECT
        id_uf,
        sigla_uf,
        nome_uf
    FROM analytics.dim_uf
    ORDER BY id_uf;
""")

with engine.connect() as conexao:
    df_dim_uf = pd.read_sql(
        consulta,
        conexao
    )

display(df_dim_uf)

In [ ]:
arquivos_senatran = sorted(
    diretorio_senatran.glob("*/*.xlsx")
)

df_arquivos_disponiveis = pd.DataFrame({
    "arquivo": [arquivo.name for arquivo in arquivos_senatran],
    "competencia": [
        arquivo.parent.name for arquivo in arquivos_senatran
    ],
    "caminho": [
        str(arquivo) for arquivo in arquivos_senatran
    ]
})

display(df_arquivos_disponiveis[
    ["competencia", "arquivo"]
])

print(f"\nTotal de arquivos: {len(arquivos_senatran)}")

In [ ]:
arquivo_jan25 = next(
    arquivo for arquivo in arquivos_senatran
    if arquivo.parent.name == "2025-01"
)

df_jan25_ufs = pd.read_excel(
    arquivo_jan25,
    sheet_name="Layout G",
    usecols=["UF"]
)

ufs_arquivo = set(df_jan25_ufs["UF"].dropna().unique())

ufs_banco = set(df_dim_uf["sigla_uf"])

print("UFs presentes no arquivo:", len(ufs_arquivo))
print("UFs cadastradas no banco:", len(ufs_banco))
print("Valores sem correspondência no banco:", sorted(ufs_arquivo - ufs_banco))
print("Valores sem ocorrência no arquivo:", sorted(ufs_banco - ufs_arquivo))

In [ ]:
import unicodedata

def normalizar_texto(valor):
    texto = str(valor).strip().upper()
    texto = unicodedata.normalize("NFKD", texto)
    return "".join(
        caractere for caractere in texto
        if not unicodedata.combining(caractere)
    )

df_jan25_validacao = pd.read_excel(
    arquivo_jan25,
    sheet_name="Layout G",
    usecols=["UF", "Tipo Veículo"]
)

df_jan25_validacao = df_jan25_validacao[
    df_jan25_validacao["Tipo Veículo"].isin(tipos_escopo)
].copy()

ufs_arquivo = {
    normalizar_texto(valor)
    for valor in df_jan25_validacao["UF"].dropna().unique()
}

ufs_banco = {
    normalizar_texto(valor)
    for valor in df_dim_uf["nome_uf"]
}

print("UFs no arquivo após filtro:", len(ufs_arquivo))
print("UFs na dimensão:", len(ufs_banco))
print("Sem correspondência no banco:", sorted(ufs_arquivo - ufs_banco))
print("Sem ocorrência no arquivo:", sorted(ufs_banco - ufs_arquivo))

In [ ]:
df_mapeamento_uf = df_dim_uf.copy()

df_mapeamento_uf["uf_normalizada"] = (
    df_mapeamento_uf["nome_uf"].apply(normalizar_texto)
)

mapeamento_uf = dict(
    zip(
        df_mapeamento_uf["uf_normalizada"],
        df_mapeamento_uf["id_uf"]
    )
)

df_mapeamento_uf = df_mapeamento_uf[
    ["id_uf", "sigla_uf", "nome_uf", "uf_normalizada"]
]

display(df_mapeamento_uf)

print(f"\nTotal de correspondências: {len(mapeamento_uf)}")
print(f"Chaves únicas: {len(set(mapeamento_uf.keys()))}")
print(f"IDs únicos: {len(set(mapeamento_uf.values()))}")

In [ ]:
competencias_carregadas = {
    "2026-08",
    "2026-09"
}

arquivos_pendentes = [
    arquivo
    for arquivo in arquivos_senatran
    if arquivo.parent.name not in competencias_carregadas
]

print(f"Arquivos pendentes: {len(arquivos_pendentes)}")

for arquivo in arquivos_pendentes:
    print(arquivo.parent.name, "-", arquivo.name)

In [ ]:
df_jan25_processado = pd.read_excel(
    arquivo_jan25,
    sheet_name="Layout G",
    usecols=[
        "UF",
        "Município",
        "Tipo Veículo",
        "Espécie Veículo",
        "Eixos",
        "Qtd. Veículos"
    ]
)

# Aplicar o escopo de veículos
df_jan25_processado = df_jan25_processado[
    df_jan25_processado["Tipo Veículo"].isin(tipos_escopo)
].copy()

# Converter o nome da UF em id_uf
df_jan25_processado["uf_normalizada"] = (
    df_jan25_processado["UF"].apply(normalizar_texto)
)

df_jan25_processado["id_uf"] = (
    df_jan25_processado["uf_normalizada"].map(mapeamento_uf)
)

# Preparar as colunas conforme a tabela fato
df_jan25_processado = df_jan25_processado.rename(columns={
    "Município": "municipio",
    "Tipo Veículo": "tipo_veiculo",
    "Espécie Veículo": "especie_veiculo",
    "Eixos": "eixos",
    "Qtd. Veículos": "qtd_veiculos"
})

df_jan25_processado["data_referencia"] = pd.Timestamp("2025-01-01").date()

# Preservar atributos textuais
for coluna in [
    "municipio",
    "tipo_veiculo",
    "especie_veiculo",
    "eixos"
]:
    df_jan25_processado[coluna] = (
        df_jan25_processado[coluna].astype("string")
    )

df_jan25_processado["qtd_veiculos"] = (
    pd.to_numeric(
        df_jan25_processado["qtd_veiculos"],
        errors="raise"
    ).astype("int64")
)

df_jan25_processado = df_jan25_processado[
    [
        "data_referencia",
        "id_uf",
        "municipio",
        "tipo_veiculo",
        "especie_veiculo",
        "eixos",
        "qtd_veiculos"
    ]
]

print("Registros:", len(df_jan25_processado))
print("Total de veículos:", df_jan25_processado["qtd_veiculos"].sum())
print("UFs sem mapeamento:", df_jan25_processado["id_uf"].isna().sum())
print("Valores nulos:", df_jan25_processado.isna().sum().sum())
print("Chaves duplicadas:", df_jan25_processado.duplicated(
    subset=[
        "data_referencia",
        "id_uf",
        "municipio",
        "tipo_veiculo",
        "especie_veiculo",
        "eixos"
    ]
).sum())

display(df_jan25_processado.head())

In [ ]:
colunas_origem = [
    "UF",
    "Município",
    "Tipo Veículo",
    "Espécie Veículo",
    "Eixos",
    "Qtd. Veículos"
]

colunas_chave = [
    "data_referencia",
    "id_uf",
    "municipio",
    "tipo_veiculo",
    "especie_veiculo",
    "eixos"
]

dados_mensais = []

for arquivo in arquivos_pendentes:
    competencia = arquivo.parent.name

    df_mes = pd.read_excel(
        arquivo,
        sheet_name="Layout G",
        usecols=colunas_origem
    )

    df_mes = df_mes[
        df_mes["Tipo Veículo"].isin(tipos_escopo)
    ].copy()

    df_mes["uf_normalizada"] = (
        df_mes["UF"].apply(normalizar_texto)
    )

    df_mes["id_uf"] = (
        df_mes["uf_normalizada"].map(mapeamento_uf)
    )

    df_mes = df_mes.rename(columns={
        "Município": "municipio",
        "Tipo Veículo": "tipo_veiculo",
        "Espécie Veículo": "especie_veiculo",
        "Eixos": "eixos",
        "Qtd. Veículos": "qtd_veiculos"
    })

    df_mes["data_referencia"] = (
        pd.Period(competencia, freq="M")
        .to_timestamp()
        .date()
    )

    for coluna in [
        "municipio",
        "tipo_veiculo",
        "especie_veiculo",
        "eixos"
    ]:
        df_mes[coluna] = df_mes[coluna].astype("string")

    df_mes["qtd_veiculos"] = (
        pd.to_numeric(
            df_mes["qtd_veiculos"],
            errors="raise"
        ).astype("int64")
    )

    df_mes = df_mes[
        [
            "data_referencia",
            "id_uf",
            "municipio",
            "tipo_veiculo",
            "especie_veiculo",
            "eixos",
            "qtd_veiculos"
        ]
    ]

    dados_mensais.append(df_mes)

df_frota_pendente = pd.concat(
    dados_mensais,
    ignore_index=True
)

resumo_processamento = (
    df_frota_pendente
    .groupby("data_referencia")
    .agg(
        quantidade_registros=("qtd_veiculos", "size"),
        total_veiculos=("qtd_veiculos", "sum")
    )
    .reset_index()
)

print("Total de registros:", len(df_frota_pendente))
print("Total de veículos:", df_frota_pendente["qtd_veiculos"].sum())
print("UFs sem mapeamento:", df_frota_pendente["id_uf"].isna().sum())
print("Valores nulos:", df_frota_pendente.isna().sum().sum())
print("Chaves duplicadas:", df_frota_pendente.duplicated(
    subset=colunas_chave
).sum())

display(resumo_processamento)

In [ ]:
consulta = text("""
    SELECT
        data_referencia,
        COUNT(*) AS quantidade_registros,
        SUM(qtd_veiculos) AS total_veiculos
    FROM analytics.fato_frota
    WHERE data_referencia >= DATE '2025-01-01'
      AND data_referencia <= DATE '2026-07-01'
    GROUP BY data_referencia
    ORDER BY data_referencia;
""")

with engine.connect() as conexao:
    df_competencias_existentes = pd.read_sql(
        consulta,
        conexao
    )

if df_competencias_existentes.empty:
    print("Nenhum registro encontrado nas 19 competências pendentes.")
else:
    display(df_competencias_existentes)

In [ ]:
with engine.begin() as conexao:
    quantidade_inserida = df_frota_pendente.to_sql(
        name="fato_frota",
        con=conexao,
        schema="analytics",
        if_exists="append",
        index=False,
        chunksize=5000,
        method="multi"
    )

print("Carga concluída.")
print(f"Registros inseridos: {quantidade_inserida:,}")

In [ ]:
consulta = text("""
    SELECT
        COUNT(*) AS quantidade_registros,
        SUM(qtd_veiculos) AS total_veiculos,
        COUNT(DISTINCT data_referencia) AS competencias
    FROM analytics.fato_frota;
""")

with engine.connect() as conexao:
    df_validacao_carga = pd.read_sql(
        consulta,
        conexao
    )

display(df_validacao_carga)

In [ ]:
consulta = text("""
    SELECT
        TO_CHAR(data_referencia, 'YYYY-MM') AS competencia,
        COUNT(*) AS quantidade_registros,
        SUM(qtd_veiculos) AS total_veiculos
    FROM analytics.fato_frota
    GROUP BY data_referencia
    ORDER BY data_referencia;
""")

with engine.connect() as conexao:
    df_validacao_mensal = pd.read_sql(
        consulta,
        conexao
    )

display(df_validacao_mensal)

print(
    "Total exato de veículos:",
    int(df_validacao_mensal["total_veiculos"].sum())
)

In [ ]:
consulta = text("""
    SELECT
        TO_CHAR(data_referencia, 'YYYY-MM') AS competencia,
        COUNT(DISTINCT id_uf) AS quantidade_ufs,
        COUNT(*) AS quantidade_registros
    FROM analytics.fato_frota
    GROUP BY data_referencia
    ORDER BY data_referencia;
""")

with engine.connect() as conexao:
    df_cobertura_geografica = pd.read_sql(
        consulta,
        conexao
    )

display(df_cobertura_geografica)

print(
    "Competências com cobertura diferente de 28:",
    len(df_cobertura_geografica[
        df_cobertura_geografica["quantidade_ufs"] != 28
    ])
)

In [ ]:
consulta = text("""
    SELECT
        COUNT(*) AS total_registros,

        COUNT(*) FILTER (
            WHERE d.id_uf IS NULL
        ) AS ufs_sem_referencia,

        COUNT(*) - COUNT(DISTINCT (
            f.data_referencia,
            f.id_uf,
            f.municipio,
            f.tipo_veiculo,
            f.especie_veiculo,
            f.eixos
        )) AS chaves_duplicadas

    FROM analytics.fato_frota f

    LEFT JOIN analytics.dim_uf d
        ON f.id_uf = d.id_uf;
""")

with engine.connect() as conexao:
    df_integridade_final = pd.read_sql(
        consulta,
        conexao
    )

display(df_integridade_final)

In [ ]:
consulta = text("""
    WITH frota_mensal AS (
        SELECT
            data_referencia,
            SUM(qtd_veiculos) AS total_veiculos
        FROM analytics.fato_frota
        GROUP BY data_referencia
    ),

    comparacoes AS (
        SELECT
            data_referencia,
            total_veiculos,

            LAG(total_veiculos, 1) OVER (
                ORDER BY data_referencia
            ) AS total_mes_anterior,

            LAG(total_veiculos, 12) OVER (
                ORDER BY data_referencia
            ) AS total_mes_ano_anterior

        FROM frota_mensal
    )

    SELECT
        TO_CHAR(data_referencia, 'YYYY-MM') AS competencia,
        total_veiculos,

        total_veiculos - total_mes_anterior
            AS variacao_mensal_absoluta,

        ROUND(
            100.0 * (total_veiculos - total_mes_anterior)
            / NULLIF(total_mes_anterior, 0),
            2
        ) AS variacao_mensal_percentual,

        total_veiculos - total_mes_ano_anterior
            AS variacao_anual_absoluta,

        ROUND(
            100.0 * (total_veiculos - total_mes_ano_anterior)
            / NULLIF(total_mes_ano_anterior, 0),
            2
        ) AS variacao_anual_percentual

    FROM comparacoes
    ORDER BY data_referencia;
""")

with engine.connect() as conexao:
    df_evolucao_frota = pd.read_sql(
        consulta,
        conexao
    )

display(df_evolucao_frota)

In [ ]:
import matplotlib.pyplot as plt

df_grafico = df_evolucao_frota.copy()

df_grafico["competencia"] = pd.to_datetime(
    df_grafico["competencia"]
)

plt.figure(figsize=(12, 5))

plt.plot(
    df_grafico["competencia"],
    df_grafico["total_veiculos"],
    marker="o"
)

plt.title("Evolução da frota de automóveis e comerciais leves")
plt.xlabel("Competência")
plt.ylabel("Quantidade de veículos")
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
from sqlalchemy import text
import pandas as pd

consulta = text("""
    WITH frota_uf AS (
        SELECT
            uf.sigla_uf,
            uf.nome_uf,
            SUM(
                CASE
                    WHEN f.data_referencia = '2025-09-01'
                    THEN f.qtd_veiculos
                    ELSE 0
                END
            ) AS frota_set_2025,
            SUM(
                CASE
                    WHEN f.data_referencia = '2026-09-01'
                    THEN f.qtd_veiculos
                    ELSE 0
                END
            ) AS frota_set_2026
        FROM analytics.fato_frota f
        JOIN analytics.dim_uf uf
            ON uf.id_uf = f.id_uf
        WHERE f.data_referencia IN (
            '2025-09-01',
            '2026-09-01'
        )
          AND uf.sigla_uf <> 'SI'
        GROUP BY
            uf.sigla_uf,
            uf.nome_uf
    )
    SELECT
        sigla_uf,
        nome_uf,
        frota_set_2025,
        frota_set_2026,
        frota_set_2026 - frota_set_2025
            AS crescimento_absoluto,
        ROUND(
            100.0 * (frota_set_2026 - frota_set_2025)
            / NULLIF(frota_set_2025, 0),
            2
        ) AS crescimento_percentual
    FROM frota_uf
    ORDER BY crescimento_absoluto DESC;
""")

df_crescimento_uf = pd.read_sql_query(consulta, engine)

display(df_crescimento_uf)

In [ ]:
consulta_regioes = text("""
    WITH frota_com_regiao AS (
        SELECT
            CASE
                WHEN uf.sigla_uf IN ('AC', 'AP', 'AM', 'PA', 'RO', 'RR', 'TO')
                    THEN 'Norte'
                WHEN uf.sigla_uf IN ('AL', 'BA', 'CE', 'MA', 'PB', 'PE', 'PI', 'RN', 'SE')
                    THEN 'Nordeste'
                WHEN uf.sigla_uf IN ('DF', 'GO', 'MT', 'MS')
                    THEN 'Centro-Oeste'
                WHEN uf.sigla_uf IN ('ES', 'MG', 'RJ', 'SP')
                    THEN 'Sudeste'
                WHEN uf.sigla_uf IN ('PR', 'RS', 'SC')
                    THEN 'Sul'
            END AS regiao,
            f.data_referencia,
            f.qtd_veiculos
        FROM analytics.fato_frota f
        JOIN analytics.dim_uf uf
            ON uf.id_uf = f.id_uf
        WHERE f.data_referencia IN (
            '2025-09-01',
            '2026-09-01'
        )
          AND uf.sigla_uf <> 'SI'
    ),
    frota_regional AS (
        SELECT
            regiao,
            SUM(
                CASE
                    WHEN data_referencia = '2025-09-01'
                    THEN qtd_veiculos ELSE 0
                END
            ) AS frota_set_2025,
            SUM(
                CASE
                    WHEN data_referencia = '2026-09-01'
                    THEN qtd_veiculos ELSE 0
                END
            ) AS frota_set_2026
        FROM frota_com_regiao
        GROUP BY regiao
    )
    SELECT
        regiao,
        frota_set_2025,
        frota_set_2026,
        frota_set_2026 - frota_set_2025 AS crescimento_absoluto,
        ROUND(
            100.0 * (frota_set_2026 - frota_set_2025)
            / NULLIF(frota_set_2025, 0),
            2
        ) AS crescimento_percentual
    FROM frota_regional
    ORDER BY crescimento_absoluto DESC;
""")

df_crescimento_regiao = pd.read_sql_query(
    consulta_regioes,
    engine
)

display(df_crescimento_regiao)

In [ ]:
consulta_tipos = text("""
    WITH frota_tipo AS (
        SELECT
            tipo_veiculo,
            SUM(
                CASE
                    WHEN data_referencia = '2025-09-01'
                    THEN qtd_veiculos ELSE 0
                END
            ) AS frota_set_2025,
            SUM(
                CASE
                    WHEN data_referencia = '2026-09-01'
                    THEN qtd_veiculos ELSE 0
                END
            ) AS frota_set_2026
        FROM analytics.fato_frota
        WHERE data_referencia IN (
            '2025-09-01',
            '2026-09-01'
        )
        GROUP BY tipo_veiculo
    )
    SELECT
        tipo_veiculo,
        frota_set_2025,
        frota_set_2026,
        frota_set_2026 - frota_set_2025 AS crescimento_absoluto,
        ROUND(
            100.0 * (frota_set_2026 - frota_set_2025)
            / NULLIF(frota_set_2025, 0),
            2
        ) AS crescimento_percentual,
        ROUND(
            100.0 * frota_set_2026
            / SUM(frota_set_2026) OVER (),
            2
        ) AS participacao_percentual_2026
    FROM frota_tipo
    ORDER BY frota_set_2026 DESC;
""")

df_composicao_tipo = pd.read_sql_query(
    consulta_tipos,
    engine
)

display(df_composicao_tipo)

In [ ]:
consulta_especies = text("""
    WITH frota_especie AS (
        SELECT
            especie_veiculo,
            SUM(
                CASE
                    WHEN data_referencia = '2025-09-01'
                    THEN qtd_veiculos ELSE 0
                END
            ) AS frota_set_2025,
            SUM(
                CASE
                    WHEN data_referencia = '2026-09-01'
                    THEN qtd_veiculos ELSE 0
                END
            ) AS frota_set_2026
        FROM analytics.fato_frota
        WHERE data_referencia IN (
            '2025-09-01',
            '2026-09-01'
        )
        GROUP BY especie_veiculo
    )
    SELECT
        especie_veiculo,
        frota_set_2025,
        frota_set_2026,
        frota_set_2026 - frota_set_2025 AS crescimento_absoluto,
        ROUND(
            100.0 * (frota_set_2026 - frota_set_2025)
            / NULLIF(frota_set_2025, 0),
            2
        ) AS crescimento_percentual,
        ROUND(
            100.0 * frota_set_2026
            / SUM(frota_set_2026) OVER (),
            2
        ) AS participacao_percentual_2026
    FROM frota_especie
    ORDER BY frota_set_2026 DESC;
""")

df_composicao_especie = pd.read_sql_query(
    consulta_especies,
    engine
)

display(df_composicao_especie)

In [ ]:
consulta_evolucao_especie = text("""
    SELECT
        data_referencia,
        especie_veiculo,
        SUM(qtd_veiculos) AS total_veiculos
    FROM analytics.fato_frota
    WHERE data_referencia BETWEEN '2025-01-01' AND '2026-09-01'
    GROUP BY
        data_referencia,
        especie_veiculo
    ORDER BY
        data_referencia,
        especie_veiculo;
""")

df_evolucao_especie = pd.read_sql_query(
    consulta_evolucao_especie,
    engine
)

display(df_evolucao_especie)

In [ ]:
df_evolucao_especie_pivot = (
    df_evolucao_especie
    .pivot(
        index="data_referencia",
        columns="especie_veiculo",
        values="total_veiculos"
    )
    .reset_index()
)

display(df_evolucao_especie_pivot)

In [ ]:
import matplotlib.pyplot as plt

especies_principais = [
    "PASSAGEIRO",
    "MISTO",
    "CARGA",
    "ESPECIAL",
    "COLECAO"
]

fig, ax = plt.subplots(figsize=(13, 6))

for especie in especies_principais:
    ax.plot(
        df_evolucao_especie_pivot["data_referencia"],
        df_evolucao_especie_pivot[especie],
        marker="o",
        markersize=3,
        label=especie
    )

ax.set_title("Evolução da frota por espécie de veículo")
ax.set_xlabel("Competência")
ax.set_ylabel("Quantidade de veículos")
ax.legend(title="Espécie")
ax.grid(True, alpha=0.3)

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.database import criar_engine

engine = criar_engine()

df_evolucao_especie = pd.read_sql_query(
    """
    SELECT
        data_referencia,
        especie_veiculo,
        SUM(qtd_veiculos) AS total_veiculos
    FROM analytics.fato_frota
    WHERE data_referencia BETWEEN '2025-01-01' AND '2026-09-01'
    GROUP BY data_referencia, especie_veiculo
    ORDER BY data_referencia, especie_veiculo;
    """,
    engine
)

df_evolucao_especie_pivot = (
    df_evolucao_especie
    .pivot(
        index="data_referencia",
        columns="especie_veiculo",
        values="total_veiculos"
    )
    .reset_index()
)

display(df_evolucao_especie_pivot)

In [ ]:
especies_principais = [
    "PASSAGEIRO",
    "MISTO",
    "CARGA",
    "ESPECIAL",
    "COLECAO"
]

df_especies_indexado = df_evolucao_especie_pivot[
    ["data_referencia"] + especies_principais
].copy()

for especie in especies_principais:
    valor_inicial = df_especies_indexado[especie].iloc[0]

    df_especies_indexado[especie] = (
        df_especies_indexado[especie] / valor_inicial * 100
    )

fig, ax = plt.subplots(figsize=(13, 6))

for especie in especies_principais:
    ax.plot(
        df_especies_indexado["data_referencia"],
        df_especies_indexado[especie],
        marker="o",
        markersize=3,
        label=especie
    )

ax.set_title(
    "Evolução indexada da frota por espécie (Jan/2025 = 100)"
)
ax.set_xlabel("Competência")
ax.set_ylabel("Índice de evolução")
ax.legend(title="Espécie")
ax.grid(True, alpha=0.3)

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
consulta_evolucao_tipo = text("""
    SELECT
        data_referencia,
        tipo_veiculo,
        SUM(qtd_veiculos) AS total_veiculos
    FROM analytics.fato_frota
    WHERE data_referencia BETWEEN '2025-01-01' AND '2026-09-01'
    GROUP BY
        data_referencia,
        tipo_veiculo
    ORDER BY
        data_referencia,
        tipo_veiculo;
""")

df_evolucao_tipo = pd.read_sql_query(
    consulta_evolucao_tipo,
    engine
)

df_evolucao_tipo_pivot = (
    df_evolucao_tipo
    .pivot(
        index="data_referencia",
        columns="tipo_veiculo",
        values="total_veiculos"
    )
    .reset_index()
)

display(df_evolucao_tipo_pivot)

In [ ]:
from sqlalchemy import text

print("Importação de text realizada com sucesso.")

In [ ]:
consulta_evolucao_tipo = text("""
    SELECT
        data_referencia,
        tipo_veiculo,
        SUM(qtd_veiculos) AS total_veiculos
    FROM analytics.fato_frota
    WHERE data_referencia BETWEEN '2025-01-01' AND '2026-09-01'
    GROUP BY
        data_referencia,
        tipo_veiculo
    ORDER BY
        data_referencia,
        tipo_veiculo;
""")

df_evolucao_tipo = pd.read_sql_query(
    consulta_evolucao_tipo,
    engine
)

df_evolucao_tipo_pivot = (
    df_evolucao_tipo
    .pivot(
        index="data_referencia",
        columns="tipo_veiculo",
        values="total_veiculos"
    )
    .reset_index()
)

display(df_evolucao_tipo_pivot)

In [ ]:
tipos_principais = [
    "AUTOMOVEL",
    "CAMINHONETE",
    "CAMIONETA",
    "UTILITARIO"
]

df_tipos_indexado = df_evolucao_tipo_pivot[
    ["data_referencia"] + tipos_principais
].copy()

for tipo in tipos_principais:
    valor_inicial = df_tipos_indexado[tipo].iloc[0]

    df_tipos_indexado[tipo] = (
        df_tipos_indexado[tipo] / valor_inicial * 100
    )

fig, ax = plt.subplots(figsize=(13, 6))

for tipo in tipos_principais:
    ax.plot(
        df_tipos_indexado["data_referencia"],
        df_tipos_indexado[tipo],
        marker="o",
        markersize=3,
        label=tipo
    )

ax.set_title(
    "Evolução indexada da frota por tipo de veículo (Jan/2025 = 100)"
)
ax.set_xlabel("Competência")
ax.set_ylabel("Índice de evolução")
ax.legend(title="Tipo de veículo")
ax.grid(True, alpha=0.3)

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
df_participacao_tipo = df_evolucao_tipo_pivot[
    ["data_referencia"] + tipos_principais
].copy()

total_frota_tipo = df_participacao_tipo[
    tipos_principais
].sum(axis=1)

for tipo in tipos_principais:
    df_participacao_tipo[tipo] = (
        df_participacao_tipo[tipo]
        / total_frota_tipo
        * 100
    )

display(
    df_participacao_tipo[
        df_participacao_tipo["data_referencia"].isin(
            pd.to_datetime(["2025-01-01", "2026-09-01"])
        )
    ].round(2)
)

In [ ]:
print("Formato da coluna:", df_participacao_tipo["data_referencia"].dtype)

print(
    df_participacao_tipo[
        ["data_referencia"] + tipos_principais
    ].tail(5).to_string(index=False)
)

print(
    "\nDatas disponíveis:",
    df_participacao_tipo["data_referencia"].min(),
    "até",
    df_participacao_tipo["data_referencia"].max()
)

In [ ]:
df_participacao_tipo["data_referencia"] = pd.to_datetime(
    df_participacao_tipo["data_referencia"]
)

display(
    df_participacao_tipo[
        df_participacao_tipo["data_referencia"].isin(
            pd.to_datetime(["2025-01-01", "2026-09-01"])
        )
    ].round(2).to_string(index=False)
)

In [ ]:
from pathlib import Path

arquivo_fenabrave = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "fenabrave"
    / "2026-10"
    / "informativo_emplacamentos_outubro_2026.pdf"
)

if arquivo_fenabrave.exists():
    tamanho_mb = arquivo_fenabrave.stat().st_size / (1024 ** 2)

    print("Arquivo encontrado!")
    print(f"Nome: {arquivo_fenabrave.name}")
    print(f"Tamanho: {tamanho_mb:.2f} MB")
    print(f"Caminho: {arquivo_fenabrave}")
else:
    print("Arquivo não encontrado.")
    print(f"Caminho esperado: {arquivo_fenabrave}")

In [ ]:
%pip install pypdf

In [ ]:
from pypdf import PdfReader

leitor_pdf = PdfReader(arquivo_fenabrave)

print(f"Total de páginas: {len(leitor_pdf.pages)}")
print("\nMetadados:")
print(leitor_pdf.metadata)

print("\nTexto das primeiras páginas:")

for i in range(min(3, len(leitor_pdf.pages))):
    texto = leitor_pdf.pages[i].extract_text()

    print(f"\n{'=' * 50}")
    print(f"PÁGINA {i + 1}")
    print('=' * 50)
    print(texto[:3000] if texto else "Nenhum texto extraído.")

In [ ]:
for i, pagina in enumerate(leitor_pdf.pages):
    texto = pagina.extract_text() or ""
    linhas = [
        linha.strip()
        for linha in texto.splitlines()
        if linha.strip()
    ]

    print(f"\n--- PÁGINA {i + 1} ---")

    for linha in linhas[:6]:
        print(linha[:180])

In [ ]:
paginas_interesse = [1, 2, 5]

for numero in paginas_interesse:
    texto = leitor_pdf.pages[numero - 1].extract_text()

    print(f"\n{'=' * 70}")
    print(f"PÁGINA {numero}")
    print('=' * 70)
    print(texto)

In [ ]:
import requests
from pathlib import Path
import pandas as pd

pasta_fenabrave = PROJECT_ROOT / "data" / "raw" / "fenabrave"
pasta_fenabrave.mkdir(parents=True, exist_ok=True)

competencias = pd.date_range(
    start="2025-01-01",
    end="2026-09-01",
    freq="MS"
)

resultados_download = []

for competencia in competencias:
    ano = competencia.year
    mes = competencia.month

    url = (
        f"https://www.fenabrave.org.br/portal/files/"
        f"{ano}_{mes:02d}_02.pdf"
    )

    pasta_mes = pasta_fenabrave / f"{ano}-{mes:02d}"
    pasta_mes.mkdir(parents=True, exist_ok=True)

    destino = (
        pasta_mes
        / f"informativo_emplacamentos_{ano}_{mes:02d}.pdf"
    )

    if destino.exists():
        resultados_download.append({
            "competencia": competencia.strftime("%Y-%m"),
            "status": "Já existente",
            "tamanho_mb": round(destino.stat().st_size / 1024**2, 2)
        })
        continue

    try:
        resposta = requests.get(url, timeout=60)

        if resposta.status_code == 200 and resposta.content.startswith(b"%PDF"):
            destino.write_bytes(resposta.content)

            resultados_download.append({
                "competencia": competencia.strftime("%Y-%m"),
                "status": "Baixado",
                "tamanho_mb": round(len(resposta.content) / 1024**2, 2)
            })
        else:
            resultados_download.append({
                "competencia": competencia.strftime("%Y-%m"),
                "status": f"Falha HTTP {resposta.status_code}",
                "tamanho_mb": None
            })

    except requests.RequestException as erro:
        resultados_download.append({
            "competencia": competencia.strftime("%Y-%m"),
            "status": f"Erro: {erro}",
            "tamanho_mb": None
        })

df_download_fenabrave = pd.DataFrame(resultados_download)

display(df_download_fenabrave)

print("\nResumo:")
print(df_download_fenabrave["status"].value_counts())

In [ ]:
from pypdf import PdfReader

registros_pdf = []

for competencia in competencias:
    ano = competencia.year
    mes = competencia.month

    arquivo = (
        pasta_fenabrave
        / f"{ano}-{mes:02d}"
        / f"informativo_emplacamentos_{ano}_{mes:02d}.pdf"
    )

    try:
        leitor = PdfReader(arquivo)

        texto_primeira_pagina = (
            leitor.pages[0].extract_text() or ""
        )

        registros_pdf.append({
            "competencia": competencia.strftime("%Y-%m"),
            "arquivo_existe": arquivo.exists(),
            "tamanho_mb": round(
                arquivo.stat().st_size / 1024**2, 2
            ),
            "paginas": len(leitor.pages),
            "texto_extraivel": bool(texto_primeira_pagina.strip()),
            "referencia_no_documento": (
                texto_primeira_pagina[:250].replace("\n", " ")
            )
        })

    except Exception as erro:
        registros_pdf.append({
            "competencia": competencia.strftime("%Y-%m"),
            "arquivo_existe": arquivo.exists(),
            "tamanho_mb": None,
            "paginas": None,
            "texto_extraivel": False,
            "referencia_no_documento": str(erro)
        })

df_auditoria_pdf = pd.DataFrame(registros_pdf)

display(df_auditoria_pdf)

In [ ]:
for competencia in competencias:
    ano = competencia.year
    mes = competencia.month

    arquivo = (
        pasta_fenabrave
        / f"{ano}-{mes:02d}"
        / f"informativo_emplacamentos_{ano}_{mes:02d}.pdf"
    )

    leitor = PdfReader(arquivo)
    texto = leitor.pages[0].extract_text() or ""

    print(f"\n{'=' * 60}")
    print(f"COMPETÊNCIA: {ano}-{mes:02d}")
    print('=' * 60)

    for linha in texto.splitlines():
        linha_limpa = linha.strip()

        if any(
            termo in linha_limpa
            for termo in ["A) Autos", "B) Com. Leves", "A + B"]
        ):
            print(linha_limpa)

In [ ]:
competencias_validacao = [
    pd.Timestamp("2025-01-01"),
    pd.Timestamp("2026-09-01")
]

for competencia in competencias_validacao:
    ano = competencia.year
    mes = competencia.month

    arquivo = (
        pasta_fenabrave
        / f"{ano}-{mes:02d}"
        / f"informativo_emplacamentos_{ano}_{mes:02d}.pdf"
    )

    leitor = PdfReader(arquivo)
    texto = leitor.pages[0].extract_text() or ""

    print(f"\n{'=' * 70}")
    print(f"RELATÓRIO: {ano}-{mes:02d}")
    print('=' * 70)
    print(texto[:3500])

In [ ]:
competencias_teste = ["2025-01", "2026-09"]

linhas_teste = []

for competencia in competencias_teste:
    ano, mes = map(int, competencia.split("-"))

    arquivo = (
        pasta_fenabrave
        / competencia
        / f"informativo_emplacamentos_{ano}_{mes:02d}.pdf"
    )

    leitor = PdfReader(arquivo)
    texto = leitor.pages[0].extract_text() or ""

    for linha in texto.splitlines():
        linha = linha.strip()

        if (
            linha.startswith("A) Autos")
            or linha.startswith("B) Com. Leves")
            or linha.startswith("A + B")
        ):
            partes = linha.split()
            categoria = " ".join(partes[:3]) if partes[0] == "B)" else " ".join(partes[:2])
            valores = partes[len(partes) - 8:]

            linhas_teste.append({
                "competencia": competencia,
                "categoria": categoria,
                "quantidade_valores": len(valores),
                "valores_extraidos": valores
            })

df_teste_parser = pd.DataFrame(linhas_teste)

display(df_teste_parser)

In [ ]:
import re

competencias_teste = ["2025-01", "2026-09"]

linhas_teste = []

for competencia in competencias_teste:
    ano, mes = map(int, competencia.split("-"))

    arquivo = (
        pasta_fenabrave
        / competencia
        / f"informativo_emplacamentos_{ano}_{mes:02d}.pdf"
    )

    leitor = PdfReader(arquivo)
    texto = leitor.pages[0].extract_text() or ""

    for linha in texto.splitlines():
        linha = linha.strip()

        if linha.startswith("A) Autos"):
            categoria = "Autos"
            conteudo = linha.replace("A) Autos", "", 1)

        elif linha.startswith("B) Com. Leves"):
            categoria = "Com. Leves"
            conteudo = linha.replace("B) Com. Leves", "", 1)

        elif linha.startswith("A + B"):
            categoria = "Autos + Com. Leves"
            conteudo = linha.replace("A + B", "", 1)

        else:
            continue

        valores = re.findall(r"-?\d[\d.]*,\d+|-?\d[\d.]*", conteudo)

        if len(valores) == 8:
            linhas_teste.append({
                "competencia": competencia,
                "categoria": categoria,
                "quantidade_valores": len(valores),
                "valores_extraidos": valores
            })

df_teste_parser = pd.DataFrame(linhas_teste)

display(df_teste_parser)

In [ ]:
arquivo = (
    pasta_fenabrave
    / "2025-01"
    / "informativo_emplacamentos_2025_01.pdf"
)

leitor = PdfReader(arquivo)
texto = leitor.pages[0].extract_text() or ""

for linha in texto.splitlines():
    linha = linha.strip()

    if any(
        linha.startswith(rotulo)
        for rotulo in ["A) Autos", "B) Com. Leves", "A + B"]
    ):
        print(repr(linha))
        print("Valores:", re.findall(r"\d[\d.,]*", linha))
        print("-" * 60)

In [ ]:
for i, linha in enumerate(texto.splitlines()):
    print(f"{i:03d}: {linha!r}")

In [ ]:
linhas = texto.splitlines()

categorias = {
    "A) Autos": "Autos",
    "B) Com. Leves": "Com. Leves",
    "A + B": "Autos + Com. Leves"
}

registros_reconstruidos = []

for i, linha in enumerate(linhas):
    linha = linha.strip()

    categoria_encontrada = next(
        (chave for chave in categorias if linha.startswith(chave)),
        None
    )

    if categoria_encontrada is None:
        continue

    valores = []

    for deslocamento in range(3):
        valores.extend(
            re.findall(
                r"-?\d[\d.]*,\d+|-?\d[\d.]*",
                linhas[i + deslocamento]
            )
        )

    registros_reconstruidos.append({
        "categoria": categorias[categoria_encontrada],
        "quantidade_valores": len(valores),
        "valores": valores
    })

df_reconstruido = pd.DataFrame(registros_reconstruidos)

display(df_reconstruido)

In [ ]:
df_emplacamentos_teste = (
    df_reconstruido[
        df_reconstruido["quantidade_valores"] == 8
    ]
    .copy()
    .reset_index(drop=True)
)

display(df_emplacamentos_teste)

In [ ]:
for _, registro in df_emplacamentos_teste.iterrows():
    print(f"\nCategoria: {registro['categoria']}")

    for posicao, valor in enumerate(registro["valores"], start=1):
        print(f"Campo {posicao}: {valor}")

In [ ]:
colunas = [
    "emplacamentos_mes",
    "emplacamentos_mes_anterior",
    "emplacamentos_acumulados",
    "emplacamentos_mes_ano_anterior",
    "emplacamentos_acumulados_ano_anterior",
    "variacao_mensal_pct",
    "variacao_anual_pct",
    "variacao_acumulada_pct"
]

registros = []

for _, linha in df_emplacamentos_teste.iterrows():
    valores = linha["valores"]

    registro = {
        "competencia": "2025-01",
        "categoria": linha["categoria"]
    }

    for i, coluna in enumerate(colunas):
        valor = valores[i]

        if i < 5:
            registro[coluna] = int(valor.replace(".", ""))
        else:
            registro[coluna] = float(
                valor.replace(".", "").replace(",", ".")
            )

    registros.append(registro)

df_emplacamentos_teste_final = pd.DataFrame(registros)

# Conferência dos percentuais calculados
df_emplacamentos_teste_final["variacao_mensal_calculada"] = (
    (
        df_emplacamentos_teste_final["emplacamentos_mes"]
        / df_emplacamentos_teste_final["emplacamentos_mes_anterior"]
        - 1
    ) * 100
).round(2)

df_emplacamentos_teste_final["variacao_anual_calculada"] = (
    (
        df_emplacamentos_teste_final["emplacamentos_mes"]
        / df_emplacamentos_teste_final["emplacamentos_mes_ano_anterior"]
        - 1
    ) * 100
).round(2)

display(df_emplacamentos_teste_final)

In [ ]:
registros_fenabrave = []
erros_extracao = []

categorias = {
    "A) Autos": "Autos",
    "B) Com. Leves": "Com. Leves",
    "A + B": "Autos + Com. Leves"
}

for competencia in competencias:
    competencia_str = competencia.strftime("%Y-%m")

    arquivo = (
        pasta_fenabrave
        / competencia_str
        / f"informativo_emplacamentos_{competencia.year}_{competencia.month:02d}.pdf"
    )

    leitor = PdfReader(arquivo)
    linhas = (leitor.pages[0].extract_text() or "").splitlines()

    registros_competencia = []

    for i, linha in enumerate(linhas):
        linha = linha.strip()

        categoria_encontrada = next(
            (
                chave for chave in categorias
                if linha.startswith(chave)
            ),
            None
        )

        if categoria_encontrada is None:
            continue

        valores = []

        for deslocamento in range(3):
            if i + deslocamento >= len(linhas):
                break

            valores.extend(
                re.findall(
                    r"-?\d[\d.]*,\d+|-?\d[\d.]*",
                    linhas[i + deslocamento]
                )
            )

        if len(valores) == 8:
            registros_competencia.append({
                "competencia": competencia_str,
                "categoria": categorias[categoria_encontrada],
                "valores": valores
            })

    if len(registros_competencia) != 3:
        erros_extracao.append({
            "competencia": competencia_str,
            "registros_encontrados": len(registros_competencia)
        })

    registros_fenabrave.extend(registros_competencia)

df_auditoria_extracao = pd.DataFrame([
    {
        "competencia": competencia_str,
        "registros": sum(
            r["competencia"] == competencia_str
            for r in registros_fenabrave
        )
    }
    for competencia_str in competencias.strftime("%Y-%m")
])

display(df_auditoria_extracao)

print("Total de registros:", len(registros_fenabrave))
print("Competências com problemas:", len(erros_extracao))

if erros_extracao:
    display(pd.DataFrame(erros_extracao))

In [ ]:
colunas_numericas = [
    "emplacamentos_mes",
    "emplacamentos_mes_anterior",
    "emplacamentos_acumulados",
    "emplacamentos_mes_ano_anterior",
    "emplacamentos_acumulados_ano_anterior",
    "variacao_mensal_pct",
    "variacao_anual_pct",
    "variacao_acumulada_pct"
]

colunas_origem = [
    "emplacamentos_mes",
    "emplacamentos_mes_anterior",
    "emplacamentos_acumulados",
    "emplacamentos_mes_ano_anterior",
    "emplacamentos_acumulados_ano_anterior",
    "variacao_mensal_pct",
    "variacao_anual_pct",
    "variacao_acumulada_pct"
]

registros_convertidos = []

for item in registros_fenabrave:
    registro = {
        "competencia": pd.to_datetime(
            item["competencia"] + "-01"
        ),
        "categoria": item["categoria"]
    }

    for i, coluna in enumerate(colunas_origem):
        valor = item["valores"][i]

        if i < 5:
            registro[coluna] = int(valor.replace(".", ""))
        else:
            registro[coluna] = float(
                valor.replace(".", "").replace(",", ".")
            )

    registros_convertidos.append(registro)

df_fenabrave = pd.DataFrame(registros_convertidos)

df_fenabrave = df_fenabrave.sort_values(
    ["competencia", "categoria"]
).reset_index(drop=True)

display(df_fenabrave.head(9))

print("\nDimensões:", df_fenabrave.shape)
print("\nTipos de dados:")
display(df_fenabrave.dtypes)

In [ ]:
df_validacao = df_fenabrave.copy()

# Separar categorias individuais e soma publicada
df_individual = df_validacao[
    df_validacao["categoria"].isin(["Autos", "Com. Leves"])
].copy()

df_soma = df_validacao[
    df_validacao["categoria"] == "Autos + Com. Leves"
].copy()

# 1. Validar acumulados
df_individual = df_individual.sort_values(
    ["categoria", "competencia"]
)

df_individual["acumulado_calculado"] = (
    df_individual.groupby("categoria")["emplacamentos_mes"].cumsum()
)

df_individual["acumulado_confere"] = (
    df_individual["emplacamentos_acumulados"]
    == df_individual["acumulado_calculado"]
)

# 2. Validar soma Autos + Com. Leves
df_soma_calculada = (
    df_individual
    .pivot(
        index="competencia",
        columns="categoria",
        values="emplacamentos_mes"
    )
)

df_soma_calculada["soma_calculada"] = (
    df_soma_calculada["Autos"]
    + df_soma_calculada["Com. Leves"]
)

df_soma_comparacao = df_soma.merge(
    df_soma_calculada[["soma_calculada"]],
    on="competencia"
)

df_soma_comparacao["soma_confere"] = (
    df_soma_comparacao["emplacamentos_mes"]
    == df_soma_comparacao["soma_calculada"]
)

# 3. Validar mês anterior, exceto janeiro/2025
df_anterior = df_individual[
    df_individual["competencia"] > pd.Timestamp("2025-01-01")
].copy()

df_anterior["mes_anterior_calculado"] = (
    df_anterior.groupby("categoria")["emplacamentos_mes"].shift(1)
)

df_anterior["mes_anterior_confere"] = (
    df_anterior["emplacamentos_mes_anterior"]
    == df_anterior["mes_anterior_calculado"]
)

print("AUDITORIA DOS ACUMULADOS")
print(df_individual["acumulado_confere"].value_counts())

print("\nAUDITORIA DA SOMA DAS CATEGORIAS")
print(df_soma_comparacao["soma_confere"].value_counts())

print("\nAUDITORIA DO MÊS ANTERIOR")
print(df_anterior["mes_anterior_confere"].value_counts())

In [ ]:
# 1. Corrigir a auditoria dos acumulados:
# o acumulado deve reiniciar em janeiro de cada ano.

df_individual["ano"] = df_individual["competencia"].dt.year

df_individual["acumulado_calculado"] = (
    df_individual
    .groupby(["categoria", "ano"])["emplacamentos_mes"]
    .cumsum()
)

df_individual["acumulado_confere"] = (
    df_individual["emplacamentos_acumulados"]
    == df_individual["acumulado_calculado"]
)

print("ACUMULADOS — AUDITORIA CORRIGIDA")
print(df_individual["acumulado_confere"].value_counts())

# 2. Detalhar divergências no campo mês anterior.

df_divergencias_anterior = df_anterior[
    ~df_anterior["mes_anterior_confere"]
].copy()

df_divergencias_anterior["diferenca"] = (
    df_divergencias_anterior["emplacamentos_mes_anterior"]
    - df_divergencias_anterior["mes_anterior_calculado"]
)

print("\nDIVERGÊNCIAS DO MÊS ANTERIOR")
display(
    df_divergencias_anterior[
        [
            "competencia",
            "categoria",
            "emplacamentos_mes_anterior",
            "mes_anterior_calculado",
            "diferenca"
        ]
    ].to_string(index=False)
)

print("\nRESUMO DAS DIFERENÇAS")
display(
    df_divergencias_anterior["diferenca"].describe()
)

In [ ]:
df_diag = df_fenabrave[
    df_fenabrave["categoria"].isin(["Autos", "Com. Leves"])
].copy()

df_diag = df_diag.sort_values(
    ["categoria", "competencia"]
).reset_index(drop=True)

df_diag["ano"] = df_diag["competencia"].dt.year

df_diag["acumulado_calculado"] = (
    df_diag
    .groupby(["categoria", "ano"])["emplacamentos_mes"]
    .cumsum()
)

df_diag["diferenca_acumulado"] = (
    df_diag["emplacamentos_acumulados"]
    - df_diag["acumulado_calculado"]
)

df_diag["acumulado_confere"] = (
    df_diag["diferenca_acumulado"] == 0
)

print("RESUMO")
print(df_diag["acumulado_confere"].value_counts())

print("\nPRIMEIROS MESES DE CADA CATEGORIA")
display(
    df_diag.groupby("categoria", group_keys=False).head(5)[
        [
            "competencia",
            "categoria",
            "emplacamentos_mes",
            "emplacamentos_acumulados",
            "acumulado_calculado",
            "diferenca_acumulado"
        ]
    ].to_string(index=False)
)

print("\nDIVERGÊNCIAS NOS ACUMULADOS")
display(
    df_diag.loc[
        ~df_diag["acumulado_confere"],
        [
            "competencia",
            "categoria",
            "emplacamentos_mes",
            "emplacamentos_acumulados",
            "acumulado_calculado",
            "diferenca_acumulado"
        ]
    ].to_string(index=False)
)

In [ ]:
df_diag = df_diag.sort_values(
    ["categoria", "competencia"]
).copy()

df_diag["incremento_acumulado"] = (
    df_diag.groupby(["categoria", "ano"])["emplacamentos_acumulados"]
    .diff()
)

df_diag["diferenca_incremento_mensal"] = (
    df_diag["incremento_acumulado"]
    - df_diag["emplacamentos_mes"]
)

print("COMPARAÇÃO ENTRE INCREMENTO DO ACUMULADO E VOLUME MENSAL")

display(
    df_diag[
        [
            "competencia",
            "categoria",
            "emplacamentos_mes",
            "emplacamentos_acumulados",
            "incremento_acumulado",
            "diferenca_incremento_mensal"
        ]
    ].to_string(index=False)
)

In [ ]:
competencias_a_verificar = [
    "2025-08",
    "2026-09"
]

for competencia in competencias_a_verificar:
    arquivo = (
        pasta_fenabrave
        / competencia
        / f"informativo_emplacamentos_{competencia.replace('-', '_')}.pdf"
    )

    leitor = PdfReader(arquivo)
    linhas = leitor.pages[0].extract_text().splitlines()

    print("\n" + "=" * 65)
    print(f"INFORMATIVO: {competencia}")
    print("=" * 65)

    for i, linha in enumerate(linhas):
        if any(
            linha.strip().startswith(rotulo)
            for rotulo in ["A) Autos", "B) Com. Leves", "A + B"]
        ):
            print("\n".join(linhas[i:i+3]))
            print("-" * 40)

In [ ]:
colunas_comparacao = [
    "emplacamentos_mes",
    "emplacamentos_mes_anterior",
    "emplacamentos_acumulados",
    "emplacamentos_mes_ano_anterior",
    "emplacamentos_acumulados_ano_anterior"
]

df_pivot = df_fenabrave.pivot(
    index="competencia",
    columns="categoria",
    values=colunas_comparacao
)

for coluna in colunas_comparacao:
    df_pivot[(coluna, "soma_calculada")] = (
        df_pivot[(coluna, "Autos")]
        + df_pivot[(coluna, "Com. Leves")]
    )

    df_pivot[(coluna, "diferenca")] = (
        df_pivot[(coluna, "Autos + Com. Leves")]
        - df_pivot[(coluna, "soma_calculada")]
    )

for coluna in colunas_comparacao:
    print(f"\n{coluna}")
    print(
        df_pivot[(coluna, "diferenca")]
        .value_counts()
        .sort_index()
    )

In [ ]:
df_pct = df_fenabrave.copy()

df_pct["variacao_mensal_calculada"] = (
    (
        df_pct["emplacamentos_mes"]
        / df_pct["emplacamentos_mes_anterior"]
    ) - 1
) * 100

df_pct["diferenca_variacao_mensal"] = (
    df_pct["variacao_mensal_pct"]
    - df_pct["variacao_mensal_calculada"]
)

df_pct["variacao_mensal_confere"] = (
    df_pct["diferenca_variacao_mensal"].abs() <= 0.02
)

print("RESULTADO DA AUDITORIA — VARIAÇÃO MENSAL")
print(df_pct["variacao_mensal_confere"].value_counts())

print("\nDIVERGÊNCIAS")
display(
    df_pct.loc[
        ~df_pct["variacao_mensal_confere"],
        [
            "competencia",
            "categoria",
            "emplacamentos_mes",
            "emplacamentos_mes_anterior",
            "variacao_mensal_pct",
            "variacao_mensal_calculada",
            "diferenca_variacao_mensal"
        ]
    ].to_string(index=False)
)

In [ ]:
df_pct["variacao_anual_calculada"] = (
    (
        df_pct["emplacamentos_mes"]
        / df_pct["emplacamentos_mes_ano_anterior"]
    ) - 1
) * 100

df_pct["diferenca_variacao_anual"] = (
    df_pct["variacao_anual_pct"]
    - df_pct["variacao_anual_calculada"]
)

df_pct["variacao_anual_confere"] = (
    df_pct["diferenca_variacao_anual"].abs() <= 0.02
)

print("RESULTADO DA AUDITORIA — VARIAÇÃO ANUAL")
print(df_pct["variacao_anual_confere"].value_counts())

print("\nDIVERGÊNCIAS")
display(
    df_pct.loc[
        ~df_pct["variacao_anual_confere"],
        [
            "competencia",
            "categoria",
            "emplacamentos_mes",
            "emplacamentos_mes_ano_anterior",
            "variacao_anual_pct",
            "variacao_anual_calculada",
            "diferenca_variacao_anual"
        ]
    ].to_string(index=False)
)

In [ ]:
df_pct["variacao_acumulada_calculada"] = (
    (
        df_pct["emplacamentos_acumulados"]
        / df_pct["emplacamentos_acumulados_ano_anterior"]
    ) - 1
) * 100

df_pct["diferenca_variacao_acumulada"] = (
    df_pct["variacao_acumulada_pct"]
    - df_pct["variacao_acumulada_calculada"]
)

df_pct["variacao_acumulada_confere"] = (
    df_pct["diferenca_variacao_acumulada"].abs() <= 0.02
)

print("RESULTADO DA AUDITORIA — VARIAÇÃO ACUMULADA")
print(df_pct["variacao_acumulada_confere"].value_counts())

print("\nDIVERGÊNCIAS")
display(
    df_pct.loc[
        ~df_pct["variacao_acumulada_confere"],
        [
            "competencia",
            "categoria",
            "emplacamentos_acumulados",
            "emplacamentos_acumulados_ano_anterior",
            "variacao_acumulada_pct",
            "variacao_acumulada_calculada",
            "diferenca_variacao_acumulada"
        ]
    ].to_string(index=False)
)

In [ ]:
print("DIMENSÕES:", df_fenabrave.shape)

print("\n1. VALORES NULOS")
display(df_fenabrave.isna().sum())

print("\n2. REGISTROS DUPLICADOS")
print(
    df_fenabrave.duplicated(
        subset=["competencia", "categoria"]
    ).sum()
)

print("\n3. CATEGORIAS")
display(
    df_fenabrave["categoria"].value_counts().sort_index()
)

print("\n4. COMPETÊNCIAS")
print("Primeira:", df_fenabrave["competencia"].min())
print("Última:", df_fenabrave["competencia"].max())
print("Total:", df_fenabrave["competencia"].nunique())

print("\n5. VALORES QUANTITATIVOS")
colunas_volumes = [
    "emplacamentos_mes",
    "emplacamentos_mes_anterior",
    "emplacamentos_acumulados",
    "emplacamentos_mes_ano_anterior",
    "emplacamentos_acumulados_ano_anterior"
]

display(df_fenabrave[colunas_volumes].describe().T)

print("\n6. CHAVES ÚNICAS")
print(
    "Chaves únicas:",
    df_fenabrave[["competencia", "categoria"]]
    .drop_duplicates()
    .shape[0]
)

In [ ]:
query_tabelas = """
SELECT
    table_schema,
    table_name
FROM information_schema.tables
WHERE table_schema IN ('staging', 'analytics')
  AND table_type = 'BASE TABLE'
ORDER BY
    table_schema,
    table_name;
"""

df_tabelas = pd.read_sql(
    query_tabelas,
    engine
)

display(df_tabelas)

In [ ]:
query_estrutura = """
SELECT
    column_name,
    data_type,
    character_maximum_length,
    numeric_precision,
    numeric_scale,
    is_nullable
FROM information_schema.columns
WHERE table_schema = 'analytics'
  AND table_name = 'fato_frota'
ORDER BY ordinal_position;
"""

df_estrutura = pd.read_sql(
    query_estrutura,
    engine
)

display(df_estrutura)

In [ ]:
query_restricoes = """
SELECT
    tc.constraint_name,
    tc.constraint_type,
    kcu.column_name,
    cc.check_clause
FROM information_schema.table_constraints tc
LEFT JOIN information_schema.key_column_usage kcu
    ON tc.constraint_name = kcu.constraint_name
    AND tc.table_schema = kcu.table_schema
    AND tc.table_name = kcu.table_name
LEFT JOIN information_schema.check_constraints cc
    ON tc.constraint_name = cc.constraint_name
    AND tc.table_schema = cc.constraint_schema
WHERE tc.table_schema = 'analytics'
  AND tc.table_name = 'fato_frota'
ORDER BY
    tc.constraint_type,
    tc.constraint_name,
    kcu.ordinal_position;
"""

df_restricoes = pd.read_sql(
    query_restricoes,
    engine
)

display(df_restricoes)

In [ ]:
ddl_staging = """
CREATE TABLE staging.fenabrave_emplacamentos (
    competencia DATE NOT NULL,
    categoria VARCHAR(30) NOT NULL,

    emplacamentos_mes BIGINT NOT NULL,
    emplacamentos_mes_anterior BIGINT NOT NULL,
    emplacamentos_acumulados BIGINT NOT NULL,
    emplacamentos_mes_ano_anterior BIGINT NOT NULL,
    emplacamentos_acumulados_ano_anterior BIGINT NOT NULL,

    variacao_mensal_pct NUMERIC(8,2) NOT NULL,
    variacao_anual_pct NUMERIC(8,2) NOT NULL,
    variacao_acumulada_pct NUMERIC(8,2) NOT NULL,

    CONSTRAINT pk_staging_fenabrave
        PRIMARY KEY (competencia, categoria),

    CONSTRAINT ck_staging_fenabrave_volumes
        CHECK (
            emplacamentos_mes > 0
            AND emplacamentos_mes_anterior > 0
            AND emplacamentos_acumulados > 0
            AND emplacamentos_mes_ano_anterior > 0
            AND emplacamentos_acumulados_ano_anterior > 0
        )
);
"""

with engine.begin() as conexao:
    conexao.execute(text(ddl_staging))

print("Comando de criação executado.")

In [ ]:
query_staging = """
SELECT
    column_name,
    data_type,
    character_maximum_length,
    numeric_precision,
    numeric_scale,
    is_nullable
FROM information_schema.columns
WHERE table_schema = 'staging'
  AND table_name = 'fenabrave_emplacamentos'
ORDER BY ordinal_position;
"""

df_estrutura_staging = pd.read_sql(query_staging, engine)
display(df_estrutura_staging)

In [ ]:
print("Dimensões:", df_fenabrave.shape)

print("\nTipos de dados:")
display(df_fenabrave.dtypes.to_frame("tipo"))

print("\nQuantidade por categoria:")
display(
    df_fenabrave["categoria"]
    .value_counts()
    .rename_axis("categoria")
    .to_frame("registros")
)

print("\nPeríodo:")
print(
    df_fenabrave["competencia"].min(),
    "a",
    df_fenabrave["competencia"].max()
)

In [ ]:
df_carga = df_fenabrave.copy()

# Converter competência para date, compatível com PostgreSQL DATE
df_carga["competencia"] = df_carga["competencia"].dt.date

registros_carga = df_carga.to_dict(orient="records")

sql_insert = """
INSERT INTO staging.fenabrave_emplacamentos (
    competencia,
    categoria,
    emplacamentos_mes,
    emplacamentos_mes_anterior,
    emplacamentos_acumulados,
    emplacamentos_mes_ano_anterior,
    emplacamentos_acumulados_ano_anterior,
    variacao_mensal_pct,
    variacao_anual_pct,
    variacao_acumulada_pct
)
VALUES (
    :competencia,
    :categoria,
    :emplacamentos_mes,
    :emplacamentos_mes_anterior,
    :emplacamentos_acumulados,
    :emplacamentos_mes_ano_anterior,
    :emplacamentos_acumulados_ano_anterior,
    :variacao_mensal_pct,
    :variacao_anual_pct,
    :variacao_acumulada_pct
);
"""

with engine.begin() as conexao:
    resultado = conexao.execute(
        text(sql_insert),
        registros_carga
    )

print(f"Registros enviados para inserção: {resultado.rowcount}")

In [ ]:
query_conferencia = """
SELECT
    categoria,
    COUNT(*) AS quantidade_registros,
    MIN(competencia) AS primeira_competencia,
    MAX(competencia) AS ultima_competencia,
    SUM(emplacamentos_mes) AS total_emplacamentos_mes
FROM staging.fenabrave_emplacamentos
GROUP BY categoria
ORDER BY categoria;
"""

df_conferencia = pd.read_sql(query_conferencia, engine)
display(df_conferencia)

query_total = """
SELECT COUNT(*) AS total_registros
FROM staging.fenabrave_emplacamentos;
"""

df_total = pd.read_sql(query_total, engine)
display(df_total)

In [ ]:
ddl_fato_emplacamentos = """
CREATE TABLE analytics.fato_emplacamentos (
    competencia DATE NOT NULL,
    categoria VARCHAR(30) NOT NULL,

    emplacamentos_mes BIGINT NOT NULL,
    emplacamentos_mes_anterior BIGINT NOT NULL,
    emplacamentos_acumulados BIGINT NOT NULL,
    emplacamentos_mes_ano_anterior BIGINT NOT NULL,
    emplacamentos_acumulados_ano_anterior BIGINT NOT NULL,

    variacao_mensal_pct NUMERIC(8,2) NOT NULL,
    variacao_anual_pct NUMERIC(8,2) NOT NULL,
    variacao_acumulada_pct NUMERIC(8,2) NOT NULL,

    CONSTRAINT pk_fato_emplacamentos
        PRIMARY KEY (competencia, categoria),

    CONSTRAINT ck_fato_emplacamentos_volumes
        CHECK (
            emplacamentos_mes > 0
            AND emplacamentos_mes_anterior > 0
            AND emplacamentos_acumulados > 0
            AND emplacamentos_mes_ano_anterior > 0
            AND emplacamentos_acumulados_ano_anterior > 0
        ),

    CONSTRAINT ck_fato_emplacamentos_categoria
        CHECK (
            categoria IN (
                'Autos',
                'Com. Leves',
                'Autos + Com. Leves'
            )
        )
);
"""

with engine.begin() as conexao:
    conexao.execute(text(ddl_fato_emplacamentos))

print("Comando de criação executado.")

In [ ]:
query_restricoes_fenabrave = """
SELECT
    tc.constraint_name,
    tc.constraint_type,
    kcu.column_name,
    cc.check_clause
FROM information_schema.table_constraints tc
LEFT JOIN information_schema.key_column_usage kcu
    ON tc.constraint_name = kcu.constraint_name
    AND tc.table_schema = kcu.table_schema
    AND tc.table_name = kcu.table_name
LEFT JOIN information_schema.check_constraints cc
    ON tc.constraint_name = cc.constraint_name
    AND tc.table_schema = cc.constraint_schema
WHERE tc.table_schema = 'analytics'
  AND tc.table_name = 'fato_emplacamentos'
ORDER BY
    tc.constraint_type,
    tc.constraint_name,
    kcu.ordinal_position;
"""

df_restricoes_fenabrave = pd.read_sql(
    query_restricoes_fenabrave,
    engine
)

display(df_restricoes_fenabrave)

In [ ]:
sql_transferencia = """
INSERT INTO analytics.fato_emplacamentos (
    competencia,
    categoria,
    emplacamentos_mes,
    emplacamentos_mes_anterior,
    emplacamentos_acumulados,
    emplacamentos_mes_ano_anterior,
    emplacamentos_acumulados_ano_anterior,
    variacao_mensal_pct,
    variacao_anual_pct,
    variacao_acumulada_pct
)
SELECT
    competencia,
    categoria,
    emplacamentos_mes,
    emplacamentos_mes_anterior,
    emplacamentos_acumulados,
    emplacamentos_mes_ano_anterior,
    emplacamentos_acumulados_ano_anterior,
    variacao_mensal_pct,
    variacao_anual_pct,
    variacao_acumulada_pct
FROM staging.fenabrave_emplacamentos;
"""

with engine.begin() as conexao:
    resultado_transferencia = conexao.execute(
        text(sql_transferencia)
    )

print(
    f"Registros transferidos: "
    f"{resultado_transferencia.rowcount}"
)

In [ ]:
query_auditoria_final = """
SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT competencia) AS total_competencias,
    COUNT(DISTINCT categoria) AS total_categorias,
    MIN(competencia) AS primeira_competencia,
    MAX(competencia) AS ultima_competencia,
    SUM(
        CASE
            WHEN NOT EXISTS (
                SELECT 1
                FROM staging.fenabrave_emplacamentos s
                WHERE s.competencia = f.competencia
                  AND s.categoria = f.categoria
                  AND s.emplacamentos_mes = f.emplacamentos_mes
                  AND s.emplacamentos_mes_anterior = f.emplacamentos_mes_anterior
                  AND s.emplacamentos_acumulados = f.emplacamentos_acumulados
                  AND s.emplacamentos_mes_ano_anterior = f.emplacamentos_mes_ano_anterior
                  AND s.emplacamentos_acumulados_ano_anterior = f.emplacamentos_acumulados_ano_anterior
                  AND s.variacao_mensal_pct = f.variacao_mensal_pct
                  AND s.variacao_anual_pct = f.variacao_anual_pct
                  AND s.variacao_acumulada_pct = f.variacao_acumulada_pct
            )
            THEN 1
            ELSE 0
        END
    ) AS registros_com_divergencia
FROM analytics.fato_emplacamentos f;
"""

df_auditoria_final = pd.read_sql(
    query_auditoria_final,
    engine
)

display(df_auditoria_final)

In [ ]:
query_dimensoes_calendario = """
SELECT
    table_name,
    column_name,
    data_type,
    ordinal_position
FROM information_schema.columns
WHERE table_schema = 'analytics'
  AND table_name IN ('dim_calendario', 'dim_competencia')
ORDER BY table_name, ordinal_position;
"""

df_dimensoes_calendario = pd.read_sql(
    query_dimensoes_calendario,
    engine
)

display(df_dimensoes_calendario)

In [ ]:
ddl_dim_calendario = """
CREATE TABLE analytics.dim_calendario (
    competencia DATE NOT NULL,
    ano SMALLINT NOT NULL,
    mes SMALLINT NOT NULL,
    nome_mes VARCHAR(20) NOT NULL,
    mes_abreviado VARCHAR(10) NOT NULL,
    trimestre SMALLINT NOT NULL,
    semestre SMALLINT NOT NULL,
    ano_mes VARCHAR(7) NOT NULL,

    CONSTRAINT pk_dim_calendario
        PRIMARY KEY (competencia),

    CONSTRAINT ck_dim_calendario_primeiro_dia
        CHECK (EXTRACT(DAY FROM competencia) = 1),

    CONSTRAINT ck_dim_calendario_mes
        CHECK (mes BETWEEN 1 AND 12),

    CONSTRAINT ck_dim_calendario_trimestre
        CHECK (trimestre BETWEEN 1 AND 4),

    CONSTRAINT ck_dim_calendario_semestre
        CHECK (semestre BETWEEN 1 AND 2)
);
"""

with engine.begin() as conexao:
    conexao.execute(text(ddl_dim_calendario))

print("Comando de criação executado.")

In [ ]:
query_estrutura_calendario = """
SELECT
    column_name,
    data_type,
    character_maximum_length,
    is_nullable
FROM information_schema.columns
WHERE table_schema = 'analytics'
  AND table_name = 'dim_calendario'
ORDER BY ordinal_position;
"""

df_estrutura_calendario = pd.read_sql(
    query_estrutura_calendario,
    engine
)

display(df_estrutura_calendario)

In [ ]:
sql_carga_calendario = """
INSERT INTO analytics.dim_calendario (
    competencia,
    ano,
    mes,
    nome_mes,
    mes_abreviado,
    trimestre,
    semestre,
    ano_mes
)
SELECT
    data_competencia::DATE,
    EXTRACT(YEAR FROM data_competencia)::SMALLINT,
    EXTRACT(MONTH FROM data_competencia)::SMALLINT,

    CASE EXTRACT(MONTH FROM data_competencia)
        WHEN 1 THEN 'Janeiro'
        WHEN 2 THEN 'Fevereiro'
        WHEN 3 THEN 'Março'
        WHEN 4 THEN 'Abril'
        WHEN 5 THEN 'Maio'
        WHEN 6 THEN 'Junho'
        WHEN 7 THEN 'Julho'
        WHEN 8 THEN 'Agosto'
        WHEN 9 THEN 'Setembro'
        WHEN 10 THEN 'Outubro'
        WHEN 11 THEN 'Novembro'
        WHEN 12 THEN 'Dezembro'
    END,

    CASE EXTRACT(MONTH FROM data_competencia)
        WHEN 1 THEN 'Jan'
        WHEN 2 THEN 'Fev'
        WHEN 3 THEN 'Mar'
        WHEN 4 THEN 'Abr'
        WHEN 5 THEN 'Mai'
        WHEN 6 THEN 'Jun'
        WHEN 7 THEN 'Jul'
        WHEN 8 THEN 'Ago'
        WHEN 9 THEN 'Set'
        WHEN 10 THEN 'Out'
        WHEN 11 THEN 'Nov'
        WHEN 12 THEN 'Dez'
    END,

    EXTRACT(QUARTER FROM data_competencia)::SMALLINT,

    CASE
        WHEN EXTRACT(MONTH FROM data_competencia) <= 6 THEN 1
        ELSE 2
    END::SMALLINT,

    TO_CHAR(data_competencia, 'YYYY-MM')

FROM GENERATE_SERIES(
    DATE '2025-01-01',
    DATE '2026-09-01',
    INTERVAL '1 month'
) AS serie(data_competencia);
"""

with engine.begin() as conexao:
    resultado_calendario = conexao.execute(
        text(sql_carga_calendario)
    )

print(f"Competências inseridas: {resultado_calendario.rowcount}")

In [ ]:
query_auditoria_calendario = """
WITH calendario AS (
    SELECT
        *,
        LAG(competencia) OVER (
            ORDER BY competencia
        ) AS competencia_anterior
    FROM analytics.dim_calendario
)
SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT competencia) AS competencias_unicas,
    MIN(competencia) AS primeira_competencia,
    MAX(competencia) AS ultima_competencia,

    SUM(
        CASE
            WHEN competencia_anterior IS NOT NULL
             AND competencia <> (
                 competencia_anterior + INTERVAL '1 month'
             )::DATE
            THEN 1
            ELSE 0
        END
    ) AS lacunas_mensais,

    SUM(
        CASE
            WHEN mes <> EXTRACT(MONTH FROM competencia)
              OR ano <> EXTRACT(YEAR FROM competencia)
              OR trimestre <> EXTRACT(QUARTER FROM competencia)
              OR semestre <> CASE
                    WHEN EXTRACT(MONTH FROM competencia) <= 6 THEN 1
                    ELSE 2
                 END
            THEN 1
            ELSE 0
        END
    ) AS inconsistencias_temporais

FROM calendario;
"""

df_auditoria_calendario = pd.read_sql(
    query_auditoria_calendario,
    engine
)

display(df_auditoria_calendario)

In [ ]:
query_integridade_temporal = """
SELECT
    'fato_frota' AS tabela,
    COUNT(DISTINCT f.data_referencia) AS competencias_na_fato,
    COUNT(DISTINCT f.data_referencia) FILTER (
        WHERE c.competencia IS NULL
    ) AS competencias_sem_calendario
FROM analytics.fato_frota f
LEFT JOIN analytics.dim_calendario c
    ON f.data_referencia = c.competencia

UNION ALL

SELECT
    'fato_emplacamentos' AS tabela,
    COUNT(DISTINCT f.competencia) AS competencias_na_fato,
    COUNT(DISTINCT f.competencia) FILTER (
        WHERE c.competencia IS NULL
    ) AS competencias_sem_calendario
FROM analytics.fato_emplacamentos f
LEFT JOIN analytics.dim_calendario c
    ON f.competencia = c.competencia;
"""

df_integridade_temporal = pd.read_sql(
    query_integridade_temporal,
    engine
)

display(df_integridade_temporal)

In [ ]:
ddl_relacionamentos_calendario = """
ALTER TABLE analytics.fato_frota
ADD CONSTRAINT fk_fato_frota_calendario
FOREIGN KEY (data_referencia)
REFERENCES analytics.dim_calendario (competencia);

ALTER TABLE analytics.fato_emplacamentos
ADD CONSTRAINT fk_fato_emplacamentos_calendario
FOREIGN KEY (competencia)
REFERENCES analytics.dim_calendario (competencia);
"""

with engine.begin() as conexao:
    for comando in ddl_relacionamentos_calendario.strip().split(";"):
        if comando.strip():
            conexao.execute(text(comando))

print("Relacionamentos criados.")

In [ ]:
query_fk_calendario = """
SELECT
    tc.constraint_name,
    tc.table_name AS tabela_fato,
    kcu.column_name AS coluna_fato,
    ccu.table_name AS tabela_referencia,
    ccu.column_name AS coluna_referencia
FROM information_schema.table_constraints tc
JOIN information_schema.key_column_usage kcu
    ON tc.constraint_name = kcu.constraint_name
    AND tc.constraint_schema = kcu.constraint_schema
JOIN information_schema.constraint_column_usage ccu
    ON tc.constraint_name = ccu.constraint_name
    AND tc.constraint_schema = ccu.constraint_schema
WHERE tc.constraint_type = 'FOREIGN KEY'
  AND tc.constraint_schema = 'analytics'
  AND tc.constraint_name IN (
      'fk_fato_frota_calendario',
      'fk_fato_emplacamentos_calendario'
  )
ORDER BY tc.table_name;
"""

df_fk_calendario = pd.read_sql(
    query_fk_calendario,
    engine
)

display(df_fk_calendario)

In [ ]:
query_dim_uf = """
SELECT
    column_name,
    data_type,
    character_maximum_length,
    is_nullable
FROM information_schema.columns
WHERE table_schema = 'analytics'
  AND table_name = 'dim_uf'
ORDER BY ordinal_position;
"""

df_estrutura_dim_uf = pd.read_sql(query_dim_uf, engine)

print("Estrutura da dimensão UF:")
display(df_estrutura_dim_uf)

query_dados_dim_uf = """
SELECT *
FROM analytics.dim_uf
ORDER BY sigla;
"""

df_dados_dim_uf = pd.read_sql(query_dados_dim_uf, engine)

print("Registros da dimensão UF:")
display(df_dados_dim_uf)

In [ ]:
query_dados_dim_uf = """
SELECT
    id_uf,
    sigla_uf,
    nome_uf,
    regiao
FROM analytics.dim_uf
ORDER BY sigla_uf;
"""

df_dados_dim_uf = pd.read_sql(
    query_dados_dim_uf,
    engine
)

display(df_dados_dim_uf)

print(f"Total de registros: {len(df_dados_dim_uf)}")

In [ ]:
query_validacao_dim_uf = """
SELECT
    regiao,
    COUNT(*) AS quantidade_ufs
FROM analytics.dim_uf
GROUP BY regiao
ORDER BY regiao;
"""

df_distribuicao_uf = pd.read_sql(
    query_validacao_dim_uf,
    engine
)

query_integridade_dim_uf = """
SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT id_uf) AS ids_distintos,
    COUNT(DISTINCT sigla_uf) AS siglas_distintas,
    (
        SELECT COUNT(*)
        FROM analytics.fato_frota f
        LEFT JOIN analytics.dim_uf d
            ON f.id_uf = d.id_uf
        WHERE d.id_uf IS NULL
    ) AS fatos_sem_dimensao
FROM analytics.dim_uf;
"""

df_integridade_dim_uf = pd.read_sql(
    query_integridade_dim_uf,
    engine
)

print("Distribuição por região:")
display(df_distribuicao_uf)

print("Integridade da dimensão:")
display(df_integridade_dim_uf)

In [ ]:
query_dim_calendario = """
SELECT
    column_name,
    data_type,
    is_nullable
FROM information_schema.columns
WHERE table_schema = 'analytics'
  AND table_name = 'dim_calendario'
ORDER BY ordinal_position;
"""

df_estrutura_calendario = pd.read_sql(
    query_dim_calendario,
    engine
)

print("Estrutura da dimensão calendário:")
display(df_estrutura_calendario)

query_dados_calendario = """
SELECT *
FROM analytics.dim_calendario
ORDER BY competencia;
"""

df_dados_calendario = pd.read_sql(
    query_dados_calendario,
    engine
)

print("Registros da dimensão calendário:")
display(df_dados_calendario)

In [ ]:
query_auditoria_calendario = """
SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT competencia) AS competencias_distintas,
    MIN(competencia) AS primeira_competencia,
    MAX(competencia) AS ultima_competencia,
    COUNT(*) FILTER (
        WHERE competencia <> DATE_TRUNC('month', competencia)::date
    ) AS competencias_fora_do_inicio_mes,
    COUNT(*) FILTER (
        WHERE ano_mes <> TO_CHAR(competencia, 'YYYY-MM')
    ) AS ano_mes_inconsistente,
    COUNT(*) FILTER (
        WHERE mes <> EXTRACT(MONTH FROM competencia)
    ) AS mes_inconsistente,
    COUNT(*) FILTER (
        WHERE trimestre <> EXTRACT(QUARTER FROM competencia)
    ) AS trimestre_inconsistente,
    COUNT(*) FILTER (
        WHERE semestre <> CASE
            WHEN EXTRACT(MONTH FROM competencia) <= 6 THEN 1
            ELSE 2
        END
    ) AS semestre_inconsistente
FROM analytics.dim_calendario;
"""

df_auditoria_calendario = pd.read_sql(
    query_auditoria_calendario,
    engine
)

display(df_auditoria_calendario)

In [ ]:
query_tabelas_analytics = """
SELECT
    table_name,
    table_type
FROM information_schema.tables
WHERE table_schema = 'analytics'
ORDER BY table_name;
"""

df_tabelas_analytics = pd.read_sql(
    query_tabelas_analytics,
    engine
)

display(df_tabelas_analytics)

In [ ]:
query_categorias_fatos = """
SELECT
    'tipo_veiculo' AS atributo,
    tipo_veiculo AS valor,
    COUNT(*) AS registros
FROM analytics.fato_frota
GROUP BY tipo_veiculo

UNION ALL

SELECT
    'especie_veiculo',
    especie_veiculo,
    COUNT(*)
FROM analytics.fato_frota
GROUP BY especie_veiculo

UNION ALL

SELECT
    'categoria_emplacamento',
    categoria,
    COUNT(*)
FROM analytics.fato_emplacamentos
GROUP BY categoria

ORDER BY atributo, valor;
"""

df_categorias_fatos = pd.read_sql(
    query_categorias_fatos,
    engine
)

display(df_categorias_fatos)

In [ ]:
query_estabilidade_categorias = """
SELECT
    'tipo_veiculo' AS atributo,
    tipo_veiculo AS valor,
    COUNT(DISTINCT data_referencia) AS competencias_presentes,
    MIN(data_referencia) AS primeira_competencia,
    MAX(data_referencia) AS ultima_competencia
FROM analytics.fato_frota
GROUP BY tipo_veiculo

UNION ALL

SELECT
    'especie_veiculo',
    especie_veiculo,
    COUNT(DISTINCT data_referencia),
    MIN(data_referencia),
    MAX(data_referencia)
FROM analytics.fato_frota
GROUP BY especie_veiculo

ORDER BY atributo, valor;
"""

df_estabilidade_categorias = pd.read_sql(
    query_estabilidade_categorias,
    engine
)

display(df_estabilidade_categorias)

In [ ]:
ddl_dim_tipo_veiculo = """
CREATE TABLE analytics.dim_tipo_veiculo (
    id_tipo_veiculo SMALLINT GENERATED ALWAYS AS IDENTITY,
    tipo_veiculo VARCHAR(50) NOT NULL,

    CONSTRAINT pk_dim_tipo_veiculo
        PRIMARY KEY (id_tipo_veiculo),

    CONSTRAINT uq_dim_tipo_veiculo
        UNIQUE (tipo_veiculo)
);
"""

with engine.begin() as conn:
    conn.execute(text(ddl_dim_tipo_veiculo))

print("Tabela analytics.dim_tipo_veiculo criada.")

In [ ]:
sql_carga_dim_tipo_veiculo = """
INSERT INTO analytics.dim_tipo_veiculo (tipo_veiculo)
SELECT DISTINCT tipo_veiculo
FROM analytics.fato_frota
ORDER BY tipo_veiculo;
"""

with engine.begin() as conn:
    resultado = conn.execute(text(sql_carga_dim_tipo_veiculo))

print(f"Registros inseridos: {resultado.rowcount}")

In [ ]:
query_dim_tipo_veiculo = """
SELECT
    id_tipo_veiculo,
    tipo_veiculo
FROM analytics.dim_tipo_veiculo
ORDER BY id_tipo_veiculo;
"""

df_dim_tipo_veiculo = pd.read_sql(
    query_dim_tipo_veiculo,
    engine
)

display(df_dim_tipo_veiculo)

print(f"Total de tipos cadastrados: {len(df_dim_tipo_veiculo)}")

In [ ]:
ddl_dim_especie_veiculo = """
CREATE TABLE analytics.dim_especie_veiculo (
    id_especie_veiculo SMALLINT GENERATED ALWAYS AS IDENTITY,
    especie_veiculo VARCHAR(50) NOT NULL,

    CONSTRAINT pk_dim_especie_veiculo
        PRIMARY KEY (id_especie_veiculo),

    CONSTRAINT uq_dim_especie_veiculo
        UNIQUE (especie_veiculo)
);
"""

with engine.begin() as conn:
    conn.execute(text(ddl_dim_especie_veiculo))

print("Tabela analytics.dim_especie_veiculo criada.")

In [ ]:
sql_carga_dim_especie_veiculo = """
INSERT INTO analytics.dim_especie_veiculo (especie_veiculo)
SELECT DISTINCT especie_veiculo
FROM analytics.fato_frota
ORDER BY especie_veiculo;
"""

with engine.begin() as conn:
    resultado = conn.execute(text(sql_carga_dim_especie_veiculo))

print(f"Registros inseridos: {resultado.rowcount}")

In [ ]:
query_dim_especie_veiculo = """
SELECT
    id_especie_veiculo,
    especie_veiculo
FROM analytics.dim_especie_veiculo
ORDER BY id_especie_veiculo;
"""

df_dim_especie_veiculo = pd.read_sql(
    query_dim_especie_veiculo,
    engine
)

display(df_dim_especie_veiculo)

print(f"Total de espécies cadastradas: {len(df_dim_especie_veiculo)}")

In [ ]:
ddl_adicionar_chaves_frota = """
ALTER TABLE analytics.fato_frota
    ADD COLUMN id_tipo_veiculo SMALLINT,
    ADD COLUMN id_especie_veiculo SMALLINT;
"""

with engine.begin() as conn:
    conn.execute(text(ddl_adicionar_chaves_frota))

print("Colunas id_tipo_veiculo e id_especie_veiculo adicionadas à fato_frota.")

In [ ]:
sql_atualizar_chaves_frota = """
UPDATE analytics.fato_frota AS f
SET id_tipo_veiculo = d.id_tipo_veiculo
FROM analytics.dim_tipo_veiculo AS d
WHERE f.tipo_veiculo = d.tipo_veiculo;

UPDATE analytics.fato_frota AS f
SET id_especie_veiculo = d.id_especie_veiculo
FROM analytics.dim_especie_veiculo AS d
WHERE f.especie_veiculo = d.especie_veiculo;
"""

with engine.begin() as conn:
    conn.execute(text(sql_atualizar_chaves_frota))

print("Atualização das chaves das dimensões concluída.")

In [ ]:
query_auditoria_chaves_frota = """
SELECT
    COUNT(*) AS total_registros,

    COUNT(id_tipo_veiculo) AS tipos_preenchidos,
    COUNT(DISTINCT id_tipo_veiculo) AS tipos_distintos,
    COUNT(*) FILTER (
        WHERE id_tipo_veiculo IS NULL
    ) AS tipos_sem_correspondencia,

    COUNT(id_especie_veiculo) AS especies_preenchidas,
    COUNT(DISTINCT id_especie_veiculo) AS especies_distintas,
    COUNT(*) FILTER (
        WHERE id_especie_veiculo IS NULL
    ) AS especies_sem_correspondencia

FROM analytics.fato_frota;
"""

df_auditoria_chaves_frota = pd.read_sql(
    query_auditoria_chaves_frota,
    engine
)

display(df_auditoria_chaves_frota)

In [ ]:
ddl_restricoes_frota = """
ALTER TABLE analytics.fato_frota
    ALTER COLUMN id_tipo_veiculo SET NOT NULL,
    ALTER COLUMN id_especie_veiculo SET NOT NULL,

    ADD CONSTRAINT fk_fato_frota_tipo_veiculo
        FOREIGN KEY (id_tipo_veiculo)
        REFERENCES analytics.dim_tipo_veiculo (id_tipo_veiculo),

    ADD CONSTRAINT fk_fato_frota_especie_veiculo
        FOREIGN KEY (id_especie_veiculo)
        REFERENCES analytics.dim_especie_veiculo (id_especie_veiculo);
"""

with engine.begin() as conn:
    conn.execute(text(ddl_restricoes_frota))

print("Restrições de integridade aplicadas à fato_frota.")

In [ ]:
query_fks_frota = """
SELECT
    tc.constraint_name,
    kcu.column_name,
    ccu.table_schema AS schema_referenciado,
    ccu.table_name AS tabela_referenciada,
    ccu.column_name AS coluna_referenciada
FROM information_schema.table_constraints AS tc
JOIN information_schema.key_column_usage AS kcu
    ON tc.constraint_name = kcu.constraint_name
    AND tc.table_schema = kcu.table_schema
JOIN information_schema.constraint_column_usage AS ccu
    ON tc.constraint_name = ccu.constraint_name
    AND tc.table_schema = ccu.table_schema
WHERE tc.constraint_type = 'FOREIGN KEY'
  AND tc.table_schema = 'analytics'
  AND tc.table_name = 'fato_frota'
ORDER BY tc.constraint_name;
"""

df_fks_frota = pd.read_sql(
    query_fks_frota,
    engine
)

display(df_fks_frota)

print(f"Total de chaves estrangeiras: {len(df_fks_frota)}")

In [ ]:
query_estrutura_fato_frota = """
SELECT
    column_name,
    data_type,
    character_maximum_length,
    is_nullable
FROM information_schema.columns
WHERE table_schema = 'analytics'
  AND table_name = 'fato_frota'
ORDER BY ordinal_position;
"""

df_estrutura_fato_frota = pd.read_sql(
    query_estrutura_fato_frota,
    engine
)

display(df_estrutura_fato_frota)

In [ ]:
ddl_remover_colunas_redundantes = """
ALTER TABLE analytics.fato_frota
    DROP COLUMN tipo_veiculo,
    DROP COLUMN especie_veiculo;
"""

with engine.begin() as conn:
    conn.execute(text(ddl_remover_colunas_redundantes))

print("Colunas textuais redundantes removidas da fato_frota.")

In [ ]:
query_validacao_fato_frota = """
SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT data_referencia) AS competencias,
    MIN(data_referencia) AS primeira_competencia,
    MAX(data_referencia) AS ultima_competencia,
    SUM(qtd_veiculos) AS total_veiculos,
    COUNT(*) FILTER (
        WHERE id_tipo_veiculo IS NULL
    ) AS tipos_sem_chave,
    COUNT(*) FILTER (
        WHERE id_especie_veiculo IS NULL
    ) AS especies_sem_chave
FROM analytics.fato_frota;
"""

df_validacao_fato_frota = pd.read_sql(
    query_validacao_fato_frota,
    engine
)

display(df_validacao_fato_frota)

In [ ]:
query_chave_primaria = """
SELECT
    tc.constraint_name,
    kcu.column_name,
    kcu.ordinal_position
FROM information_schema.table_constraints tc
JOIN information_schema.key_column_usage kcu
    ON tc.constraint_name = kcu.constraint_name
    AND tc.table_schema = kcu.table_schema
    AND tc.table_name = kcu.table_name
WHERE tc.table_schema = 'analytics'
  AND tc.table_name = 'fato_frota'
  AND tc.constraint_type = 'PRIMARY KEY'
ORDER BY kcu.ordinal_position;
"""

df_chave_primaria = pd.read_sql(
    query_chave_primaria,
    engine
)

display(df_chave_primaria)

In [ ]:
query_auditoria_nova_pk = """
SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT (
        data_referencia,
        id_uf,
        municipio,
        id_tipo_veiculo,
        id_especie_veiculo,
        eixos
    )) AS chaves_distintas
FROM analytics.fato_frota;
"""

df_auditoria_nova_pk = pd.read_sql(
    query_auditoria_nova_pk,
    engine
)

display(df_auditoria_nova_pk)

In [ ]:
ddl_criar_pk_fato_frota = """
ALTER TABLE analytics.fato_frota
ADD CONSTRAINT pk_fato_frota
PRIMARY KEY (
    data_referencia,
    id_uf,
    municipio,
    id_tipo_veiculo,
    id_especie_veiculo,
    eixos
);
"""

with engine.begin() as conn:
    conn.execute(text(ddl_criar_pk_fato_frota))

print("Nova chave primária criada com sucesso.")

In [ ]:
query_auditoria_constraints = """
SELECT
    tc.constraint_name,
    tc.constraint_type,
    kcu.column_name,
    ccu.table_schema AS schema_referenciado,
    ccu.table_name AS tabela_referenciada,
    ccu.column_name AS coluna_referenciada
FROM information_schema.table_constraints tc
LEFT JOIN information_schema.key_column_usage kcu
    ON tc.constraint_name = kcu.constraint_name
    AND tc.table_schema = kcu.table_schema
    AND tc.table_name = kcu.table_name
LEFT JOIN information_schema.constraint_column_usage ccu
    ON tc.constraint_name = ccu.constraint_name
    AND tc.table_schema = ccu.table_schema
    AND tc.constraint_name = ccu.constraint_name
WHERE tc.table_schema = 'analytics'
  AND tc.table_name = 'fato_frota'
  AND tc.constraint_type IN ('PRIMARY KEY', 'FOREIGN KEY')
ORDER BY
    tc.constraint_type,
    tc.constraint_name,
    kcu.ordinal_position;
"""

df_auditoria_constraints = pd.read_sql(
    query_auditoria_constraints,
    engine
)

display(df_auditoria_constraints)

In [ ]:
query_auditoria_constraints = """
SELECT
    tc.constraint_name,
    tc.constraint_type,
    kcu.column_name,
    kcu.ordinal_position,
    ccu.table_schema AS schema_referenciado,
    ccu.table_name AS tabela_referenciada,
    ccu.column_name AS coluna_referenciada
FROM information_schema.table_constraints tc
JOIN information_schema.key_column_usage kcu
    ON tc.constraint_catalog = kcu.constraint_catalog
    AND tc.constraint_schema = kcu.constraint_schema
    AND tc.constraint_name = kcu.constraint_name
    AND tc.table_schema = kcu.table_schema
    AND tc.table_name = kcu.table_name
LEFT JOIN information_schema.constraint_column_usage ccu
    ON kcu.constraint_catalog = ccu.constraint_catalog
    AND kcu.constraint_schema = ccu.constraint_schema
    AND kcu.constraint_name = ccu.constraint_name
    AND kcu.ordinal_position = ccu.position_in_unique_constraint
WHERE tc.table_schema = 'analytics'
  AND tc.table_name = 'fato_frota'
  AND tc.constraint_type IN ('PRIMARY KEY', 'FOREIGN KEY')
ORDER BY
    tc.constraint_type,
    tc.constraint_name,
    kcu.ordinal_position;
"""

df_auditoria_constraints = pd.read_sql(
    query_auditoria_constraints,
    engine
)

display(df_auditoria_constraints)

In [ ]:
query_auditoria_constraints = """
SELECT
    tc.constraint_name,
    tc.constraint_type,
    kcu.column_name,
    kcu.ordinal_position,
    kcu.position_in_unique_constraint
FROM information_schema.table_constraints tc
JOIN information_schema.key_column_usage kcu
    ON tc.constraint_catalog = kcu.constraint_catalog
    AND tc.constraint_schema = kcu.constraint_schema
    AND tc.constraint_name = kcu.constraint_name
    AND tc.table_schema = kcu.table_schema
    AND tc.table_name = kcu.table_name
WHERE tc.table_schema = 'analytics'
  AND tc.table_name = 'fato_frota'
  AND tc.constraint_type IN ('PRIMARY KEY', 'FOREIGN KEY')
ORDER BY
    tc.constraint_type,
    tc.constraint_name,
    kcu.ordinal_position;
"""

df_auditoria_constraints = pd.read_sql(
    query_auditoria_constraints,
    engine
)

display(df_auditoria_constraints)

In [ ]:
query_inventario_modelo = """
SELECT
    table_name,
    ordinal_position,
    column_name,
    data_type,
    character_maximum_length,
    is_nullable
FROM information_schema.columns
WHERE table_schema = 'analytics'
  AND table_name IN (
      'dim_calendario',
      'dim_uf',
      'dim_tipo_veiculo',
      'dim_especie_veiculo',
      'fato_frota',
      'fato_emplacamentos'
  )
ORDER BY
    table_name,
    ordinal_position;
"""

df_inventario_modelo = pd.read_sql(
    query_inventario_modelo,
    engine
)

display(df_inventario_modelo)

In [ ]:
query_restricoes_modelo = """
SELECT
    c.relname AS tabela,
    con.conname AS restricao,
    CASE con.contype
        WHEN 'p' THEN 'PRIMARY KEY'
        WHEN 'f' THEN 'FOREIGN KEY'
        WHEN 'u' THEN 'UNIQUE'
        WHEN 'c' THEN 'CHECK'
        ELSE con.contype::text
    END AS tipo_restricao,
    pg_get_constraintdef(con.oid) AS definicao
FROM pg_constraint con
JOIN pg_class c
    ON c.oid = con.conrelid
JOIN pg_namespace n
    ON n.oid = c.relnamespace
WHERE n.nspname = 'analytics'
  AND c.relname IN (
      'dim_calendario',
      'dim_uf',
      'dim_tipo_veiculo',
      'dim_especie_veiculo',
      'fato_frota',
      'fato_emplacamentos'
  )
ORDER BY
    c.relname,
    con.contype,
    con.conname;
"""

df_restricoes_modelo = pd.read_sql(
    query_restricoes_modelo,
    engine
)

display(df_restricoes_modelo)

In [ ]:
query_contagem_tabelas = """
SELECT 'dim_calendario' AS tabela, COUNT(*) AS registros
FROM analytics.dim_calendario

UNION ALL

SELECT 'dim_uf', COUNT(*)
FROM analytics.dim_uf

UNION ALL

SELECT 'dim_tipo_veiculo', COUNT(*)
FROM analytics.dim_tipo_veiculo

UNION ALL

SELECT 'dim_especie_veiculo', COUNT(*)
FROM analytics.dim_especie_veiculo

UNION ALL

SELECT 'fato_frota', COUNT(*)
FROM analytics.fato_frota

UNION ALL

SELECT 'fato_emplacamentos', COUNT(*)
FROM analytics.fato_emplacamentos

ORDER BY tabela;
"""

df_contagem_tabelas = pd.read_sql(
    query_contagem_tabelas,
    engine
)

display(df_contagem_tabelas)

In [ ]:
query_integridade_temporal = """
WITH auditoria AS (
    SELECT
        'fato_frota' AS tabela,
        COUNT(*) AS total_registros,
        COUNT(DISTINCT f.data_referencia) AS competencias,
        MIN(f.data_referencia) AS primeira_competencia,
        MAX(f.data_referencia) AS ultima_competencia,
        COUNT(*) FILTER (
            WHERE d.competencia IS NULL
        ) AS competencias_sem_calendario
    FROM analytics.fato_frota f
    LEFT JOIN analytics.dim_calendario d
        ON f.data_referencia = d.competencia

    UNION ALL

    SELECT
        'fato_emplacamentos',
        COUNT(*),
        COUNT(DISTINCT f.competencia),
        MIN(f.competencia),
        MAX(f.competencia),
        COUNT(*) FILTER (
            WHERE d.competencia IS NULL
        )
    FROM analytics.fato_emplacamentos f
    LEFT JOIN analytics.dim_calendario d
        ON f.competencia = d.competencia
)
SELECT *
FROM auditoria
ORDER BY tabela;
"""

df_integridade_temporal = pd.read_sql(
    query_integridade_temporal,
    engine
)

display(df_integridade_temporal)

In [ ]:
query_integridade_dimensoes = """
SELECT
    COUNT(*) AS total_registros,

    COUNT(*) FILTER (
        WHERE uf.id_uf IS NULL
    ) AS ufs_sem_correspondencia,

    COUNT(*) FILTER (
        WHERE tipo.id_tipo_veiculo IS NULL
    ) AS tipos_sem_correspondencia,

    COUNT(*) FILTER (
        WHERE especie.id_especie_veiculo IS NULL
    ) AS especies_sem_correspondencia

FROM analytics.fato_frota f

LEFT JOIN analytics.dim_uf uf
    ON f.id_uf = uf.id_uf

LEFT JOIN analytics.dim_tipo_veiculo tipo
    ON f.id_tipo_veiculo = tipo.id_tipo_veiculo

LEFT JOIN analytics.dim_especie_veiculo especie
    ON f.id_especie_veiculo = especie.id_especie_veiculo;
"""

df_integridade_dimensoes = pd.read_sql(
    query_integridade_dimensoes,
    engine
)

display(df_integridade_dimensoes)

In [ ]:
query_diagnostico_uf = """
SELECT
    (SELECT COUNT(*)
     FROM analytics.fato_frota) AS total_fato_frota,

    (SELECT COUNT(*)
     FROM analytics.fato_frota
     WHERE id_uf IS NULL) AS ids_uf_nulos,

    (SELECT COUNT(*)
     FROM analytics.fato_frota f
     WHERE EXISTS (
         SELECT 1
         FROM analytics.dim_uf d
         WHERE d.id_uf = f.id_uf
     )) AS registros_com_uf_correspondente,

    (SELECT COUNT(*)
     FROM analytics.dim_uf) AS total_dim_uf,

    (SELECT COUNT(DISTINCT id_uf)
     FROM analytics.fato_frota) AS ids_uf_distintos;
"""

df_diagnostico_uf = pd.read_sql(
    query_diagnostico_uf,
    engine
)

display(df_diagnostico_uf)

In [ ]:
query_integridade_tipo_especie = """
SELECT
    COUNT(*) AS total_fato_frota,

    COUNT(*) FILTER (
        WHERE f.id_tipo_veiculo IS NULL
    ) AS ids_tipo_nulos,

    COUNT(*) FILTER (
        WHERE tipo.id_tipo_veiculo IS NOT NULL
    ) AS registros_com_tipo_correspondente,

    COUNT(*) FILTER (
        WHERE f.id_especie_veiculo IS NULL
    ) AS ids_especie_nulos,

    COUNT(*) FILTER (
        WHERE especie.id_especie_veiculo IS NOT NULL
    ) AS registros_com_especie_correspondente

FROM analytics.fato_frota f

LEFT JOIN analytics.dim_tipo_veiculo tipo
    ON f.id_tipo_veiculo = tipo.id_tipo_veiculo

LEFT JOIN analytics.dim_especie_veiculo especie
    ON f.id_especie_veiculo = especie.id_especie_veiculo;
"""

df_integridade_tipo_especie = pd.read_sql(
    query_integridade_tipo_especie,
    engine
)

display(df_integridade_tipo_especie)

In [ ]:
query_grao_emplacamentos = """
SELECT
    COUNT(*) AS total_registros,

    COUNT(DISTINCT competencia) AS competencias,

    COUNT(DISTINCT categoria) AS categorias,

    COUNT(DISTINCT (competencia, categoria))
        AS combinacoes_competencia_categoria,

    COUNT(*) - COUNT(DISTINCT (competencia, categoria))
        AS duplicidades,

    COUNT(*) FILTER (
        WHERE categoria = 'Autos + Com. Leves'
    ) AS registros_categoria_consolidada

FROM analytics.fato_emplacamentos;
"""

df_grao_emplacamentos = pd.read_sql(
    query_grao_emplacamentos,
    engine
)

display(df_grao_emplacamentos)

In [ ]:
query_validacao_consolidado = """
WITH categorias AS (
    SELECT
        competencia,

        MAX(emplacamentos_mes)
            FILTER (WHERE categoria = 'Autos')
            AS autos,

        MAX(emplacamentos_mes)
            FILTER (WHERE categoria = 'Com. Leves')
            AS comerciais_leves,

        MAX(emplacamentos_mes)
            FILTER (WHERE categoria = 'Autos + Com. Leves')
            AS consolidado

    FROM analytics.fato_emplacamentos
    GROUP BY competencia
)
SELECT
    COUNT(*) AS competencias,

    COUNT(*) FILTER (
        WHERE consolidado = autos + comerciais_leves
    ) AS consolidados_corretos,

    COUNT(*) FILTER (
        WHERE consolidado <> autos + comerciais_leves
    ) AS divergencias,

    SUM(ABS(consolidado - autos - comerciais_leves))
        AS diferenca_absoluta_total

FROM categorias;
"""

df_validacao_consolidado = pd.read_sql(
    query_validacao_consolidado,
    engine
)

display(df_validacao_consolidado)

In [ ]:
query_relacionamentos = """
SELECT
    tc.table_name AS tabela_fato,
    kcu.column_name AS coluna_origem,
    ccu.table_name AS tabela_dimensao,
    ccu.column_name AS coluna_destino,
    tc.constraint_name AS nome_relacionamento

FROM information_schema.table_constraints tc

JOIN information_schema.key_column_usage kcu
    ON tc.constraint_schema = kcu.constraint_schema
    AND tc.constraint_name = kcu.constraint_name
    AND tc.table_name = kcu.table_name

JOIN information_schema.constraint_column_usage ccu
    ON tc.constraint_schema = ccu.constraint_schema
    AND tc.constraint_name = ccu.constraint_name
    AND kcu.ordinal_position = ccu.ordinal_position

WHERE
    tc.constraint_type = 'FOREIGN KEY'
    AND tc.constraint_schema = 'analytics'

ORDER BY
    tc.table_name,
    tc.constraint_name;
"""

df_relacionamentos = pd.read_sql(
    query_relacionamentos,
    engine
)

display(df_relacionamentos)

In [ ]:
query_relacionamentos = """
SELECT
    tc.table_name AS tabela_fato,
    kcu.column_name AS coluna_origem,
    ccu.table_name AS tabela_dimensao,
    ccu.column_name AS coluna_destino,
    tc.constraint_name AS nome_relacionamento

FROM information_schema.table_constraints tc

JOIN information_schema.key_column_usage kcu
    ON tc.constraint_schema = kcu.constraint_schema
    AND tc.constraint_name = kcu.constraint_name
    AND tc.table_name = kcu.table_name

JOIN information_schema.constraint_column_usage ccu
    ON tc.constraint_schema = ccu.constraint_schema
    AND tc.constraint_name = ccu.constraint_name
    AND tc.table_name = ccu.table_name

WHERE
    tc.constraint_type = 'FOREIGN KEY'
    AND tc.constraint_schema = 'analytics'

ORDER BY
    tc.table_name,
    tc.constraint_name;
"""

df_relacionamentos = pd.read_sql(
    query_relacionamentos,
    engine
)

display(df_relacionamentos)

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

# Identificar a raiz do projeto
PROJECT_ROOT = Path.cwd().parent

# Disponibilizar os módulos do projeto
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Reconstruir a conexão com PostgreSQL
from src.database import criar_engine

engine = criar_engine()

# Testar a conexão
from sqlalchemy import text

with engine.connect() as conn:
    resultado = conn.execute(
        text("SELECT current_database() AS banco, current_schema() AS schema_atual")
    ).fetchone()

print(f"Diretório do projeto: {PROJECT_ROOT}")
print(f"Banco conectado: {resultado.banco}")
print(f"Schema atual: {resultado.schema_atual}")
print(f"Versão do pandas: {pd.__version__}")

In [ ]:
query_cardinalidade = """
SELECT
    'dim_calendario' AS dimensao,
    COUNT(*) AS total_registros,
    COUNT(DISTINCT competencia) AS chaves_distintas,
    COUNT(*) - COUNT(DISTINCT competencia) AS duplicidades
FROM analytics.dim_calendario

UNION ALL

SELECT
    'dim_uf',
    COUNT(*),
    COUNT(DISTINCT id_uf),
    COUNT(*) - COUNT(DISTINCT id_uf)
FROM analytics.dim_uf

UNION ALL

SELECT
    'dim_tipo_veiculo',
    COUNT(*),
    COUNT(DISTINCT id_tipo_veiculo),
    COUNT(*) - COUNT(DISTINCT id_tipo_veiculo)
FROM analytics.dim_tipo_veiculo

UNION ALL

SELECT
    'dim_especie_veiculo',
    COUNT(*),
    COUNT(DISTINCT id_especie_veiculo),
    COUNT(*) - COUNT(DISTINCT id_especie_veiculo)
FROM analytics.dim_especie_veiculo

ORDER BY dimensao;
"""

df_cardinalidade = pd.read_sql(
    query_cardinalidade,
    engine
)

display(df_cardinalidade)

In [ ]:
query_cardinalidade_fatos = """
SELECT
    'fato_frota → dim_calendario' AS relacionamento,
    COUNT(*) AS total_registros,
    COUNT(DISTINCT data_referencia) AS chaves_distintas,
    MAX(frequencia) AS maximo_ocorrencias_por_chave
FROM (
    SELECT
        data_referencia,
        COUNT(*) AS frequencia
    FROM analytics.fato_frota
    GROUP BY data_referencia
) x
CROSS JOIN (
    SELECT COUNT(*) AS total
    FROM analytics.fato_frota
) t

UNION ALL

SELECT
    'fato_frota → dim_uf',
    (SELECT COUNT(*) FROM analytics.fato_frota),
    COUNT(DISTINCT id_uf),
    MAX(frequencia)
FROM (
    SELECT id_uf, COUNT(*) AS frequencia
    FROM analytics.fato_frota
    GROUP BY id_uf
) x

UNION ALL

SELECT
    'fato_frota → dim_tipo_veiculo',
    (SELECT COUNT(*) FROM analytics.fato_frota),
    COUNT(DISTINCT id_tipo_veiculo),
    MAX(frequencia)
FROM (
    SELECT id_tipo_veiculo, COUNT(*) AS frequencia
    FROM analytics.fato_frota
    GROUP BY id_tipo_veiculo
) x

UNION ALL

SELECT
    'fato_frota → dim_especie_veiculo',
    (SELECT COUNT(*) FROM analytics.fato_frota),
    COUNT(DISTINCT id_especie_veiculo),
    MAX(frequencia)
FROM (
    SELECT id_especie_veiculo, COUNT(*) AS frequencia
    FROM analytics.fato_frota
    GROUP BY id_especie_veiculo
) x

UNION ALL

SELECT
    'fato_emplacamentos → dim_calendario',
    COUNT(*),
    COUNT(DISTINCT competencia),
    MAX(frequencia)
FROM (
    SELECT competencia, COUNT(*) AS frequencia
    FROM analytics.fato_emplacamentos
    GROUP BY competencia
) x;
"""

df_cardinalidade_fatos = pd.read_sql(
    query_cardinalidade_fatos,
    engine
)

display(df_cardinalidade_fatos)

In [ ]:
query_chaves_primarias = """
SELECT
    tc.table_name AS tabela,
    tc.constraint_name AS nome_chave_primaria,
    string_agg(
        kcu.column_name,
        ', ' ORDER BY kcu.ordinal_position
    ) AS colunas_chave_primaria

FROM information_schema.table_constraints tc

JOIN information_schema.key_column_usage kcu
    ON tc.constraint_schema = kcu.constraint_schema
    AND tc.constraint_name = kcu.constraint_name
    AND tc.table_name = kcu.table_name

WHERE
    tc.constraint_type = 'PRIMARY KEY'
    AND tc.constraint_schema = 'analytics'
    AND tc.table_name IN (
        'fato_frota',
        'fato_emplacamentos'
    )

GROUP BY
    tc.table_name,
    tc.constraint_name

ORDER BY tc.table_name;
"""

df_chaves_primarias = pd.read_sql(
    query_chaves_primarias,
    engine
)

display(df_chaves_primarias)

In [ ]:
query_inventario_colunas = """
SELECT
    table_name AS tabela,
    ordinal_position AS posicao,
    column_name AS coluna,
    data_type AS tipo_dado,
    character_maximum_length AS tamanho_maximo,
    numeric_precision AS precisao_numerica,
    numeric_scale AS escala_numerica,
    is_nullable AS aceita_nulo,
    column_default AS valor_padrao

FROM information_schema.columns

WHERE
    table_schema = 'analytics'
    AND table_name IN (
        'dim_calendario',
        'dim_uf',
        'dim_tipo_veiculo',
        'dim_especie_veiculo',
        'fato_frota',
        'fato_emplacamentos'
    )

ORDER BY
    CASE table_name
        WHEN 'dim_calendario' THEN 1
        WHEN 'dim_uf' THEN 2
        WHEN 'dim_tipo_veiculo' THEN 3
        WHEN 'dim_especie_veiculo' THEN 4
        WHEN 'fato_frota' THEN 5
        WHEN 'fato_emplacamentos' THEN 6
    END,
    ordinal_position;
"""

df_inventario_colunas = pd.read_sql(
    query_inventario_colunas,
    engine
)

display(df_inventario_colunas)

In [ ]:
query_inventario_restricoes = """
SELECT
    con.conrelid::regclass::text AS tabela,
    con.conname AS nome_restricao,

    CASE con.contype
        WHEN 'p' THEN 'PRIMARY KEY'
        WHEN 'f' THEN 'FOREIGN KEY'
        WHEN 'u' THEN 'UNIQUE'
        WHEN 'c' THEN 'CHECK'
        ELSE con.contype::text
    END AS tipo_restricao,

    pg_get_constraintdef(con.oid) AS definicao

FROM pg_constraint con

JOIN pg_namespace ns
    ON ns.oid = con.connamespace

WHERE
    ns.nspname = 'analytics'
    AND con.conrelid::regclass::text IN (
        'analytics.dim_calendario',
        'analytics.dim_uf',
        'analytics.dim_tipo_veiculo',
        'analytics.dim_especie_veiculo',
        'analytics.fato_frota',
        'analytics.fato_emplacamentos'
    )

ORDER BY
    tabela,
    tipo_restricao,
    nome_restricao;
"""

df_inventario_restricoes = pd.read_sql(
    query_inventario_restricoes,
    engine
)

display(df_inventario_restricoes)

In [ ]:
query_inventario_restricoes = """
SELECT
    tabela.relname AS tabela,
    con.conname AS nome_restricao,

    CASE con.contype
        WHEN 'p' THEN 'PRIMARY KEY'
        WHEN 'f' THEN 'FOREIGN KEY'
        WHEN 'u' THEN 'UNIQUE'
        WHEN 'c' THEN 'CHECK'
        ELSE con.contype::text
    END AS tipo_restricao,

    pg_get_constraintdef(con.oid) AS definicao

FROM pg_constraint con

JOIN pg_class tabela
    ON tabela.oid = con.conrelid

JOIN pg_namespace esquema
    ON esquema.oid = tabela.relnamespace

WHERE
    esquema.nspname = 'analytics'
    AND tabela.relname IN (
        'dim_calendario',
        'dim_uf',
        'dim_tipo_veiculo',
        'dim_especie_veiculo',
        'fato_frota',
        'fato_emplacamentos'
    )

ORDER BY
    tabela.relname,
    tipo_restricao,
    con.conname;
"""

df_inventario_restricoes = pd.read_sql(
    query_inventario_restricoes,
    engine
)

display(df_inventario_restricoes)

In [ ]:
query_chaves_modelo = """
SELECT
    tabela.relname AS tabela,
    con.conname AS nome_restricao,

    CASE con.contype
        WHEN 'p' THEN 'PK'
        WHEN 'f' THEN 'FK'
    END AS tipo,

    string_agg(
        origem.attname,
        ', ' ORDER BY origem_coluna.ord
    ) AS colunas_origem,

    CASE
        WHEN con.contype = 'f'
        THEN tabela_destino.relname
    END AS tabela_referenciada,

    CASE
        WHEN con.contype = 'f'
        THEN string_agg(
            destino.attname,
            ', ' ORDER BY destino_coluna.ord
        )
    END AS colunas_referenciadas

FROM pg_constraint con

JOIN pg_class tabela
    ON tabela.oid = con.conrelid

JOIN pg_namespace esquema
    ON esquema.oid = tabela.relnamespace

JOIN pg_class tabela_destino
    ON tabela_destino.oid = con.confrelid

CROSS JOIN LATERAL
    unnest(con.conkey) WITH ORDINALITY
    AS origem_coluna(attnum, ord)

JOIN pg_attribute origem
    ON origem.attrelid = tabela.oid
    AND origem.attnum = origem_coluna.attnum

LEFT JOIN LATERAL
    unnest(con.confkey) WITH ORDINALITY
    AS destino_coluna(attnum, ord)
    ON destino_coluna.ord = origem_coluna.ord

LEFT JOIN pg_attribute destino
    ON destino.attrelid = tabela_destino.oid
    AND destino.attnum = destino_coluna.attnum

WHERE
    esquema.nspname = 'analytics'
    AND con.contype IN ('p', 'f')

GROUP BY
    tabela.relname,
    con.conname,
    con.contype,
    tabela_destino.relname

ORDER BY
    tabela.relname,
    tipo,
    con.conname;
"""

df_chaves_modelo = pd.read_sql(
    query_chaves_modelo,
    engine
)

display(df_chaves_modelo)

In [ ]:
query_chaves_modelo = """
SELECT
    tabela.relname AS tabela,
    con.conname AS nome_restricao,

    CASE con.contype
        WHEN 'p' THEN 'PK'
        WHEN 'f' THEN 'FK'
    END AS tipo,

    string_agg(
        origem.attname,
        ', ' ORDER BY origem_coluna.ord
    ) AS colunas_origem,

    tabela_destino.relname AS tabela_referenciada,

    string_agg(
        destino.attname,
        ', ' ORDER BY origem_coluna.ord
    ) FILTER (WHERE destino.attname IS NOT NULL)
        AS colunas_referenciadas

FROM pg_constraint con

JOIN pg_class tabela
    ON tabela.oid = con.conrelid

JOIN pg_namespace esquema
    ON esquema.oid = tabela.relnamespace

LEFT JOIN pg_class tabela_destino
    ON tabela_destino.oid = con.confrelid

CROSS JOIN LATERAL
    unnest(con.conkey) WITH ORDINALITY
    AS origem_coluna(attnum, ord)

JOIN pg_attribute origem
    ON origem.attrelid = tabela.oid
    AND origem.attnum = origem_coluna.attnum

LEFT JOIN LATERAL
    unnest(con.confkey) WITH ORDINALITY
    AS destino_coluna(attnum, ord)
    ON destino_coluna.ord = origem_coluna.ord

LEFT JOIN pg_attribute destino
    ON destino.attrelid = tabela_destino.oid
    AND destino.attnum = destino_coluna.attnum

WHERE
    esquema.nspname = 'analytics'
    AND con.contype IN ('p', 'f')

GROUP BY
    tabela.relname,
    con.conname,
    con.contype,
    tabela_destino.relname

ORDER BY
    tabela.relname,
    tipo,
    con.conname;
"""

df_chaves_modelo = pd.read_sql(
    query_chaves_modelo,
    engine
)

display(df_chaves_modelo)

In [ ]:
from pathlib import Path

# Diretório de documentação
diretorio_docs = PROJECT_ROOT / "docs" / "arquitetura"
diretorio_docs.mkdir(parents=True, exist_ok=True)

# Identifica colunas que fazem parte de PKs e FKs
chaves = {}

for _, linha in df_chaves_modelo.iterrows():
    tabela = linha["tabela"]
    tipo = linha["tipo"]

    colunas = [
        coluna.strip()
        for coluna in linha["colunas_origem"].split(",")
    ]

    for coluna in colunas:
        chave = (tabela, coluna)
        chaves.setdefault(chave, set()).add(tipo)

# Converte os tipos do PostgreSQL para representação no Mermaid
def formatar_tipo(linha):
    tipo = linha["tipo_dado"]

    if tipo in ("character varying", "character"):
        return f"varchar({int(linha['tamanho_maximo'])})"

    if tipo == "numeric":
        return "numeric"

    if tipo == "timestamp without time zone":
        return "timestamp"

    if tipo == "USER-DEFINED":
        return "text"

    return tipo

# Monta o diagrama
linhas = [
    "erDiagram",
    ""
]

ordem_tabelas = [
    "dim_calendario",
    "dim_uf",
    "dim_tipo_veiculo",
    "dim_especie_veiculo",
    "fato_frota",
    "fato_emplacamentos"
]

for tabela in ordem_tabelas:
    linhas.append(f"    {tabela} {{")

    colunas_tabela = df_inventario_colunas[
        df_inventario_colunas["tabela"] == tabela
    ]

    for _, coluna in colunas_tabela.iterrows():
        nome_coluna = coluna["coluna"]
        tipo = formatar_tipo(coluna)

        marcadores = chaves.get((tabela, nome_coluna), set())
        anotacoes = " ".join(sorted(marcadores))

        descricao = f"        {tipo} {nome_coluna}"

        if anotacoes:
            descricao += f" {anotacoes}"

        linhas.append(descricao)

    linhas.append("    }")
    linhas.append("")

# Relacionamentos: dimensão 1 para muitos registros na fato
relacionamentos = [
    '    dim_calendario ||--o{ fato_frota : "competencia"',
    '    dim_uf ||--o{ fato_frota : "UF"',
    '    dim_tipo_veiculo ||--o{ fato_frota : "tipo"',
    '    dim_especie_veiculo ||--o{ fato_frota : "especie"',
    '    dim_calendario ||--o{ fato_emplacamentos : "competencia"'
]

linhas.extend(relacionamentos)

# Salva o arquivo
arquivo_diagrama = diretorio_docs / "diagrama_modelo.mmd"
arquivo_diagrama.write_text(
    "\n".join(linhas) + "\n",
    encoding="utf-8"
)

print(f"Arquivo criado: {arquivo_diagrama}")
print(f"Total de entidades: {len(ordem_tabelas)}")
print(f"Total de relacionamentos: {len(relacionamentos)}")
print("\nPrévia do diagrama:\n")
print(arquivo_diagrama.read_text(encoding="utf-8"))

In [ ]:
conteudo_diagrama = arquivo_diagrama.read_text(encoding="utf-8")

conteudo_diagrama = conteudo_diagrama.replace(
    'dim_calendario ||--o{ fato_frota : "competencia"',
    'dim_calendario ||--o{ fato_frota : "data_referencia"'
)

arquivo_diagrama.write_text(
    conteudo_diagrama,
    encoding="utf-8"
)

print("Relacionamento atualizado:")
print(
    'dim_calendario ||--o{ fato_frota : "data_referencia"'
)

print("\nArquivo salvo:", arquivo_diagrama)

In [ ]:
conteudo_documentacao = r"""
# Modelo Dimensional — Inteligência Comercial do Mercado Automotivo Brasileiro

## 1. Objetivo

Este documento descreve a estrutura dimensional implementada no schema
`analytics` do banco PostgreSQL `projeto_automotivo`.

O modelo integra duas fontes oficiais:

- **SENATRAN:** informações mensais sobre o estoque de veículos registrados.
- **FENABRAVE:** informações mensais sobre o fluxo de emplacamentos.

A estrutura foi projetada para permitir análises comerciais do mercado
brasileiro de automóveis e comerciais leves, com posterior consumo no Power BI.

## 2. Escopo analítico

### 2.1 Categorias de veículos

Na análise principal, o escopo contempla:

| Fonte | Categorias incluídas |
|---|---|
| SENATRAN | AUTOMOVEL, CAMINHONETE, CAMIONETA e UTILITARIO |
| FENABRAVE | Autos, Com. Leves e Autos + Com. Leves |

Motocicletas, caminhões, ônibus, implementos e demais categorias fora
do escopo não integram a análise principal.

Na FENABRAVE, `Autos + Com. Leves` é uma categoria consolidada publicada
pela fonte. Para agregados nacionais, deve-se utilizar essa categoria,
sem somá-la novamente às categorias componentes.

### 2.2 Período

O período integrado compreende janeiro de 2025 a setembro de 2026,
totalizando 21 competências mensais.

Comparações interanuais devem considerar somente meses equivalentes
entre 2025 e 2026.

### 2.3 Abrangência geográfica

A análise geográfica da primeira versão utiliza Unidade Federativa (UF)
e região.

O município permanece armazenado na tabela fato da SENATRAN para
preservar a granularidade e a rastreabilidade da fonte, mas não será
utilizado como dimensão de análise na primeira versão do Power BI.

## 3. Arquitetura

O schema `analytics` contém seis tabelas:

### Dimensões

| Tabela | Finalidade | Registros |
|---|---|---:|
| dim_calendario | Competências mensais e atributos temporais | 21 |
| dim_uf | Unidades Federativas e regiões | 28 |
| dim_tipo_veiculo | Tipos de veículos SENATRAN | 4 |
| dim_especie_veiculo | Espécies de veículos SENATRAN | 8 |

A dimensão `dim_uf` contém 27 UFs e o registro SI (Sem Informação).

### Tabelas fato

| Tabela | Finalidade | Registros |
|---|---|---:|
| fato_frota | Estoque mensal de veículos registrados | 1.998.557 |
| fato_emplacamentos | Emplacamentos mensais publicados pela FENABRAVE | 63 |

## 4. Granularidade das tabelas fato

### 4.1 fato_frota

Representa o estoque de veículos registrados por competência,
UF, município, tipo de veículo, espécie e quantidade de eixos.

Sua granularidade física é:

**Uma linha por combinação de:**

- data_referencia;
- id_uf;
- municipio;
- id_tipo_veiculo;
- id_especie_veiculo;
- eixos.

A quantidade de veículos é armazenada em `qtd_veiculos`, do tipo `bigint`.

A chave primária composta é:

```sql
PRIMARY KEY (
    data_referencia,
    id_uf,
    municipio,
    id_tipo_veiculo,
    id_especie_veiculo,
    eixos
)

In [ ]:
arquivo_documentacao = (
    PROJECT_ROOT / "docs" / "arquitetura" / "modelo_dimensional.md"
)

print("Arquivo existe:", arquivo_documentacao.exists())

In [ ]:
conteudo_documentacao = """# Modelo Dimensional — Inteligência Comercial do Mercado Automotivo Brasileiro

## 1. Objetivo

Este documento descreve a estrutura dimensional implementada no schema
`analytics` do banco PostgreSQL `projeto_automotivo`.

O modelo integra duas fontes oficiais:

- **SENATRAN:** informações mensais sobre o estoque de veículos registrados.
- **FENABRAVE:** informações mensais sobre o fluxo de emplacamentos.

A estrutura foi projetada para permitir análises comerciais do mercado
brasileiro de automóveis e comerciais leves, com posterior consumo no Power BI.

## 2. Escopo analítico

### 2.1 Categorias de veículos

Na análise principal, o escopo contempla:

| Fonte | Categorias incluídas |
|---|---|
| SENATRAN | AUTOMOVEL, CAMINHONETE, CAMIONETA e UTILITARIO |
| FENABRAVE | Autos, Com. Leves e Autos + Com. Leves |

Motocicletas, caminhões, ônibus, implementos e demais categorias fora
do escopo não integram a análise principal.

Na FENABRAVE, `Autos + Com. Leves` é uma categoria consolidada publicada
pela fonte. Para agregados nacionais, deve-se utilizar essa categoria,
sem somá-la novamente às categorias componentes.

### 2.2 Período

O período integrado compreende janeiro de 2025 a setembro de 2026,
totalizando 21 competências mensais.

Comparações interanuais devem considerar somente meses equivalentes
entre 2025 e 2026.

### 2.3 Abrangência geográfica

A análise geográfica da primeira versão utiliza Unidade Federativa (UF)
e região.

O município permanece armazenado na tabela fato da SENATRAN para
preservar a granularidade e a rastreabilidade da fonte, mas não será
utilizado como dimensão de análise na primeira versão do Power BI.
"""

diretorio_docs = PROJECT_ROOT / "docs" / "arquitetura"
diretorio_docs.mkdir(parents=True, exist_ok=True)

arquivo_documentacao = diretorio_docs / "modelo_dimensional.md"

arquivo_documentacao.write_text(
    conteudo_documentacao,
    encoding="utf-8"
)

print("Documento inicial criado.")
print("Arquivo:", arquivo_documentacao)
print("Linhas:", len(conteudo_documentacao.splitlines()))

In [ ]:
complemento_documentacao = """

## 3. Arquitetura

O schema `analytics` contém seis tabelas:

### 3.1 Dimensões

| Tabela | Finalidade | Registros |
|---|---|---:|
| dim_calendario | Competências mensais e atributos temporais | 21 |
| dim_uf | Unidades Federativas e regiões | 28 |
| dim_tipo_veiculo | Tipos de veículos SENATRAN | 4 |
| dim_especie_veiculo | Espécies de veículos SENATRAN | 8 |

A dimensão `dim_uf` contém 27 UFs e o registro SI (Sem Informação).

### 3.2 Tabelas fato

| Tabela | Finalidade | Registros |
|---|---|---:|
| fato_frota | Estoque mensal de veículos registrados | 1.998.557 |
| fato_emplacamentos | Emplacamentos mensais publicados pela FENABRAVE | 63 |

## 4. Granularidade das tabelas fato

### 4.1 fato_frota

Representa o estoque de veículos registrados por competência,
UF, município, tipo de veículo, espécie e quantidade de eixos.

Sua granularidade física é uma linha por combinação de:

- data_referencia;
- id_uf;
- municipio;
- id_tipo_veiculo;
- id_especie_veiculo;
- eixos.

A quantidade de veículos é armazenada em `qtd_veiculos`, do tipo `bigint`.

A chave primária composta é:

```sql
PRIMARY KEY (
    data_referencia,
    id_uf,
    municipio,
    id_tipo_veiculo,
    id_especie_veiculo,
    eixos
)

In [ ]:
linhas_complemento = [
    "",
    "## 3. Arquitetura",
    "",
    "O schema `analytics` contém seis tabelas:",
    "",
    "### 3.1 Dimensões",
    "",
    "| Tabela | Finalidade | Registros |",
    "|---|---|---:|",
    "| dim_calendario | Competências mensais e atributos temporais | 21 |",
    "| dim_uf | Unidades Federativas e regiões | 28 |",
    "| dim_tipo_veiculo | Tipos de veículos SENATRAN | 4 |",
    "| dim_especie_veiculo | Espécies de veículos SENATRAN | 8 |",
    "",
    "A dimensão `dim_uf` contém 27 UFs e o registro SI (Sem Informação).",
    "",
    "### 3.2 Tabelas fato",
    "",
    "| Tabela | Finalidade | Registros |",
    "|---|---|---:|",
    "| fato_frota | Estoque mensal de veículos registrados | 1.998.557 |",
    "| fato_emplacamentos | Emplacamentos mensais publicados pela FENABRAVE | 63 |",
    "",
    "## 4. Granularidade das tabelas fato",
    "",
    "### 4.1 fato_frota",
    "",
    "Representa o estoque de veículos registrados por competência, UF, município, tipo de veículo e espécie.",
    "",
    "Sua granularidade física é uma linha por combinação de:",
    "",
    "- data_referencia;",
    "- id_uf;",
    "- municipio;",
    "- id_tipo_veiculo;",
    "- id_especie_veiculo;",
    "- eixos.",
    "",
    "A quantidade de veículos é armazenada em `qtd_veiculos`, do tipo `bigint`.",
    "",
    "Chave primária composta:",
    "",
    "    PRIMARY KEY (data_referencia, id_uf, municipio, id_tipo_veiculo, id_especie_veiculo, eixos)",
    "",
    "A medida representa estoque na data de referência mensal. Variações entre competências não devem ser denominadas emplacamentos.",
    "",
    "### 4.2 fato_emplacamentos",
    "",
    "Representa os volumes mensais de emplacamentos publicados pela FENABRAVE, por competência e categoria.",
    "",
    "Sua granularidade é uma linha por combinação de competência e categoria.",
    "",
    "Chave primária composta:",
    "",
    "    PRIMARY KEY (competencia, categoria)",
    "",
    "A tabela mantém os valores mensais, valores de comparação e percentuais publicados pela fonte:",
    "",
    "- emplacamentos_mes;",
    "- emplacamentos_mes_anterior;",
    "- emplacamentos_acumulados;",
    "- emplacamentos_mes_ano_anterior;",
    "- emplacamentos_acumulados_ano_anterior;",
    "- variacao_mensal_pct;",
    "- variacao_anual_pct;",
    "- variacao_acumulada_pct.",
    "",
    "Os valores acumulados e os valores de comparação publicados são preservados conforme a fonte, sem forçar reconciliação com a soma da série mensal."
]

with arquivo_documentacao.open("a", encoding="utf-8") as arquivo:
    arquivo.write("\n".join(linhas_complemento) + "\n")

print("Seções 3 e 4 acrescentadas.")
print("Total de linhas:", len(arquivo_documentacao.read_text(encoding="utf-8").splitlines()))

In [ ]:
linhas_complemento = [
    "",
    "## 5. Chaves e relacionamentos",
    "",
    "O modelo possui seis chaves primárias e cinco chaves estrangeiras.",
    "",
    "### 5.1 Chaves primárias",
    "",
    "| Tabela | Colunas |",
    "|---|---|",
    "| dim_calendario | competencia |",
    "| dim_uf | id_uf |",
    "| dim_tipo_veiculo | id_tipo_veiculo |",
    "| dim_especie_veiculo | id_especie_veiculo |",
    "| fato_frota | data_referencia, id_uf, municipio, id_tipo_veiculo, id_especie_veiculo, eixos |",
    "| fato_emplacamentos | competencia, categoria |",
    "",
    "### 5.2 Chaves estrangeiras",
    "",
    "| Tabela fato | Coluna de origem | Dimensão | Coluna referenciada |",
    "|---|---|---|---|",
    "| fato_frota | data_referencia | dim_calendario | competencia |",
    "| fato_frota | id_uf | dim_uf | id_uf |",
    "| fato_frota | id_tipo_veiculo | dim_tipo_veiculo | id_tipo_veiculo |",
    "| fato_frota | id_especie_veiculo | dim_especie_veiculo | id_especie_veiculo |",
    "| fato_emplacamentos | competencia | dim_calendario | competencia |",
    "",
    "### 5.3 Cardinalidade e direção de filtro",
    "",
    "Todos os relacionamentos são do tipo um-para-muitos (1:*).",
    "",
    "A dimensão ocupa o lado 1 e a tabela fato ocupa o lado muitos.",
    "",
    "A direção de filtro recomendada no Power BI é unidirecional, da dimensão para a tabela fato.",
    "",
    "Não devem ser criados relacionamentos diretos entre as duas tabelas fato.",
    "",
    "## 6. Regras de negócio e interpretação",
    "",
    "### 6.1 Estoque e fluxo",
    "",
    "- SENATRAN representa estoque de veículos registrados em uma data de referência mensal.",
    "- FENABRAVE representa fluxo de emplacamentos durante um período.",
    "- Variação da frota não equivale a volume de emplacamentos.",
    "- As duas fontes devem permanecer em tabelas fato separadas.",
    "",
    "### 6.2 Comparação interanual",
    "",
    "As comparações YoY devem utilizar competências equivalentes: janeiro de 2025 com janeiro de 2026, fevereiro com fevereiro e assim sucessivamente, até setembro.",
    "",
    "### 6.3 Categoria consolidada FENABRAVE",
    "",
    "`Autos + Com. Leves` é o consolidado publicado pela FENABRAVE.",
    "",
    "Para indicadores nacionais consolidados, utilizar diretamente essa categoria. Não somar o consolidado novamente a Autos e Com. Leves.",
    "",
    "### 6.4 Município",
    "",
    "O atributo `municipio` é mantido fisicamente em `fato_frota`, mas fica fora da experiência analítica da primeira versão.",
    "",
    "Não será criada uma dimensão municipal nesta etapa.",
    "",
    "### 6.5 Eixos",
    "",
    "O atributo `eixos` é mantido na granularidade física da fato_frota para preservar a informação original.",
    "",
    "Não constitui dimensão analítica prioritária da primeira versão."
]

with arquivo_documentacao.open("a", encoding="utf-8") as arquivo:
    arquivo.write("\n".join(linhas_complemento) + "\n")

print("Seções 5 e 6 acrescentadas.")
print("Total de linhas:", len(arquivo_documentacao.read_text(encoding="utf-8").splitlines()))

In [ ]:
linhas_complemento = [
    "",
    "## 7. Integridade e qualidade dos dados",
    "",
    "As verificações realizadas no banco confirmaram:",
    "",
    "- 21 competências na dimensão calendário.",
    "- 1.998.557 registros na fato_frota.",
    "- 63 registros na fato_emplacamentos.",
    "- Ausência de chaves primárias duplicadas.",
    "- Ausência de valores nulos nas colunas obrigatórias.",
    "- Ausência de registros órfãos nas relações verificadas.",
    "- Correspondência de todas as competências das tabelas fato com a dimensão calendário.",
    "- Correspondência dos identificadores de UF, tipo e espécie com suas respectivas dimensões.",
    "- 21 de 21 consolidações mensais da FENABRAVE validadas pela soma das categorias componentes.",
    "- 63 de 63 percentuais publicados da FENABRAVE conferidos por recálculo, dentro da tolerância de 0,02 ponto percentual.",
    "",
    "As restrições físicas incluem chaves primárias, estrangeiras, unicidade, obrigatoriedade e regras CHECK.",
    "",
    "Uma anomalia identificada no atributo `eixos` em fevereiro de 2026 foi documentada e preservada, sem correção arbitrária dos dados de origem.",
    "",
    "## 8. Orientações para o modelo semântico Power BI",
    "",
    "Na construção do modelo semântico:",
    "",
    "1. Manter as duas tabelas fato separadas.",
    "2. Relacionar cada fato às dimensões correspondentes.",
    "3. Utilizar direção de filtro unidirecional, dimensão para fato.",
    "4. Ocultar o campo `municipio` na primeira versão.",
    "5. Não disponibilizar `eixos` como atributo prioritário de análise.",
    "6. Utilizar a categoria consolidada da FENABRAVE nos indicadores nacionais de Autos + Com. Leves.",
    "7. Diferenciar claramente medidas de estoque e medidas de fluxo.",
    "8. Utilizar a dimensão calendário para filtros e comparações temporais."
]

with arquivo_documentacao.open("a", encoding="utf-8") as arquivo:
    arquivo.write("\n".join(linhas_complemento) + "\n")

print("Seções 7 e 8 acrescentadas.")
print("Total de linhas:", len(arquivo_documentacao.read_text(encoding="utf-8").splitlines()))

In [ ]:
linhas_complemento = [
    "",
    "## 9. Evolução futura",
    "",
    "Possíveis evoluções, condicionadas à necessidade analítica e à disponibilidade de dados:",
    "",
    "- Inclusão de uma dimensão municipal, caso a análise municipal passe a fazer parte do escopo.",
    "- Ampliação do histórico de competências.",
    "- Inclusão de novas fontes oficiais.",
    "- Expansão das análises comerciais e indicadores derivados.",
    "- Revisão da camada semântica conforme novas necessidades de negócio.",
    "",
    "## 10. Artefatos relacionados",
    "",
    "- `diagrama_modelo.mmd`: representação Mermaid do modelo físico.",
    "- `modelo_dimensional.md`: documentação técnica do modelo dimensional."
]

with arquivo_documentacao.open("a", encoding="utf-8") as arquivo:
    arquivo.write("\n".join(linhas_complemento) + "\n")

print("Seções 9 e 10 acrescentadas.")
print("Total de linhas:", len(arquivo_documentacao.read_text(encoding="utf-8").splitlines()))
print("Tamanho:", f"{arquivo_documentacao.stat().st_size:,} bytes")

In [ ]:
from collections import Counter

arquivos = {
    "Documentação": arquivo_documentacao,
    "Diagrama": diretorio_docs / "diagrama_modelo.mmd"
}

for nome, caminho in arquivos.items():
    print(f"\n{'=' * 70}")
    print(nome)
    print(f"{'=' * 70}")
    print("Existe:", caminho.exists())

    if caminho.exists():
        conteudo = caminho.read_text(encoding="utf-8")
        linhas = conteudo.splitlines()

        titulos = [
            linha for linha in linhas
            if linha.startswith("#")
        ]

        quantidade_blocos = sum(
            1 for linha in linhas
            if linha.strip().startswith("```")
        )

        print("Linhas:", len(linhas))
        print("Bytes:", caminho.stat().st_size)
        print("Títulos:")
        for titulo in titulos:
            print(" ", titulo)

        if nome == "Documentação":
            duplicados = [
                titulo
                for titulo, quantidade in Counter(titulos).items()
                if quantidade > 1
            ]

            print("Títulos duplicados:", duplicados)
            print("Delimitadores de código:", quantidade_blocos)
            print("Blocos de código equilibrados:", quantidade_blocos % 2 == 0)

        print("\n--- Conteúdo integral ---\n")
        print(conteudo)

In [ ]:
conteudo = arquivo_documentacao.read_text(encoding="utf-8")

# Correção 1: incluir eixos na descrição da granularidade
texto_antigo = (
    "Representa o estoque de veículos registrados por competência, "
    "UF, município, tipo de veículo e espécie."
)

texto_novo = (
    "Representa o estoque de veículos registrados por competência, "
    "UF, município, tipo de veículo, espécie e quantidade de eixos."
)

assert conteudo.count(texto_antigo) == 1
conteudo = conteudo.replace(texto_antigo, texto_novo)

# Correção 2: distinguir integridade referencial e filtro no Power BI
texto_antigo = (
    "A direção de filtro recomendada no Power BI é unidirecional, "
    "da dimensão para a tabela fato."
)

texto_novo = (
    "As chaves estrangeiras garantem a integridade referencial no PostgreSQL. "
    "Na camada semântica do Power BI, recomenda-se configurar os relacionamentos "
    "com direção de filtro unidirecional, da dimensão para a tabela fato."
)

assert conteudo.count(texto_antigo) == 1
conteudo = conteudo.replace(texto_antigo, texto_novo)

arquivo_documentacao.write_text(conteudo, encoding="utf-8")

print("Correções aplicadas.")
print("Arquivo:", arquivo_documentacao)
print("Total de linhas:", len(conteudo.splitlines()))
print("\nTrecho da granularidade:")
inicio = conteudo.index("### 4.1 fato_frota")
fim = conteudo.index("### 4.2 fato_emplacamentos")
print(conteudo[inicio:fim])

print("\nTrecho dos relacionamentos:")
inicio = conteudo.index("### 5.3 Cardinalidade")
fim = conteudo.index("## 6.")
print(conteudo[inicio:fim])

In [ ]:
from sqlalchemy import inspect

inspector = inspect(engine)

tabelas = inspector.get_table_names(schema="analytics")
views = inspector.get_view_names(schema="analytics")

print("SCHEMA: analytics")
print("=" * 60)

print("\nTABELAS:")
for tabela in sorted(tabelas):
    print(f"- {tabela}")

print("\nVIEWS:")
if views:
    for view in sorted(views):
        print(f"- {view}")
else:
    print("Nenhuma view encontrada.")

print("\nTOTAL:")
print("Tabelas:", len(tabelas))
print("Views:", len(views))

In [ ]:
sql_frota_nacional = """
SELECT
    dc.competencia,
    dc.ano_mes,
    SUM(ff.qtd_veiculos)::BIGINT AS frota_total
FROM analytics.fato_frota AS ff
INNER JOIN analytics.dim_calendario AS dc
    ON dc.competencia = ff.data_referencia
GROUP BY
    dc.competencia,
    dc.ano_mes
ORDER BY
    dc.competencia;
"""

df_frota_nacional = pd.read_sql_query(
    text(sql_frota_nacional),
    engine
)

print("FROTA NACIONAL — SÉRIE MENSAL")
print("=" * 65)
print(df_frota_nacional.to_string(index=False))

print("\nVALIDAÇÃO")
print("Quantidade de competências:", len(df_frota_nacional))
print(
    "Primeira competência:",
    df_frota_nacional["competencia"].min()
)
print(
    "Última competência:",
    df_frota_nacional["competencia"].max()
)
print(
    "Variação absoluta no período:",
    int(
        df_frota_nacional["frota_total"].iloc[-1]
        - df_frota_nacional["frota_total"].iloc[0]
    )
)

In [ ]:
sql_criar_view_frota = """
CREATE OR REPLACE VIEW analytics.vw_frota_nacional_mensal AS
SELECT
    dc.competencia,
    dc.ano,
    dc.mes,
    dc.nome_mes,
    dc.mes_abreviado,
    dc.ano_mes,
    SUM(ff.qtd_veiculos)::BIGINT AS frota_total
FROM analytics.fato_frota AS ff
INNER JOIN analytics.dim_calendario AS dc
    ON dc.competencia = ff.data_referencia
GROUP BY
    dc.competencia,
    dc.ano,
    dc.mes,
    dc.nome_mes,
    dc.mes_abreviado,
    dc.ano_mes
ORDER BY
    dc.competencia;
"""

with engine.begin() as conexao:
    conexao.execute(text(sql_criar_view_frota))

print("Comando de criação da view executado.")
print("Objeto: analytics.vw_frota_nacional_mensal")

In [ ]:
sql_validar_view = """
SELECT *
FROM analytics.vw_frota_nacional_mensal
ORDER BY competencia;
"""

df_view_frota = pd.read_sql_query(
    text(sql_validar_view),
    engine
)

print("VIEW: analytics.vw_frota_nacional_mensal")
print("=" * 70)

print(df_view_frota.to_string(index=False))

print("\nVALIDAÇÃO")
print("Registros:", len(df_view_frota))
print("Colunas:", list(df_view_frota.columns))

print(
    "Resultados idênticos à consulta original:",
    df_view_frota["frota_total"].tolist()
    == df_frota_nacional["frota_total"].tolist()
)

print(
    "Total de frota em setembro/2026:",
    int(df_view_frota.iloc[-1]["frota_total"])
)

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from sqlalchemy import text

# Recuperar a raiz do projeto
PROJECT_ROOT = Path.cwd().parent

# Disponibilizar os módulos do projeto
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Recuperar conexão com PostgreSQL
from src.database import criar_engine

engine = criar_engine()

# Verificar ambiente e conexão
with engine.connect() as conexao:
    banco = conexao.execute(
        text("SELECT current_database()")
    ).scalar()

    versao = conexao.execute(
        text("SELECT version()")
    ).scalar()

print("AMBIENTE RECUPERADO")
print("=" * 55)
print("Diretório atual:", Path.cwd())
print("Raiz do projeto:", PROJECT_ROOT)
print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("Banco conectado:", banco)
print("PostgreSQL:", versao.split(",")[0])

In [ ]:
sql_emplacamentos_nacionais = """
SELECT
    dc.competencia,
    dc.ano,
    dc.mes,
    dc.nome_mes,
    dc.ano_mes,
    fe.emplacamentos_mes,
    fe.emplacamentos_mes_anterior,
    fe.emplacamentos_acumulados,
    fe.emplacamentos_mes_ano_anterior,
    fe.emplacamentos_acumulados_ano_anterior,
    fe.variacao_mensal_pct,
    fe.variacao_anual_pct,
    fe.variacao_acumulada_pct
FROM analytics.fato_emplacamentos AS fe
INNER JOIN analytics.dim_calendario AS dc
    ON dc.competencia = fe.competencia
WHERE fe.categoria = 'Autos + Com. Leves'
ORDER BY dc.competencia;
"""

df_emplacamentos_nacionais = pd.read_sql_query(
    text(sql_emplacamentos_nacionais),
    engine
)

print("EMPLACAMENTOS NACIONAIS — FENABRAVE")
print("=" * 90)
print(df_emplacamentos_nacionais.to_string(index=False))

print("\nVALIDAÇÃO")
print("Quantidade de competências:", len(df_emplacamentos_nacionais))
print("Primeira competência:", df_emplacamentos_nacionais["competencia"].min())
print("Última competência:", df_emplacamentos_nacionais["competencia"].max())
print(
    "Total de emplacamentos mensais no período:",
    f'{df_emplacamentos_nacionais["emplacamentos_mes"].sum():,}'
)

In [ ]:
sql_criar_view_emplacamentos = """
CREATE OR REPLACE VIEW analytics.vw_emplacamentos_nacionais_mensais AS
SELECT
    dc.competencia,
    dc.ano,
    dc.mes,
    dc.nome_mes,
    dc.mes_abreviado,
    dc.ano_mes,
    fe.emplacamentos_mes,
    fe.emplacamentos_mes_anterior,
    fe.emplacamentos_acumulados,
    fe.emplacamentos_mes_ano_anterior,
    fe.emplacamentos_acumulados_ano_anterior,
    fe.variacao_mensal_pct,
    fe.variacao_anual_pct,
    fe.variacao_acumulada_pct
FROM analytics.fato_emplacamentos AS fe
INNER JOIN analytics.dim_calendario AS dc
    ON dc.competencia = fe.competencia
WHERE fe.categoria = 'Autos + Com. Leves';
"""

with engine.begin() as conexao:
    conexao.execute(text(sql_criar_view_emplacamentos))

print("Comando de criação da view executado.")
print("Objeto: analytics.vw_emplacamentos_nacionais_mensais")

In [ ]:
sql_validar_view_emplacamentos = """
SELECT *
FROM analytics.vw_emplacamentos_nacionais_mensais
ORDER BY competencia;
"""

df_view_emplacamentos = pd.read_sql_query(
    text(sql_validar_view_emplacamentos),
    engine
)

colunas_comparacao = [
    "competencia",
    "ano",
    "mes",
    "nome_mes",
    "ano_mes",
    "emplacamentos_mes",
    "emplacamentos_mes_anterior",
    "emplacamentos_acumulados",
    "emplacamentos_mes_ano_anterior",
    "emplacamentos_acumulados_ano_anterior",
    "variacao_mensal_pct",
    "variacao_anual_pct",
    "variacao_acumulada_pct"
]

print("VALIDAÇÃO DA VIEW DE EMPLACAMENTOS")
print("=" * 65)

print("Registros:", len(df_view_emplacamentos))
print("Colunas:", len(df_view_emplacamentos.columns))

print(
    "Resultados idênticos à consulta original:",
    df_view_emplacamentos[colunas_comparacao].equals(
        df_emplacamentos_nacionais[colunas_comparacao]
    )
)

print("\nPrimeiros registros:")
print(df_view_emplacamentos.head(3).to_string(index=False))

print("\nÚltimos registros:")
print(df_view_emplacamentos.tail(3).to_string(index=False))

In [ ]:
sql_comparacao_interanual = """
WITH frota AS (
    SELECT
        mes,
        MAX(frota_total) FILTER (WHERE ano = 2025) AS frota_2025,
        MAX(frota_total) FILTER (WHERE ano = 2026) AS frota_2026
    FROM analytics.vw_frota_nacional_mensal
    WHERE ano IN (2025, 2026)
      AND mes BETWEEN 1 AND 9
    GROUP BY mes
),
emplacamentos AS (
    SELECT
        mes,
        MAX(emplacamentos_mes) FILTER (WHERE ano = 2025)
            AS emplacamentos_2025,
        MAX(emplacamentos_mes) FILTER (WHERE ano = 2026)
            AS emplacamentos_2026
    FROM analytics.vw_emplacamentos_nacionais_mensais
    WHERE ano IN (2025, 2026)
      AND mes BETWEEN 1 AND 9
    GROUP BY mes
)
SELECT
    f.mes,
    f.frota_2025,
    f.frota_2026,
    ROUND(
        ((f.frota_2026::NUMERIC / NULLIF(f.frota_2025, 0)) - 1) * 100,
        2
    ) AS variacao_frota_pct,
    e.emplacamentos_2025,
    e.emplacamentos_2026,
    ROUND(
        ((e.emplacamentos_2026::NUMERIC /
          NULLIF(e.emplacamentos_2025, 0)) - 1) * 100,
        2
    ) AS variacao_emplacamentos_pct,
    ROUND(
        (
            ((e.emplacamentos_2026::NUMERIC /
              NULLIF(e.emplacamentos_2025, 0)) - 1)
            -
            ((f.frota_2026::NUMERIC /
              NULLIF(f.frota_2025, 0)) - 1)
        ) * 100,
        2
    ) AS diferenca_crescimento_pp
FROM frota AS f
INNER JOIN emplacamentos AS e
    ON e.mes = f.mes
ORDER BY f.mes;
"""

df_comparacao_interanual = pd.read_sql_query(
    text(sql_comparacao_interanual),
    engine
)

print("COMPARAÇÃO INTERANUAL — FROTA E EMPLACAMENTOS")
print("=" * 105)
print(df_comparacao_interanual.to_string(index=False))

print("\nVALIDAÇÃO")
print("Quantidade de meses:", len(df_comparacao_interanual))
print("Meses encontrados:", df_comparacao_interanual["mes"].tolist())
print(
    "Existem valores nulos:",
    df_comparacao_interanual.isnull().any().any()
)

In [ ]:
sql_auditoria_yoy_fenabrave = """
SELECT
    atual.mes,
    anterior.emplacamentos_mes AS emplacamentos_2025_serie,
    atual.emplacamentos_mes AS emplacamentos_2026,
    atual.emplacamentos_mes_ano_anterior AS referencia_2025_publicada,
    atual.emplacamentos_mes_ano_anterior
        - anterior.emplacamentos_mes AS diferenca_referencias,
    ROUND(
        (
            atual.emplacamentos_mes::NUMERIC
            / NULLIF(anterior.emplacamentos_mes, 0) - 1
        ) * 100, 2
    ) AS variacao_calculada_serie_pct,
    atual.variacao_anual_pct AS variacao_publicada_pct
FROM analytics.vw_emplacamentos_nacionais_mensais AS atual
INNER JOIN analytics.vw_emplacamentos_nacionais_mensais AS anterior
    ON anterior.mes = atual.mes
   AND anterior.ano = 2025
WHERE atual.ano = 2026
ORDER BY atual.mes;
"""

df_auditoria_yoy = pd.read_sql_query(
    text(sql_auditoria_yoy_fenabrave),
    engine
)

print("AUDITORIA DAS BASES DE COMPARAÇÃO FENABRAVE")
print("=" * 110)
print(df_auditoria_yoy.to_string(index=False))

print("\nRESUMO")
print(
    "Meses com referências diferentes:",
    int((df_auditoria_yoy["diferenca_referencias"] != 0).sum())
)
print(
    "Diferença absoluta total:",
    int(df_auditoria_yoy["diferenca_referencias"].abs().sum())
)

In [ ]:
from pathlib import Path

# Localização do projeto
PROJECT_ROOT = Path.cwd().parent
pasta_documentacao = PROJECT_ROOT / "docs" / "qualidade"
arquivo_auditoria = pasta_documentacao / "auditoria_fenabrave.md"

conteudo = r"""# Auditoria das bases de comparação — FENABRAVE

**Projeto:** Inteligência Comercial do Mercado Automotivo Brasileiro  
**Data do registro:** 09/11/2026  
**Período auditado:** janeiro a setembro de 2026  
**Categoria:** Autos + Com. Leves

## 1. Objetivo

Verificar a consistência entre os volumes mensais de 2025 armazenados na série histórica da FENABRAVE e os valores de referência de 2025 publicados nos relatórios mensais de 2026.

A auditoria busca identificar diferenças entre as bases utilizadas no cálculo da variação anual, sem pressupor que uma das fontes esteja incorreta.

## 2. Resultado da comparação

A comparação foi realizada para os nove meses equivalentes, de janeiro a setembro.

| Mês | Série 2025 | Referência publicada 2025 | Diferença | Variação pela série (%) | Variação publicada (%) |
|---|---:|---:|---:|---:|---:|
| Janeiro | 159.846 | 159.862 | 16 | 1,65 | 1,64 |
| Fevereiro | 173.809 | 173.840 | 31 | 1,72 | 1,70 |
| Março | 184.046 | 184.141 | 95 | 40,30 | 40,23 |
| Abril | 197.053 | 197.129 | 76 | 20,40 | 20,36 |
| Maio | 214.357 | 214.408 | 51 | 23,18 | 23,15 |
| Junho | 202.164 | 202.199 | 35 | 28,84 | 28,82 |
| Julho | 229.948 | 229.967 | 19 | 15,55 | 15,54 |
| Agosto | 214.490 | 214.439 | -51 | 22,64 | 22,67 |
| Setembro | 231.370 | 231.330 | -40 | 16,14 | 16,16 |

A diferença é calculada como:

`referencia_2025_publicada - emplacamentos_2025_serie`

## 3. Síntese da auditoria

- Meses auditados: 9.
- Meses com referências diferentes: 9.
- Soma das diferenças absolutas: 414 veículos.
- Diferença líquida (referência publicada menos série histórica): 232 veículos.
- Maior diferença absoluta: março, com 95 veículos.
- De janeiro a julho, as referências publicadas são maiores que os valores da série histórica.
- Em agosto e setembro, as referências publicadas são menores.

As diferenças são pequenas em relação aos volumes mensais, mas produzem divergências nos percentuais de variação anual.

## 4. Interpretação metodológica

Foram identificadas duas bases de comparação:

**Variação calculada pela série histórica**

Compara o volume mensal de 2026 com o volume do mês equivalente de 2025 armazenado na série histórica utilizada pelo projeto.

**Variação anual publicada**

Corresponde ao percentual informado diretamente pela FENABRAVE no relatório mensal de 2026, utilizando a referência de 2025 apresentada naquele relatório.

Os resultados não são necessariamente idênticos porque os valores de referência de 2025 diferem entre as duas bases.

A auditoria identifica a divergência, mas não determina sua causa. Não há evidência suficiente nesta análise para atribuí-la a revisões, ajustes ou outro procedimento específico da fonte.

## 5. Decisão para o projeto

1. Preservar os volumes mensais de 2025 da série histórica.
2. Preservar os valores de referência de 2025 publicados nos relatórios de 2026.
3. Manter `variacao_anual_pct` como indicador publicado pela FENABRAVE.
4. Não sobrescrever nem recalcular o indicador publicado para forçar igualdade com a série histórica.
5. Utilizar a série histórica quando a análise exigir comparação calculada diretamente entre competências.
6. Utilizar o percentual publicado quando o objetivo for reproduzir o indicador divulgado pela FENABRAVE.
7. Informar a existência dessa diferença em análises ou apresentações nas quais a distinção seja relevante.

## 6. Impacto na arquitetura de dados

A auditoria não identificou necessidade de alteração estrutural nas tabelas ou views existentes.

Permanecem preservadas:

- `analytics.fato_emplacamentos`
- `analytics.vw_emplacamentos_nacionais_mensais`

A estrutura atual mantém os valores publicados e os volumes mensais da série histórica necessários para distinguir as duas referências.

**Conclusão:** divergência documentada, dados preservados e nenhuma correção estrutural necessária.
"""

pasta_documentacao.mkdir(parents=True, exist_ok=True)

if arquivo_auditoria.exists():
    print("O documento já existe. Nenhum arquivo foi sobrescrito.")
    print(f"Localização: {arquivo_auditoria}")
else:
    arquivo_auditoria.write_text(conteudo, encoding="utf-8")
    print("Documento criado.")
    print(f"Localização: {arquivo_auditoria}")
    print(f"Tamanho: {arquivo_auditoria.stat().st_size:,} bytes")

In [ ]:
from pathlib import Path

arquivo_auditoria = (
    PROJECT_ROOT / "docs" / "qualidade" / "auditoria_fenabrave.md"
)

texto_auditoria = arquivo_auditoria.read_text(encoding="utf-8")

verificacoes = {
    "Objetivo documentado": "## 1. Objetivo" in texto_auditoria,
    "Tabela comparativa presente": "| Setembro | 231.370 | 231.330 | -40 |" in texto_auditoria,
    "Nove meses registrados": all(
        mes in texto_auditoria
        for mes in [
            "Janeiro", "Fevereiro", "Março", "Abril", "Maio",
            "Junho", "Julho", "Agosto", "Setembro"
        ]
    ),
    "414 diferenças absolutas": "414 veículos" in texto_auditoria,
    "Diferença líquida de 232": "232 veículos" in texto_auditoria,
    "Decisão metodológica registrada": "## 5. Decisão para o projeto" in texto_auditoria,
    "Impacto arquitetural registrado": "## 6. Impacto na arquitetura de dados" in texto_auditoria,
}

print("VALIDAÇÃO DA DOCUMENTAÇÃO")
print("=" * 65)

for verificacao, resultado in verificacoes.items():
    print(f"{'OK' if resultado else 'FALHA'} — {verificacao}")

print("=" * 65)
print(f"Total: {sum(verificacoes.values())}/{len(verificacoes)} verificações aprovadas")

In [ ]:
sql_validacao_consolidacao = """
WITH frota AS (
    SELECT
        competencia,
        COUNT(*) AS quantidade
    FROM analytics.vw_frota_nacional_mensal
    GROUP BY competencia
),
emplacamentos AS (
    SELECT
        competencia,
        COUNT(*) AS quantidade
    FROM analytics.vw_emplacamentos_nacionais_mensais
    GROUP BY competencia
)
SELECT
    COALESCE(f.competencia, e.competencia) AS competencia,
    f.quantidade AS linhas_frota,
    e.quantidade AS linhas_emplacamentos,
    CASE
        WHEN f.competencia IS NULL THEN 'AUSENTE NA FROTA'
        WHEN e.competencia IS NULL THEN 'AUSENTE NOS EMPLACAMENTOS'
        WHEN f.quantidade <> 1 OR e.quantidade <> 1 THEN 'DUPLICIDADE'
        ELSE 'OK'
    END AS situacao
FROM frota f
FULL OUTER JOIN emplacamentos e
    ON e.competencia = f.competencia
ORDER BY competencia;
"""

df_validacao_consolidacao = pd.read_sql_query(
    text(sql_validacao_consolidacao),
    engine
)

print("VALIDAÇÃO DE COMPATIBILIDADE DAS VIEWS")
print("=" * 75)
print(df_validacao_consolidacao.to_string(index=False))
print("=" * 75)
print(f"Competências encontradas: {len(df_validacao_consolidacao)}")
print(
    "Todas as competências compatíveis:",
    (df_validacao_consolidacao["situacao"] == "OK").all()
)
print(
    "Quantidade de situações diferentes de OK:",
    (df_validacao_consolidacao["situacao"] != "OK").sum()
)

In [ ]:
sql_criar_view_consolidada = """
CREATE OR REPLACE VIEW analytics.vw_indicadores_nacionais_mensais AS

WITH base AS (
    SELECT
        f.competencia,
        f.ano,
        f.mes,
        f.nome_mes,
        f.mes_abreviado,
        f.ano_mes,
        f.frota_total,
        e.emplacamentos_mes,
        e.emplacamentos_mes_anterior,
        e.emplacamentos_acumulados,
        e.emplacamentos_mes_ano_anterior,
        e.emplacamentos_acumulados_ano_anterior,
        e.variacao_mensal_pct,
        e.variacao_anual_pct AS variacao_emplacamentos_publicada_pct,
        e.variacao_acumulada_pct
    FROM analytics.vw_frota_nacional_mensal f
    INNER JOIN analytics.vw_emplacamentos_nacionais_mensais e
        ON e.competencia = f.competencia
)

SELECT
    atual.*,

    ROUND(
        100.0 * (
            atual.frota_total::numeric / anterior.frota_total - 1
        ),
        2
    ) AS variacao_frota_pct,

    ROUND(
        100.0 * (
            atual.emplacamentos_mes::numeric
            / NULLIF(anterior.emplacamentos_mes, 0) - 1
        ),
        2
    ) AS variacao_emplacamentos_serie_pct,

    ROUND(
        (
            100.0 * (
                atual.frota_total::numeric / anterior.frota_total - 1
            )
        ) - (
            100.0 * (
                atual.emplacamentos_mes::numeric
                / NULLIF(anterior.emplacamentos_mes, 0) - 1
            )
        ),
        2
    ) AS diferenca_frota_vs_emplacamentos_serie_pp,

    ROUND(
        (
            100.0 * (
                atual.frota_total::numeric / anterior.frota_total - 1
            )
        ) - atual.variacao_emplacamentos_publicada_pct,
        2
    ) AS diferenca_frota_vs_emplacamentos_publicada_pp

FROM base atual
LEFT JOIN base anterior
    ON anterior.competencia = (
        atual.competencia - INTERVAL '1 year'
    )::date;
"""

with engine.begin() as conexao:
    conexao.execute(text(sql_criar_view_consolidada))

print("View consolidada criada ou atualizada:")
print("analytics.vw_indicadores_nacionais_mensais")

In [ ]:
sql_validar_view_consolidada = """
SELECT
    competencia,
    frota_total,
    emplacamentos_mes,
    variacao_frota_pct,
    variacao_emplacamentos_serie_pct,
    variacao_emplacamentos_publicada_pct,
    diferenca_frota_vs_emplacamentos_serie_pp,
    diferenca_frota_vs_emplacamentos_publicada_pp
FROM analytics.vw_indicadores_nacionais_mensais
ORDER BY competencia;
"""

df_indicadores = pd.read_sql_query(
    text(sql_validar_view_consolidada),
    engine
)

print("VALIDAÇÃO DA VIEW CONSOLIDADA")
print("=" * 110)
print(df_indicadores.to_string(index=False))
print("=" * 110)

print(f"Total de competências: {len(df_indicadores)}")
print(
    "Competências únicas:",
    df_indicadores["competencia"].nunique()
)
print(
    "Primeira competência sem comparação interanual:",
    df_indicadores.iloc[0]["variacao_frota_pct"] is None
    or pd.isna(df_indicadores.iloc[0]["variacao_frota_pct"])
)
print(
    "Última competência:",
    df_indicadores.iloc[-1]["competencia"]
)

In [ ]:
# Converter competência para o tipo datetime
df_indicadores["competencia"] = pd.to_datetime(
    df_indicadores["competencia"]
)

# Separar os registros por ano
df_2025 = df_indicadores[
    df_indicadores["competencia"].dt.year == 2025
]

df_2026 = df_indicadores[
    df_indicadores["competencia"].dt.year == 2026
]

verificacoes = {
    "21 competências no total": len(df_indicadores) == 21,
    "21 competências únicas": df_indicadores["competencia"].nunique() == 21,
    "12 competências em 2025": len(df_2025) == 12,
    "9 competências em 2026": len(df_2026) == 9,
    "Variações calculadas nulas em 2025": (
        df_2025["variacao_frota_pct"].isna().all()
        and df_2025["variacao_emplacamentos_serie_pct"].isna().all()
    ),
    "Variações calculadas preenchidas em 2026": (
        df_2026["variacao_frota_pct"].notna().all()
        and df_2026["variacao_emplacamentos_serie_pct"].notna().all()
    ),
    "Variações publicadas preservadas": (
        df_indicadores["variacao_emplacamentos_publicada_pct"].notna().all()
    ),
    "Frota de setembro/2026 correta": (
        int(df_2026.iloc[-1]["frota_total"]) == 84607600
    ),
    "Emplacamentos de setembro/2026 corretos": (
        int(df_2026.iloc[-1]["emplacamentos_mes"]) == 268721
    ),
    "Variação da frota de setembro/2026 correta": (
        round(df_2026.iloc[-1]["variacao_frota_pct"], 2) == 3.61
    ),
    "Variação calculada dos emplacamentos correta": (
        round(df_2026.iloc[-1]["variacao_emplacamentos_serie_pct"], 2) == 16.14
    ),
    "Variação publicada dos emplacamentos correta": (
        round(df_2026.iloc[-1]["variacao_emplacamentos_publicada_pct"], 2) == 16.16
    ),
}

print("VALIDAÇÃO FINAL — VIEW CONSOLIDADA")
print("=" * 75)

for nome, resultado in verificacoes.items():
    print(f"{'OK' if resultado else 'FALHA'} — {nome}")

print("=" * 75)
print(
    f"Total: {sum(verificacoes.values())}/{len(verificacoes)} "
    "verificações aprovadas"
)

In [ ]:
sql_estrutura_geografica = """
SELECT
    table_name,
    column_name,
    data_type,
    is_nullable
FROM information_schema.columns
WHERE table_schema = 'analytics'
  AND table_name IN ('dim_uf', 'fato_frota', 'dim_tipo_veiculo')
ORDER BY table_name, ordinal_position;
"""

df_estrutura_geografica = pd.read_sql_query(
    text(sql_estrutura_geografica),
    engine
)

print("ESTRUTURA DAS TABELAS PARA ANÁLISE GEOGRÁFICA")
print("=" * 85)
print(df_estrutura_geografica.to_string(index=False))

In [ ]:
sql_criar_view_frota_uf = """
CREATE OR REPLACE VIEW analytics.vw_frota_uf_mensal AS

SELECT
    f.data_referencia AS competencia,
    EXTRACT(YEAR FROM f.data_referencia)::integer AS ano,
    EXTRACT(MONTH FROM f.data_referencia)::integer AS mes,
    u.id_uf,
    u.sigla_uf,
    u.nome_uf,
    u.regiao,
    SUM(f.qtd_veiculos)::bigint AS frota_total

FROM analytics.fato_frota f

INNER JOIN analytics.dim_uf u
    ON u.id_uf = f.id_uf

INNER JOIN analytics.dim_tipo_veiculo t
    ON t.id_tipo_veiculo = f.id_tipo_veiculo

WHERE t.tipo_veiculo IN (
    'AUTOMOVEL',
    'CAMINHONETE',
    'CAMIONETA',
    'UTILITARIO'
)

GROUP BY
    f.data_referencia,
    u.id_uf,
    u.sigla_uf,
    u.nome_uf,
    u.regiao;
"""

with engine.begin() as conexao:
    conexao.execute(text(sql_criar_view_frota_uf))

print("View criada ou atualizada:")
print("analytics.vw_frota_uf_mensal")

In [ ]:
sql_validar_frota_uf = """
WITH geografica AS (
    SELECT
        COUNT(*) AS total_linhas,
        COUNT(DISTINCT competencia) AS competencias,
        COUNT(DISTINCT id_uf) AS ufs,
        SUM(frota_total) AS frota_total
    FROM analytics.vw_frota_uf_mensal
),
nacional AS (
    SELECT
        COUNT(*) AS competencias,
        SUM(frota_total) AS frota_total
    FROM analytics.vw_frota_nacional_mensal
)
SELECT
    g.total_linhas,
    g.competencias AS competencias_geograficas,
    n.competencias AS competencias_nacionais,
    g.ufs,
    g.frota_total AS frota_geografica,
    n.frota_total AS frota_nacional,
    g.frota_total - n.frota_total AS diferenca,
    CASE
        WHEN g.competencias = n.competencias
         AND g.ufs = 28
         AND g.frota_total = n.frota_total
        THEN 'OK'
        ELSE 'VERIFICAR'
    END AS situacao
FROM geografica g
CROSS JOIN nacional n;
"""

df_validacao_frota_uf = pd.read_sql_query(
    text(sql_validar_frota_uf),
    engine
)

print("VALIDAÇÃO DA VIEW GEOGRÁFICA")
print("=" * 75)
print(df_validacao_frota_uf.to_string(index=False))

In [ ]:
sql_frota_regional_setembro = """
SELECT
    regiao,
    COUNT(DISTINCT id_uf) AS quantidade_ufs,
    SUM(frota_total) AS frota_total,
    ROUND(
        100.0 * SUM(frota_total)
        / SUM(SUM(frota_total)) OVER (),
        2
    ) AS participacao_pct
FROM analytics.vw_frota_uf_mensal
WHERE competencia = DATE '2026-09-01'
  AND sigla_uf <> 'SI'
GROUP BY regiao
ORDER BY frota_total DESC;
"""

df_frota_regional = pd.read_sql_query(
    text(sql_frota_regional_setembro),
    engine
)

print("DISTRIBUIÇÃO REGIONAL DA FROTA — SETEMBRO/2026")
print("=" * 75)
print(df_frota_regional.to_string(index=False))

print("=" * 75)
print(
    "Total das regiões:",
    f"{df_frota_regional['frota_total'].sum():,.0f}"
)

In [ ]:
sql_top_ufs = """
WITH base AS (
    SELECT
        sigla_uf,
        nome_uf,
        regiao,
        frota_total
    FROM analytics.vw_frota_uf_mensal
    WHERE competencia = DATE '2026-09-01'
      AND sigla_uf <> 'SI'
)
SELECT
    sigla_uf,
    nome_uf,
    regiao,
    frota_total,
    ROUND(
        100.0 * frota_total
        / SUM(frota_total) OVER (),
        2
    ) AS participacao_pct
FROM base
ORDER BY frota_total DESC
LIMIT 10;
"""

df_top_ufs = pd.read_sql_query(
    text(sql_top_ufs),
    engine
)

print("10 MAIORES FROTAS ESTADUAIS — SETEMBRO/2026")
print("=" * 85)
print(df_top_ufs.to_string(index=False))

print("=" * 85)
print(
    "Participação conjunta das 10 maiores UFs:",
    round(df_top_ufs["participacao_pct"].sum(), 2),
    "%"
)

In [ ]:
sql_crescimento_uf = """
WITH frota_2025 AS (
    SELECT
        id_uf,
        sigla_uf,
        nome_uf,
        regiao,
        frota_total
    FROM analytics.vw_frota_uf_mensal
    WHERE competencia = DATE '2025-09-01'
      AND sigla_uf <> 'SI'
),
frota_2026 AS (
    SELECT
        id_uf,
        frota_total
    FROM analytics.vw_frota_uf_mensal
    WHERE competencia = DATE '2026-09-01'
      AND sigla_uf <> 'SI'
),
variacoes AS (
    SELECT
        a.id_uf,
        a.sigla_uf,
        a.nome_uf,
        a.regiao,
        a.frota_total AS frota_set_2025,
        b.frota_total AS frota_set_2026,
        b.frota_total - a.frota_total AS crescimento_absoluto
    FROM frota_2025 a
    INNER JOIN frota_2026 b
        ON b.id_uf = a.id_uf
)
SELECT
    sigla_uf,
    nome_uf,
    regiao,
    frota_set_2025,
    frota_set_2026,
    crescimento_absoluto,
    ROUND(
        100.0 * crescimento_absoluto / NULLIF(frota_set_2025, 0),
        2
    ) AS crescimento_pct,
    ROUND(
        100.0 * crescimento_absoluto
        / NULLIF(SUM(crescimento_absoluto) OVER (), 0),
        2
    ) AS contribuicao_crescimento_pct
FROM variacoes
ORDER BY crescimento_absoluto DESC;
"""

df_crescimento_uf = pd.read_sql_query(
    text(sql_crescimento_uf),
    engine
)

print("CRESCIMENTO DA FROTA POR UF — SET/2025 A SET/2026")
print("=" * 115)
print(df_crescimento_uf.to_string(index=False))

print("=" * 115)
print("UFs analisadas:", len(df_crescimento_uf))
print(
    "Crescimento absoluto total:",
    f"{df_crescimento_uf['crescimento_absoluto'].sum():,.0f}"
)
print(
    "Crescimento percentual total:",
    round(
        100 * df_crescimento_uf["crescimento_absoluto"].sum()
        / df_crescimento_uf["frota_set_2025"].sum(),
        2
    ),
    "%"
)

In [ ]:
sql_variacao_si = """
SELECT
    competencia,
    frota_total
FROM analytics.vw_frota_uf_mensal
WHERE sigla_uf = 'SI'
  AND competencia IN (
      DATE '2025-09-01',
      DATE '2026-09-01'
  )
ORDER BY competencia;
"""

df_variacao_si = pd.read_sql_query(
    text(sql_variacao_si),
    engine
)

print("VARIAÇÃO DA CATEGORIA SEM INFORMAÇÃO")
print("=" * 65)
print(df_variacao_si.to_string(index=False))

if len(df_variacao_si) == 2:
    valor_2025 = int(df_variacao_si.iloc[0]["frota_total"])
    valor_2026 = int(df_variacao_si.iloc[1]["frota_total"])

    print("=" * 65)
    print(f"Setembro/2025: {valor_2025:,.0f}")
    print(f"Setembro/2026: {valor_2026:,.0f}")
    print(f"Variação absoluta: {valor_2026 - valor_2025:,.0f}")

In [ ]:
from pathlib import Path

diretorio_analises = PROJECT_ROOT / "docs" / "analises"
arquivo_geografico = diretorio_analises / "analise_geografica_frota.md"

diretorio_analises.mkdir(parents=True, exist_ok=True)

print("DIRETÓRIO:", diretorio_analises)
print("ARQUIVO:", arquivo_geografico)
print("ARQUIVO JÁ EXISTE:", arquivo_geografico.exists())

if arquivo_geografico.exists():
    print("TAMANHO ATUAL:", arquivo_geografico.stat().st_size, "bytes")
else:
    print("O arquivo poderá ser criado sem sobrescrever documentação existente.")

In [ ]:
conteudo_geografico = """# Análise Geográfica da Frota Brasileira

## 1. Objetivo

Analisar a distribuição e a evolução geográfica da frota brasileira de automóveis e veículos comerciais leves, utilizando os dados mensais da SENATRAN.

A análise contempla as 27 Unidades Federativas identificadas e acompanha separadamente os registros classificados como SI (Sem Informação).

## 2. Escopo e metodologia

- Fonte: SENATRAN.
- Período da análise de crescimento: setembro de 2025 a setembro de 2026.
- Fotografia da distribuição geográfica: setembro de 2026.
- Categorias de veículos: AUTOMOVEL, CAMINHONETE, CAMIONETA e UTILITARIO.
- Granularidade geográfica: Unidade Federativa e região.
- Município não é utilizado na análise geográfica da primeira versão.
- SI é preservada na base física, mas excluída das participações estaduais e regionais por não possuir localização geográfica identificada.

A frota representa estoque de veículos registrados na data de referência, não volume de emplacamentos do mês.

## 3. Distribuição regional — setembro de 2026

| Região | Quantidade de UFs | Frota | Participação |
|---|---:|---:|---:|
| Sudeste | 4 | 43.730.742 | 51,95% |
| Sul | 3 | 17.898.867 | 21,26% |
| Nordeste | 9 | 11.577.759 | 13,75% |
| Centro-Oeste | 4 | 7.661.961 | 9,10% |
| Norte | 7 | 3.307.313 | 3,93% |
| **Total** | **27** | **84.176.642** | **100,00% aproximadamente** |

O Sudeste concentra aproximadamente 51,95% da frota geograficamente identificada.

Sudeste e Sul, em conjunto, representam 73,21% dessa frota.

O Nordeste possui a maior quantidade de UFs (9), mas responde por 13,75% do total identificado.

## 4. Dez maiores frotas estaduais — setembro de 2026

| Posição | UF | Estado | Região | Frota | Participação |
|---:|---|---|---|---:|---:|
| 1 | SP | São Paulo | Sudeste | 26.045.872 | 30,94% |
| 2 | MG | Minas Gerais | Sudeste | 9.871.350 | 11,73% |
| 3 | PR | Paraná | Sul | 6.904.592 | 8,20% |
| 4 | RS | Rio Grande do Sul | Sul | 6.301.082 | 7,49% |
| 5 | RJ | Rio de Janeiro | Sudeste | 6.134.136 | 7,29% |
| 6 | SC | Santa Catarina | Sul | 4.693.193 | 5,58% |
| 7 | BA | Bahia | Nordeste | 3.188.677 | 3,79% |
| 8 | GO | Goiás | Centro-Oeste | 3.071.486 | 3,65% |
| 9 | PE | Pernambuco | Nordeste | 2.024.574 | 2,41% |
| 10 | DF | Distrito Federal | Centro-Oeste | 1.888.525 | 2,24% |

As dez maiores UFs concentram 83,32% da frota geograficamente identificada.

São Paulo representa sozinho 30,94% do total identificado. São Paulo e Minas Gerais, em conjunto, representam 42,67%.

## 5. Crescimento da frota por UF — setembro de 2025 a setembro de 2026

O crescimento foi calculado pela diferença entre os estoques de setembro de 2026 e setembro de 2025.

| UF | Estado | Crescimento absoluto | Crescimento percentual | Contribuição ao crescimento |
|---|---|---:|---:|---:|
| SP | São Paulo | 631.553 | 2,49% | 21,92% |
| MG | Minas Gerais | 386.380 | 4,07% | 13,41% |
| PR | Paraná | 340.000 | 5,18% | 11,80% |
| SC | Santa Catarina | 150.092 | 3,30% | 5,21% |
| RS | Rio Grande do Sul | 149.383 | 2,43% | 5,18% |
| BA | Bahia | 146.082 | 4,80% | 5,07% |
| RJ | Rio de Janeiro | 142.371 | 2,38% | 4,94% |
| GO | Goiás | 115.086 | 3,89% | 3,99% |
| ES | Espírito Santo | 96.774 | 6,11% | 3,36% |
| CE | Ceará | 79.491 | 4,41% | 2,76% |
| PE | Pernambuco | 78.048 | 4,01% | 2,71% |
| DF | Distrito Federal | 76.268 | 4,21% | 2,65% |
| MT | Mato Grosso | 64.141 | 4,54% | 2,23% |
| PA | Pará | 63.402 | 5,70% | 2,20% |
| MA | Maranhão | 48.385 | 5,91% | 1,68% |
| PB | Paraíba | 46.131 | 5,43% | 1,60% |
| RN | Rio Grande do Norte | 45.046 | 5,08% | 1,56% |
| MS | Mato Grosso do Sul | 38.087 | 3,21% | 1,32% |
| PI | Piauí | 33.839 | 5,39% | 1,17% |
| AM | Amazonas | 29.858 | 4,70% | 1,04% |
| AL | Alagoas | 26.928 | 4,64% | 0,93% |
| RO | Rondônia | 25.672 | 4,87% | 0,89% |
| SE | Sergipe | 22.732 | 4,58% | 0,79% |
| TO | Tocantins | 20.807 | 5,00% | 0,72% |
| AP | Amapá | 8.946 | 6,09% | 0,31% |
| RR | Roraima | 8.206 | 5,44% | 0,28% |
| AC | Acre | 7.679 | 4,97% | 0,27% |

O crescimento absoluto das 27 UFs foi de 2.881.387 veículos, equivalente a 3,54% sobre o estoque identificado em setembro de 2025.

São Paulo, Minas Gerais e Paraná responderam juntos por 47,13% do crescimento absoluto das 27 UFs.

O Espírito Santo apresentou o maior crescimento percentual (6,11%), seguido pelo Amapá (6,09%) e Maranhão (5,91%).

Esses resultados demonstram que os estados de maior volume não são necessariamente os de maior crescimento percentual.

## 6. Tratamento da categoria SI — Sem Informação

A categoria SI foi mantida na base para garantir a integridade e a reconciliação com o total nacional.

| Competência | Frota SI |
|---|---:|
| Setembro de 2025 | 364.097 |
| Setembro de 2026 | 430.958 |
| **Crescimento absoluto** | **66.861** |

A variação percentual da categoria SI foi de aproximadamente 18,36%.

Esse percentual não representa crescimento de uma região ou estado. Refere-se exclusivamente à evolução dos registros sem identificação geográfica.

## 7. Reconciliação com o crescimento nacional

| Componente | Crescimento absoluto |
|---|---:|
| 27 UFs identificadas | 2.881.387 |
| SI — Sem Informação | 66.861 |
| **Total nacional** | **2.948.248** |

A reconciliação fecha exatamente:

2.881.387 + 66.861 = 2.948.248 veículos.

O crescimento nacional foi de 3,61% entre setembro de 2025 e setembro de 2026.

Não existe divergência entre a análise geográfica e a nacional. A diferença decorre exclusivamente dos registros classificados como SI.

## 8. Regras para utilização no Power BI

1. Utilizar `analytics.vw_frota_uf_mensal` como fonte da análise por UF.
2. Excluir SI dos gráficos de participação estadual e regional.
3. Manter SI em indicadores de reconciliação nacional.
4. Utilizar o total das 27 UFs identificadas como denominador de participações geográficas.
5. Diferenciar crescimento absoluto de crescimento percentual.
6. Não interpretar variação do estoque da frota como volume de emplacamentos.
7. Identificar claramente a competência de referência em todos os indicadores.

## 9. View analítica utilizada

`analytics.vw_frota_uf_mensal`

A view agrega a quantidade de veículos por competência, UF e região, considerando as quatro categorias do escopo do projeto.

## 10. Limitações e observações

- A análise geográfica da primeira versão utiliza UF e região, não município.
- A categoria SI não possui atribuição territorial.
- A distribuição apresentada representa estoque de veículos, não vendas ou emplacamentos.
- Os resultados refletem as competências e os arquivos oficiais disponíveis no momento da extração.
"""

if arquivo_geografico.exists():
    raise FileExistsError(
        f"O arquivo já existe e não será sobrescrito: {arquivo_geografico}"
    )

arquivo_geografico.write_text(
    conteudo_geografico,
    encoding="utf-8"
)

print("DOCUMENTAÇÃO CRIADA")
print("=" * 70)
print("Arquivo:", arquivo_geografico)
print("Tamanho:", arquivo_geografico.stat().st_size, "bytes")
print("Linhas:", len(conteudo_geografico.splitlines()))

In [ ]:
texto_geografico = arquivo_geografico.read_text(encoding="utf-8")

verificacoes_geograficas = {
    "Objetivo documentado": "## 1. Objetivo" in texto_geografico,
    "Metodologia documentada": "## 2. Escopo e metodologia" in texto_geografico,
    "Distribuição regional registrada": "43.730.742" in texto_geografico,
    "Dez maiores UFs registradas": "26.045.872" in texto_geografico,
    "Participação das dez maiores UFs": "83,32%" in texto_geografico,
    "Crescimento das 27 UFs": "2.881.387" in texto_geografico,
    "Contribuição conjunta das três maiores": "47,13%" in texto_geografico,
    "Crescimento percentual do Espírito Santo": "6,11%" in texto_geografico,
    "Categoria SI documentada": "430.958" in texto_geografico,
    "Variação absoluta de SI": "66.861" in texto_geografico,
    "Reconciliação nacional": "2.948.248" in texto_geografico,
    "Regras para Power BI": "## 8. Regras para utilização no Power BI" in texto_geografico,
    "Limitações documentadas": "## 10. Limitações e observações" in texto_geografico,
}

print("VALIDAÇÃO DA DOCUMENTAÇÃO GEOGRÁFICA")
print("=" * 75)

for nome, resultado in verificacoes_geograficas.items():
    print(f"{'OK' if resultado else 'FALHA'} — {nome}")

print("=" * 75)
print(
    f"Total: {sum(verificacoes_geograficas.values())}/"
    f"{len(verificacoes_geograficas)} verificações aprovadas"
)

In [ ]:
sql_evolucao_regional = """
WITH frota_regional AS (
    SELECT
        competencia,
        regiao,
        SUM(frota_total)::bigint AS frota_total
    FROM analytics.vw_frota_uf_mensal
    WHERE sigla_uf <> 'SI'
    GROUP BY competencia, regiao
),
comparacao AS (
    SELECT
        competencia,
        regiao,
        frota_total,
        LAG(frota_total, 12) OVER (
            PARTITION BY regiao
            ORDER BY competencia
        ) AS frota_mes_ano_anterior
    FROM frota_regional
)
SELECT
    competencia,
    regiao,
    frota_total,
    ROUND(
        100.0 * frota_total
        / SUM(frota_total) OVER (PARTITION BY competencia),
        2
    ) AS participacao_pct,
    frota_mes_ano_anterior,
    frota_total - frota_mes_ano_anterior AS crescimento_absoluto,
    ROUND(
        100.0 * (
            frota_total::numeric
            / NULLIF(frota_mes_ano_anterior, 0) - 1
        ),
        2
    ) AS crescimento_pct
FROM comparacao
WHERE competencia BETWEEN DATE '2026-01-01' AND DATE '2026-09-01'
ORDER BY competencia, frota_total DESC;
"""

df_evolucao_regional = pd.read_sql_query(
    text(sql_evolucao_regional),
    engine
)

print("EVOLUÇÃO REGIONAL DA FROTA — JANEIRO A SETEMBRO/2026")
print("=" * 110)
print(df_evolucao_regional.to_string(index=False))

print("=" * 110)
print("Registros:", len(df_evolucao_regional))
print("Regiões:", df_evolucao_regional["regiao"].nunique())
print("Competências:", df_evolucao_regional["competencia"].nunique())
print(
    "Comparações interanuais preenchidas:",
    df_evolucao_regional["crescimento_pct"].notna().sum()
)

In [ ]:
resumo_regional = (
    df_evolucao_regional
    .sort_values("competencia")
    .groupby("regiao")
    .agg(
        frota_jan=("frota_total", "first"),
        frota_set=("frota_total", "last"),
        participacao_jan=("participacao_pct", "first"),
        participacao_set=("participacao_pct", "last"),
        crescimento_abs_set=("crescimento_absoluto", "last"),
        crescimento_pct_set=("crescimento_pct", "last"),
        crescimento_pct_jan=("crescimento_pct", "first")
    )
    .reset_index()
)

resumo_regional["variacao_participacao_pp"] = (
    resumo_regional["participacao_set"]
    - resumo_regional["participacao_jan"]
).round(2)

resumo_regional["aumento_frota_jan_set"] = (
    resumo_regional["frota_set"]
    - resumo_regional["frota_jan"]
)

resumo_regional = resumo_regional.sort_values(
    "frota_set",
    ascending=False
)

print("RESUMO REGIONAL — JANEIRO A SETEMBRO/2026")
print("=" * 115)
print(resumo_regional.to_string(index=False))

In [ ]:
# Validação do resumo regional a partir da evolução mensal

df_validacao = df_evolucao_regional.copy()
df_validacao["competencia"] = pd.to_datetime(df_validacao["competencia"])

jan = (
    df_validacao[df_validacao["competencia"] == "2026-01-01"]
    .set_index("regiao")
)

setembro = (
    df_validacao[df_validacao["competencia"] == "2026-09-01"]
    .set_index("regiao")
)

print("VALIDAÇÃO DO RESUMO REGIONAL")
print("=" * 75)

# 1. Estrutura
print("Regiões no resumo:", len(jan))
print("Regiões em setembro:", len(setembro))
print("Regiões coincidentes:", set(jan.index) == set(setembro.index))

# 2. Aumento da frota de janeiro a setembro
aumento_jan_set = (
    setembro["frota_total"] - jan["frota_total"]
)

print("\nAUMENTO DA FROTA — JANEIRO A SETEMBRO")
print(aumento_jan_set.sort_values(ascending=False).to_string())
print("Total:", f"{aumento_jan_set.sum():,}".replace(",", "."))

# 3. Crescimento interanual de setembro
crescimento_interanual = setembro["crescimento_absoluto"]

print("\nCRESCIMENTO INTERANUAL — SETEMBRO/2026")
print(crescimento_interanual.sort_values(ascending=False).to_string())
print("Total:", f"{crescimento_interanual.sum():,}".replace(",", "."))

# 4. Participação regional
variacao_participacao = (
    setembro["participacao_pct"] - jan["participacao_pct"]
)

print("\nVARIAÇÃO DA PARTICIPAÇÃO — JANEIRO A SETEMBRO")
print(variacao_participacao.sort_values().to_string())

# 5. Participação total em cada competência
soma_participacao = (
    df_validacao.groupby("competencia")["participacao_pct"].sum()
)

print("\nSOMA DAS PARTICIPAÇÕES POR MÊS")
print(soma_participacao.to_string())

# 6. Verificações finais
verificacoes = {
    "45 registros na evolução mensal": len(df_validacao) == 45,
    "5 regiões": df_validacao["regiao"].nunique() == 5,
    "9 competências": df_validacao["competencia"].nunique() == 9,
    "Aumento janeiro-setembro = 1.982.385": aumento_jan_set.sum() == 1_982_385,
    "Crescimento interanual setembro = 2.881.387": crescimento_interanual.sum() == 2_881_387,
    "Participações mensais próximas de 100%": soma_participacao.between(99.98, 100.02).all(),
}

print("\n" + "=" * 75)
for descricao, resultado in verificacoes.items():
    print(f"{'OK' if resultado else 'FALHA'} — {descricao}")

print("=" * 75)
print(f"Total: {sum(verificacoes.values())}/{len(verificacoes)} verificações aprovadas")

In [ ]:
from pathlib import Path
import pandas as pd

# Preparação dos dados
df_doc = df_evolucao_regional.copy()
df_doc["competencia"] = pd.to_datetime(df_doc["competencia"])

jan = (
    df_doc[df_doc["competencia"] == "2026-01-01"]
    .set_index("regiao")
)

setembro = (
    df_doc[df_doc["competencia"] == "2026-09-01"]
    .set_index("regiao")
)

resumo = pd.DataFrame(index=jan.index)
resumo["frota_jan"] = jan["frota_total"]
resumo["frota_set"] = setembro["frota_total"]
resumo["participacao_jan"] = jan["participacao_pct"]
resumo["participacao_set"] = setembro["participacao_pct"]
resumo["crescimento_abs_set"] = setembro["crescimento_absoluto"]
resumo["crescimento_pct_set"] = setembro["crescimento_pct"]
resumo["variacao_participacao_pp"] = (
    setembro["participacao_pct"] - jan["participacao_pct"]
)
resumo["aumento_frota_jan_set"] = (
    setembro["frota_total"] - jan["frota_total"]
)

# Formatação das tabelas em Markdown
def tabela_markdown(df, colunas, nomes):
    linhas = [
        "| " + " | ".join(nomes) + " |",
        "| " + " | ".join(["---"] * len(nomes)) + " |"
    ]

    for _, linha in df[colunas].iterrows():
        valores = []

        for coluna in colunas:
            valor = linha[coluna]

            if pd.isna(valor):
                valores.append("—")

            elif coluna == "competencia":
                valores.append(
                    pd.Timestamp(valor).strftime("%m/%Y")
                )

            elif isinstance(valor, (int, float)):
                if isinstance(valor, float):
                    valores.append(
                        f"{valor:.2f}".replace(".", ",")
                    )
                else:
                    valores.append(
                        f"{valor:,}".replace(",", ".")
                    )

            else:
                valores.append(str(valor))

        linhas.append("| " + " | ".join(valores) + " |")

    return "\n".join(linhas)


print("Função de formatação corrigida.")
print("Próximo passo: executar novamente a célula de documentação.")
tabela_mensal = tabela_markdown(
    df_doc,
    [
        "competencia", "regiao", "frota_total",
        "participacao_pct", "frota_mes_ano_anterior",
        "crescimento_absoluto", "crescimento_pct"
    ],
    [
        "Competência", "Região", "Frota",
        "Participação (%)", "Frota mesmo mês ano anterior",
        "Crescimento absoluto", "Crescimento interanual (%)"
    ]
)

tabela_resumo = tabela_markdown(
    resumo.reset_index(names="regiao"),
    [
        "regiao", "frota_jan", "frota_set",
        "participacao_jan", "participacao_set",
        "crescimento_abs_set", "crescimento_pct_set",
        "variacao_participacao_pp", "aumento_frota_jan_set"
    ],
    [
        "Região", "Frota jan/26", "Frota set/26",
        "Participação jan (%)", "Participação set (%)",
        "Crescimento interanual set.", "Crescimento interanual (%)",
        "Variação participação (p.p.)", "Aumento jan-set"
    ]
)

# Totais
total_aumento_jan_set = int(resumo["aumento_frota_jan_set"].sum())
total_crescimento_set = int(resumo["crescimento_abs_set"].sum())

conteudo = f"""# Evolução Regional da Frota — Janeiro a Setembro de 2026

## 1. Objetivo

Analisar a evolução mensal da frota de automóveis e veículos comerciais leves
nas cinco regiões brasileiras, identificando a distribuição geográfica,
a participação regional e o crescimento interanual.

## 2. Escopo e metodologia

- Fonte: dados de frota da SENATRAN, consolidados na camada analítica.
- Período analisado: janeiro a setembro de 2026.
- Comparação interanual: cada mês de 2026 contra o mesmo mês de 2025.
- Abrangência geográfica: cinco regiões brasileiras.
- A categoria `SI` (Sem Informação) foi excluída dos cálculos de participação
  e da evolução regional, pois não possui identificação geográfica.
- A frota representa estoque de veículos registrados na competência,
  não o volume de emplacamentos realizados no mês.

A participação regional utiliza como denominador a soma da frota das
27 UFs identificadas em cada competência.

## 3. Evolução mensal

A tabela apresenta, para cada região e competência:

- Frota total.
- Participação na frota das regiões identificadas.
- Frota do mesmo mês do ano anterior.
- Crescimento absoluto interanual.
- Crescimento percentual interanual.

{tabela_mensal}

## 4. Resumo comparativo — janeiro e setembro de 2026

{tabela_resumo}

## 5. Principais resultados

### Sudeste

- Permanece como a região com maior frota.
- Participação passa de 52,12% em janeiro para 51,95% em setembro,
  redução de 0,17 ponto percentual.
- Apresenta o maior crescimento absoluto interanual em setembro:
  1.257.078 veículos.

### Sul

- Mantém participação praticamente estável: 21,27% para 21,26%.
- Crescimento interanual de 3,71% em setembro.

### Nordeste

- Ganha 0,10 ponto percentual de participação, passando de 13,65% para 13,75%.
- Crescimento interanual de 4,77% em setembro.

### Centro-Oeste

- Ganha 0,02 ponto percentual de participação.
- Crescimento interanual de 3,98% em setembro.

### Norte

- Apresenta o maior crescimento percentual interanual em setembro: 5,24%.
- Participação avança de 3,88% para 3,93%.

## 6. Duas métricas de crescimento que não devem ser confundidas

**Aumento do estoque entre janeiro e setembro de 2026:**
{total_aumento_jan_set:,} veículos.

Esse indicador é a diferença entre os estoques observados em janeiro e setembro
de 2026 nas cinco regiões identificadas.

**Crescimento interanual em setembro de 2026:**
{total_crescimento_set:,} veículos.

Esse indicador compara o estoque de setembro de 2026 com o estoque de
setembro de 2025.

Nenhum dos dois indicadores deve ser denominado volume de emplacamentos.

## 7. Reconciliação e validação

- 45 registros: cinco regiões em nove competências.
- Todas as cinco regiões possuem comparação interanual preenchida.
- Aumento do estoque de janeiro a setembro: {total_aumento_jan_set:,}.
- Crescimento interanual regional em setembro: {total_crescimento_set:,}.
- Soma das participações mensais entre 99,99% e 100,01%,
  diferença decorrente do arredondamento.
- As verificações de consistência foram aprovadas.

## 8. Orientações para o Power BI

Indicadores recomendados:

- Frota regional por competência.
- Participação regional na frota identificada.
- Crescimento absoluto interanual por região.
- Crescimento percentual interanual por região.
- Variação da participação entre competências.
- Evolução do estoque entre janeiro e setembro.

Manter separados os conceitos de estoque da SENATRAN e fluxo de
emplacamentos da FENABRAVE.

## 9. Limitações

- A análise geográfica considera UF e região, não municípios.
- Registros classificados como `SI` não são atribuídos artificialmente
  a nenhuma região.
- O crescimento da frota não representa diretamente emplacamentos,
  pois o estoque também pode ser afetado por baixas, transferências,
  atualizações cadastrais e outros movimentos.
- As participações são calculadas sobre as 27 UFs identificadas,
  não sobre o total nacional que inclui `SI`.

## 10. View de origem

`analytics.vw_frota_uf_mensal`

Documento complementar: `docs/analises/analise_geografica_frota.md`
"""

# Gravação
arquivo_evolucao_regional = (
    PROJECT_ROOT
    / "docs"
    / "analises"
    / "evolucao_regional_frota.md"
)

arquivo_evolucao_regional.parent.mkdir(parents=True, exist_ok=True)
arquivo_evolucao_regional.write_text(conteudo, encoding="utf-8")

print("DOCUMENTAÇÃO CRIADA")
print("=" * 70)
print("Arquivo:", arquivo_evolucao_regional)
print("Tamanho:", arquivo_evolucao_regional.stat().st_size, "bytes")
print("Linhas:", len(conteudo.splitlines()))
print("Registros mensais documentados:", len(df_doc))
print("Regiões documentadas:", df_doc["regiao"].nunique())

In [ ]:
texto_evolucao = arquivo_evolucao_regional.read_text(encoding="utf-8")

verificacoes_documento = {
    "Objetivo e metodologia": "## 1. Objetivo" in texto_evolucao and "## 2. Escopo e metodologia" in texto_evolucao,
    "Tabela com evolução mensal": "## 3. Evolução mensal" in texto_evolucao,
    "Resumo comparativo": "## 4. Resumo comparativo" in texto_evolucao,
    "Sudeste e perda de participação": "0,17 ponto percentual" in texto_evolucao,
    "Norte com maior crescimento percentual": "5,24%" in texto_evolucao,
    "Nordeste ganha participação": "0,10 ponto percentual" in texto_evolucao,
    "Aumento do estoque jan-set": "1.982.385" in texto_evolucao,
    "Crescimento interanual de setembro": "2.881.387" in texto_evolucao,
    "Distinção entre estoque e emplacamentos": "Nenhum dos dois indicadores deve ser denominado volume de emplacamentos" in texto_evolucao,
    "Orientações para Power BI": "## 8. Orientações para o Power BI" in texto_evolucao,
    "Limitações documentadas": "## 9. Limitações" in texto_evolucao,
    "View de origem": "analytics.vw_frota_uf_mensal" in texto_evolucao,
}

print("VALIDAÇÃO DA DOCUMENTAÇÃO REGIONAL")
print("=" * 75)

for descricao, resultado in verificacoes_documento.items():
    print(f"{'OK' if resultado else 'FALHA'} — {descricao}")

print("=" * 75)
print(
    f"Total: {sum(verificacoes_documento.values())}/"
    f"{len(verificacoes_documento)} verificações aprovadas"
)

In [ ]:
for termo in ["1982385", "2881387", "1.982.385", "2.881.387"]:
    print(f"{termo}: {termo in texto_evolucao}")

print("\nTrecho da documentação:")
inicio = texto_evolucao.find("## 6. Duas métricas")
fim = texto_evolucao.find("## 7. Reconciliação")
print(texto_evolucao[inicio:fim])

In [ ]:
texto_evolucao = arquivo_evolucao_regional.read_text(encoding="utf-8")

texto_evolucao = texto_evolucao.replace("1,982,385", "1.982.385")
texto_evolucao = texto_evolucao.replace("2,881,387", "2.881.387")

arquivo_evolucao_regional.write_text(
    texto_evolucao,
    encoding="utf-8"
)

texto_evolucao = arquivo_evolucao_regional.read_text(encoding="utf-8")

print("CORREÇÃO DOS TOTAIS")
print("=" * 60)

verificacoes_totais = {
    "Aumento do estoque jan-set: 1.982.385": "1.982.385" in texto_evolucao,
    "Crescimento interanual setembro: 2.881.387": "2.881.387" in texto_evolucao,
    "Formato anterior removido": "1,982,385" not in texto_evolucao
        and "2,881,387" not in texto_evolucao,
}

for descricao, resultado in verificacoes_totais.items():
    print(f"{'OK' if resultado else 'FALHA'} — {descricao}")

print("=" * 60)
print(
    f"Total: {sum(verificacoes_totais.values())}/"
    f"{len(verificacoes_totais)} verificações aprovadas"
)

In [ ]:
texto_evolucao = arquivo_evolucao_regional.read_text(encoding="utf-8")

verificacoes_documento = {
    "Objetivo e metodologia": "## 1. Objetivo" in texto_evolucao and "## 2. Escopo e metodologia" in texto_evolucao,
    "Tabela com evolução mensal": "## 3. Evolução mensal" in texto_evolucao,
    "Resumo comparativo": "## 4. Resumo comparativo" in texto_evolucao,
    "Sudeste e perda de participação": "0,17 ponto percentual" in texto_evolucao,
    "Norte com maior crescimento percentual": "5,24%" in texto_evolucao,
    "Nordeste ganha participação": "0,10 ponto percentual" in texto_evolucao,
    "Aumento do estoque jan-set": "1.982.385" in texto_evolucao,
    "Crescimento interanual de setembro": "2.881.387" in texto_evolucao,
    "Distinção entre estoque e emplacamentos": "Nenhum dos dois indicadores deve ser denominado volume de emplacamentos" in texto_evolucao,
    "Orientações para Power BI": "## 8. Orientações para o Power BI" in texto_evolucao,
    "Limitações documentadas": "## 9. Limitações" in texto_evolucao,
    "View de origem": "analytics.vw_frota_uf_mensal" in texto_evolucao,
}

print("VALIDAÇÃO FINAL DA DOCUMENTAÇÃO REGIONAL")
print("=" * 75)

for descricao, resultado in verificacoes_documento.items():
    print(f"{'OK' if resultado else 'FALHA'} — {descricao}")

print("=" * 75)
print(
    f"Total: {sum(verificacoes_documento.values())}/"
    f"{len(verificacoes_documento)} verificações aprovadas"
)

In [ ]:
sql_frota_tipo_mensal = """
WITH frota_tipo AS (
    SELECT
        f.data_referencia AS competencia,
        t.tipo_veiculo,
        SUM(f.qtd_veiculos)::bigint AS frota_total
    FROM analytics.fato_frota f
    INNER JOIN analytics.dim_tipo_veiculo t
        ON t.id_tipo_veiculo = f.id_tipo_veiculo
    WHERE t.tipo_veiculo IN (
        'AUTOMOVEL',
        'CAMINHONETE',
        'CAMIONETA',
        'UTILITARIO'
    )
    GROUP BY
        f.data_referencia,
        t.tipo_veiculo
),
comparacao AS (
    SELECT
        competencia,
        tipo_veiculo,
        frota_total,
        LAG(frota_total, 12) OVER (
            PARTITION BY tipo_veiculo
            ORDER BY competencia
        ) AS frota_mes_ano_anterior
    FROM frota_tipo
)
SELECT
    competencia,
    tipo_veiculo,
    frota_total,
    ROUND(
        100.0 * frota_total
        / SUM(frota_total) OVER (PARTITION BY competencia),
        2
    ) AS participacao_pct,
    frota_mes_ano_anterior,
    frota_total - frota_mes_ano_anterior
        AS crescimento_absoluto,
    ROUND(
        100.0 * (
            frota_total::numeric
            / NULLIF(frota_mes_ano_anterior, 0) - 1
        ),
        2
    ) AS crescimento_pct
FROM comparacao
ORDER BY competencia, frota_total DESC;
"""

df_frota_tipo = pd.read_sql_query(
    text(sql_frota_tipo_mensal),
    engine
)

df_frota_tipo["competencia"] = pd.to_datetime(
    df_frota_tipo["competencia"]
)

print("EVOLUÇÃO DA FROTA POR TIPO — JAN/2025 A SET/2026")
print("=" * 115)
print(df_frota_tipo.to_string(index=False))

print("=" * 115)
print("Registros:", len(df_frota_tipo))
print("Tipos de veículos:", df_frota_tipo["tipo_veiculo"].nunique())
print("Competências:", df_frota_tipo["competencia"].nunique())
print(
    "Comparações interanuais preenchidas:",
    df_frota_tipo["crescimento_pct"].notna().sum()
)

In [ ]:
# Preparação
df_validacao_tipo = df_frota_tipo.copy()

df_validacao_tipo["competencia"] = pd.to_datetime(
    df_validacao_tipo["competencia"]
)

# Totais nacionais da view já validada
df_nacional_validacao = pd.read_sql_query(
    text("""
        SELECT competencia, frota_total
        FROM analytics.vw_frota_nacional_mensal
        ORDER BY competencia
    """),
    engine
)

df_nacional_validacao["competencia"] = pd.to_datetime(
    df_nacional_validacao["competencia"]
)

# Consolidação por competência
totais_tipo = (
    df_validacao_tipo
    .groupby("competencia", as_index=False)
    .agg(
        frota_soma_tipos=("frota_total", "sum"),
        participacao_soma=("participacao_pct", "sum"),
        quantidade_tipos=("tipo_veiculo", "nunique")
    )
)

conferencia = totais_tipo.merge(
    df_nacional_validacao,
    on="competencia",
    how="outer",
    validate="one_to_one"
)

conferencia["diferenca"] = (
    conferencia["frota_soma_tipos"] - conferencia["frota_total"]
)

# Verificações
verificacoes_tipo = {
    "84 registros": len(df_validacao_tipo) == 84,
    "4 tipos de veículos": df_validacao_tipo["tipo_veiculo"].nunique() == 4,
    "21 competências": df_validacao_tipo["competencia"].nunique() == 21,
    "Sem duplicidade competência/tipo": not df_validacao_tipo.duplicated(
        ["competencia", "tipo_veiculo"]
    ).any(),
    "4 tipos em todas as competências": (
        totais_tipo["quantidade_tipos"] == 4
    ).all(),
    "Soma dos tipos igual à frota nacional": (
        conferencia["diferenca"] == 0
    ).all(),
    "Participações próximas de 100%": (
        totais_tipo["participacao_soma"].between(99.98, 100.02)
    ).all(),
    "36 comparações interanuais preenchidas": (
        df_validacao_tipo["crescimento_pct"].notna().sum() == 36
    ),
    "48 comparações anteriores sem base anual": (
        df_validacao_tipo["crescimento_pct"].isna().sum() == 48
    )
}

print("VALIDAÇÃO DA FROTA POR TIPO")
print("=" * 75)

for descricao, resultado in verificacoes_tipo.items():
    print(f"{'OK' if resultado else 'FALHA'} — {descricao}")

print("=" * 75)
print(
    f"Total: {sum(verificacoes_tipo.values())}/"
    f"{len(verificacoes_tipo)} verificações aprovadas"
)

print("\nCONFERÊNCIA DOS TOTAIS MENSAIS")
print(
    conferencia[
        ["competencia", "frota_soma_tipos", "frota_total", "diferenca"]
    ].to_string(index=False)
)

In [ ]:
# Resumo comparativo por tipo de veículo

df_tipo = df_frota_tipo.copy()
df_tipo["competencia"] = pd.to_datetime(df_tipo["competencia"])

def obter_mes(data, coluna):
    return (
        df_tipo[df_tipo["competencia"] == pd.Timestamp(data)]
        .set_index("tipo_veiculo")[coluna]
    )

jan25_frota = obter_mes("2025-01-01", "frota_total")
jan25_part = obter_mes("2025-01-01", "participacao_pct")

set25_frota = obter_mes("2025-09-01", "frota_total")
set26_frota = obter_mes("2026-09-01", "frota_total")
set26_part = obter_mes("2026-09-01", "participacao_pct")

resumo_tipo = pd.DataFrame(index=set26_frota.index)

resumo_tipo["frota_set25"] = set25_frota
resumo_tipo["frota_set26"] = set26_frota

resumo_tipo["crescimento_abs_interanual"] = (
    resumo_tipo["frota_set26"] - resumo_tipo["frota_set25"]
)

resumo_tipo["crescimento_pct_interanual"] = (
    100 * (
        resumo_tipo["frota_set26"]
        / resumo_tipo["frota_set25"] - 1
    )
).round(2)

resumo_tipo["participacao_jan25_pct"] = jan25_part
resumo_tipo["participacao_set26_pct"] = set26_part

resumo_tipo["variacao_participacao_pp"] = (
    resumo_tipo["participacao_set26_pct"]
    - resumo_tipo["participacao_jan25_pct"]
).round(2)

resumo_tipo["aumento_frota_jan25_set26"] = (
    set26_frota - jan25_frota
)

total_crescimento = resumo_tipo[
    "crescimento_abs_interanual"
].sum()

resumo_tipo["contribuicao_crescimento_pct"] = (
    100 * resumo_tipo["crescimento_abs_interanual"]
    / total_crescimento
).round(2)

resumo_tipo = resumo_tipo.sort_values(
    "frota_set26",
    ascending=False
)

print("RESUMO COMPARATIVO DA FROTA POR TIPO")
print("Janeiro/2025 a setembro/2026")
print("=" * 125)
print(resumo_tipo.to_string())

print("\nTOTAIS")
print("=" * 75)
print(
    "Crescimento interanual setembro/2026:",
    f"{total_crescimento:,.0f}".replace(",", ".")
)
print(
    "Aumento do estoque jan/2025 a set/2026:",
    f"{resumo_tipo['aumento_frota_jan25_set26'].sum():,.0f}".replace(",", ".")
)

print("\nRANKING DE CRESCIMENTO PERCENTUAL INTERANUAL")
print(
    resumo_tipo["crescimento_pct_interanual"]
    .sort_values(ascending=False)
    .to_string()
)

print("\nRANKING DE CRESCIMENTO ABSOLUTO INTERANUAL")
print(
    resumo_tipo["crescimento_abs_interanual"]
    .sort_values(ascending=False)
    .to_string()
)

In [ ]:
from pathlib import Path

arquivo = PROJECT_ROOT / "docs" / "analises" / "analise_frota_tipo_veiculo.md"
arquivo.parent.mkdir(parents=True, exist_ok=True)

conteudo = """# Análise da Frota por Tipo de Veículo

## 1. Objetivo

Analisar a composição e a evolução da frota brasileira de automóveis e veículos comerciais leves, identificando diferenças de participação e crescimento entre os tipos de veículos.

## 2. Escopo e metodologia

- **Fonte:** SENATRAN — frota por UF, município, tipo, espécie e eixos.
- **Período:** janeiro de 2025 a setembro de 2026.
- **Tipos incluídos:** AUTOMOVEL, CAMINHONETE, CAMIONETA e UTILITARIO.
- **Indicador:** quantidade de veículos registrados na frota em cada competência.
- **Comparação interanual:** setembro de 2026 contra setembro de 2025.
- **Participação:** quantidade de veículos do tipo dividida pela frota total da competência.
- **Aumento do período:** diferença entre o estoque de janeiro de 2025 e o estoque de setembro de 2026.

A variação da frota representa uma mudança no estoque de veículos registrados. Não deve ser interpretada como quantidade de emplacamentos no período.

## 3. Comparativo por tipo

| Tipo | Frota set/2025 | Frota set/2026 | Crescimento absoluto | Crescimento interanual | Participação jan/2025 | Participação set/2026 | Variação da participação |
|---|---:|---:|---:|---:|---:|---:|---:|
| Automóvel | 64.410.967 | 66.145.154 | 1.734.187 | 2,69% | 79,31% | 78,18% | -1,13 p.p. |
| Caminhonete | 9.991.137 | 10.574.406 | 583.269 | 5,84% | 12,04% | 12,50% | +0,46 p.p. |
| Camioneta | 5.275.144 | 5.563.084 | 287.940 | 5,46% | 6,39% | 6,58% | +0,19 p.p. |
| Utilitário | 1.982.104 | 2.324.956 | 342.852 | 17,30% | 2,26% | 2,75% | +0,49 p.p. |

## 4. Crescimento absoluto e contribuição

O crescimento interanual da frota entre setembro de 2025 e setembro de 2026 foi de **2.948.248 veículos**.

| Tipo | Crescimento absoluto | Contribuição para o crescimento |
|---|---:|---:|
| Automóvel | 1.734.187 | 58,82% |
| Caminhonete | 583.269 | 19,78% |
| Utilitário | 342.852 | 11,63% |
| Camioneta | 287.940 | 9,77% |

O aumento do estoque entre janeiro de 2025 e setembro de 2026 foi de **4.584.170 veículos**.

## 5. Principais resultados

### 5.1 Automóveis

Os automóveis mantêm a maior participação na frota, com 78,18% em setembro de 2026. Embora tenham apresentado crescimento interanual de 2,69%, sua participação caiu 1,13 ponto percentual em relação a janeiro de 2025.

Apesar do crescimento percentual mais moderado, foram responsáveis por 58,82% do aumento absoluto da frota no comparativo entre setembro de 2025 e setembro de 2026.

### 5.2 Utilitários

Os utilitários apresentaram o maior crescimento percentual interanual, de 17,30%. Sua participação passou de 2,26% em janeiro de 2025 para 2,75% em setembro de 2026, um ganho de 0,49 ponto percentual.

É um segmento de menor participação, mas com expansão proporcional mais acelerada.

### 5.3 Caminhonetes

As caminhonetes cresceram 5,84% no comparativo interanual e ganharam 0,46 ponto percentual de participação entre janeiro de 2025 e setembro de 2026.

### 5.4 Camionetas

As camionetas cresceram 5,46% no comparativo interanual. Sua contribuição absoluta foi a menor entre os quatro tipos, correspondendo a 9,77% do crescimento interanual.

## 6. Rankings

**Crescimento percentual interanual:**

1. Utilitário: 17,30%
2. Caminhonete: 5,84%
3. Camioneta: 5,46%
4. Automóvel: 2,69%

**Crescimento absoluto interanual:**

1. Automóvel: 1.734.187
2. Caminhonete: 583.269
3. Utilitário: 342.852
4. Camioneta: 287.940

## 7. Validação dos dados

Foram aprovadas as nove verificações:

- 84 registros analisados.
- Quatro tipos de veículos.
- 21 competências mensais.
- Ausência de duplicidade por competência e tipo.
- Presença dos quatro tipos em todas as competências.
- Soma dos tipos igual à frota nacional em todos os meses.
- Soma das participações próxima de 100% em todas as competências.
- 36 comparações interanuais preenchidas.
- 48 comparações sem base anual anterior, correspondentes às competências de 2025.

A reconciliação mensal apresentou diferença zero entre a soma dos quatro tipos e a frota nacional nas 21 competências.

## 8. Recomendações para o Power BI

A análise pode alimentar:

- Gráfico de evolução mensal da frota por tipo.
- Gráfico de participação percentual por tipo.
- Indicadores de crescimento interanual absoluto e percentual.
- Ranking de tipos por crescimento.
- Comparação da composição da frota entre janeiro de 2025 e setembro de 2026.

Os indicadores devem distinguir crescimento absoluto, crescimento percentual e variação de participação em pontos percentuais.

## 9. Fonte analítica

Consulta baseada na tabela `analytics.fato_frota` e na dimensão `analytics.dim_tipo_veiculo`, com reconciliação dos totais contra `analytics.vw_frota_nacional_mensal`.

---
"""

arquivo.write_text(conteudo, encoding="utf-8")

print("DOCUMENTAÇÃO GERADA")
print("=" * 70)
print(f"Arquivo: {arquivo}")
print(f"Tamanho: {arquivo.stat().st_size:,} bytes")
print(f"Linhas: {len(conteudo.splitlines())}")
print(f"Existe: {arquivo.exists()}")

In [ ]:
conteudo_doc = arquivo.read_text(encoding="utf-8")

verificacoes_doc = {
    "Arquivo existe": arquivo.exists(),
    "Seção de metodologia": "## 2. Escopo e metodologia" in conteudo_doc,
    "Comparativo por tipo": "## 3. Comparativo por tipo" in conteudo_doc,
    "Crescimento absoluto e contribuição": "## 4. Crescimento absoluto e contribuição" in conteudo_doc,
    "Principais resultados": "## 5. Principais resultados" in conteudo_doc,
    "Rankings": "## 6. Rankings" in conteudo_doc,
    "Validação dos dados": "## 7. Validação dos dados" in conteudo_doc,
    "Recomendações para Power BI": "## 8. Recomendações para o Power BI" in conteudo_doc,
    "Total interanual registrado": "2.948.248" in conteudo_doc,
    "Aumento do período registrado": "4.584.170" in conteudo_doc,
    "Crescimento dos utilitários": "17,30%" in conteudo_doc,
    "Reconciliação sem diferenças": "diferença zero" in conteudo_doc,
}

for descricao, resultado in verificacoes_doc.items():
    print(f"{'OK' if resultado else 'FALHOU'} — {descricao}")

print("=" * 70)
print(f"Total: {sum(verificacoes_doc.values())}/{len(verificacoes_doc)} verificações aprovadas")

In [ ]:
df_frota_especie = pd.read_sql_query(
    text("""
        WITH frota_especie AS (
            SELECT
                f.data_referencia AS competencia,
                e.especie_veiculo,
                SUM(f.qtd_veiculos)::bigint AS frota_total
            FROM analytics.fato_frota f
            INNER JOIN analytics.dim_especie_veiculo e
                ON e.id_especie_veiculo = f.id_especie_veiculo
            GROUP BY
                f.data_referencia,
                e.especie_veiculo
        ),
        comparacao AS (
            SELECT
                competencia,
                especie_veiculo,
                frota_total,
                LAG(frota_total, 12) OVER (
                    PARTITION BY especie_veiculo
                    ORDER BY competencia
                ) AS frota_mes_ano_anterior
            FROM frota_especie
        )
        SELECT
            competencia,
            especie_veiculo,
            frota_total,
            ROUND(
                100.0 * frota_total /
                SUM(frota_total) OVER (PARTITION BY competencia),
                2
            ) AS participacao_pct,
            frota_mes_ano_anterior,
            frota_total - frota_mes_ano_anterior
                AS crescimento_absoluto,
            ROUND(
                100.0 * (
                    frota_total::numeric /
                    NULLIF(frota_mes_ano_anterior, 0) - 1
                ),
                2
            ) AS crescimento_pct
        FROM comparacao
        ORDER BY competencia, frota_total DESC
    """),
    engine
)

df_frota_especie["competencia"] = pd.to_datetime(
    df_frota_especie["competencia"]
)

print("ANÁLISE DA FROTA POR ESPÉCIE")
print("=" * 75)
print(f"Registros: {len(df_frota_especie):,}")
print(f"Espécies: {df_frota_especie['especie_veiculo'].nunique()}")
print(f"Competências: {df_frota_especie['competencia'].nunique()}")
print(
    "Comparações interanuais preenchidas:",
    df_frota_especie["frota_mes_ano_anterior"].notna().sum()
)

print("\nESPÉCIES E QUANTIDADE DE REGISTROS")
display(
    df_frota_especie.groupby("especie_veiculo")
    .size()
    .rename("registros")
    .to_frame()
)

print("\nCOMPOSIÇÃO EM SETEMBRO DE 2026")
display(
    df_frota_especie[
        df_frota_especie["competencia"] == "2026-09-01"
    ].sort_values("frota_total", ascending=False)
)

In [ ]:
df_validacao_especie = df_frota_especie.copy()

df_nacional_especie = pd.read_sql_query(
    text("""
        SELECT competencia, frota_total
        FROM analytics.vw_frota_nacional_mensal
        ORDER BY competencia
    """),
    engine
)

df_validacao_especie["competencia"] = pd.to_datetime(
    df_validacao_especie["competencia"]
)
df_nacional_especie["competencia"] = pd.to_datetime(
    df_nacional_especie["competencia"]
)

totais_especie = (
    df_validacao_especie
    .groupby("competencia", as_index=False)
    .agg(
        frota_soma_especies=("frota_total", "sum"),
        participacao_soma=("participacao_pct", "sum"),
        quantidade_especies=("especie_veiculo", "nunique")
    )
)

conferencia_especie = totais_especie.merge(
    df_nacional_especie,
    on="competencia",
    how="outer",
    validate="one_to_one"
)

conferencia_especie["diferenca"] = (
    conferencia_especie["frota_soma_especies"]
    - conferencia_especie["frota_total"]
)

verificacoes_especie = {
    "168 registros": len(df_validacao_especie) == 168,
    "8 espécies de veículos": (
        df_validacao_especie["especie_veiculo"].nunique() == 8
    ),
    "21 competências": (
        df_validacao_especie["competencia"].nunique() == 21
    ),
    "Sem duplicidade competência/espécie": (
        not df_validacao_especie.duplicated(
            ["competencia", "especie_veiculo"]
        ).any()
    ),
    "8 espécies em todas as competências": (
        conferencia_especie["quantidade_especies"] == 8
    ).all(),
    "Soma das espécies igual à frota nacional": (
        conferencia_especie["diferenca"] == 0
    ).all(),
    "Participações próximas de 100%": (
        conferencia_especie["participacao_soma"]
        .between(99.98, 100.02)
    ).all(),
    "72 comparações interanuais preenchidas": (
        df_validacao_especie["frota_mes_ano_anterior"].notna().sum() == 72
    ),
    "96 comparações sem base anual anterior": (
        df_validacao_especie["frota_mes_ano_anterior"].isna().sum() == 96
    )
}

print("VALIDAÇÃO DA FROTA POR ESPÉCIE")
print("=" * 75)

for descricao, resultado in verificacoes_especie.items():
    print(f"{'OK' if resultado else 'FALHOU'} — {descricao}")

print("=" * 75)
print(
    f"Total: {sum(verificacoes_especie.values())}/"
    f"{len(verificacoes_especie)} verificações aprovadas"
)

print("\nCONFERÊNCIA DOS TOTAIS MENSAIS")
display(
    conferencia_especie[
        ["competencia", "frota_soma_especies", "frota_total", "diferenca"]
    ]
)

In [ ]:
# Seleciona as competências necessárias
df_especie_resumo = df_frota_especie.copy()

datas = {
    "jan25": pd.Timestamp("2025-01-01"),
    "set25": pd.Timestamp("2025-09-01"),
    "set26": pd.Timestamp("2026-09-01")
}

# Frota em setembro de 2025 e setembro de 2026
set25 = (
    df_especie_resumo[
        df_especie_resumo["competencia"] == datas["set25"]
    ][["especie_veiculo", "frota_total"]]
    .rename(columns={"frota_total": "frota_set25"})
)

set26 = (
    df_especie_resumo[
        df_especie_resumo["competencia"] == datas["set26"]
    ][["especie_veiculo", "frota_total", "participacao_pct"]]
    .rename(columns={
        "frota_total": "frota_set26",
        "participacao_pct": "participacao_set26_pct"
    })
)

# Participação em janeiro de 2025
jan25 = (
    df_especie_resumo[
        df_especie_resumo["competencia"] == datas["jan25"]
    ][["especie_veiculo", "participacao_pct", "frota_total"]]
    .rename(columns={
        "participacao_pct": "participacao_jan25_pct",
        "frota_total": "frota_jan25"
    })
)

# Consolidação
resumo_especie = (
    set25
    .merge(set26, on="especie_veiculo", validate="one_to_one")
    .merge(jan25, on="especie_veiculo", validate="one_to_one")
    .set_index("especie_veiculo")
)

resumo_especie["crescimento_abs_interanual"] = (
    resumo_especie["frota_set26"] - resumo_especie["frota_set25"]
)

resumo_especie["crescimento_pct_interanual"] = (
    100 * (
        resumo_especie["frota_set26"] /
        resumo_especie["frota_set25"] - 1
    )
).round(2)

resumo_especie["variacao_participacao_pp"] = (
    resumo_especie["participacao_set26_pct"] -
    resumo_especie["participacao_jan25_pct"]
).round(2)

resumo_especie["aumento_frota_jan25_set26"] = (
    resumo_especie["frota_set26"] -
    resumo_especie["frota_jan25"]
)

crescimento_total = resumo_especie[
    "crescimento_abs_interanual"
].sum()

resumo_especie["contribuicao_crescimento_pct"] = (
    100 * resumo_especie["crescimento_abs_interanual"] /
    crescimento_total
).round(2)

# Ordenação por crescimento absoluto
resumo_especie = resumo_especie.sort_values(
    "crescimento_abs_interanual",
    ascending=False
)

print("RESUMO COMPARATIVO DA FROTA POR ESPÉCIE")
print("Janeiro/2025 a setembro/2026")
print("=" * 125)
display(resumo_especie.round(2))

print("\nTOTAIS")
print("=" * 75)
print(f"Crescimento interanual setembro/2026: {crescimento_total:,.0f}")
print(
    "Aumento do estoque jan/2025 a set/2026:",
    f"{resumo_especie['aumento_frota_jan25_set26'].sum():,.0f}"
)

print("\nRANKING DE CRESCIMENTO PERCENTUAL INTERANUAL")
display(
    resumo_especie["crescimento_pct_interanual"]
    .sort_values(ascending=False)
)

print("\nRANKING DE CRESCIMENTO ABSOLUTO INTERANUAL")
display(
    resumo_especie["crescimento_abs_interanual"]
    .sort_values(ascending=False)
)

In [ ]:
from pathlib import Path

arquivo_especie = (
    PROJECT_ROOT / "docs" / "analises" /
    "analise_frota_especie_veiculo.md"
)

conteudo_especie = """# Análise da Frota por Espécie de Veículo

## 1. Objetivo

Analisar a composição e a evolução da frota brasileira por espécie de veículo, identificando diferenças de participação, crescimento absoluto e crescimento percentual.

## 2. Escopo e metodologia

- **Fonte:** SENATRAN — frota por UF, município, tipo, espécie e eixos.
- **Período:** janeiro de 2025 a setembro de 2026.
- **Espécies:** Passageiro, Misto, Carga, Especial, Coleção, Corrida, Tração e Sem Informação.
- **Comparação interanual:** setembro de 2026 contra setembro de 2025.
- **Participação:** quantidade de veículos da espécie dividida pela frota total da competência.
- **Aumento do período:** diferença entre o estoque de janeiro de 2025 e o estoque de setembro de 2026.

A variação da frota representa uma mudança no estoque de veículos registrados, não a quantidade de emplacamentos no período.

## 3. Comparativo por espécie

| Espécie | Frota set/2025 | Frota set/2026 | Crescimento absoluto | Crescimento interanual | Participação jan/2025 | Participação set/2026 | Variação da participação |
|---|---:|---:|---:|---:|---:|---:|---:|
| Passageiro | 63.642.350 | 65.352.665 | 1.710.315 | 2,69% | 78,37% | 77,24% | -1,13 p.p. |
| Misto | 7.264.244 | 7.893.758 | 629.514 | 8,67% | 8,66% | 9,33% | +0,67 p.p. |
| Especial | 4.467.673 | 4.834.935 | 367.262 | 8,22% | 5,31% | 5,71% | +0,40 p.p. |
| Carga | 6.142.027 | 6.349.431 | 207.404 | 3,38% | 7,51% | 7,50% | -0,01 p.p. |
| Coleção | 126.454 | 160.243 | 33.789 | 26,72% | 0,13% | 0,19% | +0,06 p.p. |
| Corrida | 740 | 739 | -1 | -0,14% | 0,00% | 0,00% | 0,00 p.p. |
| Tração | 145 | 143 | -2 | -1,38% | 0,00% | 0,00% | 0,00 p.p. |
| Sem Informação | 15.719 | 15.686 | -33 | -0,21% | 0,02% | 0,02% | 0,00 p.p. |

## 4. Crescimento absoluto e contribuição

O crescimento interanual da frota entre setembro de 2025 e setembro de 2026 foi de **2.948.248 veículos**.

| Espécie | Crescimento absoluto | Contribuição para o crescimento |
|---|---:|---:|
| Passageiro | 1.710.315 | 58,01% |
| Misto | 629.514 | 21,35% |
| Especial | 367.262 | 12,46% |
| Carga | 207.404 | 7,03% |
| Coleção | 33.789 | 1,15% |
| Corrida | -1 | aproximadamente 0,00% |
| Tração | -2 | aproximadamente 0,00% |
| Sem Informação | -33 | aproximadamente 0,00% |

O aumento do estoque entre janeiro de 2025 e setembro de 2026 foi de **4.584.170 veículos**.

## 5. Principais resultados

### 5.1 Passageiro

Passageiro representa 77,24% da frota em setembro de 2026. Apresentou crescimento interanual de 2,69%, com aumento absoluto de 1.710.315 veículos.

Apesar de continuar dominante e responder por 58,01% do crescimento interanual, sua participação caiu 1,13 ponto percentual em relação a janeiro de 2025.

### 5.2 Misto

Misto apresentou crescimento interanual de 8,67%, alcançando participação de 9,33%. Ganhou 0,67 ponto percentual desde janeiro de 2025 e respondeu por 21,35% do crescimento absoluto interanual.

### 5.3 Especial

Especial cresceu 8,22% no comparativo interanual e ganhou 0,40 ponto percentual de participação, chegando a 5,71% da frota.

### 5.4 Carga

Carga cresceu 3,38% no comparativo interanual. Sua participação permaneceu praticamente estável, passando de 7,51% para 7,50%.

### 5.5 Coleção

Coleção apresentou o maior crescimento percentual, de 26,72%. Entretanto, sua participação foi de apenas 0,19% em setembro de 2026. O resultado deve ser interpretado considerando a pequena base de veículos.

### 5.6 Demais espécies

Corrida, Tração e Sem Informação apresentaram pequenas reduções absolutas. Seus volumes são muito reduzidos em relação à frota total, portanto suas variações têm impacto limitado sobre a composição nacional.

## 6. Rankings

**Crescimento percentual interanual:**

1. Coleção: 26,72%
2. Misto: 8,67%
3. Especial: 8,22%
4. Carga: 3,38%
5. Passageiro: 2,69%
6. Corrida: -0,14%
7. Sem Informação: -0,21%
8. Tração: -1,38%

**Crescimento absoluto interanual:**

1. Passageiro: 1.710.315
2. Misto: 629.514
3. Especial: 367.262
4. Carga: 207.404
5. Coleção: 33.789
6. Corrida: -1
7. Tração: -2
8. Sem Informação: -33

## 7. Validação dos dados

Foram aprovadas as nove verificações:

- 168 registros analisados.
- Oito espécies de veículos.
- 21 competências mensais.
- Ausência de duplicidade por competência e espécie.
- Presença das oito espécies em todas as competências.
- Soma das espécies igual à frota nacional em todos os meses.
- Soma das participações próxima de 100% em todas as competências.
- 72 comparações interanuais preenchidas.
- 96 comparações sem base anual anterior, correspondentes às competências de 2025.

A reconciliação mensal apresentou diferença zero entre a soma das oito espécies e a frota nacional nas 21 competências.

## 8. Recomendações para o Power BI

A análise pode alimentar:

- Evolução mensal da frota por espécie.
- Participação percentual das espécies na frota.
- Crescimento interanual absoluto e percentual.
- Ranking de espécies por crescimento.
- Comparação da composição entre janeiro de 2025 e setembro de 2026.

Recomenda-se dar destaque visual às espécies Passageiro, Misto, Especial e Carga. Coleção pode ser apresentada em indicador ou visual complementar, com atenção à sua pequena participação.

Os indicadores devem distinguir crescimento absoluto, crescimento percentual e variação de participação em pontos percentuais.

## 9. Fonte analítica

Consulta baseada na tabela `analytics.fato_frota` e na dimensão `analytics.dim_especie_veiculo`, com reconciliação dos totais contra `analytics.vw_frota_nacional_mensal`.

---
"""

arquivo_especie.write_text(conteudo_especie, encoding="utf-8")

print("DOCUMENTAÇÃO GERADA")
print("=" * 70)
print(f"Arquivo: {arquivo_especie}")
print(f"Tamanho: {arquivo_especie.stat().st_size:,} bytes")
print(f"Linhas: {len(conteudo_especie.splitlines())}")
print(f"Existe: {arquivo_especie.exists()}")

In [ ]:
conteudo_doc_especie = arquivo_especie.read_text(encoding="utf-8")

verificacoes_doc_especie = {
    "Arquivo existe": arquivo_especie.exists(),
    "Seção de metodologia": "## 2. Escopo e metodologia" in conteudo_doc_especie,
    "Comparativo por espécie": "## 3. Comparativo por espécie" in conteudo_doc_especie,
    "Crescimento absoluto e contribuição": "## 4. Crescimento absoluto e contribuição" in conteudo_doc_especie,
    "Principais resultados": "## 5. Principais resultados" in conteudo_doc_especie,
    "Rankings": "## 6. Rankings" in conteudo_doc_especie,
    "Validação dos dados": "## 7. Validação dos dados" in conteudo_doc_especie,
    "Recomendações para Power BI": "## 8. Recomendações para o Power BI" in conteudo_doc_especie,
    "Crescimento interanual total": "2.948.248" in conteudo_doc_especie,
    "Aumento do período": "4.584.170" in conteudo_doc_especie,
    "Crescimento de Coleção": "26,72%" in conteudo_doc_especie,
    "Reconciliação sem diferenças": "diferença zero" in conteudo_doc_especie,
}

for descricao, resultado in verificacoes_doc_especie.items():
    print(f"{'OK' if resultado else 'FALHOU'} — {descricao}")

print("=" * 70)
print(
    f"Total: {sum(verificacoes_doc_especie.values())}/"
    f"{len(verificacoes_doc_especie)} verificações aprovadas"
)

In [ ]:
df_views = pd.read_sql_query(
    text("""
        SELECT
            table_name AS view_name,
            COUNT(*) AS quantidade_colunas,
            STRING_AGG(
                column_name,
                ', ' ORDER BY ordinal_position
            ) AS colunas
        FROM information_schema.columns
        WHERE table_schema = 'analytics'
          AND table_name IN (
              SELECT table_name
              FROM information_schema.views
              WHERE table_schema = 'analytics'
          )
        GROUP BY table_name
        ORDER BY table_name
    """),
    engine
)

print("INVENTÁRIO DAS VIEWS ANALÍTICAS")
print("=" * 90)
print(f"Total de views: {len(df_views)}")
display(df_views)

In [ ]:
df_definicoes = pd.read_sql_query(
    text("""
        SELECT
            viewname AS nome_view,
            definition AS definicao_sql
        FROM pg_views
        WHERE schemaname = 'analytics'
        ORDER BY viewname
    """),
    engine
)

for _, linha in df_definicoes.iterrows():
    print("=" * 100)
    print(f"VIEW: {linha['nome_view']}")
    print("=" * 100)
    print(linha["definicao_sql"])
    print()

In [ ]:
sql_view_tipo = """
CREATE OR REPLACE VIEW analytics.vw_frota_tipo_mensal AS
WITH frota_tipo AS (
    SELECT
        dc.competencia,
        dc.ano,
        dc.mes,
        dc.nome_mes,
        dc.mes_abreviado,
        dc.ano_mes,
        t.id_tipo_veiculo,
        t.tipo_veiculo,
        SUM(f.qtd_veiculos)::bigint AS frota_total
    FROM analytics.fato_frota f
    INNER JOIN analytics.dim_calendario dc
        ON dc.competencia = f.data_referencia
    INNER JOIN analytics.dim_tipo_veiculo t
        ON t.id_tipo_veiculo = f.id_tipo_veiculo
    WHERE t.tipo_veiculo IN (
        'AUTOMOVEL',
        'CAMINHONETE',
        'CAMIONETA',
        'UTILITARIO'
    )
    GROUP BY
        dc.competencia,
        dc.ano,
        dc.mes,
        dc.nome_mes,
        dc.mes_abreviado,
        dc.ano_mes,
        t.id_tipo_veiculo,
        t.tipo_veiculo
),
comparacao AS (
    SELECT
        *,
        LAG(frota_total, 12) OVER (
            PARTITION BY id_tipo_veiculo
            ORDER BY competencia
        ) AS frota_mes_ano_anterior
    FROM frota_tipo
)
SELECT
    competencia,
    ano,
    mes,
    nome_mes,
    mes_abreviado,
    ano_mes,
    id_tipo_veiculo,
    tipo_veiculo,
    frota_total,
    ROUND(
        100.0 * frota_total /
        NULLIF(SUM(frota_total) OVER (PARTITION BY competencia), 0),
        2
    ) AS participacao_pct,
    frota_mes_ano_anterior,
    frota_total - frota_mes_ano_anterior AS crescimento_absoluto,
    ROUND(
        100.0 * (
            frota_total::numeric /
            NULLIF(frota_mes_ano_anterior, 0) - 1
        ),
        2
    ) AS crescimento_pct
FROM comparacao;
"""

with engine.begin() as conn:
    conn.execute(text(sql_view_tipo))

df_view_tipo = pd.read_sql_query(
    text("""
        SELECT *
        FROM analytics.vw_frota_tipo_mensal
        ORDER BY competencia, frota_total DESC
    """),
    engine
)

print("VIEW CRIADA E CONSULTADA")
print("=" * 75)
print(f"Registros: {len(df_view_tipo)}")
print(f"Tipos: {df_view_tipo['tipo_veiculo'].nunique()}")
print(f"Competências: {df_view_tipo['competencia'].nunique()}")
print("\nAMOSTRA — SETEMBRO DE 2026")

display(
    df_view_tipo[
        pd.to_datetime(df_view_tipo["competencia"])
        == pd.Timestamp("2026-09-01")
    ]
)

In [ ]:
# Preparar os dados de referência
df_referencia_tipo = df_frota_tipo.copy()
df_referencia_tipo["competencia"] = pd.to_datetime(
    df_referencia_tipo["competencia"]
)

df_view_tipo["competencia"] = pd.to_datetime(
    df_view_tipo["competencia"]
)

colunas_comparacao = [
    "competencia",
    "tipo_veiculo",
    "frota_total",
    "participacao_pct",
    "frota_mes_ano_anterior",
    "crescimento_absoluto",
    "crescimento_pct"
]

referencia = df_referencia_tipo[colunas_comparacao].sort_values(
    ["competencia", "tipo_veiculo"]
).reset_index(drop=True)

view = df_view_tipo[colunas_comparacao].sort_values(
    ["competencia", "tipo_veiculo"]
).reset_index(drop=True)

# Comparar valores, tratando NaN como equivalentes
comparacao_tipo = referencia.compare(
    view,
    keep_shape=False,
    keep_equal=False
)

verificacoes_view_tipo = {
    "84 registros na view": len(view) == 84,
    "Mesmas competências": (
        referencia["competencia"].equals(view["competencia"])
    ),
    "Mesmos tipos": (
        referencia["tipo_veiculo"].equals(view["tipo_veiculo"])
    ),
    "Mesmas quantidades": (
        referencia["frota_total"].equals(view["frota_total"])
    ),
    "Mesmas participações": (
        referencia["participacao_pct"].equals(view["participacao_pct"])
    ),
    "Mesma base interanual": (
        referencia["frota_mes_ano_anterior"].equals(
            view["frota_mes_ano_anterior"]
        )
    ),
    "Mesmo crescimento absoluto": (
        referencia["crescimento_absoluto"].equals(
            view["crescimento_absoluto"]
        )
    ),
    "Mesmo crescimento percentual": (
        referencia["crescimento_pct"].equals(view["crescimento_pct"])
    ),
    "Equivalência integral dos dados": comparacao_tipo.empty
}

print("VALIDAÇÃO DA VIEW — FROTA POR TIPO")
print("=" * 75)

for descricao, resultado in verificacoes_view_tipo.items():
    print(f"{'OK' if resultado else 'FALHOU'} — {descricao}")

print("=" * 75)
print(
    f"Total: {sum(verificacoes_view_tipo.values())}/"
    f"{len(verificacoes_view_tipo)} verificações aprovadas"
)

if not comparacao_tipo.empty:
    print("\nDIFERENÇAS ENCONTRADAS")
    display(comparacao_tipo)

In [ ]:
sql_view_especie = """
CREATE OR REPLACE VIEW analytics.vw_frota_especie_mensal AS
WITH frota_especie AS (
    SELECT
        dc.competencia,
        dc.ano,
        dc.mes,
        dc.nome_mes,
        dc.mes_abreviado,
        dc.ano_mes,
        e.id_especie_veiculo,
        e.especie_veiculo,
        SUM(f.qtd_veiculos)::bigint AS frota_total
    FROM analytics.fato_frota f
    INNER JOIN analytics.dim_calendario dc
        ON dc.competencia = f.data_referencia
    INNER JOIN analytics.dim_especie_veiculo e
        ON e.id_especie_veiculo = f.id_especie_veiculo
    GROUP BY
        dc.competencia,
        dc.ano,
        dc.mes,
        dc.nome_mes,
        dc.mes_abreviado,
        dc.ano_mes,
        e.id_especie_veiculo,
        e.especie_veiculo
),
comparacao AS (
    SELECT
        *,
        LAG(frota_total, 12) OVER (
            PARTITION BY id_especie_veiculo
            ORDER BY competencia
        ) AS frota_mes_ano_anterior
    FROM frota_especie
)
SELECT
    competencia,
    ano,
    mes,
    nome_mes,
    mes_abreviado,
    ano_mes,
    id_especie_veiculo,
    especie_veiculo,
    frota_total,
    ROUND(
        100.0 * frota_total /
        NULLIF(SUM(frota_total) OVER (PARTITION BY competencia), 0),
        2
    ) AS participacao_pct,
    frota_mes_ano_anterior,
    frota_total - frota_mes_ano_anterior AS crescimento_absoluto,
    ROUND(
        100.0 * (
            frota_total::numeric /
            NULLIF(frota_mes_ano_anterior, 0) - 1
        ),
        2
    ) AS crescimento_pct
FROM comparacao;
"""

with engine.begin() as conn:
    conn.execute(text(sql_view_especie))

df_view_especie = pd.read_sql_query(
    text("""
        SELECT *
        FROM analytics.vw_frota_especie_mensal
        ORDER BY competencia, frota_total DESC
    """),
    engine
)

print("VIEW CRIADA E CONSULTADA")
print("=" * 75)
print(f"Registros: {len(df_view_especie)}")
print(f"Espécies: {df_view_especie['especie_veiculo'].nunique()}")
print(f"Competências: {df_view_especie['competencia'].nunique()}")

print("\nAMOSTRA — SETEMBRO DE 2026")

display(
    df_view_especie[
        pd.to_datetime(df_view_especie["competencia"])
        == pd.Timestamp("2026-09-01")
    ]
)

In [ ]:
# Preparar os dados de referência
df_referencia_especie = df_frota_especie.copy()

df_referencia_especie["competencia"] = pd.to_datetime(
    df_referencia_especie["competencia"]
)

df_view_especie["competencia"] = pd.to_datetime(
    df_view_especie["competencia"]
)

colunas_comparacao = [
    "competencia",
    "especie_veiculo",
    "frota_total",
    "participacao_pct",
    "frota_mes_ano_anterior",
    "crescimento_absoluto",
    "crescimento_pct"
]

referencia = df_referencia_especie[
    colunas_comparacao
].sort_values(
    ["competencia", "especie_veiculo"]
).reset_index(drop=True)

view = df_view_especie[
    colunas_comparacao
].sort_values(
    ["competencia", "especie_veiculo"]
).reset_index(drop=True)

# Comparar valores, tratando NaN como equivalentes
comparacao_especie = referencia.compare(
    view,
    keep_shape=False,
    keep_equal=False
)

verificacoes_view_especie = {
    "168 registros na view": len(view) == 168,
    "8 espécies": view["especie_veiculo"].nunique() == 8,
    "21 competências": view["competencia"].nunique() == 21,
    "Mesmas competências": (
        referencia["competencia"].equals(view["competencia"])
    ),
    "Mesmas espécies": (
        referencia["especie_veiculo"].equals(view["especie_veiculo"])
    ),
    "Mesmas quantidades": (
        referencia["frota_total"].equals(view["frota_total"])
    ),
    "Mesmas participações": (
        referencia["participacao_pct"].equals(view["participacao_pct"])
    ),
    "Mesma base interanual": (
        referencia["frota_mes_ano_anterior"].equals(
            view["frota_mes_ano_anterior"]
        )
    ),
    "Mesmo crescimento absoluto": (
        referencia["crescimento_absoluto"].equals(
            view["crescimento_absoluto"]
        )
    ),
    "Mesmo crescimento percentual": (
        referencia["crescimento_pct"].equals(
            view["crescimento_pct"]
        )
    ),
    "Equivalência integral dos dados": comparacao_especie.empty
}

print("VALIDAÇÃO DA VIEW — FROTA POR ESPÉCIE")
print("=" * 75)

for descricao, resultado in verificacoes_view_especie.items():
    print(f"{'OK' if resultado else 'FALHOU'} — {descricao}")

print("=" * 75)
print(
    f"Total: {sum(verificacoes_view_especie.values())}/"
    f"{len(verificacoes_view_especie)} verificações aprovadas"
)

if not comparacao_especie.empty:
    print("\nDIFERENÇAS ENCONTRADAS")
    display(comparacao_especie)

In [ ]:
# Views esperadas na camada analítica
views_esperadas = {
    "vw_frota_nacional_mensal": 21,
    "vw_emplacamentos_nacionais_mensais": 21,
    "vw_frota_uf_mensal": 588,
    "vw_indicadores_nacionais_mensais": 21,
    "vw_frota_tipo_mensal": 84,
    "vw_frota_especie_mensal": 168
}

# Consultar views existentes e respectivas colunas
df_inventario = pd.read_sql_query(
    text("""
        SELECT
            table_name AS view_name,
            column_name,
            ordinal_position
        FROM information_schema.columns
        WHERE table_schema = 'analytics'
          AND table_name IN (
              SELECT table_name
              FROM information_schema.views
              WHERE table_schema = 'analytics'
          )
        ORDER BY table_name, ordinal_position
    """),
    engine
)

# Contar registros de cada view
resultados_auditoria = []

for nome_view, quantidade_esperada in views_esperadas.items():
    existe = nome_view in df_inventario["view_name"].unique()

    if existe:
        quantidade_real = pd.read_sql_query(
            text(f"SELECT COUNT(*) AS total FROM analytics.{nome_view}"),
            engine
        ).iloc[0]["total"]

        quantidade_colunas = len(
            df_inventario[
                df_inventario["view_name"] == nome_view
            ]
        )
    else:
        quantidade_real = None
        quantidade_colunas = 0

    resultados_auditoria.append({
        "view": nome_view,
        "existe": existe,
        "registros_esperados": quantidade_esperada,
        "registros_reais": quantidade_real,
        "quantidade_colunas": quantidade_colunas,
        "validacao": (
            existe and quantidade_real == quantidade_esperada
        )
    })

df_auditoria_views = pd.DataFrame(resultados_auditoria)

print("AUDITORIA CONSOLIDADA — CAMADA ANALÍTICA")
print("=" * 90)

display(df_auditoria_views)

print("\nRESUMO")
print("=" * 90)
print(
    "Views encontradas:",
    df_auditoria_views["existe"].sum(),
    "de",
    len(views_esperadas)
)
print(
    "Views com volume correto:",
    df_auditoria_views["validacao"].sum(),
    "de",
    len(views_esperadas)
)
print(
    "Total de registros nas seis views:",
    df_auditoria_views["registros_reais"].sum()
)

print("\nESTRUTURA — COLUNAS POR VIEW")
print("=" * 90)

for nome_view in views_esperadas:
    colunas = df_inventario.loc[
        df_inventario["view_name"] == nome_view,
        "column_name"
    ].tolist()

    print(f"\n{nome_view} ({len(colunas)} colunas)")
    print(", ".join(colunas))

In [ ]:
auditorias = []

# 1. Verificar duplicidades nas chaves esperadas
chaves_views = {
    "vw_frota_nacional_mensal": ["competencia"],
    "vw_emplacamentos_nacionais_mensais": ["competencia"],
    "vw_frota_uf_mensal": ["competencia", "id_uf"],
    "vw_indicadores_nacionais_mensais": ["competencia"],
    "vw_frota_tipo_mensal": ["competencia", "id_tipo_veiculo"],
    "vw_frota_especie_mensal": ["competencia", "id_especie_veiculo"]
}

for nome_view, chaves in chaves_views.items():
    colunas = ", ".join(chaves)

    resultado = pd.read_sql_query(
        text(f"""
            SELECT
                COUNT(*) AS registros,
                COUNT(DISTINCT ({colunas})) AS chaves_distintas
            FROM analytics.{nome_view}
        """),
        engine
    ).iloc[0]

    auditorias.append({
        "verificacao": f"Unicidade — {nome_view}",
        "resultado": int(resultado["registros"]) ==
                     int(resultado["chaves_distintas"]),
        "detalhe": (
            f"{resultado['chaves_distintas']} chaves distintas "
            f"em {resultado['registros']} registros"
        )
    })

# 2. Comparar totais de frota por UF, tipo e espécie
comparacoes_totais = {
    "UF": """
        SELECT COUNT(*) AS divergencias
        FROM (
            SELECT
                n.competencia,
                n.frota_total AS total_nacional,
                SUM(u.frota_total) AS total_agregado
            FROM analytics.vw_frota_nacional_mensal n
            JOIN analytics.vw_frota_uf_mensal u
                ON u.competencia = n.competencia
            GROUP BY n.competencia, n.frota_total
            HAVING n.frota_total <> SUM(u.frota_total)
        ) x
    """,
    "Tipo": """
        SELECT COUNT(*) AS divergencias
        FROM (
            SELECT
                n.competencia,
                n.frota_total AS total_nacional,
                SUM(t.frota_total) AS total_agregado
            FROM analytics.vw_frota_nacional_mensal n
            JOIN analytics.vw_frota_tipo_mensal t
                ON t.competencia = n.competencia
            GROUP BY n.competencia, n.frota_total
            HAVING n.frota_total <> SUM(t.frota_total)
        ) x
    """,
    "Espécie": """
        SELECT COUNT(*) AS divergencias
        FROM (
            SELECT
                n.competencia,
                n.frota_total AS total_nacional,
                SUM(e.frota_total) AS total_agregado
            FROM analytics.vw_frota_nacional_mensal n
            JOIN analytics.vw_frota_especie_mensal e
                ON e.competencia = n.competencia
            GROUP BY n.competencia, n.frota_total
            HAVING n.frota_total <> SUM(e.frota_total)
        ) x
    """
}

for dimensao, sql in comparacoes_totais.items():
    divergencias = pd.read_sql_query(
        text(sql), engine
    ).iloc[0]["divergencias"]

    auditorias.append({
        "verificacao": f"Reconciliação nacional × {dimensao}",
        "resultado": int(divergencias) == 0,
        "detalhe": f"{divergencias} competências divergentes"
    })

# 3. Verificar integração dos indicadores nacionais
divergencias_integracao = pd.read_sql_query(
    text("""
        SELECT COUNT(*) AS divergencias
        FROM analytics.vw_indicadores_nacionais_mensais i
        JOIN analytics.vw_frota_nacional_mensal f
            ON f.competencia = i.competencia
        JOIN analytics.vw_emplacamentos_nacionais_mensais e
            ON e.competencia = i.competencia
        WHERE i.frota_total <> f.frota_total
           OR i.emplacamentos_mes <> e.emplacamentos_mes
    """),
    engine
).iloc[0]["divergencias"]

auditorias.append({
    "verificacao": "Integração dos indicadores nacionais",
    "resultado": int(divergencias_integracao) == 0,
    "detalhe": (
        f"{divergencias_integracao} competências divergentes"
    )
})

# Exibir resultado
df_auditoria_granularidade = pd.DataFrame(auditorias)

print("AUDITORIA — GRANULARIDADE E CONSISTÊNCIA")
print("=" * 90)

for _, linha in df_auditoria_granularidade.iterrows():
    status = "OK" if linha["resultado"] else "FALHOU"
    print(f"{status} — {linha['verificacao']}")
    print(f"       {linha['detalhe']}")

print("=" * 90)
print(
    f"Resultado: {df_auditoria_granularidade['resultado'].sum()}/"
    f"{len(df_auditoria_granularidade)} verificações aprovadas"
)

In [ ]:
# Campos essenciais: não podem conter valores nulos
campos_essenciais = {
    "vw_frota_nacional_mensal": [
        "competencia", "frota_total"
    ],
    "vw_emplacamentos_nacionais_mensais": [
        "competencia", "emplacamentos_mes"
    ],
    "vw_frota_uf_mensal": [
        "competencia", "id_uf", "frota_total"
    ],
    "vw_indicadores_nacionais_mensais": [
        "competencia", "frota_total", "emplacamentos_mes"
    ],
    "vw_frota_tipo_mensal": [
        "competencia", "id_tipo_veiculo", "tipo_veiculo",
        "frota_total"
    ],
    "vw_frota_especie_mensal": [
        "competencia", "id_especie_veiculo", "especie_veiculo",
        "frota_total"
    ]
}

resultados_cobertura = []
resultados_nulos = []

for nome_view in views_esperadas:

    # Cobertura temporal
    cobertura = pd.read_sql_query(
        text(f"""
            SELECT
                COUNT(*) AS registros,
                COUNT(DISTINCT competencia) AS competencias,
                MIN(competencia) AS primeira_competencia,
                MAX(competencia) AS ultima_competencia
            FROM analytics.{nome_view}
        """),
        engine
    ).iloc[0]

    cobertura_ok = (
        cobertura["competencias"] == 21
        and pd.Timestamp(cobertura["primeira_competencia"])
            == pd.Timestamp("2025-01-01")
        and pd.Timestamp(cobertura["ultima_competencia"])
            == pd.Timestamp("2026-09-01")
    )

    resultados_cobertura.append({
        "view": nome_view,
        "competencias": cobertura["competencias"],
        "primeira": cobertura["primeira_competencia"],
        "ultima": cobertura["ultima_competencia"],
        "validacao": cobertura_ok
    })

    # Obter todas as colunas da view
    colunas = df_inventario.loc[
        df_inventario["view_name"] == nome_view,
        "column_name"
    ].tolist()

    # Contar valores nulos em cada coluna
    expressoes = [
        f'COUNT(*) FILTER (WHERE "{coluna}" IS NULL) AS "{coluna}"'
        for coluna in colunas
    ]

    sql_nulos = f"""
        SELECT {", ".join(expressoes)}
        FROM analytics.{nome_view}
    """

    contagem_nulos = pd.read_sql_query(
        text(sql_nulos),
        engine
    ).iloc[0]

    for coluna, quantidade in contagem_nulos.items():
        if quantidade > 0:
            resultados_nulos.append({
                "view": nome_view,
                "coluna": coluna,
                "nulos": int(quantidade),
                "campo_essencial": (
                    coluna in campos_essenciais[nome_view]
                )
            })

# Resultados de cobertura
df_cobertura = pd.DataFrame(resultados_cobertura)

print("COBERTURA TEMPORAL")
print("=" * 90)
display(df_cobertura)

print(
    "Views com cobertura correta:",
    df_cobertura["validacao"].sum(),
    "de",
    len(df_cobertura)
)

# Resultados de valores nulos
df_nulos = pd.DataFrame(resultados_nulos)

print("\nVALORES NULOS ENCONTRADOS")
print("=" * 90)

if df_nulos.empty:
    print("Nenhum valor nulo encontrado nas seis views.")
else:
    display(df_nulos)

# Verificar apenas campos essenciais
if df_nulos.empty:
    nulos_essenciais = 0
else:
    nulos_essenciais = df_nulos[
        df_nulos["campo_essencial"]
    ]["nulos"].sum()

print("\nRESUMO DA AUDITORIA")
print("=" * 90)
print(
    f"Cobertura temporal: "
    f"{df_cobertura['validacao'].sum()}/{len(df_cobertura)}"
)
print(f"Campos essenciais com nulos: {int(nulos_essenciais)}")
print(
    "Verificação dos campos essenciais:",
    "APROVADA" if nulos_essenciais == 0 else "REVISAR"
)

In [ ]:
from pathlib import Path

diretorio_docs = PROJECT_ROOT / "docs" / "arquitetura"
diretorio_docs.mkdir(parents=True, exist_ok=True)

conteudo = r"""# Camada Analítica — Inteligência Comercial do Mercado Automotivo Brasileiro

## 1. Objetivo

A camada analítica do projeto organiza os dados tratados no PostgreSQL em views destinadas à análise de negócio e à integração com o Power BI.

Seu objetivo é disponibilizar indicadores consistentes sobre:

- Evolução mensal da frota brasileira de automóveis e comerciais leves.
- Distribuição geográfica da frota por Unidade da Federação.
- Composição da frota por tipo e espécie de veículo.
- Evolução mensal dos emplacamentos de automóveis e comerciais leves.
- Comparação entre a evolução do estoque da frota e o fluxo de emplacamentos.

As views são construídas sobre as tabelas do schema `analytics`, cuja estrutura dimensional está documentada em `modelo_dimensional.md`.

## 2. Escopo analítico

### 2.1 Período

- Início: janeiro de 2025.
- Fim: setembro de 2026.
- Total: 21 competências mensais.

Outubro de 2026 é tratado como informação suplementar, fora da versão integrada principal.

### 2.2 Categorias de veículos

O escopo considera:

- AUTOMOVEL
- CAMINHONETE
- CAMIONETA
- UTILITARIO

Na fonte FENABRAVE, são utilizadas as categorias originais `Autos` e `Com. Leves`, além da categoria consolidada `Autos + Com. Leves`.

Motocicletas, caminhões, ônibus e demais categorias fora do escopo não integram as análises principais.

### 2.3 Fontes e conceitos

**SENATRAN — Frota**

Representa o estoque de veículos registrados na data de referência mensal.

Variações da frota representam crescimento ou redução do estoque. Não devem ser denominadas emplacamentos.

**FENABRAVE — Emplacamentos**

Representa o fluxo de veículos emplacados durante determinado período.

Os indicadores publicados pela entidade são preservados conforme a fonte, inclusive quando apresentam diferenças em relação a cálculos derivados da série mensal histórica.

## 3. Inventário das views

Todas as views estão no schema `analytics`.

| View | Granularidade | Registros |
|---|---|---:|
| `vw_frota_nacional_mensal` | Uma linha por competência | 21 |
| `vw_emplacamentos_nacionais_mensais` | Uma linha por competência | 21 |
| `vw_frota_uf_mensal` | Competência e UF | 588 |
| `vw_indicadores_nacionais_mensais` | Uma linha por competência | 21 |
| `vw_frota_tipo_mensal` | Competência e tipo de veículo | 84 |
| `vw_frota_especie_mensal` | Competência e espécie de veículo | 168 |

## 4. Descrição das views

### 4.1 vw_frota_nacional_mensal

Apresenta a evolução mensal da frota nacional.

**Granularidade:** uma linha por competência.

Principais campos:

- `competencia`: data de referência mensal.
- `ano`, `mes`: componentes temporais.
- `nome_mes`, `mes_abreviado`, `ano_mes`: atributos para apresentação e ordenação.
- `frota_total`: quantidade total de veículos da frota considerada.

A view consolida os registros da tabela `fato_frota`.

### 4.2 vw_emplacamentos_nacionais_mensais

Apresenta os emplacamentos nacionais mensais da categoria consolidada `Autos + Com. Leves`.

**Granularidade:** uma linha por competência.

Principais campos:

- `competencia`, `ano`, `mes`, `nome_mes`, `mes_abreviado`, `ano_mes`.
- `emplacamentos_mes`.
- `emplacamentos_mes_anterior`.
- `emplacamentos_acumulados`.
- `emplacamentos_mes_ano_anterior`.
- `emplacamentos_acumulados_ano_anterior`.
- `variacao_mensal_pct`.
- `variacao_anual_pct`.
- `variacao_acumulada_pct`.

Os valores publicados pela FENABRAVE são preservados. Não se força a reconciliação entre acumulados publicados e a soma dos valores mensais da série histórica.

### 4.3 vw_frota_uf_mensal

Apresenta a distribuição da frota por Unidade da Federação ao longo do período.

**Granularidade:** uma linha por competência e UF.

Principais campos:

- `competencia`, `ano`, `mes`.
- `id_uf`, `sigla_uf`, `nome_uf`.
- `regiao`.
- `frota_total`.

A view possui 588 registros, correspondentes a 28 identificações geográficas por 21 competências. A identificação `SI` representa registros sem informação de UF.

A análise geográfica da primeira versão utiliza UF e região, sem detalhamento municipal.

### 4.4 vw_indicadores_nacionais_mensais

Integra os indicadores nacionais de frota e emplacamentos.

**Granularidade:** uma linha por competência.

Principais campos:

- Atributos temporais.
- `frota_total`.
- Indicadores mensais, acumulados e publicados de emplacamentos.
- `variacao_frota_pct`.
- `variacao_emplacamentos_serie_pct`.
- `diferenca_frota_vs_emplacamentos_serie_pp`.
- `diferenca_frota_vs_emplacamentos_publicada_pp`.

A view permite comparar a evolução do estoque da frota com o fluxo de emplacamentos.

As diferenças em pontos percentuais devem ser interpretadas conforme a base de comparação utilizada: série mensal histórica ou variação anual publicada pela FENABRAVE.

Os indicadores interanuais calculados para 2025 permanecem nulos quando dependem de uma base de 2024 não disponível no escopo.

### 4.5 vw_frota_tipo_mensal

Apresenta a composição e a evolução da frota por tipo de veículo.

**Granularidade:** uma linha por competência e tipo.

Principais campos:

- Atributos temporais.
- `id_tipo_veiculo`, `tipo_veiculo`.
- `frota_total`.
- `participacao_pct`.
- `frota_mes_ano_anterior`.
- `crescimento_absoluto`.
- `crescimento_pct`.

A participação percentual é calculada sobre o total da frota na mesma competência.

A comparação interanual utiliza o mesmo mês do ano anterior, por meio de defasagem de 12 competências.

### 4.6 vw_frota_especie_mensal

Apresenta a composição e a evolução da frota por espécie de veículo.

**Granularidade:** uma linha por competência e espécie.

Principais campos:

- Atributos temporais.
- `id_especie_veiculo`, `especie_veiculo`.
- `frota_total`.
- `participacao_pct`.
- `frota_mes_ano_anterior`.
- `crescimento_absoluto`.
- `crescimento_pct`.

A view mantém as oito espécies existentes na dimensão, inclusive `Sem Informação`, `CORRIDA` e `TRACAO`.

A participação percentual utiliza o total da frota na mesma competência. O crescimento interanual compara competências equivalentes.

## 5. Regras metodológicas

### 5.1 Estoque versus fluxo

Frota e emplacamentos representam fenômenos diferentes:

- Frota: estoque de veículos registrados em uma data.
- Emplacamentos: fluxo de registros realizados durante um período.

Não se deve interpretar o crescimento da frota como equivalente ao volume de emplacamentos.

### 5.2 Comparação interanual

As comparações interanuais utilizam somente meses equivalentes entre 2025 e 2026.

Não há comparação interanual calculada para 2025 quando a base de 2024 não está disponível.

### 5.3 Participação percentual

Nas views por tipo e espécie, a participação é calculada dividindo a frota de cada grupo pelo total da frota na respectiva competência.

### 5.4 Crescimento

- Crescimento absoluto: diferença entre a frota atual e a frota do mesmo mês do ano anterior.
- Crescimento percentual: variação relativa entre essas duas quantidades.

### 5.5 Valores publicados

Os indicadores publicados pela FENABRAVE são preservados como fornecidos pela fonte. Diferenças entre valores publicados e valores recalculados a partir da série mensal não são corrigidas artificialmente.

## 6. Auditoria e validação

As seis views passaram pelas verificações realizadas no PostgreSQL.

### 6.1 Existência e volume

| View | Volume esperado | Resultado |
|---|---:|---|
| `vw_frota_nacional_mensal` | 21 | Aprovado |
| `vw_emplacamentos_nacionais_mensais` | 21 | Aprovado |
| `vw_frota_uf_mensal` | 588 | Aprovado |
| `vw_indicadores_nacionais_mensais` | 21 | Aprovado |
| `vw_frota_tipo_mensal` | 84 | Aprovado |
| `vw_frota_especie_mensal` | 168 | Aprovado |

### 6.2 Granularidade e consistência

- Unicidade das chaves esperadas nas seis views: aprovada.
- Reconciliação dos totais nacionais com a agregação por UF: aprovada.
- Reconciliação dos totais nacionais com a agregação por tipo: aprovada.
- Reconciliação dos totais nacionais com a agregação por espécie: aprovada.
- Integração entre as views nacionais de frota, emplacamentos e indicadores: aprovada.

Resultado: 10 de 10 verificações aprovadas.

### 6.3 Cobertura e valores nulos

- Cobertura de janeiro de 2025 a setembro de 2026 nas seis views: aprovada.
- Campos essenciais sem valores nulos: aprovado.
- Nulos em indicadores interanuais sem base de comparação: permitidos conforme a metodologia.

### 6.4 Equivalência com análises anteriores

- `vw_frota_tipo_mensal`: 9 de 9 verificações aprovadas.
- `vw_frota_especie_mensal`: 11 de 11 verificações aprovadas.

As duas views reproduzem integralmente os resultados das análises realizadas anteriormente no notebook.

## 7. Orientações para o Power BI

- Utilizar as views do schema `analytics` como fontes para os relatórios.
- Priorizar as views agregadas conforme a pergunta analítica.
- Manter a coluna `competencia` como referência temporal.
- Ordenar nomes de meses por seus respectivos números ou utilizar `ano_mes` para eixos cronológicos.
- Não somar novamente categorias consolidadas aos seus componentes.
- Não relacionar diretamente fatos independentes.
- Preservar a distinção entre estoque de frota e fluxo de emplacamentos.
- Tratar valores nulos interanuais de 2025 como ausência de base histórica, e não como zero.

A conexão será realizada ao PostgreSQL instalado no Fedora, quando o ambiente estiver acessível a partir do Windows, onde será utilizado o Power BI Desktop.

## 8. Artefatos relacionados

- `modelo_dimensional.md`: documentação do modelo dimensional.
- `diagrama_modelo.mmd`: diagrama do modelo dimensional.
- `docs/analises/analise_geografica_frota.md`: análise geográfica.
- `docs/analises/evolucao_regional_frota.md`: evolução regional.
- `docs/analises/analise_frota_tipo_veiculo.md`: análise por tipo.
- `docs/analises/analise_frota_especie_veiculo.md`: análise por espécie.
- `docs/qualidade/auditoria_fenabrave.md`: auditoria dos dados FENABRAVE.
"""

arquivo_documentacao = diretorio_docs / "camada_analitica.md"
arquivo_documentacao.write_text(conteudo, encoding="utf-8")

print("DOCUMENTAÇÃO CRIADA")
print("=" * 75)
print(f"Arquivo: {arquivo_documentacao}")
print(f"Linhas: {len(conteudo.splitlines())}")
print(f"Tamanho: {arquivo_documentacao.stat().st_size:,} bytes")
print(f"Existe: {arquivo_documentacao.exists()}")

In [ ]:
arquivo_documentacao = (
    PROJECT_ROOT
    / "docs"
    / "arquitetura"
    / "camada_analitica.md"
)

documento = arquivo_documentacao.read_text(encoding="utf-8")

secoes_esperadas = [
    "## 1. Objetivo",
    "## 2. Escopo analítico",
    "## 3. Inventário das views",
    "## 4. Descrição das views",
    "## 5. Regras metodológicas",
    "## 6. Auditoria e validação",
    "## 7. Orientações para o Power BI",
    "## 8. Artefatos relacionados"
]

views_esperadas = [
    "vw_frota_nacional_mensal",
    "vw_emplacamentos_nacionais_mensais",
    "vw_frota_uf_mensal",
    "vw_indicadores_nacionais_mensais",
    "vw_frota_tipo_mensal",
    "vw_frota_especie_mensal"
]

verificacoes_documento = {
    "Arquivo existe": arquivo_documentacao.exists(),
    "Oito seções principais": all(
        secao in documento for secao in secoes_esperadas
    ),
    "Seis views documentadas": all(
        view in documento for view in views_esperadas
    ),
    "Distinção entre estoque e fluxo": (
        "Estoque versus fluxo" in documento
    ),
    "Regra de comparação interanual": (
        "Comparação interanual" in documento
    ),
    "Orientações para Power BI": (
        "Orientações para o Power BI" in documento
    ),
    "Referência ao modelo dimensional": (
        "modelo_dimensional.md" in documento
    ),
    "Referência ao diagrama": (
        "diagrama_modelo.mmd" in documento
    )
}

print("REVISÃO DA DOCUMENTAÇÃO — CAMADA ANALÍTICA")
print("=" * 75)

for descricao, resultado in verificacoes_documento.items():
    print(f"{'OK' if resultado else 'FALHOU'} — {descricao}")

print("=" * 75)
print(
    f"Resultado: {sum(verificacoes_documento.values())}/"
    f"{len(verificacoes_documento)} verificações aprovadas"
)

print("\nMETADADOS")
print("=" * 75)
print(f"Linhas: {len(documento.splitlines())}")
print(f"Tamanho: {arquivo_documentacao.stat().st_size:,} bytes")

In [ ]:
import subprocess
from pathlib import Path

resultado = subprocess.run(
    ["git", "status", "--short", "--branch"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True
)

print("Código de saída:", resultado.returncode)
print(resultado.stdout if resultado.stdout else "(sem saída)")

if resultado.stderr:
    print("\nMensagens:")
    print(resultado.stderr)

In [ ]:
from pathlib import Path

arquivo_gitignore = PROJECT_ROOT / ".gitignore"

if arquivo_gitignore.exists():
    print("CONTEÚDO ATUAL DO .gitignore:\n")
    print(arquivo_gitignore.read_text(encoding="utf-8"))
else:
    print("O arquivo .gitignore ainda não existe.")

In [ ]:
from pathlib import Path

arquivo_gitignore = PROJECT_ROOT / ".gitignore"

regras_adicionais = """

# IDEs e editores
.vscode/
.idea/

# Jupyter - arquivos temporários
*.ipynb_checkpoints

# PostgreSQL - backups e dumps
*.dump
*.backup
*.sql.gz
*.tar
"""

conteudo_atual = arquivo_gitignore.read_text(encoding="utf-8")

if "# IDEs e editores" not in conteudo_atual:
    arquivo_gitignore.write_text(
        conteudo_atual.rstrip() + "\n" + regras_adicionais,
        encoding="utf-8"
    )
    print("Regras adicionais incluídas.")
else:
    print("As regras adicionais já existem. Nenhuma alteração realizada.")

print("\nConteúdo atualizado:\n")
print(arquivo_gitignore.read_text(encoding="utf-8"))

In [ ]:
import subprocess

resultado = subprocess.run(
    ["git", "status", "--short", "--untracked-files=all"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True
)

print("Arquivos pendentes:\n")
print(resultado.stdout if resultado.stdout else "(nenhum arquivo pendente)")

if resultado.stderr:
    print("\nErros:")
    print(resultado.stderr)

In [ ]:
import json

notebooks = [
    PROJECT_ROOT / "notebooks/00_validacao_ambiente.ipynb",
    PROJECT_ROOT / "notebooks/01_ inspecao_dados_senatran.ipynb",
]

for arquivo in notebooks:
    print(f"\n{'=' * 65}")
    print(f"Arquivo: {arquivo.name}")
    print(f"Tamanho: {arquivo.stat().st_size / 1024:.2f} KB")

    with arquivo.open("r", encoding="utf-8") as f:
        notebook = json.load(f)

    total_celulas = len(notebook.get("cells", []))
    celulas_com_saida = sum(
        bool(celula.get("outputs"))
        for celula in notebook.get("cells", [])
    )

    print(f"Total de células: {total_celulas}")
    print(f"Células com saídas armazenadas: {celulas_com_saida}")

In [ ]:
import json
import shutil
import tempfile
from pathlib import Path

notebooks = [
    PROJECT_ROOT / "notebooks/00_validacao_ambiente.ipynb",
    PROJECT_ROOT / "notebooks/01_ inspecao_dados_senatran.ipynb",
]

pasta_backup = Path(
    tempfile.mkdtemp(prefix="backup_notebooks_automotivo_")
)

print("Backup:", pasta_backup)

for arquivo in notebooks:
    # Preserva uma cópia original com todas as saídas
    shutil.copy2(
        arquivo,
        pasta_backup / arquivo.name
    )

    # Limpa as saídas da versão que será versionada
    with arquivo.open("r", encoding="utf-8") as f:
        notebook = json.load(f)

    for celula in notebook.get("cells", []):
        if celula.get("cell_type") == "code":
            celula["outputs"] = []
            celula["execution_count"] = None

    with arquivo.open("w", encoding="utf-8") as f:
        json.dump(
            notebook,
            f,
            ensure_ascii=False,
            indent=1
        )
        f.write("\n")

    print(f"Saídas limpas: {arquivo.name}")

print("\nCópias originais preservadas:", len(notebooks))